# Sentiment + Sarcasm Impact on Stock Market Predictions — **Hybrid (10-year)**

Pipeline end-to-end qui quantifie l'impact du sentiment (news + social, **sarcasme & emotion** inclus)
sur la direction des rendements boursiers, sur **~10 ans (2015–2025)** et avec un **ensemble hybride
arbres + reseau sequentiel**.

> **Corrections majeures par rapport a la version d'origine**
> 1. Le notebook contenait **3 cellules « CELL 5 » concurrentes** (collecte de texte) dont 2 mal placees
>    au milieu du pipeline → execution « Run All » cassee (le nettoyage tournait *avant* la collecte).
>    Ne reste qu'**une seule** cellule de collecte, a la bonne place.
> 2. **Chemins unifies** : chaque cellule reutilise les constantes `PATH_*`, `ckpt`, `mem`, `pip_install`
>    de CELL 1/2 (avant, 3 conventions de chemins incompatibles `/kaggle/working/...` cohabitaient).
> 3. **Fenetre etendue a 10 ans** (2015→2025) + split temporel revu (train ≤ 2021, val = 2022, test ≥ 2023).
> 4. **Collecte ancree sur les 15 datasets Kaggle attaches** (fiable, hors-ligne) + FNSPID/RSS optionnels
>    pour la recence 2023–2025.
> 5. **Modeles hybrides** : LightGBM + XGBoost + CatBoost **+ reseau sequentiel BiGRU-attention**,
>    combines par **stacking (meta-modele logistique) + calibration isotone**.
> 6. **Validation croisee temporelle corrigee** (tri chronologique global) + backtest enrichi
>    (Sharpe, max drawdown, couts de transaction).

**⚙️ Avant de lancer :** active le **GPU (T4 x2)** dans *Settings → Accelerator* (FinBERT + BiGRU).
**Pipeline :** 1 Config · 2 Helpers · 3 Credentials · 4 Prix · 5 Collecte texte · 6 Nettoyage+sarcasme ·
7 FinBERT · 8 Sarcasme ML+Emotion · 9 Agregation · 10 Indicateurs · 11 Merge+targets ·
12 Features+split · 13 Arbres · **13b BiGRU** · **13c Stacking** · 14 Ensemble+backtest ·
15 Causalite · 16 Temps reel · 17 Visualisation · 18 Rapport.


## Cell 1 — Configuration & paths (10 ans + hybride)

In [12]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION & PATHS  (Hybrid v2 — 10 ans)            ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, sys, gc, re, json, pickle, hashlib, warnings, subprocess
import datetime as _dt
from pathlib import Path

import numpy as np
import pandas as pd
import os
BASE_DIR   = os.environ.get("BASE_DIR", "/kaggle/working/sentiment_pipeline")
OUTPUT_DIR = f"{BASE_DIR}/data"
MODELS_DIR = f"{BASE_DIR}/models"

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

# ── Directories ────────────────────────────────────────────────────
BASE_DIR      = "/kaggle/working/sentiment_pipeline"
OUTPUT_DIR    = f"{BASE_DIR}/data"
MODELS_DIR    = f"{BASE_DIR}/models"
DASHBOARD_DIR = f"{BASE_DIR}/dashboard"
CACHE_DIR     = f"{BASE_DIR}/cache"
SHARDS_DIR    = f"{CACHE_DIR}/text_shards"     # spill-to-disk de la collecte
for d in (OUTPUT_DIR, MODELS_DIR, DASHBOARD_DIR, CACHE_DIR, SHARDS_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Universe ───────────────────────────────────────────────────────
STOCKS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA"]
STOCKS_SET = set(STOCKS)
NAMES = {
    "AAPL":  "Apple",     "MSFT": "Microsoft", "GOOGL": "Alphabet",
    "AMZN":  "Amazon",    "META": "Meta",      "TSLA":  "Tesla",
    "NVDA":  "Nvidia",
}
# ── Fenetre d'etude : ~10 ans ──────────────────────────────────────
YEAR_START, YEAR_END = 2015, 2025
ALL_YEARS = list(range(YEAR_START, YEAR_END + 1))

# ── Output files (un par etape) ────────────────────────────────────
PATH_PRICES   = f"{OUTPUT_DIR}/01_prices.parquet"
PATH_RAW      = f"{OUTPUT_DIR}/02_text_raw.parquet"
PATH_CLEAN    = f"{OUTPUT_DIR}/03_text_clean.parquet"
PATH_SENT     = f"{OUTPUT_DIR}/04_sentiment.parquet"
PATH_DAILY    = f"{OUTPUT_DIR}/05_daily.parquet"
PATH_TECH     = f"{OUTPUT_DIR}/06_prices_tech.parquet"
PATH_MERGED   = f"{OUTPUT_DIR}/07_merged.parquet"
PATH_FEATURES = f"{OUTPUT_DIR}/08_features.parquet"
PATH_TRAIN    = f"{OUTPUT_DIR}/09_train.parquet"
PATH_VAL      = f"{OUTPUT_DIR}/09_val.parquet"
PATH_TEST     = f"{OUTPUT_DIR}/09_test.parquet"
PATH_PREDS    = f"{OUTPUT_DIR}/10_predictions.parquet"
PATH_BACKTEST = f"{OUTPUT_DIR}/11_backtest.parquet"

# ── Split temporel (sur ~10 ans : 7 ans train / 1 val / >=1 test) ──
# Le coeur de l'etude (correlation, Granger, SHAP) couvre TOUTE la fenetre ;
# le split sert au modele predictif. Test = periode la mieux couverte par
# les datasets locaux (>= 2023) — extensible si du texte recent existe.
TRAIN_END = pd.Timestamp("2020-12-31")
VAL_END   = pd.Timestamp("2021-12-31")
TEST_END  = pd.Timestamp("2022-12-31")
# 2024-2025 restent INGEREES (YEAR_END=2025) mais HORS test tant que la
# couverture texte y est faible. Quand tu auras du volume 2024-2025, avance
# simplement TEST_END (ex. 2025-12-31).

# ── MASSIVE-DATA MODE ──────────────────────────────────────────────
# True  : vise des millions de lignes (GPU recommande pour CELL 7/8).
# False : run CPU-friendly (~200k lignes).
MASSIVE_MODE = True

def quota_news(yr):
    if MASSIVE_MODE:
        return 600_000
    return 4000

def quota_social(yr):
    if MASSIVE_MODE:
        return 400_000
    return 15000

PER_SOURCE_HF_MAX = 5_000_000 if MASSIVE_MODE else 2_000_000
HARD_TOTAL_CAP    = 5_000_000 if MASSIVE_MODE else 1_000_000   # sur en 1 session (collecte+lectures séquentielles)
MAX_TEXT_CHARS    = 1000   # troncature texte a ingestion (FinBERT ne lit que ~512 tokens)
#   -> 1000 chars suffit pour le sentiment ; baisse a 600 si disque encore juste,
#      ou monte HARD_TOTAL_CAP a 5-6M maintenant que le texte est tronque (surveille le disque).
SHARD_FLUSH_EVERY = 250_000   if MASSIVE_MODE else 100_000

# Budget temps de collecte (s) — borne le run pour rester < limite Kaggle.
COLLECT_TIME_BUDGET_S = 60 * 60 * 3

# ── PARAMETRES DU MODELE HYBRIDE ───────────────────────────────────
USE_SEQ_MODEL = True     # reseau sequentiel BiGRU-attention (CELL 13b)
USE_STACKING  = True     # meta-modele de stacking + calibration (CELL 13c)
USE_FNSPID    = True     # SOURCE B FNSPID (HF streaming) : ON (enrichit 1999-2023)
#   Astuce: un MIROIR FNSPID en dataset Kaggle (Input) serait lu en LOCAL par la
#   SOURCE A (bien plus rapide que le streaming). Cherche "FNSPID" dans Add Input.
USE_RSS       = True     # SOURCE C RSS : ON (complement recent, ~dernieres semaines)
FNSPID_MAX_ROWS    = 5_000_000  # borne disque (lignes RETENUES)
FNSPID_MAX_SECONDS = 5400       # chrono securite FNSPID = 90 min max

# SOURCE D : datasets HuggingFace (chargement direct)
USE_HF        = True
HF_DATASETS   = ["m-ric/financial-news-2024"]  # ajoute des IDs HF supplementaires ici
HF_MAX_ROWS   = 4_000_000
HF_MAX_SECONDS= 1500            # 25 min max par run HF

# SOURCE E : scraping API news historique (Finnhub) -> remplit 2024-2025
# Requiert un secret FINNHUB_TOKEN (Add-ons -> Secrets). Sans cle : saute.
USE_NEWSAPI         = True
NEWSAPI_FROM        = "2024-01-01"
NEWSAPI_TO          = "2025-12-31"
NEWSAPI_MAX_SECONDS = 1500

# SOURCE F : Alpha Vantage News & Sentiment (news datee + score sentiment)
# Requiert un secret AV_KEY (Add-ons -> Secrets, cle gratuite alphavantage.co).
# Free tier = ~25 requetes/jour : on borne le nombre de requetes.
USE_AV         = True
AV_FROM        = "2022-01-01"   # archive news AV commence ~2022
AV_TO          = "2025-12-31"
AV_MAX_REQ     = 24             # reste sous le quota gratuit (25/jour)
AV_MAX_SECONDS = 900

# Disque : supprime les parquets ligne-par-ligne une fois les features bâties
# (raw/clean/finbert/sent, les + gros). Mets False seulement si tu veux pouvoir
# reprendre ces etapes sans recalcul (au prix de ~plusieurs Go).
FREE_ROWLEVEL_AFTER_FEATURES = True

# ── Horizon de prédiction (papier) ─────────────────────────────────
# 1 = direction à 1 jour (quasi 50/50, très dur).
# 5/10/20 = horizons plus longs → accuracy honnête plus élevée (momentum).
# Doit figurer dans CONFIG["horizons"] ci-dessus.
HORIZON = 10

# Ablation source : ne garder que la NEWS (signal plus propre que les tweets WSB).
# False = tout (news + social). True = news seulement.
NEWS_ONLY = False

# ── Type de cible (papier) ─────────────────────────────────────────
# "direction"  = monter/descendre à +HORIZON jours (quasi 50/50, très dur).
# "volatility" = forte volatilité à venir (oui/non). BEAUCOUP plus prévisible
#                (la volatilité se regroupe dans le temps) → accuracy honnête
#                nettement plus élevée. Sans fuite : vol FUTURE prédite via un
#                seuil basé UNIQUEMENT sur le passé.
TARGET = "direction"
# NB: si tu ajoutes un MIROIR FNSPID en dataset Kaggle (Input), la SOURCE A
#     le lira en local (rapide, sans streaming) -> tu peux alors remettre
#     USE_FNSPID=False. Le local est toujours preferable au streaming HF.
SEQ_LEN       = 20       # fenetre temporelle (jours de bourse) du BiGRU
SEQ_EPOCHS    = 25
SEQ_BATCH     = 512

# ── Global CONFIG ──────────────────────────────────────────────────
CONFIG = {
    "stocks":                    NAMES,
    "year_start":                YEAR_START,
    "year_end":                  YEAR_END,
    "horizons":                  [1, 2, 3, 5, 10, 20],
    "date_start":                f"{YEAR_START}-01-01",
    "date_end":                  f"{YEAR_END}-12-31",
    "massive_mode":              MASSIVE_MODE,
    "per_source_hf_max":         PER_SOURCE_HF_MAX,
    "hard_total_cap":            HARD_TOTAL_CAP,
    "shard_flush_every":         SHARD_FLUSH_EVERY,
    "seq_len":                   SEQ_LEN,
    # Jamais de donnees synthetiques : on echoue bruyamment pour corriger le setup.
    "allow_synthetic_fallback":  False,
}

print("=" * 65)
print(f"  CONFIG  |  {len(STOCKS)} stocks  |  {YEAR_START}-{YEAR_END}  (~{YEAR_END-YEAR_START+1} ans)")
print("=" * 65)
print(f"  OUTPUT_DIR    : {OUTPUT_DIR}")
print(f"  MODELS_DIR    : {MODELS_DIR}")
print(f"  Stocks        : {STOCKS}")
print(f"  Train  end    : {TRAIN_END.date()}   (>= {YEAR_START})")
print(f"  Val    end    : {VAL_END.date()}")
print(f"  Test  window  : 2023-01-01 .. {TEST_END.date()}")
print(f"  Ingest end    : {YEAR_END} (2024-2025 ingerees, hors test)")
print(f"  Hybride       : seq={USE_SEQ_MODEL}  stacking={USE_STACKING}  SEQ_LEN={SEQ_LEN}")
print(f"  MASSIVE_MODE  : {MASSIVE_MODE}  |  hard cap = {HARD_TOTAL_CAP:,} rows")
print("=" * 65)


  CONFIG  |  7 stocks  |  2015-2025  (~11 ans)
  OUTPUT_DIR    : /kaggle/working/sentiment_pipeline/data
  MODELS_DIR    : /kaggle/working/sentiment_pipeline/models
  Stocks        : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA']
  Train  end    : 2020-12-31   (>= 2015)
  Val    end    : 2021-12-31
  Test  window  : 2023-01-01 .. 2022-12-31
  Ingest end    : 2025 (2024-2025 ingerees, hors test)
  Hybride       : seq=True  stacking=True  SEQ_LEN=20
  MASSIVE_MODE  : True  |  hard cap = 5,000,000 rows


## Cell 2 — Helpers (dates, tickers, memoire, checkpoints)

In [13]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — HELPERS                                                ║
# ╚══════════════════════════════════════════════════════════════════╝
from dateutil import parser as _dparser

# ── Date normalization ─────────────────────────────────────────────
def ndate(s):
    # Normalize any string to YYYY-MM-DD, return None on failure.
    if s is None: return None
    s = str(s).strip()
    if s in ("", "nan", "None", "NaT", "NaN"): return None
    s = re.sub(r"\s*UTC\s*$", "", s)
    s = re.sub(r"\+\d{2}:\d{2}$", "", s)
    # Unix timestamp
    try:
        ts = float(s)
        if 1e9 < ts < 2e9:
            return _dt.datetime.utcfromtimestamp(ts).strftime("%Y-%m-%d")
    except (ValueError, TypeError):
        pass
    # Standard formats
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%m/%d/%Y",
                "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%B %d, %Y", "%b %d, %Y"):
        try:
            return _dt.datetime.strptime(s[:26], fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    # Last resort: fuzzy parse
    try:
        return _dparser.parse(s, fuzzy=True).strftime("%Y-%m-%d")
    except Exception:
        return None

def normalize_date_col(series):
    # Robustly turn any date column into tz-naive midnight datetime64[ns].
    s = pd.to_datetime(series, errors="coerce", utc=False)
    if hasattr(s.dt, "tz") and s.dt.tz is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s.dt.normalize()

# ── Hashing for dedup ──────────────────────────────────────────────
def text_hash(text, n=200):
    return hashlib.md5(text[:n].encode("utf-8", errors="ignore")).hexdigest()

# ── Ticker detection ───────────────────────────────────────────────
SYM_MAP = {
    "APPLE": "AAPL", "APPLE INC": "AAPL", "$AAPL": "AAPL",
    "MICROSOFT": "MSFT", "MICROSOFT CORP": "MSFT", "$MSFT": "MSFT",
    "GOOGLE": "GOOGL", "ALPHABET": "GOOGL", "GOOG": "GOOGL",
    "ALPHABET INC": "GOOGL", "$GOOGL": "GOOGL", "$GOOG": "GOOGL",
    "AMAZON": "AMZN", "AMAZON.COM": "AMZN", "AMAZON.COM INC": "AMZN",
    "$AMZN": "AMZN",
    "FACEBOOK": "META", "FB": "META", "META PLATFORMS": "META",
    "META PLATFORMS INC": "META", "$META": "META", "$FB": "META",
    "TESLA": "TSLA", "TESLA INC": "TSLA", "TESLA MOTORS": "TSLA",
    "$TSLA": "TSLA",
    "NVIDIA": "NVDA", "NVIDIA CORP": "NVDA", "$NVDA": "NVDA",
}
DETECT_RE = {
    "AAPL":  re.compile(r"\b(AAPL|\$AAPL|Apple(?:\s+Inc\.?)?)\b", re.I),
    "MSFT":  re.compile(r"\b(MSFT|\$MSFT|Microsoft(?:\s+Corp\.?)?)\b", re.I),
    "GOOGL": re.compile(r"\b(GOOGL?|\$GOOGL?|Google|Alphabet(?:\s+Inc\.?)?)\b", re.I),
    "AMZN":  re.compile(r"\b(AMZN|\$AMZN|Amazon(?:\.com)?(?:\s+Inc\.?)?)\b", re.I),
    "META":  re.compile(r"\b(META|\$META|Meta\s+Platforms|Facebook|\$FB)\b", re.I),
    "TSLA":  re.compile(r"\b(TSLA|\$TSLA|Tesla(?:\s+(?:Inc\.|Motors))?)\b", re.I),
    "NVDA":  re.compile(r"\b(NVDA|\$NVDA|Nvidia(?:\s+Corp\.?)?)\b", re.I),
}

def detect_tickers(text):
    if not text:
        return []
    return [tk for tk, rx in DETECT_RE.items() if rx.search(text)]

# ── Memory + GC ────────────────────────────────────────────────────
class MemManager:
    @staticmethod
    def free():
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
    @staticmethod
    def info():
        try:
            import psutil
            v = psutil.virtual_memory()
            print(f"  RAM: {v.percent:.0f}% used | {v.available/1024**3:.1f} GB free")
        except Exception:
            pass
    @staticmethod
    def optimize_df(df, verbose=False):
        # Downcast numeric dtypes to save memory.
        before = df.memory_usage(deep=True).sum() / 1024**2
        for col in df.select_dtypes(include=["float64"]).columns:
            df[col] = df[col].astype(np.float32)
        for col in df.select_dtypes(include=["int64"]).columns:
            mn, mx = df[col].min(), df[col].max()
            if pd.isna(mn) or pd.isna(mx): continue
            for dt in (np.int8, np.int16, np.int32):
                if np.iinfo(dt).min <= mn and mx <= np.iinfo(dt).max:
                    df[col] = df[col].astype(dt)
                    break
        after = df.memory_usage(deep=True).sum() / 1024**2
        if verbose:
            print(f"  Memory: {before:.1f} -> {after:.1f} MB ({100*(1-after/before):.0f}% saved)")
        return df

mem = MemManager()

# ── Checkpoint manager ─────────────────────────────────────────────
class CheckpointMgr:
    @staticmethod
    def save(df, path, verbose=True):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(path, index=False, compression="snappy")
        if verbose:
            sz = Path(path).stat().st_size / 1024**2
            print(f"  💾 saved {Path(path).name} ({len(df):,} rows | {sz:.1f} MB)")

    @staticmethod
    def load(path, verbose=True):
        df = pd.read_parquet(path)
        if verbose:
            print(f"  📂 loaded {Path(path).name} ({len(df):,} rows)")
        return df

    @staticmethod
    def exists_and_valid(path, min_rows=1):
        if not Path(path).exists():
            return False
        try:
            import pyarrow.parquet as pq
            return pq.read_metadata(path).num_rows >= min_rows
        except Exception:
            return False

ckpt = CheckpointMgr()

# ── Pip install helper ─────────────────────────────────────────────
def pip_install(pkgs, quiet=True):
    if isinstance(pkgs, str):
        pkgs = [pkgs]
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet: cmd.append("-q")
    cmd.extend(pkgs)
    subprocess.run(cmd, check=False)

print("✅ Helpers loaded")


✅ Helpers loaded


## Cell 3 — Credentials (HuggingFace + Kaggle, optionnel)

In [14]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — CREDENTIALS (HF + Kaggle + APIs)  v2                  ║
# ║  Corrections: FINNHUB_TOKEN, UserSecretsClient mutualisé,        ║
# ║  CONFIG guard, imports, clarification Kaggle API                 ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, json
from pathlib import Path

if "CONFIG" not in dir():
    CONFIG = {}

KAGGLE_OK = False

# ── Instanciation unique de UserSecretsClient ─────────────────────
_sc = None
try:
    from kaggle_secrets import UserSecretsClient
    _sc = UserSecretsClient()
except ImportError:
    pass  # Hors Kaggle

def _read_secret(*names):
    for n in names:
        v = os.environ.get(n)
        if v and v.strip(): return v.strip()
    if _sc:
        for n in names:
            try:
                v = _sc.get_secret(n)
                if v and v.strip(): return v.strip()
            except Exception: pass
    return None

# ── HuggingFace token ─────────────────────────────────────────────
HF_TOKEN = _read_secret("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print(f"✅ HF token set  ({HF_TOKEN[:8]}…) — gated datasets accessibles")
else:
    print("○  HF_TOKEN non défini — datasets PUBLICS accessibles sans token")

# ── Kaggle API (optionnel sur Kaggle natif) ───────────────────────
# Sur Kaggle, /kaggle/input est accessible SANS ces secrets.
# Utile uniquement pour télécharger programmatiquement via l'API.
try:
    KU = _read_secret("KAGGLE_USERNAME")
    KK = _read_secret("KAGGLE_KEY")
    if KU and KK:
        os.environ["KAGGLE_USERNAME"] = KU
        os.environ["KAGGLE_KEY"]      = KK
        kdir = Path.home() / ".kaggle"
        kdir.mkdir(exist_ok=True)
        kf = kdir / "kaggle.json"
        kf.write_text(json.dumps({"username": KU, "key": KK}))
        kf.chmod(0o600)
        try:
            import io, contextlib
            _buf = io.StringIO()
            with contextlib.redirect_stdout(_buf), contextlib.redirect_stderr(_buf):
                import kaggle
                kaggle.api.authenticate()
            KAGGLE_OK = True
            print(f"✅ Kaggle API OK  ({KU})")
        except Exception as e:
            print(f"⚠️  Kaggle auth failed: {str(e)[:80]}")
    else:
        print("○  KAGGLE_USERNAME/KEY non définis — /kaggle/input accessible normalement")
except Exception:
    print("○  Kaggle API désactivée — /kaggle/input toujours accessible")

# ── Clés APIs news ────────────────────────────────────────────────
GUARDIAN_KEY  = _read_secret("GUARDIAN_API_KEY",   "GUARDIAN_KEY")
NYT_KEY       = _read_secret("NYT_API_KEY",        "NYTIMES_API_KEY",      "NYT_KEY")
AV_KEY        = _read_secret("ALPHAVANTAGE_KEY",   "ALPHAVANTAGE_API_KEY", "ALPHA_VANTAGE_KEY")
POLYGON_KEY   = _read_secret("POLYGON_API_KEY",    "POLYGON_KEY")
FINNHUB_TOKEN = _read_secret("FINNHUB_TOKEN",      "FINNHUB_API_TOKEN")    # ← ajouté

for label, key, url in [
    ("Guardian API key",   GUARDIAN_KEY,  "https://open-platform.theguardian.com/access/"),
    ("NYT API key",        NYT_KEY,       "https://developer.nytimes.com/"),
    ("Alpha Vantage key",  AV_KEY,        "https://www.alphavantage.co/support/#api-key"),
    ("Polygon.io key",     POLYGON_KEY,   "https://polygon.io/"),
    ("Finnhub token",      FINNHUB_TOKEN, "https://finnhub.io/"),
]:
    if key: print(f"✅ {label} set")
    else:   print(f"○  {label} missing → free at {url}")

# ── Push CONFIG + os.environ ──────────────────────────────────────
CONFIG["hf_ok"]     = True
CONFIG["hf_token"]  = HF_TOKEN is not None
CONFIG["kaggle_ok"] = KAGGLE_OK

for k, v in [("GUARDIAN_API_KEY",  GUARDIAN_KEY),
             ("NYT_API_KEY",       NYT_KEY),
             ("ALPHAVANTAGE_KEY",  AV_KEY),
             ("POLYGON_API_KEY",   POLYGON_KEY),
             ("FINNHUB_TOKEN",     FINNHUB_TOKEN)]:
    if v:
        os.environ[k] = v

print(f"\n  CONFIG → hf_ok={CONFIG['hf_ok']}  "
      f"hf_token={CONFIG['hf_token']}  "
      f"kaggle_ok={CONFIG['kaggle_ok']}")

✅ HF token set  (hf_Vmaug…) — gated datasets accessibles
✅ Kaggle API OK  (abdelkhaleqelhaddad)
✅ Guardian API key set
✅ NYT API key set
✅ Alpha Vantage key set
✅ Polygon.io key set
✅ Finnhub token set

  CONFIG → hf_ok=True  hf_token=True  kaggle_ok=True


## Cell 4 — Stock prices (yfinance)

Telecharge l'OHLCV journalier 2015→2025. Idempotent.

In [15]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — STOCK PRICES (yfinance)                                ║
# ╚══════════════════════════════════════════════════════════════════╝
try:
    import yfinance as yf
except ImportError:
    pip_install("yfinance")
    import yfinance as yf

print(f"yfinance {yf.__version__}")

def normalize_price_df(df, ticker):
    # yfinance returns vary across versions - robustly extract OHLCV.
    if df is None or df.empty: return None
    df = df.copy()
    # Flatten MultiIndex columns (newer yfinance)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    # Detect date column
    date_col = next((c for c in df.columns
                     if str(c).lower() in ("date", "datetime", "index")), None)
    if date_col is None: return None
    df["date"] = pd.to_datetime(df[date_col], errors="coerce", utc=False)
    df = df.dropna(subset=["date"])
    if hasattr(df["date"].dt, "tz") and df["date"].dt.tz is not None:
        df["date"] = df["date"].dt.tz_localize(None)
    # Standard column names (case-insensitive)
    rename = {}
    for c in df.columns:
        cl = str(c).lower().strip()
        if cl in ("open",):        rename[c] = "open"
        elif cl in ("high",):      rename[c] = "high"
        elif cl in ("low",):       rename[c] = "low"
        elif cl in ("close", "adj close"): rename[c] = "close"
        elif cl == "volume":       rename[c] = "volume"
    df = df.rename(columns=rename)
    df["ticker"] = ticker
    # Year filter
    df = df[(df["date"].dt.year >= YEAR_START) & (df["date"].dt.year <= YEAR_END)]
    keep = [c for c in ["ticker","date","open","high","low","close","volume"]
            if c in df.columns]
    df = df[keep].drop_duplicates(subset=["ticker","date"])
    df = df.sort_values("date").reset_index(drop=True)
    for c in ("open","high","low","close"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "volume" in df.columns:
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0).astype("int64")
    return df if not df.empty else None


if ckpt.exists_and_valid(PATH_PRICES, min_rows=100):
    print(f"\n  ✅ {Path(PATH_PRICES).name} exists - skipping download")
    df_prices = ckpt.load(PATH_PRICES)
else:
    print(f"\n  Downloading {YEAR_START}-{YEAR_END} OHLCV for {len(STOCKS)} tickers...\n")
    dfs, failed = [], []
    for tk in STOCKS:
        try:
            raw = yf.download(
                tk,
                start=f"{YEAR_START}-01-01",
                end=f"{YEAR_END}-12-31",
                progress=False,
                auto_adjust=True,
                actions=False,
            )
            df = normalize_price_df(raw, tk)
            if df is not None:
                dfs.append(df)
                print(f"  ✅ {tk:<6}  {len(df):>5} rows  "
                      f"{df['date'].min().date()} -> {df['date'].max().date()}")
            else:
                failed.append(tk)
                print(f"  ⚠️  {tk:<6}  empty after normalize")
        except Exception as e:
            failed.append(tk)
            print(f"  ❌ {tk:<6}  {str(e)[:80]}")

    if not dfs:
        raise RuntimeError("No prices downloaded - check internet")

    df_prices = pd.concat(dfs, ignore_index=True)
    df_prices = df_prices.sort_values(["ticker", "date"]).reset_index(drop=True)

    # Daily return
    df_prices["return_1d"] = (
        df_prices.groupby("ticker")["close"].pct_change().round(6)
    )

    ckpt.save(df_prices, PATH_PRICES)
    print(f"\n  Failed tickers: {failed if failed else 'none'}")

# Summary
print(f"\n  Rows total : {len(df_prices):,}")
print(f"  Tickers    : {df_prices['ticker'].nunique()}")
print(f"  Period     : {df_prices['date'].min().date()} -> {df_prices['date'].max().date()}")


yfinance 0.2.66


  ✅ AAPL     2765 rows  2015-01-02 -> 2025-12-30
  ✅ MSFT     2765 rows  2015-01-02 -> 2025-12-30
  ✅ GOOGL    2765 rows  2015-01-02 -> 2025-12-30
  ✅ AMZN     2765 rows  2015-01-02 -> 2025-12-30
  ✅ META     2765 rows  2015-01-02 -> 2025-12-30
  ✅ TSLA     2765 rows  2015-01-02 -> 2025-12-30
  ✅ NVDA     2765 rows  2015-01-02 -> 2025-12-30
  💾 saved 01_prices.parquet (19,355 rows | 1.0 MB)

  Failed tickers: none

  Rows total : 19,355
  Tickers    : 7
  Period     : 2015-01-02 -> 2025-12-30


## Cell 5 — Collecte de texte sur 10 ans (robuste, multi-source)

**Source primaire = les 15 datasets Kaggle attaches** (`/kaggle/input`, hors-ligne, fiables, 2007→2024) :
headlines, all-the-news, S&P-500-news-2008-2024, financial-tweets, WallStreetBets, tweets-top-companies-2015-2020, etc.
Chaque fichier est lu avec **detection automatique** des colonnes texte / date / ticker, classification
news vs social, filtrage sur nos 7 tickers et la fenetre 2015–2025.

**Sources de recence optionnelles (best-effort, jamais bloquantes)** : FNSPID en streaming (HF) +
RSS Yahoo/Google News pour 2023–2025. Sortie unique → `PATH_RAW` avec le schema attendu par CELL 6 :
`text, ticker, date, source, dtype`.

In [16]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — COLLECTE TEXTE 10 ANS  (Kaggle-input + HF/RSS optionnels)║
# ║  Sortie : PATH_RAW  [text, ticker, date, source, dtype]           ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, gc, re, io, json, time, urllib.request, urllib.parse
from pathlib import Path
import numpy as np
import pandas as pd

RAW_SHD_DIR = Path(SHARDS_DIR)
RAW_SHD_DIR.mkdir(parents=True, exist_ok=True)

def free_ram():
    """Vide la RAM Python ET rend la memoire a l OS (malloc_trim)."""
    import gc as _gc, ctypes as _ct
    _gc.collect()
    try: _ct.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass

free_ram()   # on part d un etat memoire propre

if ckpt.exists_and_valid(PATH_RAW, min_rows=1000):
    import pyarrow.parquet as _pqm
    _nrows = _pqm.ParquetFile(PATH_RAW).metadata.num_rows
    df_raw = None   # on NE charge PAS toutes les lignes en RAM
    print(f"  ✅ {Path(PATH_RAW).name} existe ({_nrows:,} rows) — collecte sautee")
else:
    t_start = time.time()
    SYMBOL_TO_TICKER = {
        "AAPL": "AAPL", "MSFT": "MSFT", "GOOGL": "GOOGL", "GOOG": "GOOGL",
        "AMZN": "AMZN", "META": "META", "FB": "META", "TSLA": "TSLA", "NVDA": "NVDA",
    }
    SOCIAL_HINTS = ("tweet", "reddit", "wallstreet", "wsb", "social", "stocktwits")

    _buffer, _stats = [], {}
    _shard_n, _total = [0], [0]

    def _flush():
        if not _buffer:
            return
        pd.DataFrame(_buffer).to_parquet(
            RAW_SHD_DIR / f"raw_{_shard_n[0]:06d}.parquet",
            index=False, compression="snappy")
        _shard_n[0] += 1
        _buffer.clear()
        gc.collect()

    def budget_left():
        return (_total[0] < HARD_TOTAL_CAP and
                (time.time() - t_start) < COLLECT_TIME_BUDGET_S)

    def add_row(text, ticker, date, source, dtype):
        if not text:
            return False
        text = str(text).strip()
        if len(text) < 20:
            return False
        d = ndate(date)
        if d is None:
            return False
        yr = int(d[:4])
        if yr < YEAR_START or yr > YEAR_END:
            return False
        if ticker not in STOCKS_SET:
            return False
        _buffer.append({"text": text[:globals().get("MAX_TEXT_CHARS", 1000)], "ticker": ticker, "date": d,
                        "source": source, "dtype": dtype})
        _total[0] += 1
        _stats[source] = _stats.get(source, 0) + 1
        if len(_buffer) >= SHARD_FLUSH_EVERY:
            _flush()
        return True

    # ── Detection de colonnes generique ──────────────────────────────
    TEXT_KEYS = ("text", "headline", "title", "article", "content", "body",
                 "tweet", "message", "news", "summary", "top1", "selftext")
    DATE_KEYS = ("date", "datetime", "published", "publish_date", "created_at",
                 "created_utc", "time", "timestamp", "datetime_utc", "pub_date")
    SYM_KEYS  = ("ticker", "symbol", "stock", "stock_symbol", "stocks", "company",
                 "ticker_symbol", "tickers")

    def _pick(cols_lower, keys):
        for k in keys:
            if k in cols_lower:
                return cols_lower[k]
        # fuzzy contains
        for k in keys:
            for cl, orig in cols_lower.items():
                if k in cl:
                    return orig
        return None

    def _ingest_frame(df, source, default_dtype):
        if df is None or len(df) == 0:
            return 0
        cols_lower = {str(c).lower().strip(): c for c in df.columns}
        tcol = _pick(cols_lower, TEXT_KEYS)
        dcol = _pick(cols_lower, DATE_KEYS)
        scol = _pick(cols_lower, SYM_KEYS)
        if tcol is None or dcol is None:
            return 0
        n0 = _total[0]
        texts = df[tcol].astype(str).tolist()
        dates = df[dcol].astype(str).tolist()
        syms  = df[scol].astype(str).tolist() if scol is not None else [None] * len(df)
        for txt, dt, sym in zip(texts, dates, syms):
            if not budget_left():
                break
            tk = None
            if sym is not None:
                tk = SYMBOL_TO_TICKER.get(str(sym).upper().strip().lstrip("$"))
            if tk is None:                      # sinon detection dans le texte
                det = detect_tickers(txt)
                if len(det) == 1:               # mono-ticker only (evite le bruit)
                    tk = det[0]
            if tk is not None:
                add_row(txt, tk, dt, source, default_dtype)
        return _total[0] - n0

    # ── SOURCE A : datasets Kaggle attaches (primaire, hors-ligne) ────
    def collect_kaggle_inputs():
        root = Path("/kaggle/input")
        if not root.exists():
            print("  /kaggle/input absent — aucun dataset attache.")
            return
        files = [p for p in root.rglob("*")
                 if p.suffix.lower() in (".csv", ".parquet", ".json")
                 and p.stat().st_size > 1024]
        files.sort(key=lambda p: p.stat().st_size)   # petits d'abord
        print(f"  /kaggle/input : {len(files)} fichier(s) candidat(s)")
        for p in files:
            if not budget_left():
                break
            slug   = str(p.relative_to(root)).split(os.sep)[0]
            source = f"kaggle:{slug}"[:40]
            is_social = any(h in str(p).lower() for h in SOCIAL_HINTS)
            dtype = "social" if is_social else "news"
            try:
                if p.suffix.lower() == ".parquet":
                    df = pd.read_parquet(p)
                    got = _ingest_frame(df, source, dtype)
                    del df; gc.collect()
                elif p.suffix.lower() == ".json":
                    try:
                        df = pd.read_json(p, lines=True)
                    except Exception:
                        df = pd.read_json(p)
                    got = _ingest_frame(df, source, dtype)
                    del df; gc.collect()
                else:  # CSV — lecture par morceaux pour les gros fichiers
                    got = 0
                    for chunk in pd.read_csv(p, chunksize=500_000,
                                             on_bad_lines="skip",
                                             encoding_errors="ignore",
                                             low_memory=False):
                        got += _ingest_frame(chunk, source, dtype)
                        del chunk; gc.collect()
                        if not budget_left():
                            break
                if got:
                    print(f"    ✅ {p.name[:48]:<48} +{got:,}  [{dtype}]")
            except Exception as e:
                print(f"    ⚠️  {p.name[:48]:<48} {str(e)[:50]}")

    # ── SOURCE B : FNSPID streaming (HF, optionnel/best-effort) ───────
    def collect_fnspid():
        if not budget_left():
            return
        try:
            from datasets import load_dataset
        except Exception:
            try:
                pip_install(["datasets", "huggingface_hub"])
                from datasets import load_dataset
            except Exception as e:
                print("  FNSPID: datasets indisponible —", str(e)[:60]); return
        import logging as _hflog
        for _ln in ("huggingface_hub", "datasets", "huggingface_hub.repocard", "datasets.builder"):
            try: _hflog.getLogger(_ln).setLevel(_hflog.ERROR)
            except Exception: pass
        try:
            ds = load_dataset("Zihan1004/FNSPID", split="train", streaming=True)
        except Exception as e:
            print("  FNSPID indisponible —", str(e)[:80]); return
        n = 0; scanned = 0
        t_fns = time.time()
        print("  FNSPID: streaming demarre (scan en cours, soyez patient)…")
        try:
            for ex in ds:
                if not budget_left():
                    break
                if n >= FNSPID_MAX_ROWS or (time.time() - t_fns) > FNSPID_MAX_SECONDS:
                    print(f"  FNSPID: cap atteint ({n:,} gardes / {scanned:,} scannes / {int(time.time()-t_fns)}s) -> on passe")
                    break
                scanned += 1
                if scanned % 250_000 == 0:
                    print(f"    FNSPID… scannes={scanned:,}  gardes={n:,}  t={int(time.time()-t_fns)}s")
                sym = str(ex.get("Stock_symbol") or ex.get("symbol") or "").upper().strip()
                tk  = SYMBOL_TO_TICKER.get(sym)
                title = ex.get("Article_title") or ex.get("title") or ""
                summ  = ex.get("Lsa_summary") or ex.get("Article") or ""
                date  = ex.get("Date") or ex.get("date") or ""
                txt = (str(title) + ". " + str(summ)).strip(". ")
                if tk is None:
                    det = detect_tickers(txt)
                    tk = det[0] if len(det) == 1 else None
                if tk and add_row(txt, tk, date, "FNSPID", "news"):
                    n += 1
        except Exception as e:
            print("  FNSPID interrompu —", str(e)[:60])
        print(f"  FNSPID: +{n:,}")

    # ── SOURCE C : RSS recent (Yahoo + Google News) best-effort ───────
    def collect_rss():
        feeds = []
        for tk in STOCKS:
            feeds.append((tk, f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={tk}&region=US&lang=en-US"))
            q = urllib.parse.quote(f"{NAMES[tk]} stock")
            feeds.append((tk, f"https://news.google.com/rss/search?q={q}&hl=en-US&gl=US&ceid=US:en"))
        n = 0
        for tk, url in feeds:
            if not budget_left():
                break
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                xml = urllib.request.urlopen(req, timeout=20).read().decode("utf-8", "ignore")
            except Exception:
                continue
            for it in re.findall(r"<item>(.*?)</item>", xml, re.S):
                tm = re.search(r"<title>(.*?)</title>", it, re.S)
                dm = re.search(r"<pubDate>(.*?)</pubDate>", it, re.S)
                if not tm:
                    continue
                title = re.sub(r"<[^>]+>", "", tm.group(1)).strip()
                date  = dm.group(1) if dm else ""
                if add_row(title, tk, date, "RSS", "news"):
                    n += 1
        print(f"  RSS: +{n:,}")

    # ── SOURCE D : datasets HuggingFace (generique, optionnel) ────────
    def collect_hf():
        try:
            from datasets import load_dataset
        except Exception:
            try:
                pip_install(["datasets", "huggingface_hub"])
                from datasets import load_dataset
            except Exception as e:
                print("  HF: datasets indisponible —", str(e)[:60]); return
        for ds_id in HF_DATASETS:
            if not budget_left():
                break
            print(f"  HF: chargement {ds_id} (streaming) …")
            import logging as _hflog2
            for _ln in ("huggingface_hub", "datasets", "huggingface_hub.repocard", "datasets.builder"):
                try: _hflog2.getLogger(_ln).setLevel(_hflog2.ERROR)
                except Exception: pass
            try:
                ds = load_dataset(ds_id, split="train", streaming=True)
            except Exception as e:
                print(f"  HF: {ds_id} indisponible — {str(e)[:70]}"); continue
            batch, n0, t_hf, scanned = [], _total[0], time.time(), 0
            try:
                for ex in ds:
                    if not budget_left():
                        break
                    if (time.time() - t_hf) > HF_MAX_SECONDS or (_total[0] - n0) >= HF_MAX_ROWS:
                        print(f"    cap {ds_id} atteint"); break
                    scanned += 1
                    if scanned % 100_000 == 0:
                        print(f"    HF {ds_id}… lus={scanned:,}  gardes={_total[0]-n0:,}  t={int(time.time()-t_hf)}s")
                    batch.append(ex)
                    if len(batch) >= 50_000:
                        _ingest_frame(pd.DataFrame(batch), f"hf:{ds_id}"[:40], "news")
                        batch.clear(); gc.collect()
                if batch:
                    _ingest_frame(pd.DataFrame(batch), f"hf:{ds_id}"[:40], "news")
                    batch.clear(); gc.collect()
            except Exception as e:
                print(f"    HF {ds_id} interrompu — {str(e)[:60]}")
            print(f"  HF {ds_id}: +{_total[0]-n0:,}")

    # ── SOURCE E : scraping API news historique Finnhub (2024-2025) ───
    def collect_newsapi():
        token = globals().get("FINNHUB_TOKEN")
        if not token:
            print("  NewsAPI: pas de FINNHUB_TOKEN (Add-ons->Secrets) — saute"); return
        import datetime as _dt
        try:
            d_from = _dt.date.fromisoformat(str(NEWSAPI_FROM))
            d_to   = _dt.date.fromisoformat(str(NEWSAPI_TO))
        except Exception:
            d_from, d_to = _dt.date(2024, 1, 1), _dt.date.today()
        if d_to > _dt.date.today():
            d_to = _dt.date.today()
        n0, t_api = _total[0], time.time()
        for tk in STOCKS:
            if not budget_left() or (time.time() - t_api) > NEWSAPI_MAX_SECONDS:
                break
            cur = d_from
            while cur < d_to:
                if not budget_left() or (time.time() - t_api) > NEWSAPI_MAX_SECONDS:
                    break
                nxt = min(cur + _dt.timedelta(days=30), d_to)
                url = (f"https://finnhub.io/api/v1/company-news?symbol={tk}"
                       f"&from={cur}&to={nxt}&token={token}")
                try:
                    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                    raw = urllib.request.urlopen(req, timeout=25).read().decode("utf-8", "ignore")
                    arr = json.loads(raw)
                except Exception:
                    arr = []
                if isinstance(arr, list):
                    for it in arr:
                        head = (str(it.get("headline", "")) + ". " +
                                str(it.get("summary", ""))).strip(". ")
                        ts = it.get("datetime")
                        try:
                            d = _dt.datetime.utcfromtimestamp(int(ts)).strftime("%Y-%m-%d") if ts else ""
                        except Exception:
                            d = ""
                        add_row(head, tk, d, "FINNHUB", "news")
                time.sleep(1.1)   # respecter le rate-limit free (~60 req/min)
                cur = nxt
        print(f"  NewsAPI(Finnhub): +{_total[0]-n0:,}")

    # ── SOURCE F : Alpha Vantage News & Sentiment (datee + sentiment) ─
    def collect_av():
        key = globals().get("AV_KEY")
        if not key:
            print("  AlphaVantage: pas de AV_KEY (Add-ons->Secrets) — saute"); return
        import datetime as _dt
        try:
            d_from = _dt.date.fromisoformat(str(AV_FROM))
            d_to   = _dt.date.fromisoformat(str(AV_TO))
        except Exception:
            d_from, d_to = _dt.date(2022, 1, 1), _dt.date.today()
        if d_to > _dt.date.today():
            d_to = _dt.date.today()
        n0, t_av, reqs = _total[0], time.time(), 0
        for tk in STOCKS:
            if not budget_left() or reqs >= AV_MAX_REQ or (time.time()-t_av) > AV_MAX_SECONDS:
                break
            cur = d_from
            while cur < d_to:
                if not budget_left() or reqs >= AV_MAX_REQ or (time.time()-t_av) > AV_MAX_SECONDS:
                    break
                nxt = min(cur + _dt.timedelta(days=90), d_to)
                tf = cur.strftime("%Y%m%dT0000")
                tt = nxt.strftime("%Y%m%dT2359")
                url = ("https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
                       f"&tickers={tk}&time_from={tf}&time_to={tt}"
                       f"&limit=1000&sort=EARLIEST&apikey={key}")
                try:
                    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                    raw = urllib.request.urlopen(req, timeout=30).read().decode("utf-8", "ignore")
                    obj = json.loads(raw)
                except Exception:
                    obj = {}
                reqs += 1
                if isinstance(obj, dict) and (obj.get("Note") or obj.get("Information")):
                    print("  AlphaVantage: quota/limite atteint — arret de la source")
                    print(f"  AlphaVantage: +{_total[0]-n0:,}  ({reqs} requetes)")
                    return
                feed = obj.get("feed") if isinstance(obj, dict) else None
                if feed:
                    for it in feed:
                        head = (str(it.get("title", "")) + ". " +
                                str(it.get("summary", ""))).strip(". ")
                        tp = str(it.get("time_published", ""))[:8]   # YYYYMMDD
                        try:
                            d = _dt.datetime.strptime(tp, "%Y%m%d").strftime("%Y-%m-%d") if tp else ""
                        except Exception:
                            d = ""
                        add_row(head, tk, d, "ALPHAVANTAGE", "news")
                time.sleep(13)   # free tier ~5 req/min
                cur = nxt
        print(f"  AlphaVantage: +{_total[0]-n0:,}  ({reqs} requetes)")

    # ── Reglages sources optionnelles (B/C) ───────────────────────────
    # Off par defaut : SOURCE A (tes 15 datasets) suffit. Mets True dans la
    # CELL 1 (ou ici) pour rallumer, avec garde-fous anti-blocage ci-dessous.
    USE_FNSPID        = globals().get("USE_FNSPID", False)
    USE_RSS           = globals().get("USE_RSS", False)
    FNSPID_MAX_ROWS   = globals().get("FNSPID_MAX_ROWS", 50_000)
    FNSPID_MAX_SECONDS= globals().get("FNSPID_MAX_SECONDS", 600)
    USE_HF            = globals().get("USE_HF", False)
    HF_DATASETS       = globals().get("HF_DATASETS", ["m-ric/financial-news-2024"])
    HF_MAX_ROWS       = globals().get("HF_MAX_ROWS", 2_000_000)
    HF_MAX_SECONDS    = globals().get("HF_MAX_SECONDS", 1200)
    USE_NEWSAPI       = globals().get("USE_NEWSAPI", False)
    NEWSAPI_FROM      = globals().get("NEWSAPI_FROM", "2024-01-01")
    NEWSAPI_TO        = globals().get("NEWSAPI_TO", "2025-12-31")
    NEWSAPI_MAX_SECONDS = globals().get("NEWSAPI_MAX_SECONDS", 1200)
    USE_AV            = globals().get("USE_AV", False)
    AV_FROM           = globals().get("AV_FROM", "2022-01-01")
    AV_TO             = globals().get("AV_TO", "2025-12-31")
    AV_MAX_REQ        = globals().get("AV_MAX_REQ", 24)
    AV_MAX_SECONDS    = globals().get("AV_MAX_SECONDS", 900)

    # ── Execution des sources ─────────────────────────────────────────
    print("\n── SOURCE A : datasets Kaggle attaches ──────────────────────")
    collect_kaggle_inputs()
    print(f"  cumul apres Kaggle : {_total[0]:,}")

    if USE_FNSPID and budget_left():
        print("\n── SOURCE B : FNSPID (HF streaming, optionnel) ──────────────")
        collect_fnspid()
    else:
        print("\n── SOURCE B : FNSPID désactivée (USE_FNSPID=False) ──────────")
    if USE_RSS and budget_left():
        print("\n── SOURCE C : RSS recence 2023-2025 (optionnel) ─────────────")
        collect_rss()
    else:
        print("── SOURCE C : RSS désactivée (USE_RSS=False) ────────────────")
    if USE_HF and budget_left():
        print("\n── SOURCE D : datasets HuggingFace (optionnel) ──────────────")
        collect_hf()
    else:
        print("── SOURCE D : HF désactivée (USE_HF=False) ──────────────────")
    if USE_NEWSAPI and budget_left():
        print("\n── SOURCE E : scraping API Finnhub 2024-2025 (optionnel) ────")
        collect_newsapi()
    else:
        print("── SOURCE E : NewsAPI désactivée (USE_NEWSAPI=False) ─────────")
    if USE_AV and budget_left():
        print("\n── SOURCE F : Alpha Vantage News+Sentiment (optionnel) ──────")
        collect_av()
    else:
        print("── SOURCE F : AlphaVantage désactivée (USE_AV=False) ─────────")

    _flush()

    # ── Consolidation + dedup + filtre ────────────────────────────────
    print("\n── Consolidation des shards ─────────────────────────────────")
    shards = sorted(RAW_SHD_DIR.glob("raw_*.parquet"))
    if not shards:
        if not CONFIG["allow_synthetic_fallback"]:
            raise RuntimeError(
                "Collecte vide : aucun texte recolte. Verifie que des datasets "
                "sont bien attaches dans /kaggle/input (Add Data) ou active le reseau "
                "pour FNSPID/RSS. (allow_synthetic_fallback=False → pas de fausses donnees.)")
    # ── Consolidation SÉQUENTIELLE : shard par shard, jamais tout en RAM ──
    import pyarrow as _pa, pyarrow.parquet as _pq
    import shutil as _sh
    _SCHEMA = _pa.schema([("text", _pa.string()), ("ticker", _pa.string()),
                          ("date", _pa.timestamp("ns")), ("source", _pa.string()),
                          ("dtype", _pa.string())])
    _writer = _pq.ParquetWriter(PATH_RAW, _SCHEMA, compression="snappy")
    _seen = set()          # hash des textes vus (borne RAM ~1 Go a 10M)
    pre, kept = 0, 0
    try:
        for _sp in shards:
            try:
                _d = pd.read_parquet(_sp)
            except Exception:
                try: _sp.unlink()
                except Exception: pass
                continue
            pre += len(_d)
            _d["_h"] = _d["text"].astype(str).str[:200].apply(text_hash)
            _d = _d[~_d["_h"].isin(_seen)]
            _seen.update(_d["_h"].tolist())
            _d = _d.drop(columns="_h")
            _d["date"] = pd.to_datetime(_d["date"], errors="coerce")
            _d = _d.dropna(subset=["date"])
            _d = _d[(_d["date"].dt.year >= YEAR_START) & (_d["date"].dt.year <= YEAR_END)]
            if len(_d):
                for _c in ("text", "ticker", "source", "dtype"):
                    _d[_c] = _d[_c].astype(str)
                _tbl = _pa.Table.from_pandas(
                    _d[["text", "ticker", "date", "source", "dtype"]],
                    schema=_SCHEMA, preserve_index=False)
                _writer.write_table(_tbl)
                kept += len(_d)
            try: _sp.unlink()          # libère le disque IMMÉDIATEMENT
            except Exception: pass
            del _d; gc.collect()
    finally:
        _writer.close()
    del _seen; free_ram()
    try: _sh.rmtree(RAW_SHD_DIR, ignore_errors=True)
    except Exception: pass
    df_raw = None        # rien en RAM
    print("  🧹 shards supprimés au fil de l eau (disque + RAM bornés)")
    print(f"  Dedup/filtre : {pre:,} → {kept:,}")
    print("  Par source :")
    for s, n in sorted(_stats.items(), key=lambda x: -x[1]):
        print(f"    {s:<40} {n:>10,}")

# ── Rapport ──────────────────────────────────────────────────────────
print("\n" + "═" * 65)
try:
    _rep = pd.read_parquet(PATH_RAW, columns=["ticker", "date", "dtype"])  # SANS le texte = léger
    _rep["date"] = pd.to_datetime(_rep["date"], errors="coerce")
    print(f"  RAW rows     : {len(_rep):,}")
    print(f"  News/Social  : {(_rep['dtype']=='news').sum():,} / {(_rep['dtype']=='social').sum():,}")
    print(f"  Periode      : {_rep['date'].min().date()} → {_rep['date'].max().date()}")
    print("  Par ticker :")
    for tk, n in _rep.groupby("ticker").size().sort_values(ascending=False).items():
        print(f"    {tk:<6} {n:>10,}")
    del _rep; free_ram()
except Exception as _e:
    print(f"  (rapport indisponible: {str(_e)[:50]})")
print("═" * 65)



── SOURCE A : datasets Kaggle attaches ──────────────────────
  /kaggle/input : 47 fichier(s) candidat(s)
    ⚠️  catboost_training.json                           Mixing dicts with non-Series may lead to ambiguous
    ✅ cnbc_headlines.csv                               +214  [news]
    ✅ guardian_headlines.csv                           +452  [news]
    ✅ sp500_headlines_2008_2024.csv                    +579  [news]
    ✅ Combined_News_DJIA.csv                           +12  [news]
    ✅ stockerbot-export1.csv                           +1,280  [social]
    ✅ stockerbot-export.csv                            +1,273  [social]
    ✅ RedditNews.csv                                   +200  [social]
    ✅ reuters_headlines.csv                            +2,467  [news]
    ✅ stock_tweets.csv                                 +58,553  [social]
    ✅ reduced_dataset-release.csv                      +6,252  [social]
    ✅ reddit_wsb.csv                                   +840  [social]
    ✅ analyst_r

README.md: 0.00B [00:00, ?B/s]

  FNSPID: streaming demarre (scan en cours, soyez patient)…
    FNSPID… scannes=250,000  gardes=2,285  t=10s
    FNSPID… scannes=500,000  gardes=4,307  t=20s
    FNSPID… scannes=750,000  gardes=8,557  t=30s
    FNSPID… scannes=1,000,000  gardes=13,058  t=41s
    FNSPID… scannes=1,250,000  gardes=14,564  t=51s
    FNSPID… scannes=1,500,000  gardes=17,856  t=92s
    FNSPID… scannes=1,750,000  gardes=17,856  t=196s
    FNSPID… scannes=2,000,000  gardes=21,117  t=308s
    FNSPID… scannes=2,250,000  gardes=33,012  t=395s
    FNSPID… scannes=2,500,000  gardes=36,084  t=404s
    FNSPID… scannes=2,750,000  gardes=39,277  t=413s
    FNSPID… scannes=3,000,000  gardes=42,707  t=422s
    FNSPID… scannes=3,250,000  gardes=46,704  t=430s
    FNSPID… scannes=3,500,000  gardes=49,977  t=439s
    FNSPID… scannes=3,750,000  gardes=53,586  t=448s
    FNSPID… scannes=4,000,000  gardes=57,667  t=458s
    FNSPID… scannes=4,250,000  gardes=58,896  t=466s
    FNSPID… scannes=4,500,000  gardes=59,043  t=475s
 

README.md:   0%|          | 0.00/348 [00:00<?, ?B/s]

  HF m-ric/financial-news-2024: +1,706

── SOURCE E : scraping API Finnhub 2024-2025 (optionnel) ────
  NewsAPI(Finnhub): +11,798

── SOURCE F : Alpha Vantage News+Sentiment (optionnel) ──────
  AlphaVantage: +6,642  (24 requetes)

── Consolidation des shards ─────────────────────────────────
  🧹 shards supprimés au fil de l eau (disque + RAM bornés)
  Dedup/filtre : 4,068,888 → 3,812,968
  Par source :
    kaggle:datasets                           3,719,012
    FNSPID                                      329,730
    FINNHUB                                      11,798
    ALPHAVANTAGE                                  6,642
    hf:m-ric/financial-news-2024                  1,706

═════════════════════════════════════════════════════════════════
  RAW rows     : 3,812,968
  News/Social  : 425,173 / 3,387,795
  Periode      : 2015-01-01 → 2025-12-31
  Par ticker :
    AAPL    1,205,257
    TSLA    1,089,629
    AMZN      578,627
    GOOGL     483,098
    MSFT      292,027
    META      11

## Cell 6 — Nettoyage + score de sarcasme emoji-aware

In [17]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — TEXT CLEANING + EMOJI-AWARE SARCASM SCORE (v3)        ║
# ║                                                                  ║
# ║  Optimisations vs v2:                                           ║
# ║   • Traitement SÉQUENTIEL par chunks (≤500k rows/chunk)         ║
# ║     → RAM constante ~2 GB quelle que soit la taille du dataset  ║
# ║   • Checkpoint par shard de sortie → reprise si crash           ║
# ║   • joblib Parallel appliqué chunk par chunk (pas sur 6M rows)  ║
# ║   • Pré-filtre longueur AVANT le demojize (opération lente)     ║
# ║   • Flush parquet par shard → consolidation finale légère       ║
# ║   • Sarcasm: vecteur numpy pré-compilé (3× plus rapide)         ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, gc, re, time, hashlib
from pathlib import Path
import numpy  as np
import pandas as pd

# ── Paths CANONIQUES (definis en CELL 1/2) ───────────────────────
# PATH_RAW / PATH_CLEAN / ckpt / pip_install proviennent des cellules 1 & 2.
CLEAN_SHD_DIR = Path(CACHE_DIR) / "clean_shards"
CLEAN_SHD_DIR.mkdir(parents=True, exist_ok=True)

def _read_rows(_path, _start, _end):
    """Lit SEULEMENT les row-groups chevauchant [start,end) — jamais tout le fichier."""
    import pyarrow.parquet as _pq, pandas as _pd
    _pf = _pq.ParquetFile(_path); _md = _pf.metadata
    _out, _pos = [], 0
    for _rg in range(_md.num_row_groups):
        _n = _md.row_group(_rg).num_rows
        if _pos + _n <= _start: _pos += _n; continue
        if _pos >= _end: break
        _t = _pf.read_row_group(_rg)
        _lo = max(0, _start - _pos); _hi = min(_n, _end - _pos)
        _out.append(_t.slice(_lo, _hi - _lo).to_pandas())
        _pos += _n
    return _pd.concat(_out, ignore_index=True) if _out else _pd.DataFrame()

CHUNK_SIZE = 500_000   # rows lus à la fois — ajuster selon RAM disponible

# ── Vérif checkpoint final ────────────────────────────────────────
if ckpt.exists_and_valid(PATH_CLEAN, min_rows=10):
    print(f"  ✅ {Path(PATH_CLEAN).name} exists ({pd.read_parquet(PATH_CLEAN).shape[0]:,} rows) — skipping")
    df_clean = ckpt.load(PATH_CLEAN)
else:
    # ── Dépendances ──────────────────────────────────────────────
    try: import emoji
    except ImportError: pip_install("emoji"); import emoji
    try: from joblib import Parallel, delayed
    except ImportError: pip_install("joblib"); from joblib import Parallel, delayed

    n_workers = max(1, (os.cpu_count() or 2) - 1)
    print(f"  joblib workers: {n_workers}")

    # ── Regex pré-compilés ────────────────────────────────────────
    _URL_RE     = re.compile(r"https?://\S+|www\.\S+")
    _MENTION_RE = re.compile(r"@\w+")
    _HASHTAG_RE = re.compile(r"#(\w+)")
    _DOLLAR_RE  = re.compile(r"\$([A-Z]{2,5})")
    _HTML_RE    = re.compile(r"<[^>]+>")
    _MULTI_WS   = re.compile(r"\s+")
    _CTRL_RE    = re.compile(r"[\x00-\x1F\x7F]")
    _NOISE_RE   = re.compile(
        r"^(rt\s|buy now|click here|free trial|sign up|^\s*\d+\s*$)", re.I)
    _SHORT_RE   = re.compile(r"^.{0,19}$", re.DOTALL)  # < 20 chars

    def clean_news(t):
        t = _HTML_RE.sub(" ", t)
        t = _URL_RE.sub(" ", t)
        t = emoji.demojize(t, delimiters=(" :", ": "))
        t = _CTRL_RE.sub(" ", t)
        return _MULTI_WS.sub(" ", t).strip()

    def clean_social(t):
        t = _URL_RE.sub(" ", t)
        t = _MENTION_RE.sub(" ", t)
        t = _HASHTAG_RE.sub(r"\1", t)
        t = _DOLLAR_RE.sub(r"\1", t)
        t = emoji.demojize(t, delimiters=(" :", ": "))
        t = _CTRL_RE.sub(" ", t)
        return _MULTI_WS.sub(" ", t).strip()

    def _clean_one(args):
        text, is_social = args
        return clean_social(text) if is_social else clean_news(text)

    # ── Patterns sarcasme ──────────────────────────────────────────
    # Compilés une seule fois, réutilisés sur tous les chunks
    SARC_PATTERNS = [
        # Lexical / phrasal
        (re.compile(r'"[\w\s]+"', re.I),                                         0.15),
        (re.compile(r"\b(oh|wow|yeah|sure|totally)\s+(great|awesome|wonderful|amazing|brilliant)", re.I), 0.30),
        (re.compile(r"\bthanks?\s+a\s+(lot|bunch)\b", re.I),                     0.25),
        (re.compile(r"\bobviously\b", re.I),                                     0.10),
        (re.compile(r"\b(genius|brilliant)\s+(move|idea|plan)", re.I),           0.30),
        (re.compile(r"\bcouldn'?t\s+be\s+(better|happier|prouder)", re.I),       0.25),
        (re.compile(r"\bwhat\s+a\s+(surprise|shocker)", re.I),                   0.35),
        (re.compile(r"\b/s\b"),                                                  0.50),
        (re.compile(r"\b(yeah\s+)?right\b\s*[.!]"),                              0.15),
        (re.compile(r"!{2,}|\?{2,}"),                                            0.10),
        (re.compile(r"\b(definitely|absolutely)\s+(not|won'?t)", re.I),          0.10),
        (re.compile(r"\b(moon|rocket|tendies|stonks|yolo|apes?)\b", re.I),       0.20),
        (re.compile(r"\bto\s+the\s+moon\b", re.I),                              0.30),
        (re.compile(r"\b(this\s+is\s+fine|totally\s+normal|nothing\s+to\s+see\s+here)\b", re.I), 0.40),
        (re.compile(r"\b(just\s+a\s+dip|buy\s+the\s+dip)\b", re.I),             0.15),
        (re.compile(r"\bhow\s+convenient\b", re.I),                              0.35),
        # WSB / finance meme
        (re.compile(r"\b(diamond\s+hands?|paper\s+hands?)\b", re.I),            0.20),
        (re.compile(r"\b(calls?|puts?)\s+(printing|ripping|mooning)\b", re.I),  0.15),
        # Demojized emoji irony signals
        (re.compile(r":face_with_rolling_eyes:"),                               0.50),
        (re.compile(r":clown_face:"),                                           0.55),
        (re.compile(r":upside.down_face:"),                                     0.55),
        (re.compile(r":smirking_face:"),                                        0.35),
        (re.compile(r":winking_face:.{0,40}(great|amazing|awesome|sure)", re.I),0.30),
        (re.compile(r":(rocket|full_moon|crescent_moon|new_moon|waxing_crescent_moon):"),  0.15),
        (re.compile(r":(skull|skull_and_crossbones):"),                         0.30),
        (re.compile(r":(thinking_face|monocle_face):"),                         0.20),
        (re.compile(r":(see-no-evil_monkey|hear-no-evil_monkey|speak-no-evil_monkey):"),  0.25),
        (re.compile(r":fire:.{0,30}:fire:"),                                    0.10),
        (re.compile(r":(face_with_tears_of_joy|rolling_on_the_floor_laughing):"
                    r".{0,40}(loss|crash|down|bear|tank|dump|sell.?off)", re.I),0.45),
        (re.compile(r":(grinning_face|beaming_face_with_smiling_eyes|smiling_face_with_smiling_eyes):"
                    r".{0,40}(disaster|bankruptcy|fired|loss|crash)", re.I),    0.40),
        # Anger / fear lexicon (bonus pour futurs modèles)
        (re.compile(r"\b(panic|crash|plunge|collapse|disaster|catastrophe)\b", re.I), 0.05),
        (re.compile(r"\b(furious|outraged|enraged|infuriated|disgusted)\b", re.I),    0.05),
    ]

    # Vectorisé: score sarcasme en batch sur Series pandas
    def sarcasm_score_batch(texts: pd.Series) -> np.ndarray:
        """Applique tous les patterns sur une Series → ndarray float32."""
        scores = np.zeros(len(texts), dtype=np.float32)
        arr    = texts.to_numpy()       # numpy string array, lecture rapide
        for rx, w in SARC_PATTERNS:
            mask = np.array([bool(rx.search(t)) for t in arr], dtype=bool)
            scores[mask] += w
        return np.clip(scores, 0.0, 1.0).astype(np.float32)

    # ── Découverte du nombre total de rows dans raw ───────────────
    print(f"\n  Lecture metadata de {Path(PATH_RAW).name}...")
    raw_schema   = pd.read_parquet(PATH_RAW, columns=["ticker"])  # lecture légère
    total_rows   = len(raw_schema)
    del raw_schema; gc.collect()

    n_chunks     = (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f"  Total rows  : {total_rows:,}")
    print(f"  Chunk size  : {CHUNK_SIZE:,}")
    print(f"  Nb chunks   : {n_chunks}")

    # ── Reprise: compter les shards déjà produits ─────────────────
    done_shards = sorted(CLEAN_SHD_DIR.glob("clean_shard_*.parquet"))
    start_chunk = len(done_shards)
    if start_chunk > 0:
        print(f"\n  ♻️  {start_chunk} shard(s) déjà traités — reprise au chunk {start_chunk}")

    t_total = time.time()
    rows_processed = start_chunk * CHUNK_SIZE

    # ── Boucle principale: chunk par chunk ───────────────────────
    for chunk_idx in range(start_chunk, n_chunks):
        row_start = chunk_idx * CHUNK_SIZE
        row_end   = min(row_start + CHUNK_SIZE, total_rows)

        t0 = time.time()
        print(f"\n  ── Chunk {chunk_idx+1}/{n_chunks}  "
              f"[rows {row_start:,} → {row_end:,}] ──────────────────")

        # Lecture sélective du chunk (parquet row-group skipping)
        df = _read_rows(PATH_RAW, row_start, row_end).copy()   # lecture séquentielle, RAM bornée
        # ↑ Pour les très grands fichiers, préférer:
        # df = pq.read_table(PATH_RAW, filters=[...]).to_pandas().iloc[...]
        # Mais iloc suffît si PATH_RAW tient en RAM séquentiellement

        gc.collect()
        n_raw = len(df)

        # ── 1. Pré-filtre longueur (avant demojize = économise du temps) ─
        df = df[df["text"].str.len() >= 20].copy()
        n_prefilter = n_raw - len(df)

        # ── 2. Cleaning parallèle ─────────────────────────────────
        texts       = df["text"].tolist()
        is_social_l = (df["dtype"] == "social").tolist()

        cleaned = Parallel(n_jobs=n_workers, batch_size=5_000, backend="loky")(
            delayed(_clean_one)((t, s)) for t, s in zip(texts, is_social_l)
        )
        df["text"] = cleaned
        del texts, is_social_l, cleaned; gc.collect()

        # ── 3. Filtre post-cleaning ───────────────────────────────
        df = df[df["text"].str.len() >= 20].copy()
        df = df[~df["text"].str.contains(_NOISE_RE, na=False)].copy()
        n_after_clean = len(df)

        # ── 4. Sarcasm score (batch vectorisé) ────────────────────
        df["sarcasm_score"] = sarcasm_score_batch(df["text"])
        # Score de sentiment EMOJI (distinct de FinBERT) — sur tokens demojizés ":name:"
        _EMO_BULL = r":(rocket|chart_increasing|money_bag|money_with_wings|dollar_banknote|fire|gem_stone|raising_hands|flexed_biceps|party_popper|full_moon|thumbs_up|green_circle|green_heart|star_struck|hundred_points|partying_face|upwards_button):"
        _EMO_BEAR = r":(chart_decreasing|skull|loudly_crying_face|crying_face|bear|red_circle|thumbs_down|broken_heart|fearful_face|weary_face|nauseated_face|pile_of_poo|downwards_button|anxious_face_with_sweat):"
        _etxt = df["text"].astype(str)
        _nbull = _etxt.str.count(_EMO_BULL); _nbear = _etxt.str.count(_EMO_BEAR)
        df["emoji_score"] = ((_nbull - _nbear) / (_nbull + _nbear + 1)).astype("float32")

        # ── 5. ID stable ─────────────────────────────────────────
        df = df.reset_index(drop=True)
        df["id"] = np.arange(row_start, row_start + len(df), dtype=np.int64)

        # ── 6. Optimisation mémoire ───────────────────────────────
        for col in ["ticker", "source", "dtype"]:
            if col in df.columns:
                df[col] = df[col].astype("category")
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")

        # ── 7. Sauvegarde du shard ────────────────────────────────
        shard_path = str(CLEAN_SHD_DIR / f"clean_shard_{chunk_idx:05d}.parquet")
        df.to_parquet(shard_path, index=False, compression="snappy")

        elapsed = time.time() - t0
        rows_processed += n_after_clean
        n_sarc_05 = (df["sarcasm_score"] > 0.5).sum()

        print(f"    raw={n_raw:,}  pre-filtered={n_prefilter:,}  "
              f"kept={n_after_clean:,}  sarc>0.5={n_sarc_05:,}")
        print(f"    ⏱ {elapsed:.1f}s  |  cumul: {rows_processed:,} rows")
        del df; gc.collect()

    # ── Consolidation finale des shards ──────────────────────────
    print(f"\n── Consolidation des shards ─────────────────────────────")
    all_shards = sorted(CLEAN_SHD_DIR.glob("clean_shard_*.parquet"))
    print(f"  {len(all_shards)} shards → lecture...")

    # Lecture par batches de 10 shards pour ne pas saturer la RAM
    BATCH_SHARDS = 10
    consolidated = []
    for i in range(0, len(all_shards), BATCH_SHARDS):
        batch = all_shards[i : i + BATCH_SHARDS]
        parts = [pd.read_parquet(str(p)) for p in batch]
        df_b  = pd.concat(parts, ignore_index=True)
        del parts; gc.collect()
        consolidated.append(df_b)
        del df_b; gc.collect()

    print("  Concat final...")
    df_clean = pd.concat(consolidated, ignore_index=True)
    del consolidated; gc.collect()

    # Déduplication globale sur hash du texte (cross-shard)
    pre_dedup = len(df_clean)
    df_clean["_h"] = df_clean["text"].str[:200].apply(
        lambda s: hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest())
    df_clean = df_clean.drop_duplicates(subset=["_h"]).drop(columns=["_h"])
    print(f"  Dedup: {pre_dedup:,} → {len(df_clean):,}")

    # Tri final + ré-indexation des IDs
    df_clean = (df_clean
                .sort_values(["ticker", "date"])
                .reset_index(drop=True))
    df_clean["id"] = np.arange(len(df_clean), dtype=np.int64)

    # Sauvegarde finale
    ckpt.save(df_clean, PATH_CLEAN)
    try:
        import shutil as _sh; _sh.rmtree(CLEAN_SHD_DIR, ignore_errors=True)
        print("  🧹 clean_shards supprimés (espace libéré)")
    except Exception: pass
    total_elapsed = (time.time() - t_total) / 60
    print(f"\n  ✅ Sauvegardé → {Path(PATH_CLEAN).name}")
    print(f"  ⏱ Temps total: {total_elapsed:.1f} min")

    # Nettoyage shards intermédiaires (optionnel — décommenter si espace limité)
    # for p in all_shards: p.unlink()

# ── Rapport final ─────────────────────────────────────────────────
print("\n" + "═"*65)
print(f"  Clean rows      : {len(df_clean):,}")
print(f"  News            : {(df_clean['dtype']=='news').sum():,}")
print(f"  Social          : {(df_clean['dtype']=='social').sum():,}")
print(f"  Sources         : {df_clean['source'].nunique()}")
if "date" in df_clean.columns:
    dates = pd.to_datetime(df_clean["date"], errors="coerce")
    print(f"  Date range      : {dates.min().date()} → {dates.max().date()}")
n_s05 = (df_clean["sarcasm_score"] > 0.5).sum()
n_s03 = (df_clean["sarcasm_score"] > 0.3).sum()
print(f"\n  Sarcasm > 0.3   : {n_s03:,} ({n_s03/len(df_clean)*100:.1f}%)")
print(f"  Sarcasm > 0.5   : {n_s05:,} ({n_s05/len(df_clean)*100:.1f}%)")
print(f"  Mean sarc score : {df_clean['sarcasm_score'].mean():.4f}")
if n_s05 > 0:
    ex = df_clean[df_clean["sarcasm_score"] > 0.5]["text"].iloc[0][:140]
    print(f"  Exemple sarc>0.5: {ex!r}")
print(f"\n  Par ticker:")
for tk, n in df_clean.groupby("ticker").size().sort_values(ascending=False).items():
    nn = df_clean[(df_clean["ticker"]==tk) & (df_clean["dtype"]=="news")].shape[0]
    ns = df_clean[(df_clean["ticker"]==tk) & (df_clean["dtype"]=="social")].shape[0]
    print(f"    {tk:<6}  {n:>9,}   news {nn:>8,}   social {ns:>8,}")
print(f"\n  Colonnes        : {list(df_clean.columns)}")
print("═"*65)

  joblib workers: 3

  Lecture metadata de 02_text_raw.parquet...
  Total rows  : 3,812,968
  Chunk size  : 500,000
  Nb chunks   : 8

  ── Chunk 1/8  [rows 0 → 500,000] ──────────────────
    raw=500,000  pre-filtered=0  kept=408,789  sarc>0.5=452
    ⏱ 48.7s  |  cumul: 408,789 rows

  ── Chunk 2/8  [rows 500,000 → 1,000,000] ──────────────────
    raw=500,000  pre-filtered=0  kept=492,807  sarc>0.5=7
    ⏱ 44.6s  |  cumul: 901,596 rows

  ── Chunk 3/8  [rows 1,000,000 → 1,500,000] ──────────────────
    raw=500,000  pre-filtered=0  kept=496,592  sarc>0.5=3
    ⏱ 45.7s  |  cumul: 1,398,188 rows

  ── Chunk 4/8  [rows 1,500,000 → 2,000,000] ──────────────────
    raw=500,000  pre-filtered=0  kept=496,057  sarc>0.5=13
    ⏱ 48.1s  |  cumul: 1,894,245 rows

  ── Chunk 5/8  [rows 2,000,000 → 2,500,000] ──────────────────
    raw=500,000  pre-filtered=0  kept=495,420  sarc>0.5=38
    ⏱ 63.0s  |  cumul: 2,389,665 rows

  ── Chunk 6/8  [rows 2,500,000 → 3,000,000] ──────────────────
    raw=

## Cell 7 — FinBERT sentiment (`ProsusAI/finbert`)

In [18]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — FINBERT SENTIMENT (v3 — SÉQUENTIEL + DATASET COMPLET) ║
# ║                                                                  ║
# ║  Optimisations vs v1:                                           ║
# ║   • Traitement COMPLET du dataset (plus de sample 80k sur CPU)  ║
# ║   • Boucle chunks de 100k rows → RAM constante ~3 GB            ║
# ║   • Checkpoint par shard → reprise au chunk N+1 si crash        ║
# ║   • GPU: batch 256 + fp16 + torch.compile() si dispo            ║
# ║   • CPU: batch 64 + quantization INT8 (2× speed)                ║
# ║   • Truncation à 64 tokens (headlines/tweets < 280 chars)       ║
# ║   • Pas de mem.optimize_df() par chunk → évite re-import        ║
# ║   • Consolidation finale légère (10 shards à la fois)           ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, gc, time
from pathlib import Path
import numpy  as np
import pandas as pd

# ── Paths CANONIQUES (CELL 1/2) ──────────────────────────────────
# PATH_CLEAN / OUTPUT_DIR / ckpt / pip_install proviennent des cellules 1 & 2.
PATH_FB_OUT = f"{OUTPUT_DIR}/04a_finbert.parquet"
def _read_rows(_path, _start, _end):
    """Lit SEULEMENT les row-groups chevauchant [start,end) — jamais tout le fichier."""
    import pyarrow.parquet as _pq, pandas as _pd
    _pf = _pq.ParquetFile(_path); _md = _pf.metadata
    _out, _pos = [], 0
    for _rg in range(_md.num_row_groups):
        _n = _md.row_group(_rg).num_rows
        if _pos + _n <= _start: _pos += _n; continue
        if _pos >= _end: break
        _t = _pf.read_row_group(_rg)
        _lo = max(0, _start - _pos); _hi = min(_n, _end - _pos)
        _out.append(_t.slice(_lo, _hi - _lo).to_pandas())
        _pos += _n
    return _pd.concat(_out, ignore_index=True) if _out else _pd.DataFrame()
FB_SHD_DIR  = Path(CACHE_DIR) / "finbert_shards"
FB_SHD_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000   # rows par chunk — réduire à 50_000 si OOM sur CPU
import glob, shutil
from pathlib import Path
_cands = glob.glob("/kaggle/input/**/04a_finbert.parquet", recursive=True)
if _cands and not Path(PATH_FB_OUT).exists():
    Path(PATH_FB_OUT).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(_cands[0], PATH_FB_OUT)
    print(f"  ♻️  FinBERT rechargé depuis le Dataset (aucun recalcul) : {_cands[0]}")

# ── Checkpoint final ──────────────────────────────────────────────
if ckpt.exists_and_valid(PATH_FB_OUT, min_rows=10):
    n = pd.read_parquet(PATH_FB_OUT).shape[0]
    print(f"  ✅ 04a_finbert.parquet exists ({n:,} rows) — skipping")
    df_fb = ckpt.load(PATH_FB_OUT)
else:
    # ── Dépendances ──────────────────────────────────────────────
    try: import torch
    except ImportError: pip_install("torch"); import torch
    try: from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                                    pipeline as hf_pipeline)
    except ImportError:
        pip_install(["transformers", "tokenizers", "accelerate"])
        from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                                   pipeline as hf_pipeline)
    try: from tqdm.auto import tqdm
    except ImportError: pip_install("tqdm"); from tqdm.auto import tqdm

    # ── Détection device ─────────────────────────────────────────
    device    = 0 if torch.cuda.is_available() else -1
    is_gpu    = device == 0
    use_fp16  = False

    if is_gpu:
        gpu_name  = torch.cuda.get_device_name(0)
        vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
        use_fp16  = vram_gb >= 8          # fp16 safe sur T4/P100/A100
        BATCH     = 256 if vram_gb >= 16 else 128 if vram_gb >= 8 else 64
        print(f"  Device  : {gpu_name} ({vram_gb:.1f} GB VRAM)")
        print(f"  fp16    : {use_fp16}  |  batch/inférence: {BATCH}")
    else:
        BATCH = 64
        print(f"  Device  : CPU  |  batch: {BATCH}  |  INT8 quantization activée")

    # ── Chargement modèle ─────────────────────────────────────────
    MODEL_NAME = "ProsusAI/finbert"
    print(f"\n  Loading {MODEL_NAME}...")
    t_load = time.time()

    if is_gpu:
        dtype_arg = torch.float16 if use_fp16 else torch.float32
        fb = hf_pipeline(
            "sentiment-analysis",
            model=MODEL_NAME,
            device=device,
            max_length=64,
            truncation=True,
            torch_dtype=dtype_arg,
        )
        # torch.compile : OFF par defaut (sur T4 les recompilations CUDAGraph
        # coutent souvent plus qu elles ne rapportent). Mets USE_TORCH_COMPILE=True
        # dans la CELL 1 pour le reactiver.
        if globals().get("USE_TORCH_COMPILE", False):
            try:
                fb.model = torch.compile(fb.model, mode="reduce-overhead")
                print("  torch.compile: activé")
            except Exception:
                pass
        else:
            print("  torch.compile: désactivé (plus stable/rapide sur T4)")
    else:
        # CPU: quantization INT8 via bitsandbytes ou fallback pipeline standard
        try:
            from transformers import BitsAndBytesConfig
            qconfig = BitsAndBytesConfig(load_in_8bit=True)
            fb = hf_pipeline(
                "sentiment-analysis",
                model=MODEL_NAME,
                device_map="cpu",
                max_length=64,
                truncation=True,
                model_kwargs={"quantization_config": qconfig},
            )
            print("  INT8 quantization: activée (bitsandbytes)")
        except Exception:
            # Fallback: pipeline standard CPU
            fb = hf_pipeline(
                "sentiment-analysis",
                model=MODEL_NAME,
                device=-1,
                max_length=64,
                truncation=True,
            )
            print("  INT8 quantization: non disponible — pipeline standard CPU")

    print(f"  ✅ Modèle chargé en {time.time()-t_load:.1f}s\n")

    # ── Mapping labels → score numérique ─────────────────────────
    def results_to_arrays(results):
        labels = []
        scores = []
        for r in results:
            lbl = r["label"].lower()
            sc  = float(r["score"])
            labels.append(lbl)
            scores.append(+sc if lbl == "positive"
                          else -sc if lbl == "negative"
                          else 0.0)
        return labels, np.array(scores, dtype=np.float32)

    # ── Découverte du total de rows ───────────────────────────────
    print(f"  Lecture metadata {Path(PATH_CLEAN).name}...")
    _meta       = pd.read_parquet(PATH_CLEAN, columns=["id"])
    total_rows  = len(_meta)
    del _meta; gc.collect()

    n_chunks    = (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
    done_shards = sorted(FB_SHD_DIR.glob("fb_shard_*.parquet"))
    start_chunk = len(done_shards)

    print(f"  Total rows   : {total_rows:,}")
    print(f"  Chunk size   : {CHUNK_SIZE:,}")
    print(f"  Nb chunks    : {n_chunks}")
    if start_chunk > 0:
        print(f"  ♻️  Reprise   : chunk {start_chunk}/{n_chunks} "
              f"({start_chunk*CHUNK_SIZE:,} rows déjà traités)\n")

    t_total      = time.time()
    rows_done    = start_chunk * CHUNK_SIZE
    label_counts = {"positive": 0, "negative": 0, "neutral": 0}

    # ── Boucle principale ─────────────────────────────────────────
    for chunk_idx in range(start_chunk, n_chunks):
        row_start = chunk_idx * CHUNK_SIZE
        row_end   = min(row_start + CHUNK_SIZE, total_rows)
        n_chunk   = row_end - row_start

        t0 = time.time()
        print(f"  ── Chunk {chunk_idx+1}/{n_chunks}  "
              f"[{row_start:,} → {row_end:,}  ({n_chunk:,} rows)] ───")

        # Lecture du chunk
        df_chunk = _read_rows(PATH_CLEAN, row_start, row_end).copy()   # lecture séquentielle, RAM bornée
        gc.collect()

        # Truncation texte → 280 chars (≈ 64 tokens pour FinBERT)
        texts = df_chunk["text"].astype(str).str.slice(0, 280).tolist()

        # ⚡ ACCÉLÉRATION : on ne score que les textes UNIQUES (mêmes textes =
        #    mêmes scores). Les titres de news se répètent beaucoup → gros gain.
        uniq = list(dict.fromkeys(texts))
        if texts:
            print(f"    {len(uniq):,} uniques / {len(texts):,} "
                  f"({100*len(uniq)/len(texts):.0f}%) → FinBERT sur les uniques")

        uniq_out = []
        t_inf    = time.time()
        for i in tqdm(range(0, len(uniq), BATCH),
                      desc=f"  FinBERT chunk {chunk_idx+1}/{n_chunks}",
                      leave=False):
            batch = uniq[i : i + BATCH]
            try:
                out = fb(batch, batch_size=len(batch))
                uniq_out.extend(out)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and is_gpu:
                    BATCH = max(16, BATCH // 2)
                    print(f"\n  ⚠️ OOM → batch réduit à {BATCH}")
                    torch.cuda.empty_cache()
                    try:
                        out = fb(batch, batch_size=BATCH)
                        uniq_out.extend(out)
                    except Exception:
                        uniq_out.extend([{"label": "neutral", "score": 0.0}] * len(batch))
                else:
                    uniq_out.extend([{"label": "neutral", "score": 0.0}] * len(batch))

        inf_elapsed = time.time() - t_inf

        # Remap uniques → toutes les lignes
        _fbmap  = {t: r for t, r in zip(uniq, uniq_out)}
        results = [_fbmap.get(t, {"label": "neutral", "score": 0.0}) for t in texts]
        del _fbmap, uniq, uniq_out

        # Conversion résultats
        labels, scores = results_to_arrays(results)
        df_chunk["finbert_label"] = labels
        df_chunk["finbert_score"] = scores

        # Optimisation types avant sauvegarde
        df_chunk["finbert_label"] = df_chunk["finbert_label"].astype("category")
        df_chunk["finbert_score"] = df_chunk["finbert_score"].astype(np.float32)

        # Statistiques chunk
        for lbl in ["positive", "negative", "neutral"]:
            label_counts[lbl] += (df_chunk["finbert_label"] == lbl).sum()

        rows_done  += n_chunk
        throughput  = n_chunk / max(inf_elapsed, 1)
        remaining   = (total_rows - rows_done) / max(throughput, 1)

        print(f"    pos={label_counts['positive']:,}  "
              f"neg={label_counts['negative']:,}  "
              f"neu={label_counts['neutral']:,}")
        print(f"    ⏱ {inf_elapsed:.1f}s  |  {throughput:,.0f} rows/s  |  "
              f"ETA: {remaining/60:.1f} min  |  cumul: {rows_done:,}/{total_rows:,}")

        # Sauvegarde shard
        shard_path = str(FB_SHD_DIR / f"fb_shard_{chunk_idx:05d}.parquet")
        df_chunk.to_parquet(shard_path, index=False, compression="snappy")
        del df_chunk, texts, results, labels, scores; gc.collect()

        if is_gpu:
            torch.cuda.empty_cache()

    # ── Libération modèle ─────────────────────────────────────────
    del fb; gc.collect()
    if is_gpu: torch.cuda.empty_cache()
    print(f"\n  ✅ Inférence terminée en {(time.time()-t_total)/60:.1f} min")

    # ── Consolidation des shards ──────────────────────────────────
    print(f"\n  Consolidation des shards (streaming, RAM bornée)...")
    all_shards = sorted(FB_SHD_DIR.glob("fb_shard_*.parquet"))
    import pyarrow as _pa, pyarrow.parquet as _pq
    _w, _schema = None, None
    for _sp in all_shards:
        _t = _pq.read_table(str(_sp))
        if _w is None:
            _schema = _t.schema
            _w = _pq.ParquetWriter(PATH_FB_OUT, _schema, compression="snappy")
        else:
            try: _t = _t.cast(_schema)
            except Exception: pass
        _w.write_table(_t)
        del _t; gc.collect()
    if _w is not None: _w.close()
    df_fb = None
    try:
        import shutil as _sh; _sh.rmtree(FB_SHD_DIR, ignore_errors=True) #
        print("  🧹 finbert_shards supprimés (espace libéré)")
    except Exception: pass
    print(f"  ✅ Sauvegardé → {Path(PATH_FB_OUT).name}")

    # Nettoyage shards (décommenter si espace disque limité)
    # for p in all_shards: p.unlink()

# ── Rapport final (lecture légère : sans le texte) ─────────────────
print("\n" + "═"*65)
_rep = pd.read_parquet(PATH_FB_OUT, columns=["ticker", "finbert_label", "finbert_score"])
_N = len(_rep)
print(f"  Output rows      : {_N:,}")
print(f"\n  Distribution labels:")
for lbl, n in _rep["finbert_label"].value_counts().items():
    bar = "█" * int(n / max(_N,1) * 40)
    print(f"    {lbl:<10} {n:>10,}  {n/max(_N,1)*100:5.1f}%  {bar}")
print(f"\n  Score stats:")
print(f"    Mean  : {_rep['finbert_score'].mean():+.4f}")
print(f"    Std   : {_rep['finbert_score'].std():.4f}")
print(f"    Min   : {_rep['finbert_score'].min():+.4f}")
print(f"    Max   : {_rep['finbert_score'].max():+.4f}")
print(f"\n  Par ticker:")
for tk, grp in _rep.groupby("ticker"):
    pos = (grp["finbert_label"] == "positive").sum()
    neg = (grp["finbert_label"] == "negative").sum()
    neu = (grp["finbert_label"] == "neutral").sum()
    print(f"    {tk:<6}  total={len(grp):>8,}  "
          f"pos={pos:>7,}  neg={neg:>7,}  neu={neu:>7,}  "
          f"mean={grp['finbert_score'].mean():+.3f}")
del _rep; gc.collect()
print("═"*65)

  Device  : Tesla T4 (15.6 GB VRAM)
  fp16    : True  |  batch/inférence: 128

  Loading ProsusAI/finbert...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

    pos=334,403  neg=384,574  neu=2,181,023
    ⏱ 104.8s  |  954 rows/s  |  ETA: 3.0 min  |  cumul: 2,900,000/3,069,583
  ── Chunk 30/31  [2,900,000 → 3,000,000  (100,000 rows)] ───
    100,000 uniques / 100,000 (100%) → FinBERT sur les uniques


  FinBERT chunk 30/31:   0%|          | 0/782 [00:00<?, ?it/s]

    pos=341,022  neg=395,270  neu=2,263,708
    ⏱ 100.5s  |  995 rows/s  |  ETA: 1.2 min  |  cumul: 3,000,000/3,069,583
  ── Chunk 31/31  [3,000,000 → 3,069,583  (69,583 rows)] ───
    69,583 uniques / 69,583 (100%) → FinBERT sur les uniques


  FinBERT chunk 31/31:   0%|          | 0/544 [00:00<?, ?it/s]

    pos=350,542  neg=405,752  neu=2,313,289
    ⏱ 75.4s  |  923 rows/s  |  ETA: 0.0 min  |  cumul: 3,069,583/3,069,583

  ✅ Inférence terminée en 49.0 min

  Consolidation des shards (streaming, RAM bornée)...
  🧹 finbert_shards supprimés (espace libéré)
  ✅ Sauvegardé → 04a_finbert.parquet

═════════════════════════════════════════════════════════════════
  Output rows      : 3,069,583

  Distribution labels:
    neutral     2,313,289   75.4%  ██████████████████████████████
    negative      405,752   13.2%  █████
    positive      350,542   11.4%  ████

  Score stats:
    Mean  : -0.0170
    Std   : 0.3954
    Min   : -0.9787
    Max   : +0.9633

  Par ticker:
    AAPL    total= 894,509  pos=113,371  neg=116,846  neu=664,292  mean=-0.008
    AMZN    total= 468,438  pos= 68,918  neg= 56,770  neu=342,750  mean=+0.017
    GOOGL   total= 372,464  pos= 33,786  neg= 40,952  neu=297,726  mean=-0.017
    META    total=  70,118  pos=  4,697  neg= 14,814  neu= 50,607  mean=-0.117
    MSFT    t

## Cell 8 — Sarcasme ML + Emotion + categorie 4 classes

In [19]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — SARCASM ML + EMOTION + 4-CLASS SENTIMENT CATEGORY      ║
# ║                                                                  ║
# ║  New in v2:                                                      ║
# ║   • `sentiment_category` ∈ {positive, negative, neutral,         ║
# ║     sarcastic} — a 4th category that overrides FinBERT when      ║
# ║     sarcasm_final is strong, capturing "ironic positive" /       ║
# ║     "ironic negative" cases that FinBERT alone gets wrong.       ║
# ║   • Composite formula tuned to use the 4-class signal.           ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_SENT, min_rows=10):
    print(f"  ✅ {Path(PATH_SENT).name} exists - skipping")
    df_sent = ckpt.load(PATH_SENT)
else:
    import torch
    from transformers import pipeline as hf_pipeline
    from tqdm.auto import tqdm

    df = ckpt.load(f"{OUTPUT_DIR}/04a_finbert.parquet")
    print(f"  Input rows: {len(df):,}")

    device  = 0 if torch.cuda.is_available() else -1
    is_cpu  = device == -1
    BATCH   = 16 if is_cpu else 32

    # ── Sarcasm ML (only on candidates) ───────────────────────────
    sarc_candidates = df[df["sarcasm_score"] > 0.25].copy()
    print(f"\n  Sarcasm ML candidates (rule-score > 0.25): "
          f"{len(sarc_candidates):,}")

    if len(sarc_candidates) > 0:
        print("  Loading cardiffnlp/twitter-roberta-base-irony...")
        try:
            sarc_model = hf_pipeline(
                "text-classification",
                model="cardiffnlp/twitter-roberta-base-irony",
                device=device, max_length=128, truncation=True,
            )
            if is_cpu and len(sarc_candidates) > 50_000:
                sarc_candidates = sarc_candidates.sample(50_000, random_state=42)
            texts = sarc_candidates["text"].tolist()
            results = []
            for i in tqdm(range(0, len(texts), BATCH), desc="Sarcasm ML"):
                batch = texts[i:i+BATCH]
                try:
                    results.extend(sarc_model(batch, batch_size=len(batch)))
                except Exception:
                    results.extend([{"label": "non_irony", "score": 0.0}] * len(batch))
            sarc_candidates["sarcasm_ml"] = np.array([
                r["score"] if "iron" in r["label"].lower() else 0.0
                for r in results
            ], dtype=np.float32)
            df = df.merge(sarc_candidates[["id", "sarcasm_ml"]], on="id", how="left")
            df["sarcasm_ml"] = df["sarcasm_ml"].fillna(0.0).astype(np.float32)
            del sarc_model; mem.free()
            print(f"  ✅ Sarcasm ML done | mean ML score: "
                  f"{df['sarcasm_ml'].mean():.4f}")
        except Exception as e:
            print(f"  ⚠️  Sarcasm ML failed: {str(e)[:100]} - using rule score only")
            df["sarcasm_ml"] = np.float32(0.0)
    else:
        df["sarcasm_ml"] = np.float32(0.0)

    # Final sarcasm signal: blend of rule and ML, [0, 1].
    df["sarcasm_final"] = (
        df["sarcasm_score"] * 0.3 + df["sarcasm_ml"] * 0.7
    ).clip(0, 1).astype(np.float32)

    # ── Emotion (on a sample to bound runtime) ────────────────────
    df_emo_sample = df
    if is_cpu and len(df) > 80_000:
        df_emo_sample = df.sample(80_000, random_state=42)
    elif not is_cpu and len(df) > 400_000:
        df_emo_sample = df.sample(400_000, random_state=42)

    print(f"\n  Emotion detection on {len(df_emo_sample):,} rows...")
    try:
        emo_model = hf_pipeline(
            "text-classification",
            model="j-hartmann/emotion-english-distilroberta-base",
            device=device, max_length=128, truncation=True,
            top_k=None,
        )
        texts = df_emo_sample["text"].tolist()
        emo_results = []
        for i in tqdm(range(0, len(texts), BATCH), desc="Emotion"):
            batch = texts[i:i+BATCH]
            try:
                emo_results.extend(emo_model(batch, batch_size=len(batch)))
            except Exception:
                emo_results.extend([[{"label": "neutral", "score": 1.0}]] * len(batch))

        def extract_emo(res):
            if not isinstance(res, list): res = [res]
            out = {"emo_fear": 0.0, "emo_anger": 0.0, "emo_joy": 0.0,
                   "emo_sadness": 0.0, "emo_surprise": 0.0, "emo_disgust": 0.0}
            for e in res:
                lab = e["label"].lower()
                sc  = float(e["score"])
                if "fear"      in lab: out["emo_fear"]     = sc
                elif "anger"   in lab: out["emo_anger"]    = sc
                elif "joy"     in lab: out["emo_joy"]      = sc
                elif "sad"     in lab: out["emo_sadness"]  = sc
                elif "surpris" in lab: out["emo_surprise"] = sc
                elif "disgust" in lab: out["emo_disgust"]  = sc
            return out

        emo_df = pd.DataFrame([extract_emo(r) for r in emo_results])
        emo_df["id"] = df_emo_sample["id"].values
        df = df.merge(emo_df, on="id", how="left")
        for c in ["emo_fear","emo_anger","emo_joy","emo_sadness","emo_surprise","emo_disgust"]:
            df[c] = df[c].fillna(0.0).astype(np.float32)
        del emo_model; mem.free()
        print(f"  ✅ Emotion done")
    except Exception as e:
        print(f"  ⚠️  Emotion failed: {str(e)[:100]} - defaulting to zeros")
        for c in ["emo_fear","emo_anger","emo_joy","emo_sadness","emo_surprise","emo_disgust"]:
            df[c] = np.float32(0.0)

    # ── 4-CLASS SENTIMENT CATEGORY ────────────────────────────────
    # Logic:
    #   - sarcasm_final > 0.50  → "sarcastic" (always wins, since the
    #     literal FinBERT label is then unreliable)
    #   - else use FinBERT label (positive / negative / neutral)
    # We also store an integer encoding for ML features.
    SARC_THRESHOLD = 0.50

    def assign_cat(row):
        if row["sarcasm_final"] > SARC_THRESHOLD:
            return "sarcastic"
        lab = str(row["finbert_label"]).lower()
        if lab in ("positive", "negative", "neutral"):
            return lab
        return "neutral"

    # Vectorisé (apply axis=1 serait très lent sur des millions de lignes)
    _lab = df["finbert_label"].astype(str).str.lower()
    _lab = _lab.where(_lab.isin(["positive", "negative", "neutral"]), "neutral")
    _lab = _lab.mask(df["sarcasm_final"] > SARC_THRESHOLD, "sarcastic")
    df["sentiment_category"] = _lab.astype("category")

    CAT_TO_INT = {"negative": 0, "neutral": 1, "positive": 2, "sarcastic": 3}
    df["sentiment_cat_id"] = df["sentiment_category"].map(CAT_TO_INT).astype(np.int8)

    # One-hot indicators (used by daily aggregation in Cell 9 to compute
    # the proportion of each category per ticker × day).
    for cat in ("negative", "neutral", "positive", "sarcastic"):
        df[f"is_{cat}"] = (df["sentiment_category"] == cat).astype(np.int8)

    # ── COMPOSITE SENTIMENT (now sarcasm-flip-aware) ───────────────
    # Sarcastic positive language usually conveys a NEGATIVE underlying view
    # (and vice versa). We dampen + partially invert finbert when sarcasm is
    # strong, then add emotion deltas.
    flip_factor = 1.0 - 1.6 * df["sarcasm_final"]   # ∈ [-0.6, 1.0]
    df["sentiment_composite"] = (
        0.55 * df["finbert_score"] * flip_factor
        + 0.25 * (df["emo_joy"] - df["emo_fear"] - df["emo_anger"])
        - 0.10 * df["sarcasm_final"]                # global pessimism prior
    ).clip(-1.0, 1.0).astype(np.float32)

    # Stats
    print(f"\n  ── 4-class category counts ──")
    print(df["sentiment_category"].value_counts().to_string())
    print(f"\n  Composite mean : {df['sentiment_composite'].mean():+.4f}")
    print(f"  Composite std  : {df['sentiment_composite'].std():.4f}")

    # Le texte ne sert plus en aval (agrégation journalière = numérique) :
    # on le retire → PATH_SENT beaucoup plus léger, RAM des étapes suivantes bornée.
    df = df.drop(columns=[c for c in ("text",) if c in df.columns])
    df = mem.optimize_df(df)
    ckpt.save(df, PATH_SENT)
    df_sent = df

print(f"\n  Final cols: {list(df_sent.columns)}")


  📂 loaded 04a_finbert.parquet (3,069,583 rows)
  Input rows: 3,069,583

  Sarcasm ML candidates (rule-score > 0.25): 12,771
  Loading cardiffnlp/twitter-roberta-base-irony...


config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-irony
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Emotion:   0%|          | 0/12500 [00:00<?, ?it/s]

  ✅ Emotion done

  ── 4-class category counts ──
sentiment_category
neutral      2302119
negative      405046
positive      350038
sarcastic      12380

  Composite mean : -0.0108
  Composite std  : 0.2202
  💾 saved 04_sentiment.parquet (3,069,583 rows | 52.9 MB)

  Final cols: ['ticker', 'date', 'source', 'dtype', 'sarcasm_score', 'emoji_score', 'id', 'finbert_label', 'finbert_score', 'sarcasm_ml', 'sarcasm_final', 'emo_fear', 'emo_anger', 'emo_joy', 'emo_sadness', 'emo_surprise', 'emo_disgust', 'sentiment_category', 'sentiment_cat_id', 'is_negative', 'is_neutral', 'is_positive', 'is_sarcastic', 'sentiment_composite']


## Cell 9 — Agregation journaliere (ticker × jour)

In [20]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — DAILY AGGREGATION                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_DAILY, min_rows=10):
    print(f"  ✅ {Path(PATH_DAILY).name} exists - skipping")
    df_daily = ckpt.load(PATH_DAILY)
else:
    # ── Agrégation CHUNKÉE (RAM bornée) : sommes/sommes de carrés/comptes ──
    #    par (ticker,date) → moyennes & std identiques à pandas (ddof=1).
    import numpy as _np, pyarrow.parquet as _pqd
    if "_read_rows" not in globals():
        def _read_rows(_p, _a, _b):
            import pyarrow.parquet as _pq, pandas as _pd
            _pf=_pq.ParquetFile(_p); _md=_pf.metadata; _o=[]; _pos=0
            for _rg in range(_md.num_row_groups):
                _n=_md.row_group(_rg).num_rows
                if _pos+_n<=_a: _pos+=_n; continue
                if _pos>=_b: break
                _t=_pf.read_row_group(_rg); _lo=max(0,_a-_pos); _hi=min(_n,_b-_pos)
                _o.append(_t.slice(_lo,_hi-_lo).to_pandas()); _pos+=_n
            return _pd.concat(_o, ignore_index=True) if _o else _pd.DataFrame()
    _nrows = _pqd.ParquetFile(PATH_SENT).metadata.num_rows
    print(f"  Input rows : {_nrows:,}  (agrégation par chunks)")
    _CH = 1_000_000
    _parts = []
    _need = ["ticker","date","sentiment_composite","finbert_score","sarcasm_final",
             "emo_fear","emo_anger","emo_joy","emo_sadness",
             "is_positive","is_negative","is_neutral","is_sarcastic","emoji_score"]
    for _s0 in range(0, _nrows, _CH):
        ch = _read_rows(PATH_SENT, _s0, min(_s0 + _CH, _nrows))
        if not len(ch): continue
        if globals().get("NEWS_ONLY", False) and "dtype" in ch.columns:
            ch = ch[ch["dtype"] == "news"]
            if not len(ch): continue
        ch = ch[[c for c in _need if c in ch.columns]].copy()
        if "emoji_score" not in ch.columns:
            ch["emoji_score"] = 0.0   # robustesse : vieux checkpoint sans emoji
        ch["date"] = pd.to_datetime(ch["date"], errors="coerce").dt.normalize()
        ch = ch.dropna(subset=["date"])
        ch["_sc2"] = ch["sentiment_composite"] ** 2
        ch["_fb2"] = ch["finbert_score"] ** 2
        g = ch.groupby(["ticker", "date"])
        _parts.append(pd.DataFrame({
            "sc_sum": g["sentiment_composite"].sum(), "sc_sq": g["_sc2"].sum(),
            "sc_min": g["sentiment_composite"].min(), "sc_max": g["sentiment_composite"].max(),
            "sc_n":   g["sentiment_composite"].count(),
            "fb_sum": g["finbert_score"].sum(), "fb_sq": g["_fb2"].sum(),
            "fb_n":   g["finbert_score"].count(),
            "sarc_sum": g["sarcasm_final"].sum(), "sarc_max": g["sarcasm_final"].max(),
            "sarc_n":   g["sarcasm_final"].count(),
            "emoj_sum": g["emoji_score"].sum(), "emoj_n": g["emoji_score"].count(),
            "fear_sum": g["emo_fear"].sum(),   "fear_n": g["emo_fear"].count(),
            "anger_sum": g["emo_anger"].sum(), "anger_n": g["emo_anger"].count(),
            "joy_sum": g["emo_joy"].sum(),     "joy_n": g["emo_joy"].count(),
            "sad_sum": g["emo_sadness"].sum(), "sad_n": g["emo_sadness"].count(),
            "pos_sum": g["is_positive"].sum(), "pos_n": g["is_positive"].count(),
            "neg_sum": g["is_negative"].sum(), "neg_n": g["is_negative"].count(),
            "neu_sum": g["is_neutral"].sum(),  "neu_n": g["is_neutral"].count(),
            "sar_sum": g["is_sarcastic"].sum(),"sar_n": g["is_sarcastic"].count(),
        }).reset_index())
        del ch, g; gc.collect()

    allp = pd.concat(_parts, ignore_index=True); del _parts; gc.collect()
    _agg = {k: ("min" if k.endswith("_min") else "max" if k.endswith("_max") else "sum")
            for k in allp.columns if k not in ("ticker", "date")}
    G = allp.groupby(["ticker", "date"], as_index=False).agg(_agg)

    def _std(s_sum, s_sq, n):
        n = n.astype("float64")
        var = (s_sq - (s_sum ** 2) / n) / (n - 1.0)
        var = var.where(n > 1, 0.0)
        return _np.sqrt(var.clip(lower=0))

    df_daily = pd.DataFrame({
        "ticker": G["ticker"], "date": G["date"],
        "sent_mean": G["sc_sum"] / G["sc_n"],
        "sent_std":  _std(G["sc_sum"], G["sc_sq"], G["sc_n"]),
        "sent_min":  G["sc_min"], "sent_max": G["sc_max"],
        "mentions":  G["sc_n"].astype("int64"),
        "finbert_mean": G["fb_sum"] / G["fb_n"],
        "finbert_std":  _std(G["fb_sum"], G["fb_sq"], G["fb_n"]),
        "sarcasm_mean": G["sarc_sum"] / G["sarc_n"],
        "sarcasm_max":  G["sarc_max"],
        "emoji_mean": G["emoj_sum"] / G["emoj_n"],
        "fear_mean": G["fear_sum"] / G["fear_n"],
        "anger_mean": G["anger_sum"] / G["anger_n"],
        "joy_mean": G["joy_sum"] / G["joy_n"],
        "sadness_mean": G["sad_sum"] / G["sad_n"],
        "share_positive": G["pos_sum"] / G["pos_n"],
        "share_negative": G["neg_sum"] / G["neg_n"],
        "share_neutral":  G["neu_sum"] / G["neu_n"],
        "share_sarcastic": G["sar_sum"] / G["sar_n"],
    })
    del allp, G; gc.collect()

    for c in ["sent_std", "finbert_std"]:
        df_daily[c] = df_daily[c].fillna(0)

    df_daily = df_daily.sort_values(["ticker", "date"]).reset_index(drop=True)
    df_daily = mem.optimize_df(df_daily)
    ckpt.save(df_daily, PATH_DAILY)

print(f"\n  Daily rows  : {len(df_daily):,}")
print(f"  Date range  : {df_daily['date'].min().date()} -> {df_daily['date'].max().date()}")
print(f"  Tickers     : {sorted(df_daily['ticker'].unique())}")
print(f"\n  Mentions per ticker:")
for tk, n in df_daily.groupby('ticker')['mentions'].sum().sort_values(ascending=False).items():
    print(f"    {tk:<6} {int(n):>10,}")
print(f"\n  Mean sentiment per ticker:")
for tk, s in df_daily.groupby('ticker')['sent_mean'].mean().sort_values(ascending=False).items():
    print(f"    {tk:<6} {s:+.4f}")


  Input rows : 3,069,583  (agrégation par chunks)
  💾 saved 05_daily.parquet (23,258 rows | 1.5 MB)

  Daily rows  : 23,258
  Date range  : 2015-01-01 -> 2025-12-31
  Tickers     : ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']

  Mentions per ticker:
    TSLA    1,007,111
    AAPL      894,509
    AMZN      468,438
    GOOGL     372,464
    MSFT      228,719
    META       70,118
    NVDA       28,224

  Mean sentiment per ticker:
    NVDA   +0.0386
    MSFT   +0.0157
    AMZN   +0.0058
    AAPL   +0.0051
    GOOGL  -0.0067
    TSLA   -0.0131
    META   -0.0638


## Cell 10 — Indicateurs techniques

In [21]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — TECHNICAL INDICATORS                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_TECH, min_rows=10):
    print(f"  ✅ {Path(PATH_TECH).name} exists - skipping")
    df_tech = ckpt.load(PATH_TECH)
else:
    df = ckpt.load(PATH_PRICES)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    print(f"  Input rows: {len(df):,}")

    def add_tech(g):
        g = g.sort_values("date").copy()
        c = g["close"]
        # Returns
        for n in (1, 3, 5, 10, 20):
            g[f"ret_{n}d"] = c.pct_change(n)
        # Moving averages
        for w in (5, 10, 20, 50):
            g[f"sma_{w}"] = c.rolling(w, min_periods=1).mean()
            g[f"ema_{w}"] = c.ewm(span=w, adjust=False).mean()
        g["price_vs_sma20"] = (c / g["sma_20"] - 1)
        g["price_vs_sma50"] = (c / g["sma_50"] - 1)
        # Volatility
        g["vol_5d"]  = g["ret_1d"].rolling(5,  min_periods=1).std()
        g["vol_20d"] = g["ret_1d"].rolling(20, min_periods=1).std()
        # RSI (14)
        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(14, min_periods=1).mean()
        loss  = (-delta.clip(upper=0)).rolling(14, min_periods=1).mean()
        rs    = gain / loss.replace(0, np.nan)
        g["rsi_14"] = (100 - 100 / (1 + rs)).fillna(50)
        # MACD
        ema12 = c.ewm(span=12, adjust=False).mean()
        ema26 = c.ewm(span=26, adjust=False).mean()
        g["macd"]        = ema12 - ema26
        g["macd_signal"] = g["macd"].ewm(span=9, adjust=False).mean()
        g["macd_hist"]   = g["macd"] - g["macd_signal"]
        # Bollinger
        sma20 = c.rolling(20, min_periods=1).mean()
        std20 = c.rolling(20, min_periods=1).std()
        g["bb_upper"] = sma20 + 2 * std20
        g["bb_lower"] = sma20 - 2 * std20
        g["bb_width"] = ((g["bb_upper"] - g["bb_lower"]) / sma20).fillna(0)
        g["bb_pos"]   = ((c - g["bb_lower"]) /
                          (g["bb_upper"] - g["bb_lower"] + 1e-9)).clip(0, 1)
        # ATR (14)
        hl  = g["high"] - g["low"]
        hc  = (g["high"] - c.shift()).abs()
        lc  = (g["low"]  - c.shift()).abs()
        tr  = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        g["atr_14"] = tr.rolling(14, min_periods=1).mean()
        # Volume
        g["vol_ma20"]    = g["volume"].rolling(20, min_periods=1).mean()
        g["vol_ratio"]   = (g["volume"] / g["vol_ma20"].replace(0, np.nan)).fillna(1)
        # High-Low range
        g["hl_range"]    = ((g["high"] - g["low"]) / c).fillna(0)
        return g

    print("  Computing indicators per ticker...")
    parts = []
    for tk, g in df.groupby("ticker"):
        parts.append(add_tech(g))
    df_tech = pd.concat(parts, ignore_index=True)
    df_tech = df_tech.sort_values(["ticker", "date"]).reset_index(drop=True)

    # Fill remaining NaNs (early rows where rolling windows are not full)
    num_cols = df_tech.select_dtypes(include=[np.number]).columns
    df_tech[num_cols] = df_tech.groupby("ticker", observed=True)[num_cols].transform(
        lambda x: x.ffill()
    )
    df_tech[num_cols] = df_tech[num_cols].fillna(0)

    df_tech = mem.optimize_df(df_tech)
    ckpt.save(df_tech, PATH_TECH)

print(f"\n  Tech rows : {len(df_tech):,}")
print(f"  Columns   : {len(df_tech.columns)}")
indicator_cols = [c for c in df_tech.columns if c not in ('ticker','date','open','high','low','close','volume','return_1d')]
print(f"  Indicators added: {indicator_cols[:15]} ...")


  📂 loaded 01_prices.parquet (19,355 rows)
  Input rows: 19,355
  Computing indicators per ticker...
  💾 saved 06_prices_tech.parquet (19,355 rows | 3.9 MB)

  Tech rows : 19,355
  Columns   : 37
  Indicators added: ['ret_1d', 'ret_3d', 'ret_5d', 'ret_10d', 'ret_20d', 'sma_5', 'ema_5', 'sma_10', 'ema_10', 'sma_20', 'ema_20', 'sma_50', 'ema_50', 'price_vs_sma20', 'price_vs_sma50'] ...


# ═══════════ ORCHESTRATION MULTI-CONFIGS (auto) ═══════════
4 runs enchaînés : **2023 H=1 → 2023 H=20 → 2022 H=10 → 2023 H=10** (le dernier laisse l'état final = run principal du papier). Chaque run purge les checkpoints dépendants du split/horizon, ré-exécute Cells 11→15, puis consigne ses métriques dans `RESULTS.txt`.


## ▶ RUN test 2023 — horizon 1 j


In [22]:
# ═══ BASCULE CONFIG — test 2023, horizon 1 j ═══
import pandas as pd, glob, os
TRAIN_END = pd.Timestamp("2021-12-31")
VAL_END   = pd.Timestamp("2022-12-31")
TEST_END  = pd.Timestamp("2023-12-31")
HORIZON   = 1
try:
    CONFIG["horizon"] = HORIZON
except Exception:
    pass
KEEP = ("_2022", "_2023", "_h1.", "_h10.", "_h20.")
for pat in ["07_", "08_", "09_train", "09_val", "09_test", "10_predictions", "11_backtest"]:
    for f in glob.glob(f"{OUTPUT_DIR}/{pat}*.parquet"):
        b = os.path.basename(f)
        if any(s in b for s in KEEP):
            continue
        os.remove(f); print("rm", b)
for f in glob.glob(f"{MODELS_DIR}/*"):
    os.remove(f)
print(f"✅ CONFIG ACTIVE → train≤{TRAIN_END.date()}  val≤{VAL_END.date()}  test≤{TEST_END.date()}  H={HORIZON}j")


✅ CONFIG ACTIVE → train≤2021-12-31  val≤2022-12-31  test≤2023-12-31  H=1j


## Cell 11 — Merge sentiment ⨉ prix + targets

In [23]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — MERGE SENTIMENT + PRICES + TARGETS                    ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_MERGED, min_rows=10):
    print(f"  ✅ {Path(PATH_MERGED).name} exists - skipping")
    df_merged = ckpt.load(PATH_MERGED)
else:
    df_sent  = ckpt.load(PATH_DAILY)
    df_price = ckpt.load(PATH_TECH)

    df_sent["date"]  = normalize_date_col(df_sent["date"])
    df_price["date"] = normalize_date_col(df_price["date"])
    df_sent["ticker"]  = df_sent["ticker"].astype(str)
    df_price["ticker"] = df_price["ticker"].astype(str)

    print(f"  Sentiment rows : {len(df_sent):,}")
    print(f"  Price    rows  : {len(df_price):,}")

    # Left join: keep every trading day, fill sentiment with neutral if missing
    df = df_price.merge(df_sent, on=["ticker", "date"], how="left")

    # Sentiment columns to fill
    sent_cols = ["sent_mean","sent_std","sent_min","sent_max","mentions",
                 "finbert_mean","finbert_std","sarcasm_mean","sarcasm_max",
                 "fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic",
                 "emoji_mean"]
    for c in sent_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    print(f"  Merged rows    : {len(df):,}")
    days_with_sent = (df["mentions"] > 0).sum()
    print(f"  Trading days WITH sentiment: {days_with_sent:,} "
          f"({100*days_with_sent/len(df):.1f}%)")

    # ── TARGETS ────────────────────────────────────────────────────
    # Forward returns at multiple horizons (per ticker, no cross-leakage)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    for h in CONFIG["horizons"]:
        df[f"close_t+{h}"]     = df.groupby("ticker", observed=True)["close"].shift(-h)
        df[f"return_t+{h}"]    = (df[f"close_t+{h}"] - df["close"]) / df["close"]
        df[f"direction_t+{h}"] = (df[f"return_t+{h}"] > 0).astype(np.int8)
    # Primary target
    _H = globals().get("HORIZON", 1)
    if f"return_t+{_H}" not in df.columns:
        print(f"  ⚠️ horizon {_H} absent de CONFIG[horizons] → repli sur 1")
        _H = 1
    _TARGET = globals().get("TARGET", "direction")
    if _TARGET == "volatility":
        # Rendement journalier (depuis close, robuste au nom de colonne)
        df["_r1"] = df.groupby("ticker", observed=True)["close"].pct_change()
        # Volatilité réalisée FORWARD sur t+1..t+H (= la cible, légitimement future)
        _fwd_sq = None
        for _k in range(1, _H + 1):
            _rk = df.groupby("ticker", observed=True)["_r1"].shift(-_k)
            _fwd_sq = _rk.pow(2) if _fwd_sq is None else _fwd_sq + _rk.pow(2)
        df["_fwd_vol"] = (_fwd_sq / _H) ** 0.5
        # Seuil = médiane GLISSANTE PASSÉE de la vol, fenêtre courte (~1 trimestre)
        # → cible "vol élevée RELATIVE au régime récent" : classes ~50/50 à toute
        # période (évite l effet régime calme/turbulent), et aucune info du futur.
        _VOL_WIN = 63
        _past = df.groupby("ticker", observed=True)["_r1"].transform(
            lambda x: x.rolling(_H, min_periods=max(3, _H // 2)).std())
        df["_thr"] = _past.groupby(df["ticker"], observed=True).transform(
            lambda x: x.rolling(_VOL_WIN, min_periods=20).median())
        df["target_return"]    = df["_fwd_vol"].where(df["_thr"].notna())
        df["target_direction"] = (df["_fwd_vol"] > df["_thr"]).astype(np.int8)
        df = df.drop(columns=["_r1", "_fwd_vol", "_thr"])
        print(f"  🎯 Cible = VOLATILITÉ à +{_H}j (1 = vol future > médiane glissante passée)")
    else:
        df["target_return"]    = df[f"return_t+{_H}"]
        df["target_direction"] = df[f"direction_t+{_H}"]
        print(f"  🎯 Cible = direction à +{_H} jour(s)")
    # Drop rows without target_return (last few rows per ticker)
    df = df[df["target_return"].notna()].copy()

    df = mem.optimize_df(df)
    ckpt.save(df, PATH_MERGED)
    df_merged = df

print(f"\n  Final rows : {len(df_merged):,}")
print(f"  Columns    : {len(df_merged.columns)}")
print(f"\n  Direction balance (t+1):")
print(f"    Up   : {(df_merged['target_direction']==1).sum():,}  "
      f"({100*(df_merged['target_direction']==1).mean():.1f}%)")
print(f"    Down : {(df_merged['target_direction']==0).sum():,}  "
      f"({100*(df_merged['target_direction']==0).mean():.1f}%)")


  📂 loaded 05_daily.parquet (23,258 rows)
  📂 loaded 06_prices_tech.parquet (19,355 rows)
  Sentiment rows : 23,258
  Price    rows  : 19,355
  Merged rows    : 19,355
  Trading days WITH sentiment: 16,506 (85.3%)
  🎯 Cible = direction à +1 jour(s)
  💾 saved 07_merged.parquet (19,348 rows | 6.4 MB)

  Final rows : 19,348
  Columns    : 75

  Direction balance (t+1):
    Up   : 10,299  (53.2%)
    Down : 9,049  (46.8%)


## Cell 12 — Feature engineering + split temporel

In [24]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — FEATURE ENGINEERING + TEMPORAL SPLIT                  ║
# ╚══════════════════════════════════════════════════════════════════╝
if all(ckpt.exists_and_valid(p) for p in (PATH_FEATURES, PATH_TRAIN, PATH_VAL, PATH_TEST)):
    print(f"  ✅ Feature splits exist - skipping")
    df_feat = ckpt.load(PATH_FEATURES)
else:
    df = ckpt.load(PATH_MERGED)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    print(f"  Input rows : {len(df):,}")

    SENT_COLS = ["sent_mean","sent_std","mentions","finbert_mean","emoji_mean",
                 "sarcasm_mean","fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic"]

    print("\n  Lagging sentiment features (anti-leakage)...")
    # Shift each sentiment column by 1 day per ticker -> "yesterday's sentiment"
    for c in SENT_COLS:
        if c not in df.columns:
            continue   # robustesse : colonne absente (vieux checkpoint)
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # Rolling sentiment features (shift + rolling BOTH inside groupby to avoid
    # cross-ticker leakage at boundaries — transform() preserves original index).
    print("  Building sentiment rolling features...")
    for w in (3, 7, 14, 30):
        df[f"sent_ma_{w}d"] = (
            df.groupby("ticker", observed=True)["sent_mean"]
              .transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean())
        ).fillna(0)

    # Sentiment momentum (lagged)
    df["sent_momentum"] = (df["sent_ma_3d"] - df["sent_ma_14d"]).fillna(0)

    # Sarcasm-adjusted sentiment (lagged)
    df["sent_adj_lag1"] = (
        df["sent_mean_lag1"] * (1 - 0.5 * df["sarcasm_mean_lag1"])
    ).fillna(0).astype(np.float32)

    # Calendar features
    df["dow"]        = df["date"].dt.dayofweek.astype(np.int8)
    df["month"]      = df["date"].dt.month.astype(np.int8)
    df["quarter"]    = df["date"].dt.quarter.astype(np.int8)
    df["is_monday"]  = (df["dow"] == 0).astype(np.int8)
    df["is_friday"]  = (df["dow"] == 4).astype(np.int8)

    # Ticker as integer
    stock_map = {s: i for i, s in enumerate(sorted(STOCKS))}
    df["stock_id"] = df["ticker"].map(stock_map).astype(np.int8)

    # Fill NaN from lags (first row per ticker)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # Drop rows where target is missing
    df = df.dropna(subset=["target_return", "target_direction"]).copy()

    # ── MARKET-WIDE SENTIMENT FEATURES (cross-ticker) ─────────────
    # On any given day, what was the AVERAGE sentiment across ALL tickers?
    # This captures macro mood that no single ticker can show. We then
    # compute each ticker's DEVIATION from the market mood = their unique
    # signal.
    print("\n  Building market-wide cross-ticker features...")
    market_daily = (df.groupby("date", as_index=False)
                      .agg(market_sent_mean = ("sent_mean",        "mean"),
                           market_share_neg = ("share_negative",   "mean"),
                           market_share_pos = ("share_positive",   "mean"),
                           market_share_sarc= ("share_sarcastic",  "mean"),
                           market_mentions  = ("mentions",         "sum")))
    df = df.merge(market_daily, on="date", how="left")

    # Per-ticker deviation from market mean (negative → ticker more
    # pessimistic than the market, positive → more optimistic)
    df["sent_vs_market"] = (df["sent_mean"] - df["market_sent_mean"]).astype(np.float32)

    # Lag market features by 1 day (anti-leakage like the rest)
    for c in ("market_sent_mean", "market_share_neg", "market_share_pos",
              "market_share_sarc", "sent_vs_market"):
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # ── FEATURE LIST ──────────────────────────────────────────────
    FEATURE_COLS = [
        # Technical
        "ret_1d","ret_3d","ret_5d","ret_10d","ret_20d",
        "vol_5d","vol_20d", "rsi_14",
        "macd","macd_signal","macd_hist",
        "bb_width","bb_pos","atr_14",
        "price_vs_sma20","price_vs_sma50",
        "vol_ratio","hl_range",
        # Sentiment (all lagged or rolling - no same-day)
        "sent_mean_lag1","sent_std_lag1","mentions_lag1",
        "finbert_mean_lag1","sarcasm_mean_lag1",
        "fear_mean_lag1","anger_mean_lag1",
        "joy_mean_lag1","sadness_mean_lag1",
        # 4-class category proportions (lagged) — captures whether
        # the previous day's chatter was bullish, bearish, neutral or ironic
        "share_positive_lag1","share_negative_lag1",
        "share_neutral_lag1","share_sarcastic_lag1",
    "emoji_mean_lag1",
        # Market-wide cross-ticker features (NEW)
        "market_sent_mean_lag1","market_share_neg_lag1",
        "market_share_pos_lag1","market_share_sarc_lag1",
        "sent_vs_market_lag1",
        # Rolling sentiment trends
        "sent_ma_3d","sent_ma_7d","sent_ma_14d","sent_ma_30d",
        "sent_momentum","sent_adj_lag1",
        # Calendar
        "dow","month","quarter","is_monday","is_friday",
        # Identifier
        "stock_id",
    ]
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    print(f"\n  Features  : {len(FEATURE_COLS)}")

    # Save list
    with open(f"{MODELS_DIR}/feature_names.json", "w") as f:
        json.dump({"features": FEATURE_COLS}, f, indent=2)

    # ── TEMPORAL SPLIT ────────────────────────────────────────────
    train = df[df["date"] <= TRAIN_END].copy()
    val   = df[(df["date"] > TRAIN_END) & (df["date"] <= VAL_END)].copy()
    test  = df[(df["date"] > VAL_END) & (df["date"] <= TEST_END)].copy()

    print(f"\n  Train : {len(train):>7,} rows   "
          f"({train['date'].min().date()} -> {train['date'].max().date()})")
    print(f"  Val   : {len(val):>7,} rows   "
          f"({val['date'].min().date()} -> {val['date'].max().date()})")
    print(f"  Test  : {len(test):>7,} rows   "
          f"({test['date'].min().date()} -> {test['date'].max().date()})")

    # Save everything
    df = mem.optimize_df(df)
    ckpt.save(df,    PATH_FEATURES)
    ckpt.save(train, PATH_TRAIN)
    ckpt.save(val,   PATH_VAL)
    ckpt.save(test,  PATH_TEST)

    # ── Liberation disque : les parquets ligne-par-ligne (raw/clean/finbert/sent)
    #    ne sont plus lus en aval (tout passe par les agregats journaliers). ──
    if globals().get("FREE_ROWLEVEL_AFTER_FEATURES", True):
        import os as _os
        # Sauvegarde d un petit résumé "volume par source" pour la Fig 9
        # (sinon perdu quand on supprime les parquets ligne-par-ligne).
        try:
            _srcp = globals().get("PATH_SENT", "") or PATH_CLEAN
            if _srcp and _os.path.exists(_srcp):
                _sc = pd.read_parquet(_srcp, columns=["source", "dtype"])
                _scg = _sc.groupby(["source", "dtype"]).size().reset_index(name="n")
                _scg.to_parquet(f"{OUTPUT_DIR}/source_counts.parquet", index=False)
                del _sc, _scg
        except Exception as _e:
            print(f"  (résumé sources non sauvé: {str(_e)[:40]})")
        for _p in (PATH_RAW, PATH_CLEAN, PATH_FB_OUT, globals().get("PATH_SENT", "")):
            try:
                if _p and _os.path.exists(_p):
                    _sz = _os.path.getsize(_p) / 1e9
                    _os.remove(_p)
                    print(f"  🧹 {Path(_p).name} supprimé (~{_sz:.2f} GB libérés)")
            except Exception: pass

    df_feat = df

print(f"\n  Features file : {Path(PATH_FEATURES).name} ({len(df_feat):,} rows)")


# ════════════════════════════════════════════════════════════════════
#  AUDIT ANTI-FUITE (leakage) — pour la section méthodo du papier
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 64)
print("  AUDIT ANTI-FUITE (look-ahead leakage)")
print("═" * 64)
import json as _json
_audit_ok = True
try:
    with open(f"{MODELS_DIR}/feature_names.json") as _f:
        _FC = _json.load(_f)["features"]
except Exception:
    _FC = FEATURE_COLS

# (a) Aucune colonne dérivée du futur/cible ne doit être une feature
_forbidden = ("target_", "return_t+", "close_t+", "direction_t+")
_leaky = [c for c in _FC if any(k in c for k in _forbidden)]
if _leaky:
    _audit_ok = False
    print(f"  ❌ Features suspectes (dérivées de la cible) : {_leaky}")
else:
    print(f"  ✅ Aucune feature dérivée de la cible ({len(_FC)} features)")

# (b) Intégrité temporelle du split (train < val < test)
try:
    _tr = ckpt.load(PATH_TRAIN); _va = ckpt.load(PATH_VAL); _te = ckpt.load(PATH_TEST)
    _trmax, _vamin = _tr["date"].max(), _va["date"].min()
    _vamax, _temin = _va["date"].max(), _te["date"].min()
    if _trmax < _vamin and _vamax < _temin:
        print(f"  ✅ Split chronologique strict : "
              f"train≤{_trmax.date()} < val≤{_vamax.date()} < test≤{_te['date'].max().date()}")
    else:
        _audit_ok = False
        print(f"  ❌ Chevauchement temporel : trmax={_trmax}, vamin={_vamin}, "
              f"vamax={_vamax}, temin={_temin}")

    # (c) Corrélation feature↔cible sur le TRAIN : flag |r|>0.30 (signe de fuite)
    import numpy as _np
    _y = _tr["target_direction"].astype(float).values
    _susp = []
    for _c in _FC:
        try:
            _x = _tr[_c].astype(float).values
            if _np.nanstd(_x) == 0: continue
            _r = _np.corrcoef(_np.nan_to_num(_x), _y)[0, 1]
            _voltgt = globals().get("TARGET", "direction") == "volatility"
            _volfam = ("vol_", "atr", "bb_", "rsi", "macd", "ret_", "return_", "hl_range")
            # En cible "volatility", les features de vol/momentum SONT légitimement
            # prédictives (la vol se regroupe) → ce n est pas une fuite.
            if abs(_r) > 0.30 and not (_voltgt and any(_w in _c for _w in _volfam)):
                _susp.append((_c, round(float(_r), 3)))
        except Exception:
            pass
    if _susp:
        _audit_ok = False
        print(f"  ⚠️ Features très corrélées à la cible (|r|>0.30) — à vérifier :")
        for _c, _r in sorted(_susp, key=lambda t: -abs(t[1]))[:10]:
            print(f"       {_c:<28} r={_r:+.3f}")
    else:
        print(f"  ✅ Aucune feature anormalement corrélée à la cible (|r|≤0.30)")
    del _tr, _va, _te
except Exception as _e:
    print(f"  (audit partiel : {str(_e)[:60]})")

print("─" * 64)
print(f"  VERDICT : {'✅ PAS DE FUITE DÉTECTÉE' if _audit_ok else '⚠️ POINTS À VÉRIFIER (voir ci-dessus)'}")
print("═" * 64)


  📂 loaded 07_merged.parquet (19,348 rows)
  Input rows : 19,348

  Lagging sentiment features (anti-leakage)...
  Building sentiment rolling features...

  Building market-wide cross-ticker features...

  Features  : 49

  Train :  12,341 rows   (2015-01-02 -> 2021-12-31)
  Val   :   1,757 rows   (2022-01-03 -> 2022-12-30)
  Test  :   1,750 rows   (2023-01-03 -> 2023-12-29)
  💾 saved 08_features.parquet (19,348 rows | 8.2 MB)
  💾 saved 09_train.parquet (12,341 rows | 5.7 MB)
  💾 saved 09_val.parquet (1,757 rows | 0.8 MB)
  💾 saved 09_test.parquet (1,750 rows | 0.8 MB)
  🧹 02_text_raw.parquet supprimé (~0.34 GB libérés)
  🧹 03_text_clean.parquet supprimé (~0.26 GB libérés)
  🧹 04a_finbert.parquet supprimé (~0.27 GB libérés)
  🧹 04_sentiment.parquet supprimé (~0.06 GB libérés)

  Features file : 08_features.parquet (19,348 rows)

════════════════════════════════════════════════════════════════
  AUDIT ANTI-FUITE (look-ahead leakage)
══════════════════════════════════════════════════════

## Cell 13 — Arbres : LightGBM + XGBoost + CatBoost

In [25]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — TRAIN MODELS                                          ║
# ╚══════════════════════════════════════════════════════════════════╝
import time

# Lazy installs
for pkg in ("lightgbm", "xgboost", "catboost"):
    try:
        __import__(pkg)
    except ImportError:
        pip_install(pkg)

import lightgbm as lgb
import xgboost  as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import joblib

# Load
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test  = ckpt.load(PATH_TEST)

# CORRECTION ANTI-FUITE : tri chronologique GLOBAL avant TimeSeriesSplit.
# (les fichiers sont tries [ticker, date] → la CV temporelle melangerait
#  les tickers et casserait l'ordre du temps.)
train = train.sort_values("date").reset_index(drop=True)
val   = val.sort_values("date").reset_index(drop=True)
test  = test.sort_values("date").reset_index(drop=True)

X_tr, y_tr = train[FEATURE_COLS].values, train["target_direction"].values.astype(int)
X_va, y_va = val[FEATURE_COLS].values,   val["target_direction"].values.astype(int)
X_te, y_te = test[FEATURE_COLS].values,  test["target_direction"].values.astype(int)

print(f"  Train : {X_tr.shape}  |  pos rate = {y_tr.mean():.3f}")
print(f"  Val   : {X_va.shape}  |  pos rate = {y_va.mean():.3f}")
print(f"  Test  : {X_te.shape}  |  pos rate = {y_te.mean():.3f}")

sw_tr = compute_sample_weight("balanced", y_tr)

# ── 1. LIGHTGBM ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  LightGBM")
print("─" * 60)
lgb_params = dict(
    objective="binary", metric="auc",
    n_estimators=800, learning_rate=0.03,
    num_leaves=63, max_depth=8,
    min_child_samples=40,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1,
)
tscv = TimeSeriesSplit(n_splits=5)
cv_lgb = []
clf_lgb = lgb.LGBMClassifier(**lgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_lgb.fit(
        X_tr[tr_i], y_tr[tr_i],
        sample_weight=sw_tr[tr_i],
        eval_set=[(X_tr[val_i], y_tr[val_i])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    auc = roc_auc_score(y_tr[val_i], clf_lgb.predict_proba(X_tr[val_i])[:, 1])
    cv_lgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_lgb):.4f} ± {np.std(cv_lgb):.4f}")
# Final fit
clf_lgb.fit(X_tr, y_tr, sample_weight=sw_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)])
joblib.dump(clf_lgb, f"{MODELS_DIR}/model_lgb.joblib")
auc_lgb_val = roc_auc_score(y_va, clf_lgb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_lgb_val:.4f}")

# ── 2. XGBOOST ─────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  XGBoost")
print("─" * 60)
neg, pos = int((y_tr == 0).sum()), int((y_tr == 1).sum())
spw = neg / max(pos, 1)
xgb_params = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=600, learning_rate=0.05,
    max_depth=7, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw,
    random_state=42, n_jobs=-1, verbosity=0,
    early_stopping_rounds=50,
)
cv_xgb = []
clf_xgb = xgb.XGBClassifier(**xgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_xgb.fit(X_tr[tr_i], y_tr[tr_i],
                eval_set=[(X_tr[val_i], y_tr[val_i])],
                verbose=False)
    auc = roc_auc_score(y_tr[val_i], clf_xgb.predict_proba(X_tr[val_i])[:, 1])
    cv_xgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_xgb):.4f} ± {np.std(cv_xgb):.4f}")
# Final fit (without early-stopping callback now)
xgb_params_final = {k: v for k, v in xgb_params.items() if k != "early_stopping_rounds"}
clf_xgb_final = xgb.XGBClassifier(**xgb_params_final)
clf_xgb_final.fit(X_tr, y_tr)
joblib.dump(clf_xgb_final, f"{MODELS_DIR}/model_xgb.joblib")
auc_xgb_val = roc_auc_score(y_va, clf_xgb_final.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_xgb_val:.4f}")

# ── 3. CATBOOST ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  CatBoost")
print("─" * 60)
clf_cb = CatBoostClassifier(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=42, verbose=0,
    early_stopping_rounds=50,
)
clf_cb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=(X_va, y_va), use_best_model=True)
clf_cb.save_model(f"{MODELS_DIR}/model_cb.cbm")
auc_cb_val = roc_auc_score(y_va, clf_cb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_cb_val:.4f}")
print(f"  Best iter     = {clf_cb.best_iteration_}")

# Save model summary
model_summary = {
    "lgb": {"cv_auc_mean": float(np.mean(cv_lgb)),
            "cv_auc_std":  float(np.std(cv_lgb)),
            "val_auc":     float(auc_lgb_val)},
    "xgb": {"cv_auc_mean": float(np.mean(cv_xgb)),
            "cv_auc_std":  float(np.std(cv_xgb)),
            "val_auc":     float(auc_xgb_val)},
    "cb":  {"val_auc":     float(auc_cb_val),
            "best_iter":   int(clf_cb.best_iteration_)},
}
with open(f"{MODELS_DIR}/training_summary.json", "w") as f:
    json.dump(model_summary, f, indent=2)

print("\n" + "=" * 60)
print(f"  ✅ 3 models trained and saved to {MODELS_DIR}")
print("=" * 60)
mem.free()


  📂 loaded 09_train.parquet (12,341 rows)
  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  Train : (12341, 49)  |  pos rate = 0.537
  Val   : (1757, 49)  |  pos rate = 0.469
  Test  : (1750, 49)  |  pos rate = 0.549

────────────────────────────────────────────────────────────
  LightGBM
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.5051
  fold 2/5  AUC=0.5321
  fold 3/5  AUC=0.5206
  fold 4/5  AUC=0.5202
  fold 5/5  AUC=0.5322
  CV AUC mean   = 0.5220 ± 0.0100
  Val AUC       = 0.5253

────────────────────────────────────────────────────────────
  XGBoost
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.5072
  fold 2/5  AUC=0.5470
  fold 3/5  AUC=0.5181
  fold 4/5  AUC=0.5427
  fold 5/5  AUC=0.5327
  CV AUC mean   = 0.5295 ± 0.0150
  Val AUC       = 0.5309

────────────────────────────────────────────────────────────
  CatBoost
────────────────────────────────────────────────────────────
  V

## Cell 13b — Modele sequentiel hybride (BiGRU + attention)  ⭐ NOUVEAU

Apporte la composante « deep » de l'ensemble hybride. Pour chaque (ticker, jour) on construit une
**fenetre glissante de `SEQ_LEN` jours** du vecteur de features (sentiment lagge + techniques), normalisee
sur le **train uniquement** (anti-fuite). Un **BiGRU bidirectionnel + attention** apprend les dynamiques
temporelles que les arbres ne capturent pas. Les probabilites val/test sont sauvegardees pour le stacking.
GPU recommande ; degrade proprement en CPU (ou se desactive si torch absent).

In [26]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13b — BiGRU + ATTENTION (modele sequentiel)                ║
# ║  Sortie : {OUTPUT_DIR}/seq_proba.parquet  [ticker,date,p_seq,split]║
# ╚══════════════════════════════════════════════════════════════════╝
import json, time
import numpy as np
import pandas as pd
from pathlib import Path

SEQ_OUT = f"{OUTPUT_DIR}/seq_proba.parquet"

if not USE_SEQ_MODEL:
    print("  USE_SEQ_MODEL=False → modele sequentiel desactive.")
elif Path(SEQ_OUT).exists():
    print(f"  ✅ {Path(SEQ_OUT).name} existe — skip BiGRU")
else:
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        pip_install("torch"); import torch; import torch.nn as nn

    from sklearn.metrics import roc_auc_score

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device : {device}")

    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    df = ckpt.load(PATH_FEATURES)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # ── Standardisation (stats du TRAIN uniquement) ──────────────────
    train_mask = df["date"] <= TRAIN_END
    mu = df.loc[train_mask, FEATURE_COLS].mean()
    sd = df.loc[train_mask, FEATURE_COLS].std().replace(0, 1.0)
    Xall = ((df[FEATURE_COLS] - mu) / sd).fillna(0.0).values.astype("float32")
    y    = df["target_direction"].values.astype("float32")
    tks  = df["ticker"].values
    dts  = df["date"].values
    n_feat = Xall.shape[1]
    L = int(SEQ_LEN)

    # ── Construction des fenetres glissantes par ticker ──────────────
    print(f"  Construction des fenetres (L={L}, n_feat={n_feat})...")
    W, yy, wt, wd = [], [], [], []
    for tk in sorted(set(tks)):
        idx = np.where(tks == tk)[0]
        Xi, yi, di = Xall[idx], y[idx], dts[idx]
        for j in range(len(idx)):
            lo = max(0, j - L + 1)
            w = Xi[lo:j + 1]
            if len(w) < L:                       # padding par la 1ere ligne
                w = np.vstack([np.repeat(w[:1], L - len(w), axis=0), w])
            W.append(w); yy.append(yi[j]); wt.append(tk); wd.append(di[j])
    W  = np.asarray(W, dtype="float32")
    yy = np.asarray(yy, dtype="float32")
    wd = pd.to_datetime(pd.Series(wd))
    print(f"  Fenetres : {W.shape}")

    tr_m = (wd <= TRAIN_END).values
    va_m = ((wd > TRAIN_END) & (wd <= VAL_END)).values
    te_m = ((wd > VAL_END) & (wd <= TEST_END)).values

    # Borne memoire : sous-echantillonne le train si gigantesque (CPU)
    MAX_TR = 400_000 if device.type == "cuda" else 120_000
    tr_idx = np.where(tr_m)[0]
    if len(tr_idx) > MAX_TR:
        rng = np.random.default_rng(42)
        tr_idx = rng.choice(tr_idx, MAX_TR, replace=False)
    va_idx = np.where(va_m)[0]
    te_idx = np.where(te_m)[0]
    print(f"  train={len(tr_idx):,}  val={len(va_idx):,}  test={len(te_idx):,}")

    # ── Modele : BiGRU bidirectionnel + attention additive ───────────
    class BiGRUAttn(nn.Module):
        def __init__(self, n_feat, hidden=64, layers=1, p=0.3):
            super().__init__()
            self.gru = nn.GRU(n_feat, hidden, num_layers=layers,
                              batch_first=True, bidirectional=True,
                              dropout=p if layers > 1 else 0.0)
            self.attn = nn.Linear(hidden * 2, 1)
            self.head = nn.Sequential(
                nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(p),
                nn.Linear(64, 1))
        def forward(self, x):
            h, _ = self.gru(x)                       # (B, L, 2H)
            a = torch.softmax(self.attn(h).squeeze(-1), dim=1)  # (B, L)
            ctx = (h * a.unsqueeze(-1)).sum(dim=1)   # (B, 2H)
            return self.head(ctx).squeeze(-1)

    model = BiGRUAttn(n_feat).to(device)
    pos_w = float((yy[tr_idx] == 0).sum()) / max(float((yy[tr_idx] == 1).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    Wt = torch.from_numpy(W)   # garde en CPU, on transfere par batch

    def run_eval(idx):
        model.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(idx), SEQ_BATCH):
                b = idx[i:i + SEQ_BATCH]
                xb = Wt[b].to(device)
                outs.append(torch.sigmoid(model(xb)).cpu().numpy())
        return np.concatenate(outs) if outs else np.array([])

    best_auc, best_state, patience, bad = -1.0, None, 5, 0
    print("  Entrainement BiGRU...")
    for ep in range(1, int(SEQ_EPOCHS) + 1):
        model.train()
        rng = np.random.default_rng(ep)
        perm = rng.permutation(tr_idx)
        t0 = time.time()
        for i in range(0, len(perm), SEQ_BATCH):
            b = perm[i:i + SEQ_BATCH]
            xb = Wt[b].to(device)
            yb = torch.from_numpy(yy[b]).to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        p_va = run_eval(va_idx)
        auc = roc_auc_score(yy[va_idx], p_va) if len(set(yy[va_idx])) > 1 else 0.5
        print(f"    epoch {ep:>2}  val_AUC={auc:.4f}  ({time.time()-t0:.1f}s)")
        if auc > best_auc:
            best_auc, best_state, bad = auc, {k: v.cpu().clone()
                                              for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print("    early stopping"); break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  ✅ meilleur val_AUC = {best_auc:.4f}")

    # ── Probabilites val + test → parquet pour le stacking ───────────
    out = pd.DataFrame({
        "ticker": np.r_[np.array(wt)[va_idx], np.array(wt)[te_idx]],
        "date":   pd.to_datetime(np.r_[wd.values[va_idx], wd.values[te_idx]]),
        "p_seq":  np.r_[run_eval(va_idx), run_eval(te_idx)].astype("float32"),
        "split":  (["val"] * len(va_idx)) + (["test"] * len(te_idx)),
    })
    out["date"] = out["date"].dt.normalize()
    out.to_parquet(SEQ_OUT, index=False)
    torch.save(model.state_dict(), f"{MODELS_DIR}/model_seq.pt")
    with open(f"{MODELS_DIR}/seq_summary.json", "w") as f:
        json.dump({"val_auc": float(best_auc), "seq_len": L,
                   "n_feat": int(n_feat)}, f, indent=2)
    print(f"  💾 {Path(SEQ_OUT).name}  ({len(out):,} rows)  + model_seq.pt")
    mem.free()


  Device : cuda
  📂 loaded 08_features.parquet (19,348 rows)
  Construction des fenetres (L=20, n_feat=49)...
  Fenetres : (19348, 20, 49)
  train=12,341  val=1,757  test=1,750
  Entrainement BiGRU...
    epoch  1  val_AUC=0.4762  (0.6s)
    epoch  2  val_AUC=0.4796  (0.2s)
    epoch  3  val_AUC=0.4816  (0.2s)
    epoch  4  val_AUC=0.4897  (0.2s)
    epoch  5  val_AUC=0.4950  (0.1s)
    epoch  6  val_AUC=0.4985  (0.2s)
    epoch  7  val_AUC=0.4842  (0.2s)
    epoch  8  val_AUC=0.4929  (0.2s)
    epoch  9  val_AUC=0.5082  (0.2s)
    epoch 10  val_AUC=0.5010  (0.1s)
    epoch 11  val_AUC=0.4929  (0.2s)
    epoch 12  val_AUC=0.5147  (0.2s)
    epoch 13  val_AUC=0.4902  (0.2s)
    epoch 14  val_AUC=0.4947  (0.2s)
    epoch 15  val_AUC=0.4968  (0.2s)
    epoch 16  val_AUC=0.5135  (0.2s)
    epoch 17  val_AUC=0.5045  (0.2s)
    early stopping
  ✅ meilleur val_AUC = 0.5147
  💾 seq_proba.parquet  (3,507 rows)  + model_seq.pt


## Cell 13c — Stacking hybride + calibration  ⭐ NOUVEAU

Combine les **4 modeles de base** (LightGBM, XGBoost, CatBoost, BiGRU) via un **meta-modele logistique**
ajuste sur la validation, puis applique une **calibration isotone** (probabilites mieux calibrees → meilleures
decisions de trading). Produit `stack_proba.parquet` (proba finale alignee sur les lignes de test) que la
CELL 14 utilise automatiquement.

In [27]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13c — STACKING (meta-logistique) + CALIBRATION ISOTONE     ║
# ╚══════════════════════════════════════════════════════════════════╝
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from catboost import CatBoostClassifier

if not USE_STACKING:
    print("  USE_STACKING=False → stacking desactive (CELL 14 utilisera la moyenne fixe).")
else:
    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    val  = ckpt.load(PATH_VAL).sort_values(["ticker", "date"]).reset_index(drop=True)
    test = ckpt.load(PATH_TEST).sort_values(["ticker", "date"]).reset_index(drop=True)
    for d in (val, test):
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()

    clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
    clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
    clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

    def tree_probas(d):
        X = d[FEATURE_COLS].values
        return np.column_stack([
            clf_lgb.predict_proba(X)[:, 1],
            clf_xgb.predict_proba(X)[:, 1],
            clf_cb.predict_proba(X)[:, 1],
        ])

    Pva, Pte = tree_probas(val), tree_probas(test)
    base_names = ["lgb", "xgb", "cb"]

    # ── Ajout du BiGRU si disponible ─────────────────────────────────
    seqp = f"{OUTPUT_DIR}/seq_proba.parquet"
    if Path(seqp).exists():
        sp = pd.read_parquet(seqp)
        sp["date"] = pd.to_datetime(sp["date"]).dt.normalize()
        def add_seq(d, P):
            m = d[["ticker", "date"]].merge(
                sp[["ticker", "date", "p_seq"]], on=["ticker", "date"], how="left")
            return np.column_stack([P, m["p_seq"].fillna(0.5).values])
        Pva, Pte = add_seq(val, Pva), add_seq(test, Pte)
        base_names.append("seq")
        print("  ✅ BiGRU integre au stacking")
    else:
        print("  ℹ️  Pas de seq_proba.parquet — stacking sur les 3 arbres seulement")

    yva = val["target_direction"].astype(int).values
    yte = test["target_direction"].astype(int).values

    # ── Meta-modele logistique (ajuste sur la validation) ────────────
    meta = LogisticRegression(max_iter=2000, C=1.0)
    meta.fit(Pva, yva)
    raw_va = meta.predict_proba(Pva)[:, 1]
    raw_te = meta.predict_proba(Pte)[:, 1]

    # ── Calibration isotone (sur la validation) ──────────────────────
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va, yva)
    p_stack    = iso.transform(raw_te).astype("float32")
    p_stack_va = iso.transform(raw_va).astype("float32")
    # Seuil de décision calibré sur la VALIDATION (quantile = taux de hausse val).
    # Évite l effondrement de l ensemble (sinon il prédit presque tout "baisse").
    _thr_stack = float(np.quantile(p_stack_va, 1.0 - float(np.mean(yva))))
    print(f"  Seuil ensemble calibré (val) : {_thr_stack:.4f}")

    # ── Rapport AUC base vs stack ────────────────────────────────────
    print("\n  AUC (test) par modele de base :")
    for i, nm in enumerate(base_names):
        print(f"    {nm:<5} {roc_auc_score(yte, Pte[:, i]):.4f}")
    print(f"    {'STACK':<5} {roc_auc_score(yte, p_stack):.4f}  "
          f"(acc={accuracy_score(yte, (p_stack>0.5).astype(int)):.4f})")
    print(f"  Poids meta (logistique) : "
          f"{dict(zip(base_names, np.round(meta.coef_[0], 3)))}")

    # ── Sauvegardes ──────────────────────────────────────────────────
    out = test[["ticker", "date"]].copy()
    out["p_stack"] = p_stack
    out.to_parquet(f"{OUTPUT_DIR}/stack_proba.parquet", index=False)
    joblib.dump({"meta": meta, "iso": iso, "names": base_names},
                f"{MODELS_DIR}/model_stack.joblib")
    with open(f"{MODELS_DIR}/stack_summary.json", "w") as f:
        json.dump({"base_names": base_names,
                   "weights": dict(zip(base_names, meta.coef_[0].tolist())),
                   "stack_test_auc": float(roc_auc_score(yte, p_stack)),
                   "threshold": _thr_stack}, f, indent=2)
    with open(f"{MODELS_DIR}/stack_threshold.json", "w") as f:
        json.dump({"threshold": _thr_stack,
                   "val_pos_rate": float(np.mean(yva))}, f)
    print(f"  💾 stack_proba.parquet + model_stack.joblib")


  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  ✅ BiGRU integre au stacking
  Seuil ensemble calibré (val) : 0.5000

  AUC (test) par modele de base :
    lgb   0.4970
    xgb   0.5141
    cb    0.5134
    seq   0.5216
    STACK 0.5134  (acc=0.5183)
  Poids meta (logistique) : {'lgb': np.float64(0.173), 'xgb': np.float64(0.446), 'cb': np.float64(0.203), 'seq': np.float64(0.293)}
  💾 stack_proba.parquet + model_stack.joblib


## Cell 14 — Ensemble (stacking hybride), evaluation & backtest

In [28]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — ENSEMBLE + EVAL + BACKTEST                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import joblib
from catboost import CatBoostClassifier

with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

test = ckpt.load(PATH_TEST)
X_te = test[FEATURE_COLS].values
y_te = test["target_direction"].values.astype(int)

# Load models
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

# Probabilities
p_lgb = clf_lgb.predict_proba(X_te)[:, 1]
p_xgb = clf_xgb.predict_proba(X_te)[:, 1]
p_cb  = clf_cb.predict_proba(X_te)[:, 1]

# ── Ensemble HYBRIDE : on privilegie la pile apprise (CELL 13c :
#    arbres + BiGRU + meta-logistique calibree) si elle existe, sinon
#    on retombe sur la moyenne ponderee fixe.
from pathlib import Path as _P
_stack_path = f"{OUTPUT_DIR}/stack_proba.parquet"
_p_fixed = 0.4 * p_lgb + 0.3 * p_xgb + 0.3 * p_cb
if _P(_stack_path).exists():
    _sp = pd.read_parquet(_stack_path)
    _sp["date"] = pd.to_datetime(_sp["date"]).dt.normalize()
    _key = test[["ticker", "date"]].copy()
    _key["date"] = pd.to_datetime(_key["date"]).dt.normalize()
    _m = _key.merge(_sp[["ticker", "date", "p_stack"]], on=["ticker", "date"], how="left")
    p_ens = _m["p_stack"].fillna(pd.Series(_p_fixed)).values
    print("  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)")
else:
    p_ens = _p_fixed
    print("  ℹ️  Ensemble = moyenne ponderee fixe (stacking absent)")

# Seuil ensemble par CALAGE DE TAUX : on aligne la proportion de "hausse"
# prédite sur le taux de hausse de la VALIDATION (a priori connu, pas de
# fuite des labels test). Robuste au fort déséquilibre des horizons longs.
try:
    with open(f"{MODELS_DIR}/stack_threshold.json") as _f:
        _vpr = float(json.load(_f).get("val_pos_rate", 0.5))
    _THR_ENS = float(np.quantile(p_ens, 1.0 - _vpr))
except Exception:
    _THR_ENS = 0.5

# Per-model metrics (seuil par modèle : ensemble = seuil calibré)
def metrics_row(name, p, thr=0.5):
    yp = (p > thr).astype(int)
    return {
        "name":      name,
        "accuracy":  float(accuracy_score(y_te, yp)),
        "auc":       float(roc_auc_score(y_te, p)),
        "precision": float(precision_score(y_te, yp, zero_division=0)),
        "recall":    float(recall_score(y_te, yp, zero_division=0)),
        "f1":        float(f1_score(y_te, yp, zero_division=0)),
    }
rows = [metrics_row("lightgbm", p_lgb), metrics_row("xgboost", p_xgb),
        metrics_row("catboost", p_cb)]
# Ensemble (stacking) exclu : dégénère sous fort déséquilibre et n'apporte
# rien (AUC ≈ modèles de base). Modèles retenus : LightGBM/XGBoost/CatBoost.
results = pd.DataFrame(rows).set_index("name").round(4)

print("=" * 60)
print("  TEST SET METRICS")
print("=" * 60)
print(results.to_string())

# ── BACKTEST ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BACKTEST  (threshold 0.55 long / 0.45 short)")
print("=" * 60)
_is_vol = globals().get("TARGET", "direction") == "volatility"
if _is_vol:
    print("  ⚠️ Cible = VOLATILITÉ : ce backtest directionnel (long/short) n a PAS")
    print("     de sens (on prédit l amplitude, pas le sens du mouvement).")
    print("     → À IGNORER pour le papier. Utiliser AUC + ablation à la place.")
    print("     (Calculé seulement pour ne pas casser la figure du dashboard.)")
bt = test[["ticker", "date", "close"]].copy()
bt["pred_ens"] = p_lgb   # backtest sur LightGBM (modèle principal)
bt = bt.sort_values(["ticker", "date"]).reset_index(drop=True)
# Rendement RÉALISÉ à 1 jour (close-to-close). On n'utilise JAMAIS le target
# à H jours pour composer : sinon on cumulerait un rendement H-jours chaque
# jour (fenêtres chevauchantes) → explosion irréaliste du cumul.
bt["ret_1d"] = bt.groupby("ticker", observed=True)["close"].pct_change().fillna(0.0)
bt["signal"]   = 0
bt.loc[bt["pred_ens"] > 0.55, "signal"] = 1
bt.loc[bt["pred_ens"] < 0.45, "signal"] = -1

# Strategy return = signal de la veille × rendement réalisé du jour
bt["signal_lag"]      = bt.groupby("ticker", observed=True)["signal"].shift(1).fillna(0)
bt["strategy_return"] = bt["signal_lag"] * bt["ret_1d"]

# ── Transaction costs (10 bps per round-trip = realistic retail broker)
TXN_COST_BPS = 10
bt["pos_change"]      = (bt["signal"] != bt["signal_lag"]).astype(np.int8)
bt["txn_cost"]        = bt["pos_change"] * (TXN_COST_BPS / 10000.0)
bt["strategy_net"]    = bt["strategy_return"] - bt["txn_cost"]

bt["cum_strategy"]    = bt.groupby("ticker", observed=True)["strategy_net"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())
bt["cum_market"]      = bt.groupby("ticker", observed=True)["ret_1d"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())

# Per-ticker summary
backtest_summary = {}
for tk, g in bt.groupby("ticker", observed=True):
    ret_s   = (g["cum_strategy"].iloc[-1] - 1) * 100
    ret_m   = (g["cum_market"].iloc[-1]   - 1) * 100
    sr_std  = g["strategy_net"].std()
    sharpe  = (g["strategy_net"].mean() / sr_std * np.sqrt(252)) if sr_std > 0 else 0.0
    n_tr    = int(g["pos_change"].sum())
    act     = g[g["signal"] != 0]["strategy_net"]
    win_rt  = float((act > 0).mean()) if len(act) else 0.0
    txn_drag = float(g["txn_cost"].sum() * 100)
    backtest_summary[tk] = {
        "return_strategy":  round(float(ret_s), 2),
        "return_market":    round(float(ret_m), 2),
        "alpha":            round(float(ret_s - ret_m), 2),
        "sharpe":           round(float(sharpe), 3),
        "n_trades":         n_tr,
        "win_rate":         round(win_rt, 4),
        "txn_cost_drag":    round(txn_drag, 2),     # cumulative cost in %
    }

# Aggregate
total_s = (bt.groupby("date")["strategy_net"].mean().fillna(0) + 1).cumprod().iloc[-1] - 1
total_m = (bt.groupby("date")["ret_1d"].mean().fillna(0)   + 1).cumprod().iloc[-1] - 1
_daily_strat = bt.groupby("date")["strategy_net"].mean().fillna(0)
_eq = (1 + _daily_strat).cumprod()
_max_dd = float((_eq / _eq.cummax() - 1).min())
_sh_overall = (float(_daily_strat.mean() / _daily_strat.std() * np.sqrt(252))
               if _daily_strat.std() > 0 else 0.0)
backtest_summary["__overall__"] = {
    "return_strategy": round(float(total_s * 100), 2),
    "return_market":   round(float(total_m * 100), 2),
    "alpha":           round(float((total_s - total_m) * 100), 2),
    "sharpe":          round(_sh_overall, 3),
    "max_drawdown":    round(_max_dd * 100, 2),
}

print(f"\n  Strategy total : {total_s*100:+.2f}%")
print(f"  Market   total : {total_m*100:+.2f}%")
print(f"  Alpha          : {(total_s - total_m)*100:+.2f}%\n")
print(f"  {'Ticker':<8} {'Strategy':>10} {'Market':>10} {'Alpha':>9} {'Sharpe':>7} {'Trades':>7} {'WinR':>6}")
print("  " + "─" * 60)
for tk in STOCKS:
    if tk not in backtest_summary: continue
    s = backtest_summary[tk]
    print(f"  {tk:<8} {s['return_strategy']:>+9.2f}% {s['return_market']:>+9.2f}% "
          f"{s['alpha']:>+8.2f}% {s['sharpe']:>7.2f} {s['n_trades']:>7d} {s['win_rate']*100:>5.1f}%")

# Save
ckpt.save(bt, PATH_BACKTEST)
results.to_json(f"{MODELS_DIR}/test_metrics.json", indent=2)
with open(f"{MODELS_DIR}/backtest_summary.json", "w") as f:
    json.dump(backtest_summary, f, indent=2)

# Confusion matrix (LightGBM)
print("\n  Confusion matrix (LightGBM):")
yp_ens = (p_lgb > 0.5).astype(int)
cm = confusion_matrix(y_te, yp_ens)
print(f"             pred_down  pred_up")
print(f"  true_down   {cm[0,0]:>7,}  {cm[0,1]:>7,}")
print(f"  true_up     {cm[1,0]:>7,}  {cm[1,1]:>7,}")

mem.free()


# ════════════════════════════════════════════════════════════════════
#  BASELINES + SIGNIFICATIVITÉ STATISTIQUE — pour l'acceptation du papier
# ════════════════════════════════════════════════════════════════════
import numpy as _np
from sklearn.metrics import roc_auc_score as _auc

print("\n" + "═" * 64)
print("  BASELINES & SIGNIFICATIVITÉ (test set)")
print("═" * 64)

# Baselines naïves
_base_up   = float((y_te == 1).mean())          # "toujours hausse"
_base_maj  = max(_base_up, 1 - _base_up)         # classe majoritaire
print(f"  Baseline 'toujours hausse' : acc = {_base_up:.4f}")
print(f"  Baseline classe majoritaire: acc = {_base_maj:.4f}")
print(f"  (un modèle utile doit battre {_base_maj:.4f} de façon significative)")

def _auc_safe(yt, p):
    try: return _auc(yt, p)
    except Exception: return 0.5

# Bootstrap IC 95% + test de permutation pour l'AUC de chaque modèle
_rng = _np.random.default_rng(42)
_N_BOOT, _N_PERM = 1000, 1000
print(f"\n  {'Modèle':<11} {'AUC':>7}  {'IC 95% bootstrap':>22}  {'p (perm)':>9}")
print("  " + "-" * 56)
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    _p = _np.asarray(_p, dtype=float)
    _obs = _auc_safe(y_te, _p)
    # bootstrap
    _bs = []
    _n = len(y_te)
    for _ in range(_N_BOOT):
        _idx = _rng.integers(0, _n, _n)
        if len(_np.unique(y_te[_idx])) < 2: continue
        _bs.append(_auc_safe(y_te[_idx], _p[_idx]))
    _lo, _hi = (_np.percentile(_bs, [2.5, 97.5]) if _bs else (_np.nan, _np.nan))
    # permutation : H0 = pas de lien (on mélange y)
    _null = _np.empty(_N_PERM)
    for _i in range(_N_PERM):
        _null[_i] = _auc_safe(_rng.permutation(y_te), _p)
    _pval = float((_np.sum(_np.abs(_null - 0.5) >= abs(_obs - 0.5)) + 1) / (_N_PERM + 1))
    _star = "***" if _pval < 0.01 else "**" if _pval < 0.05 else "*" if _pval < 0.10 else "ns"
    print(f"  {_name:<11} {_obs:>7.4f}  [{_lo:.4f}, {_hi:.4f}]  {_pval:>8.4f} {_star}")

print("\n  Lecture : un AUC est crédible si son IC 95% exclut 0.50 et p<0.05.")
print("  *** p<0.01  ** p<0.05  * p<0.10  ns = non significatif")
print("═" * 64)


# ════════════════════════════════════════════════════════════════════
#  MODE ABSTENTION — accuracy vs couverture (sélectivité de confiance)
#  Le modèle n'agit que sur ses prédictions les plus confiantes.
#  Courbe publiable : "accuracy croît quand on réduit la couverture".
# ════════════════════════════════════════════════════════════════════
import numpy as _np
print("\n" + "═" * 64)
print("  MODE ABSTENTION — accuracy sur les prédictions les + confiantes")
print("═" * 64)

def _conf_curve(name, p, thr=0.5):
    p = _np.asarray(p, dtype=float)
    conf = _np.abs(p - thr)                 # distance au seuil de décision = confiance
    order = _np.argsort(-conf)              # plus confiant d'abord
    yp_all = (p > thr).astype(int)
    rows = []
    for cov in (1.00, 0.50, 0.30, 0.20, 0.10, 0.05):
        k = max(1, int(round(len(p) * cov)))
        idx = order[:k]
        acc = float((yp_all[idx] == y_te[idx]).mean())
        rows.append((cov, k, acc))
    return rows

print(f"  {'Couverture':>10} {'N trades':>9} {'Accuracy':>9}")
print("  " + "-" * 32)
_abst_out = {}
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    print(f"  ── {_name} ──")
    _thr_a = _THR_ENS if _name == "ensemble" else 0.5
    _rows = _conf_curve(_name, _p, _thr_a)
    _abst_out[_name] = [{"coverage": c, "n": k, "accuracy": a} for c, k, a in _rows]
    for cov, k, acc in _rows:
        print(f"  {cov*100:>9.0f}% {k:>9,} {acc:>9.4f}")

try:
    with open(f"{MODELS_DIR}/abstention_curve.json", "w") as _f:
        json.dump(_abst_out, _f, indent=2)
except Exception:
    pass

print("\n  Lecture : si l'accuracy monte quand la couverture baisse, le modèle")
print("  'sait quand il sait' — argument fort pour le papier (trading sélectif).")
print("  À reporter en figure : accuracy (y) vs couverture (x), par modèle.")
print("═" * 64)


  📂 loaded 09_test.parquet (1,750 rows)
  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)
  TEST SET METRICS
          accuracy     auc  precision  recall      f1
name                                                 
lightgbm    0.5011  0.4970     0.5410  0.6046  0.5710
xgboost     0.5160  0.5141     0.5462  0.7014  0.6141
catboost    0.5189  0.5134     0.5527  0.6493  0.5971

  BACKTEST  (threshold 0.55 long / 0.45 short)

  Strategy total : +0.88%
  Market   total : +112.33%
  Alpha          : -111.45%

  Ticker     Strategy     Market     Alpha  Sharpe  Trades   WinR
  ────────────────────────────────────────────────────────────
  AAPL         -1.88%    +54.80%   -56.68%   -0.16      96  12.9%
  MSFT         +0.90%    +58.35%   -57.45%    0.14      96  12.9%
  GOOGL        +4.16%    +56.74%   -52.58%    0.33      95  18.9%
  AMZN         -2.71%    +77.04%   -79.76%   -0.10     103  17.6%
  META         +8.61%   +183.76%  -175.15%    0.50     102  16.4%
  TSLA        -12.54% 

## Cell 15 — Analyse causale (Pearson lags + Granger + SHAP + Ablation)

In [29]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — CAUSAL ANALYSIS                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
for pkg in ("statsmodels", "shap"):
    try: __import__(pkg)
    except ImportError: pip_install(pkg)

from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
import shap

df = ckpt.load(PATH_FEATURES)
df["date"] = pd.to_datetime(df["date"])

# ── A. PEARSON LAG CORRELATIONS (per ticker) ──────────────────────
print("=" * 60)
print("  A. PEARSON LAG CORRELATIONS")
print("=" * 60)
lag_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 50: continue
    for lag in range(0, 6):
        s_lagged = ds["sent_mean"].shift(lag).dropna()
        ret_aligned = ds["return_1d"].iloc[lag:].reset_index(drop=True)
        s_lagged    = s_lagged.reset_index(drop=True)
        n = min(len(s_lagged), len(ret_aligned))
        if n < 30:
            r_val, p_val = np.nan, np.nan
        else:
            r_val, p_val = stats.pearsonr(s_lagged[:n], ret_aligned[:n])
        lag_rows.append({
            "ticker":      tk,
            "lag_days":    lag,
            "correlation": round(float(r_val), 4) if not np.isnan(r_val) else None,
            "p_value":     round(float(p_val), 4) if not np.isnan(p_val) else None,
            "significant": (p_val < 0.05) if not np.isnan(p_val) else False,
        })
df_lags = pd.DataFrame(lag_rows)
df_lags.to_csv(f"{MODELS_DIR}/lag_correlations.csv", index=False)
print("\n  Lag-correlation summary:")
print(df_lags.groupby("lag_days")["correlation"].describe()[
      ["mean", "std", "min", "max"]].round(4).to_string())
print(f"\n  Significant lags (p<0.05): {df_lags['significant'].sum()} / {len(df_lags)}")

# ── B. GRANGER CAUSALITY ──────────────────────────────────────────
print("\n" + "=" * 60)
print("  B. GRANGER CAUSALITY  (sent -> return)")
print("=" * 60)
granger_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 100: continue
    try:
        data = ds[["return_1d", "sent_mean"]].values
        gc_res = grangercausalitytests(data, maxlag=5, verbose=False)
        for lag, tests in gc_res.items():
            granger_rows.append({
                "ticker":  tk,
                "lag":     lag,
                "p_value": round(float(tests[0]["ssr_ftest"][1]), 4),
                "causal":  bool(tests[0]["ssr_ftest"][1] < 0.05),
            })
    except Exception as e:
        print(f"  ⚠️  {tk}: {str(e)[:80]}")
df_granger = pd.DataFrame(granger_rows)
df_granger.to_csv(f"{MODELS_DIR}/granger_results.csv", index=False)
print("  Verdict per ticker (significant lags / 5):")
for tk in STOCKS:
    g = df_granger[df_granger["ticker"] == tk]
    if g.empty: continue
    n_sig = int(g["causal"].sum())
    verdict = "✅ causal" if n_sig >= 2 else "⚠️  weak" if n_sig == 1 else "❌ not causal"
    print(f"    {tk:<6} {n_sig}/5   {verdict}")

# ── C. SHAP VALUES ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  C. SHAP VALUES (LightGBM)")
print("=" * 60)
import joblib
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
test = ckpt.load(PATH_TEST)
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

# Sample for speed
sample_n   = min(3000, len(test))
X_sample   = test[FEATURE_COLS].sample(sample_n, random_state=42).values
explainer  = shap.TreeExplainer(clf_lgb)
shap_vals  = explainer.shap_values(X_sample)
if isinstance(shap_vals, list):    # for binary classifier returns [neg, pos]
    sv = shap_vals[1]
else:
    sv = shap_vals
mean_abs   = np.abs(sv).mean(axis=0)
shap_df    = pd.DataFrame({"feature": FEATURE_COLS, "mean_abs_shap": mean_abs}) \
                .sort_values("mean_abs_shap", ascending=False)
shap_df.to_csv(f"{MODELS_DIR}/shap_importance.csv", index=False)

print("\n  Top 15 features:")
total = shap_df["mean_abs_shap"].sum()
SENT_KEY = ("sent", "finbert", "sarcasm", "fear", "anger", "joy", "sadness", "mentions", "emoji")
sent_share = shap_df[shap_df["feature"].str.contains("|".join(SENT_KEY))]["mean_abs_shap"].sum()
for _, r in shap_df.head(15).iterrows():
    tag = " 🟣NLP" if any(k in r["feature"] for k in SENT_KEY) else ""
    bar = "█" * int(r["mean_abs_shap"] / shap_df["mean_abs_shap"].max() * 30)
    print(f"    {r['feature']:<25} {r['mean_abs_shap']:.5f}  {bar}{tag}")
print(f"\n  Sentiment/NLP share of total SHAP : {sent_share/total*100:.1f}%")

# ── D. ABLATION STUDY ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  D. ABLATION  (with vs without sentiment features)")
print("=" * 60)
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

tech_only = [c for c in FEATURE_COLS if not any(k in c for k in SENT_KEY)]
sent_only = [c for c in FEATURE_COLS if any(k in c for k in SENT_KEY)]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test_ = test  # alias

ablation = {}
for label, cols in (
    ("technical_only", tech_only),
    ("sentiment_only", sent_only),
    ("full_model",     FEATURE_COLS),
):
    if not cols:
        continue
    X_tr_a = train[cols].values
    y_tr_a = train["target_direction"].values.astype(int)
    X_te_a = test_[cols].values
    y_te_a = test_["target_direction"].values.astype(int)
    clf = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    clf.fit(X_tr_a, y_tr_a)
    auc = roc_auc_score(y_te_a, clf.predict_proba(X_te_a)[:, 1])
    ablation[label] = {
        "n_features": len(cols),
        "auc":        round(float(auc), 4),
    }
    print(f"  {label:<20} ({len(cols):>2} feats)   AUC = {auc:.4f}")

if "technical_only" in ablation and "full_model" in ablation:
    lift = ablation["full_model"]["auc"] - ablation["technical_only"]["auc"]
    ablation["sentiment_lift"] = round(float(lift), 4)
    print(f"\n  ▶  Sentiment lift over price-only: {lift:+.4f} AUC points "
          f"({lift*100:+.2f}%)")

with open(f"{MODELS_DIR}/ablation.json", "w") as f:
    json.dump(ablation, f, indent=2)
# ── E. PLACEBO : sentiment remplacé par du bruit (sanity check) ───
print("\n" + "=" * 60)
print("  E. PLACEBO  (sentiment features = bruit aléatoire)")
print("=" * 60)
rng = np.random.default_rng(42)
train_p, test_p = train.copy(), test_.copy()
for c in sent_only:                      # mêmes colonnes NLP, remplacées par du bruit
    mu, sd = train[c].mean(), train[c].std() + 1e-9
    train_p[c] = rng.normal(mu, sd, len(train_p))
    test_p[c]  = rng.normal(mu, sd, len(test_p))

clf_p = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    random_state=42, n_jobs=-1, verbose=-1,
)
clf_p.fit(train_p[FEATURE_COLS].values, train_p["target_direction"].values.astype(int))
auc_placebo = roc_auc_score(
    test_p["target_direction"].values.astype(int),
    clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1],
)
auc_full = ablation.get("full_model", {}).get("auc", float("nan"))
auc_tech = ablation.get("technical_only", {}).get("auc", float("nan"))

# ── F. SAUVEGARDE PROBA PLACEBO (sentiment=bruit) pour le Projet 2 ──
# clf_p et test_p viennent de la section E ci-dessus.
proba_placebo = clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1]
out_P = test_[["ticker","date"]].copy()
out_P["pred_placebo"] = proba_placebo

# nom daté selon l'année de test courante
yr = int(pd.to_datetime(test_["date"]).dt.year.mode()[0])
out_P.to_parquet(f"{OUTPUT_DIR}/proba_placebo_{yr}.parquet", index=False)

# ── G. PLACEBO MULTI-GRAINES ──────────────────────────────────────
print("\n" + "="*60); print("  G. PLACEBO MULTI-GRAINES (10 tirages)"); print("="*60)
seeds = range(42, 52); records = []
for seed in seeds:
    rng_g = np.random.default_rng(seed)
    tr_p, te_p = train.copy(), test_.copy()
    for c in sent_only:
        mu, sd = train[c].mean(), train[c].std() + 1e-9
        tr_p[c] = rng_g.normal(mu, sd, len(tr_p)); te_p[c] = rng_g.normal(mu, sd, len(te_p))
    clf_g = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               random_state=42, n_jobs=-1, verbose=-1)
    clf_g.fit(tr_p[FEATURE_COLS].values, tr_p["target_direction"].values.astype(int))
    records.append(clf_g.predict_proba(te_p[FEATURE_COLS].values)[:, 1])
proba_mean = np.mean(records, axis=0)
out_Pm = test_[["ticker","date"]].copy(); out_Pm["pred_placebo"] = proba_mean
out_Pm.to_parquet(f"{OUTPUT_DIR}/proba_placebo_multi_{yr}.parquet", index=False)
print(f"✅ proba_placebo_multi_{yr}.parquet sauvegardé")
print(f"✅ proba_placebo_{yr}.parquet sauvegardé : {out_P.shape}")
print("Vérif année :", sorted(pd.to_datetime(out_P['date']).dt.year.unique()))
print(f"  technical_only      AUC = {auc_tech:.4f}")
print(f"  full_model (réel)   AUC = {auc_full:.4f}")
print(f"  full_model PLACEBO  AUC = {auc_placebo:.4f}  (sentiment = bruit)")
print(f"\n  ▶  Lift réel    : {(auc_full-auc_tech)*100:+.2f}%")
print(f"  ▶  Lift placebo : {(auc_placebo-auc_tech)*100:+.2f}%")
print("  Lecture : si le lift placebo ≈ 0 alors que le lift réel est positif,")
print("            le gain vient bien du sentiment réel (pas d'un simple ajout de colonnes).")

  📂 loaded 08_features.parquet (19,348 rows)
  A. PEARSON LAG CORRELATIONS

  Lag-correlation summary:
            mean     std     min     max
lag_days                                
0         0.1844  0.0487  0.1426  0.2674
1         0.0285  0.0232 -0.0013  0.0741
2         0.0026  0.0286 -0.0461  0.0385
3         0.0028  0.0292 -0.0273  0.0562
4         0.0124  0.0175 -0.0182  0.0314
5        -0.0039  0.0128 -0.0231  0.0172

  Significant lags (p<0.05): 11 / 42

  B. GRANGER CAUSALITY  (sent -> return)
  Verdict per ticker (significant lags / 5):
    AAPL   2/5   ✅ causal
    MSFT   3/5   ✅ causal
    GOOGL  0/5   ❌ not causal
    AMZN   0/5   ❌ not causal
    META   5/5   ✅ causal
    TSLA   0/5   ❌ not causal
    NVDA   3/5   ✅ causal

  C. SHAP VALUES (LightGBM)
  📂 loaded 09_test.parquet (1,750 rows)

  Top 15 features:
    market_share_pos_lag1     0.09062  ██████████████████████████████
    ret_1d                    0.01662  █████
    dow                       0.01533  █████
 

In [30]:
# ═══ SNAPSHOT — consigne les métriques de CE run dans RESULTS.txt ═══
import json, glob, os, shutil, traceback
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score
try:
    YEAR = int(pd.Timestamp(TEST_END).year); H = int(HORIZON)
    RES  = f"{BASE_DIR}/RESULTS.txt"
    test = pd.read_parquet(f"{OUTPUT_DIR}/09_test.parquet")
    with open(f"{MODELS_DIR}/feature_names.json") as fh:
        FEATS = json.load(fh)["features"]
    ycol = "target_direction"
    X = test[FEATS]; y = test[ycol].values.astype(int)
    rng = np.random.default_rng(42)
    def boot_ci(y_, p_, n=1000):
        idx = np.arange(len(y_)); a = []
        for _ in range(n):
            b = rng.choice(idx, len(idx), replace=True)
            if y_[b].min() == y_[b].max(): continue
            a.append(roc_auc_score(y_[b], p_[b]))
        return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))
    def perm_p(y_, p_, n=500):
        base = roc_auc_score(y_, p_); c = 0
        for _ in range(n):
            yp = rng.permutation(y_)
            if yp.min() == yp.max(): continue
            if abs(roc_auc_score(yp, p_) - 0.5) >= abs(base - 0.5): c += 1
        return (c + 1) / (n + 1)
    L = ["", "="*66,
         f"RUN  test_year={YEAR}  horizon={H}j   n_test={len(y)}  pos_rate={y.mean():.3f}",
         "-"*66,
         f"baseline classe majoritaire (acc) = {max(y.mean(), 1-y.mean()):.4f}"]
    for path in sorted(glob.glob(f"{MODELS_DIR}/*.joblib")):
        name = os.path.basename(path).replace(".joblib", "")
        try:
            mdl = joblib.load(path); p = mdl.predict_proba(X)[:, 1]
        except Exception as e:
            L.append(f"{name:22s} (skip: {type(e).__name__})"); continue
        auc = roc_auc_score(y, p); lo, hi = boot_ci(y, p); pv = perm_p(y, p)
        L.append(f"{name:22s} AUC={auc:.4f}  IC95=[{lo:.4f}, {hi:.4f}]  p_perm={pv:.4f}")
    try:
        bt = pd.read_parquet(f"{OUTPUT_DIR}/11_backtest.parquet")
        bt["y"] = (bt["ret_1d"] > 0).astype(int)
        per = bt.groupby("ticker").apply(
            lambda g: roc_auc_score(g["y"], g["pred_ens"]) if g["y"].nunique() > 1 else np.nan)
        L.append("per-ticker AUC (ensemble, ret_1d) : "
                 + "  ".join(f"{t}={v:.3f}" for t, v in per.sort_values().items()))
    except Exception as e:
        L.append(f"per-ticker indisponible ({type(e).__name__}: {e})")
    tagfull = f"_{YEAR}_h{H}"
    for f_ in ["09_test.parquet", "10_predictions.parquet", "11_backtest.parquet"]:
        s = f"{OUTPUT_DIR}/{f_}"
        if os.path.exists(s):
            shutil.copy2(s, s.replace(".parquet", tagfull + ".parquet"))
    phase2_tag = "" if (YEAR == 2023 and H == 10) else ("_2022" if (YEAR == 2022 and H == 10) else None)
    if phase2_tag is not None:
        if phase2_tag:
            shutil.copy2(f"{OUTPUT_DIR}/11_backtest.parquet",
                         f"{OUTPUT_DIR}/11_backtest{phase2_tag}.parquet")
        import lightgbm as lgb
        from sklearn.utils.class_weight import compute_sample_weight
        SENT_KEY = ("sent","finbert","sarcasm","fear","anger","joy","sadness","mentions","emoji")
        tech = [c for c in FEATS if not any(k in c for k in SENT_KEY)]
        tr = pd.read_parquet(f"{OUTPUT_DIR}/09_train.parquet")
        va = pd.read_parquet(f"{OUTPUT_DIR}/09_val.parquet")
        trv = pd.concat([tr, va])
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               max_depth=8, subsample=0.8, colsample_bytree=0.7,
                               random_state=42, verbosity=-1)
        m.fit(trv[tech], trv[ycol].astype(int),
              sample_weight=compute_sample_weight("balanced", trv[ycol].astype(int)))
        out = test[["ticker", "date"]].copy()
        out["pred_tech"] = m.predict_proba(test[tech])[:, 1]
        out.to_parquet(f"{OUTPUT_DIR}/proba_technical_only{phase2_tag}.parquet")
        L.append(f"proba_technical_only{phase2_tag}.parquet sauvegardé "
                 f"({len(out)} lignes, {len(tech)} features techniques)")
    txt = "\n".join(L)
    print(txt)
    with open(RES, "a") as fh:
        fh.write(txt + "\n")
except Exception:
    traceback.print_exc()



RUN  test_year=2023  horizon=1j   n_test=1750  pos_rate=0.549
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.5491
model_lgb              AUC=0.4970  IC95=[0.4705, 0.5250]  p_perm=0.8104
model_stack            (skip: AttributeError)
model_xgb              AUC=0.5141  IC95=[0.4869, 0.5411]  p_perm=0.2994
per-ticker AUC (ensemble, ret_1d) : AAPL=0.370  TSLA=0.385  MSFT=0.412  NVDA=0.436  AMZN=0.455  GOOGL=0.456  META=0.456


## ▶ RUN test 2023 — horizon 20 j


In [31]:
# ═══ BASCULE CONFIG — test 2023, horizon 20 j ═══
import pandas as pd, glob, os
TRAIN_END = pd.Timestamp("2021-12-31")
VAL_END   = pd.Timestamp("2022-12-31")
TEST_END  = pd.Timestamp("2023-12-31")
HORIZON   = 20
try:
    CONFIG["horizon"] = HORIZON
except Exception:
    pass
KEEP = ("_2022", "_2023", "_h1.", "_h10.", "_h20.")
for pat in ["07_", "08_", "09_train", "09_val", "09_test", "10_predictions", "11_backtest"]:
    for f in glob.glob(f"{OUTPUT_DIR}/{pat}*.parquet"):
        b = os.path.basename(f)
        if any(s in b for s in KEEP):
            continue
        os.remove(f); print("rm", b)
for f in glob.glob(f"{MODELS_DIR}/*"):
    os.remove(f)
print(f"✅ CONFIG ACTIVE → train≤{TRAIN_END.date()}  val≤{VAL_END.date()}  test≤{TEST_END.date()}  H={HORIZON}j")


rm 07_merged.parquet
rm 08_features.parquet
rm 09_train.parquet
rm 09_val.parquet
rm 09_test.parquet
rm 11_backtest.parquet
✅ CONFIG ACTIVE → train≤2021-12-31  val≤2022-12-31  test≤2023-12-31  H=20j


## Cell 11 — Merge sentiment ⨉ prix + targets

In [32]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — MERGE SENTIMENT + PRICES + TARGETS                    ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_MERGED, min_rows=10):
    print(f"  ✅ {Path(PATH_MERGED).name} exists - skipping")
    df_merged = ckpt.load(PATH_MERGED)
else:
    df_sent  = ckpt.load(PATH_DAILY)
    df_price = ckpt.load(PATH_TECH)

    df_sent["date"]  = normalize_date_col(df_sent["date"])
    df_price["date"] = normalize_date_col(df_price["date"])
    df_sent["ticker"]  = df_sent["ticker"].astype(str)
    df_price["ticker"] = df_price["ticker"].astype(str)

    print(f"  Sentiment rows : {len(df_sent):,}")
    print(f"  Price    rows  : {len(df_price):,}")

    # Left join: keep every trading day, fill sentiment with neutral if missing
    df = df_price.merge(df_sent, on=["ticker", "date"], how="left")

    # Sentiment columns to fill
    sent_cols = ["sent_mean","sent_std","sent_min","sent_max","mentions",
                 "finbert_mean","finbert_std","sarcasm_mean","sarcasm_max",
                 "fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic",
                 "emoji_mean"]
    for c in sent_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    print(f"  Merged rows    : {len(df):,}")
    days_with_sent = (df["mentions"] > 0).sum()
    print(f"  Trading days WITH sentiment: {days_with_sent:,} "
          f"({100*days_with_sent/len(df):.1f}%)")

    # ── TARGETS ────────────────────────────────────────────────────
    # Forward returns at multiple horizons (per ticker, no cross-leakage)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    for h in CONFIG["horizons"]:
        df[f"close_t+{h}"]     = df.groupby("ticker", observed=True)["close"].shift(-h)
        df[f"return_t+{h}"]    = (df[f"close_t+{h}"] - df["close"]) / df["close"]
        df[f"direction_t+{h}"] = (df[f"return_t+{h}"] > 0).astype(np.int8)
    # Primary target
    _H = globals().get("HORIZON", 1)
    if f"return_t+{_H}" not in df.columns:
        print(f"  ⚠️ horizon {_H} absent de CONFIG[horizons] → repli sur 1")
        _H = 1
    _TARGET = globals().get("TARGET", "direction")
    if _TARGET == "volatility":
        # Rendement journalier (depuis close, robuste au nom de colonne)
        df["_r1"] = df.groupby("ticker", observed=True)["close"].pct_change()
        # Volatilité réalisée FORWARD sur t+1..t+H (= la cible, légitimement future)
        _fwd_sq = None
        for _k in range(1, _H + 1):
            _rk = df.groupby("ticker", observed=True)["_r1"].shift(-_k)
            _fwd_sq = _rk.pow(2) if _fwd_sq is None else _fwd_sq + _rk.pow(2)
        df["_fwd_vol"] = (_fwd_sq / _H) ** 0.5
        # Seuil = médiane GLISSANTE PASSÉE de la vol, fenêtre courte (~1 trimestre)
        # → cible "vol élevée RELATIVE au régime récent" : classes ~50/50 à toute
        # période (évite l effet régime calme/turbulent), et aucune info du futur.
        _VOL_WIN = 63
        _past = df.groupby("ticker", observed=True)["_r1"].transform(
            lambda x: x.rolling(_H, min_periods=max(3, _H // 2)).std())
        df["_thr"] = _past.groupby(df["ticker"], observed=True).transform(
            lambda x: x.rolling(_VOL_WIN, min_periods=20).median())
        df["target_return"]    = df["_fwd_vol"].where(df["_thr"].notna())
        df["target_direction"] = (df["_fwd_vol"] > df["_thr"]).astype(np.int8)
        df = df.drop(columns=["_r1", "_fwd_vol", "_thr"])
        print(f"  🎯 Cible = VOLATILITÉ à +{_H}j (1 = vol future > médiane glissante passée)")
    else:
        df["target_return"]    = df[f"return_t+{_H}"]
        df["target_direction"] = df[f"direction_t+{_H}"]
        print(f"  🎯 Cible = direction à +{_H} jour(s)")
    # Drop rows without target_return (last few rows per ticker)
    df = df[df["target_return"].notna()].copy()

    df = mem.optimize_df(df)
    ckpt.save(df, PATH_MERGED)
    df_merged = df

print(f"\n  Final rows : {len(df_merged):,}")
print(f"  Columns    : {len(df_merged.columns)}")
print(f"\n  Direction balance (t+1):")
print(f"    Up   : {(df_merged['target_direction']==1).sum():,}  "
      f"({100*(df_merged['target_direction']==1).mean():.1f}%)")
print(f"    Down : {(df_merged['target_direction']==0).sum():,}  "
      f"({100*(df_merged['target_direction']==0).mean():.1f}%)")


  📂 loaded 05_daily.parquet (23,258 rows)
  📂 loaded 06_prices_tech.parquet (19,355 rows)
  Sentiment rows : 23,258
  Price    rows  : 19,355
  Merged rows    : 19,355
  Trading days WITH sentiment: 16,506 (85.3%)
  🎯 Cible = direction à +20 jour(s)
  💾 saved 07_merged.parquet (19,215 rows | 6.4 MB)

  Final rows : 19,215
  Columns    : 75

  Direction balance (t+1):
    Up   : 12,149  (63.2%)
    Down : 7,066  (36.8%)


## Cell 12 — Feature engineering + split temporel

In [33]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — FEATURE ENGINEERING + TEMPORAL SPLIT                  ║
# ╚══════════════════════════════════════════════════════════════════╝
if all(ckpt.exists_and_valid(p) for p in (PATH_FEATURES, PATH_TRAIN, PATH_VAL, PATH_TEST)):
    print(f"  ✅ Feature splits exist - skipping")
    df_feat = ckpt.load(PATH_FEATURES)
else:
    df = ckpt.load(PATH_MERGED)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    print(f"  Input rows : {len(df):,}")

    SENT_COLS = ["sent_mean","sent_std","mentions","finbert_mean","emoji_mean",
                 "sarcasm_mean","fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic"]

    print("\n  Lagging sentiment features (anti-leakage)...")
    # Shift each sentiment column by 1 day per ticker -> "yesterday's sentiment"
    for c in SENT_COLS:
        if c not in df.columns:
            continue   # robustesse : colonne absente (vieux checkpoint)
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # Rolling sentiment features (shift + rolling BOTH inside groupby to avoid
    # cross-ticker leakage at boundaries — transform() preserves original index).
    print("  Building sentiment rolling features...")
    for w in (3, 7, 14, 30):
        df[f"sent_ma_{w}d"] = (
            df.groupby("ticker", observed=True)["sent_mean"]
              .transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean())
        ).fillna(0)

    # Sentiment momentum (lagged)
    df["sent_momentum"] = (df["sent_ma_3d"] - df["sent_ma_14d"]).fillna(0)

    # Sarcasm-adjusted sentiment (lagged)
    df["sent_adj_lag1"] = (
        df["sent_mean_lag1"] * (1 - 0.5 * df["sarcasm_mean_lag1"])
    ).fillna(0).astype(np.float32)

    # Calendar features
    df["dow"]        = df["date"].dt.dayofweek.astype(np.int8)
    df["month"]      = df["date"].dt.month.astype(np.int8)
    df["quarter"]    = df["date"].dt.quarter.astype(np.int8)
    df["is_monday"]  = (df["dow"] == 0).astype(np.int8)
    df["is_friday"]  = (df["dow"] == 4).astype(np.int8)

    # Ticker as integer
    stock_map = {s: i for i, s in enumerate(sorted(STOCKS))}
    df["stock_id"] = df["ticker"].map(stock_map).astype(np.int8)

    # Fill NaN from lags (first row per ticker)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # Drop rows where target is missing
    df = df.dropna(subset=["target_return", "target_direction"]).copy()

    # ── MARKET-WIDE SENTIMENT FEATURES (cross-ticker) ─────────────
    # On any given day, what was the AVERAGE sentiment across ALL tickers?
    # This captures macro mood that no single ticker can show. We then
    # compute each ticker's DEVIATION from the market mood = their unique
    # signal.
    print("\n  Building market-wide cross-ticker features...")
    market_daily = (df.groupby("date", as_index=False)
                      .agg(market_sent_mean = ("sent_mean",        "mean"),
                           market_share_neg = ("share_negative",   "mean"),
                           market_share_pos = ("share_positive",   "mean"),
                           market_share_sarc= ("share_sarcastic",  "mean"),
                           market_mentions  = ("mentions",         "sum")))
    df = df.merge(market_daily, on="date", how="left")

    # Per-ticker deviation from market mean (negative → ticker more
    # pessimistic than the market, positive → more optimistic)
    df["sent_vs_market"] = (df["sent_mean"] - df["market_sent_mean"]).astype(np.float32)

    # Lag market features by 1 day (anti-leakage like the rest)
    for c in ("market_sent_mean", "market_share_neg", "market_share_pos",
              "market_share_sarc", "sent_vs_market"):
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # ── FEATURE LIST ──────────────────────────────────────────────
    FEATURE_COLS = [
        # Technical
        "ret_1d","ret_3d","ret_5d","ret_10d","ret_20d",
        "vol_5d","vol_20d", "rsi_14",
        "macd","macd_signal","macd_hist",
        "bb_width","bb_pos","atr_14",
        "price_vs_sma20","price_vs_sma50",
        "vol_ratio","hl_range",
        # Sentiment (all lagged or rolling - no same-day)
        "sent_mean_lag1","sent_std_lag1","mentions_lag1",
        "finbert_mean_lag1","sarcasm_mean_lag1",
        "fear_mean_lag1","anger_mean_lag1",
        "joy_mean_lag1","sadness_mean_lag1",
        # 4-class category proportions (lagged) — captures whether
        # the previous day's chatter was bullish, bearish, neutral or ironic
        "share_positive_lag1","share_negative_lag1",
        "share_neutral_lag1","share_sarcastic_lag1",
    "emoji_mean_lag1",
        # Market-wide cross-ticker features (NEW)
        "market_sent_mean_lag1","market_share_neg_lag1",
        "market_share_pos_lag1","market_share_sarc_lag1",
        "sent_vs_market_lag1",
        # Rolling sentiment trends
        "sent_ma_3d","sent_ma_7d","sent_ma_14d","sent_ma_30d",
        "sent_momentum","sent_adj_lag1",
        # Calendar
        "dow","month","quarter","is_monday","is_friday",
        # Identifier
        "stock_id",
    ]
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    print(f"\n  Features  : {len(FEATURE_COLS)}")

    # Save list
    with open(f"{MODELS_DIR}/feature_names.json", "w") as f:
        json.dump({"features": FEATURE_COLS}, f, indent=2)

    # ── TEMPORAL SPLIT ────────────────────────────────────────────
    train = df[df["date"] <= TRAIN_END].copy()
    val   = df[(df["date"] > TRAIN_END) & (df["date"] <= VAL_END)].copy()
    test  = df[(df["date"] > VAL_END) & (df["date"] <= TEST_END)].copy()

    print(f"\n  Train : {len(train):>7,} rows   "
          f"({train['date'].min().date()} -> {train['date'].max().date()})")
    print(f"  Val   : {len(val):>7,} rows   "
          f"({val['date'].min().date()} -> {val['date'].max().date()})")
    print(f"  Test  : {len(test):>7,} rows   "
          f"({test['date'].min().date()} -> {test['date'].max().date()})")

    # Save everything
    df = mem.optimize_df(df)
    ckpt.save(df,    PATH_FEATURES)
    ckpt.save(train, PATH_TRAIN)
    ckpt.save(val,   PATH_VAL)
    ckpt.save(test,  PATH_TEST)

    # ── Liberation disque : les parquets ligne-par-ligne (raw/clean/finbert/sent)
    #    ne sont plus lus en aval (tout passe par les agregats journaliers). ──
    if globals().get("FREE_ROWLEVEL_AFTER_FEATURES", True):
        import os as _os
        # Sauvegarde d un petit résumé "volume par source" pour la Fig 9
        # (sinon perdu quand on supprime les parquets ligne-par-ligne).
        try:
            _srcp = globals().get("PATH_SENT", "") or PATH_CLEAN
            if _srcp and _os.path.exists(_srcp):
                _sc = pd.read_parquet(_srcp, columns=["source", "dtype"])
                _scg = _sc.groupby(["source", "dtype"]).size().reset_index(name="n")
                _scg.to_parquet(f"{OUTPUT_DIR}/source_counts.parquet", index=False)
                del _sc, _scg
        except Exception as _e:
            print(f"  (résumé sources non sauvé: {str(_e)[:40]})")
        for _p in (PATH_RAW, PATH_CLEAN, PATH_FB_OUT, globals().get("PATH_SENT", "")):
            try:
                if _p and _os.path.exists(_p):
                    _sz = _os.path.getsize(_p) / 1e9
                    _os.remove(_p)
                    print(f"  🧹 {Path(_p).name} supprimé (~{_sz:.2f} GB libérés)")
            except Exception: pass

    df_feat = df

print(f"\n  Features file : {Path(PATH_FEATURES).name} ({len(df_feat):,} rows)")


# ════════════════════════════════════════════════════════════════════
#  AUDIT ANTI-FUITE (leakage) — pour la section méthodo du papier
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 64)
print("  AUDIT ANTI-FUITE (look-ahead leakage)")
print("═" * 64)
import json as _json
_audit_ok = True
try:
    with open(f"{MODELS_DIR}/feature_names.json") as _f:
        _FC = _json.load(_f)["features"]
except Exception:
    _FC = FEATURE_COLS

# (a) Aucune colonne dérivée du futur/cible ne doit être une feature
_forbidden = ("target_", "return_t+", "close_t+", "direction_t+")
_leaky = [c for c in _FC if any(k in c for k in _forbidden)]
if _leaky:
    _audit_ok = False
    print(f"  ❌ Features suspectes (dérivées de la cible) : {_leaky}")
else:
    print(f"  ✅ Aucune feature dérivée de la cible ({len(_FC)} features)")

# (b) Intégrité temporelle du split (train < val < test)
try:
    _tr = ckpt.load(PATH_TRAIN); _va = ckpt.load(PATH_VAL); _te = ckpt.load(PATH_TEST)
    _trmax, _vamin = _tr["date"].max(), _va["date"].min()
    _vamax, _temin = _va["date"].max(), _te["date"].min()
    if _trmax < _vamin and _vamax < _temin:
        print(f"  ✅ Split chronologique strict : "
              f"train≤{_trmax.date()} < val≤{_vamax.date()} < test≤{_te['date'].max().date()}")
    else:
        _audit_ok = False
        print(f"  ❌ Chevauchement temporel : trmax={_trmax}, vamin={_vamin}, "
              f"vamax={_vamax}, temin={_temin}")

    # (c) Corrélation feature↔cible sur le TRAIN : flag |r|>0.30 (signe de fuite)
    import numpy as _np
    _y = _tr["target_direction"].astype(float).values
    _susp = []
    for _c in _FC:
        try:
            _x = _tr[_c].astype(float).values
            if _np.nanstd(_x) == 0: continue
            _r = _np.corrcoef(_np.nan_to_num(_x), _y)[0, 1]
            _voltgt = globals().get("TARGET", "direction") == "volatility"
            _volfam = ("vol_", "atr", "bb_", "rsi", "macd", "ret_", "return_", "hl_range")
            # En cible "volatility", les features de vol/momentum SONT légitimement
            # prédictives (la vol se regroupe) → ce n est pas une fuite.
            if abs(_r) > 0.30 and not (_voltgt and any(_w in _c for _w in _volfam)):
                _susp.append((_c, round(float(_r), 3)))
        except Exception:
            pass
    if _susp:
        _audit_ok = False
        print(f"  ⚠️ Features très corrélées à la cible (|r|>0.30) — à vérifier :")
        for _c, _r in sorted(_susp, key=lambda t: -abs(t[1]))[:10]:
            print(f"       {_c:<28} r={_r:+.3f}")
    else:
        print(f"  ✅ Aucune feature anormalement corrélée à la cible (|r|≤0.30)")
    del _tr, _va, _te
except Exception as _e:
    print(f"  (audit partiel : {str(_e)[:60]})")

print("─" * 64)
print(f"  VERDICT : {'✅ PAS DE FUITE DÉTECTÉE' if _audit_ok else '⚠️ POINTS À VÉRIFIER (voir ci-dessus)'}")
print("═" * 64)


  📂 loaded 07_merged.parquet (19,215 rows)
  Input rows : 19,215

  Lagging sentiment features (anti-leakage)...
  Building sentiment rolling features...

  Building market-wide cross-ticker features...

  Features  : 49

  Train :  12,341 rows   (2015-01-02 -> 2021-12-31)
  Val   :   1,757 rows   (2022-01-03 -> 2022-12-30)
  Test  :   1,750 rows   (2023-01-03 -> 2023-12-29)
  💾 saved 08_features.parquet (19,215 rows | 8.1 MB)
  💾 saved 09_train.parquet (12,341 rows | 5.7 MB)
  💾 saved 09_val.parquet (1,757 rows | 0.8 MB)
  💾 saved 09_test.parquet (1,750 rows | 0.8 MB)

  Features file : 08_features.parquet (19,215 rows)

════════════════════════════════════════════════════════════════
  AUDIT ANTI-FUITE (look-ahead leakage)
════════════════════════════════════════════════════════════════
  ✅ Aucune feature dérivée de la cible (49 features)
  📂 loaded 09_train.parquet (12,341 rows)
  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  ✅ Split chronologique st

## Cell 13 — Arbres : LightGBM + XGBoost + CatBoost

In [34]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — TRAIN MODELS                                          ║
# ╚══════════════════════════════════════════════════════════════════╝
import time

# Lazy installs
for pkg in ("lightgbm", "xgboost", "catboost"):
    try:
        __import__(pkg)
    except ImportError:
        pip_install(pkg)

import lightgbm as lgb
import xgboost  as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import joblib

# Load
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test  = ckpt.load(PATH_TEST)

# CORRECTION ANTI-FUITE : tri chronologique GLOBAL avant TimeSeriesSplit.
# (les fichiers sont tries [ticker, date] → la CV temporelle melangerait
#  les tickers et casserait l'ordre du temps.)
train = train.sort_values("date").reset_index(drop=True)
val   = val.sort_values("date").reset_index(drop=True)
test  = test.sort_values("date").reset_index(drop=True)

X_tr, y_tr = train[FEATURE_COLS].values, train["target_direction"].values.astype(int)
X_va, y_va = val[FEATURE_COLS].values,   val["target_direction"].values.astype(int)
X_te, y_te = test[FEATURE_COLS].values,  test["target_direction"].values.astype(int)

print(f"  Train : {X_tr.shape}  |  pos rate = {y_tr.mean():.3f}")
print(f"  Val   : {X_va.shape}  |  pos rate = {y_va.mean():.3f}")
print(f"  Test  : {X_te.shape}  |  pos rate = {y_te.mean():.3f}")

sw_tr = compute_sample_weight("balanced", y_tr)

# ── 1. LIGHTGBM ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  LightGBM")
print("─" * 60)
lgb_params = dict(
    objective="binary", metric="auc",
    n_estimators=800, learning_rate=0.03,
    num_leaves=63, max_depth=8,
    min_child_samples=40,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1,
)
tscv = TimeSeriesSplit(n_splits=5)
cv_lgb = []
clf_lgb = lgb.LGBMClassifier(**lgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_lgb.fit(
        X_tr[tr_i], y_tr[tr_i],
        sample_weight=sw_tr[tr_i],
        eval_set=[(X_tr[val_i], y_tr[val_i])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    auc = roc_auc_score(y_tr[val_i], clf_lgb.predict_proba(X_tr[val_i])[:, 1])
    cv_lgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_lgb):.4f} ± {np.std(cv_lgb):.4f}")
# Final fit
clf_lgb.fit(X_tr, y_tr, sample_weight=sw_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)])
joblib.dump(clf_lgb, f"{MODELS_DIR}/model_lgb.joblib")
auc_lgb_val = roc_auc_score(y_va, clf_lgb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_lgb_val:.4f}")

# ── 2. XGBOOST ─────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  XGBoost")
print("─" * 60)
neg, pos = int((y_tr == 0).sum()), int((y_tr == 1).sum())
spw = neg / max(pos, 1)
xgb_params = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=600, learning_rate=0.05,
    max_depth=7, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw,
    random_state=42, n_jobs=-1, verbosity=0,
    early_stopping_rounds=50,
)
cv_xgb = []
clf_xgb = xgb.XGBClassifier(**xgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_xgb.fit(X_tr[tr_i], y_tr[tr_i],
                eval_set=[(X_tr[val_i], y_tr[val_i])],
                verbose=False)
    auc = roc_auc_score(y_tr[val_i], clf_xgb.predict_proba(X_tr[val_i])[:, 1])
    cv_xgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_xgb):.4f} ± {np.std(cv_xgb):.4f}")
# Final fit (without early-stopping callback now)
xgb_params_final = {k: v for k, v in xgb_params.items() if k != "early_stopping_rounds"}
clf_xgb_final = xgb.XGBClassifier(**xgb_params_final)
clf_xgb_final.fit(X_tr, y_tr)
joblib.dump(clf_xgb_final, f"{MODELS_DIR}/model_xgb.joblib")
auc_xgb_val = roc_auc_score(y_va, clf_xgb_final.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_xgb_val:.4f}")

# ── 3. CATBOOST ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  CatBoost")
print("─" * 60)
clf_cb = CatBoostClassifier(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=42, verbose=0,
    early_stopping_rounds=50,
)
clf_cb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=(X_va, y_va), use_best_model=True)
clf_cb.save_model(f"{MODELS_DIR}/model_cb.cbm")
auc_cb_val = roc_auc_score(y_va, clf_cb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_cb_val:.4f}")
print(f"  Best iter     = {clf_cb.best_iteration_}")

# Save model summary
model_summary = {
    "lgb": {"cv_auc_mean": float(np.mean(cv_lgb)),
            "cv_auc_std":  float(np.std(cv_lgb)),
            "val_auc":     float(auc_lgb_val)},
    "xgb": {"cv_auc_mean": float(np.mean(cv_xgb)),
            "cv_auc_std":  float(np.std(cv_xgb)),
            "val_auc":     float(auc_xgb_val)},
    "cb":  {"val_auc":     float(auc_cb_val),
            "best_iter":   int(clf_cb.best_iteration_)},
}
with open(f"{MODELS_DIR}/training_summary.json", "w") as f:
    json.dump(model_summary, f, indent=2)

print("\n" + "=" * 60)
print(f"  ✅ 3 models trained and saved to {MODELS_DIR}")
print("=" * 60)
mem.free()


  📂 loaded 09_train.parquet (12,341 rows)
  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  Train : (12341, 49)  |  pos rate = 0.658
  Val   : (1757, 49)  |  pos rate = 0.372
  Test  : (1750, 49)  |  pos rate = 0.733

────────────────────────────────────────────────────────────
  LightGBM
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.5117
  fold 2/5  AUC=0.5125
  fold 3/5  AUC=0.4806
  fold 4/5  AUC=0.4576
  fold 5/5  AUC=0.5535
  CV AUC mean   = 0.5032 ± 0.0325
  Val AUC       = 0.6024

────────────────────────────────────────────────────────────
  XGBoost
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.4767
  fold 2/5  AUC=0.5112
  fold 3/5  AUC=0.4427
  fold 4/5  AUC=0.4950
  fold 5/5  AUC=0.5565
  CV AUC mean   = 0.4964 ± 0.0377
  Val AUC       = 0.6067

────────────────────────────────────────────────────────────
  CatBoost
────────────────────────────────────────────────────────────
  V

## Cell 13b — Modele sequentiel hybride (BiGRU + attention)  ⭐ NOUVEAU

Apporte la composante « deep » de l'ensemble hybride. Pour chaque (ticker, jour) on construit une
**fenetre glissante de `SEQ_LEN` jours** du vecteur de features (sentiment lagge + techniques), normalisee
sur le **train uniquement** (anti-fuite). Un **BiGRU bidirectionnel + attention** apprend les dynamiques
temporelles que les arbres ne capturent pas. Les probabilites val/test sont sauvegardees pour le stacking.
GPU recommande ; degrade proprement en CPU (ou se desactive si torch absent).

In [35]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13b — BiGRU + ATTENTION (modele sequentiel)                ║
# ║  Sortie : {OUTPUT_DIR}/seq_proba.parquet  [ticker,date,p_seq,split]║
# ╚══════════════════════════════════════════════════════════════════╝
import json, time
import numpy as np
import pandas as pd
from pathlib import Path

SEQ_OUT = f"{OUTPUT_DIR}/seq_proba.parquet"

if not USE_SEQ_MODEL:
    print("  USE_SEQ_MODEL=False → modele sequentiel desactive.")
elif Path(SEQ_OUT).exists():
    print(f"  ✅ {Path(SEQ_OUT).name} existe — skip BiGRU")
else:
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        pip_install("torch"); import torch; import torch.nn as nn

    from sklearn.metrics import roc_auc_score

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device : {device}")

    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    df = ckpt.load(PATH_FEATURES)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # ── Standardisation (stats du TRAIN uniquement) ──────────────────
    train_mask = df["date"] <= TRAIN_END
    mu = df.loc[train_mask, FEATURE_COLS].mean()
    sd = df.loc[train_mask, FEATURE_COLS].std().replace(0, 1.0)
    Xall = ((df[FEATURE_COLS] - mu) / sd).fillna(0.0).values.astype("float32")
    y    = df["target_direction"].values.astype("float32")
    tks  = df["ticker"].values
    dts  = df["date"].values
    n_feat = Xall.shape[1]
    L = int(SEQ_LEN)

    # ── Construction des fenetres glissantes par ticker ──────────────
    print(f"  Construction des fenetres (L={L}, n_feat={n_feat})...")
    W, yy, wt, wd = [], [], [], []
    for tk in sorted(set(tks)):
        idx = np.where(tks == tk)[0]
        Xi, yi, di = Xall[idx], y[idx], dts[idx]
        for j in range(len(idx)):
            lo = max(0, j - L + 1)
            w = Xi[lo:j + 1]
            if len(w) < L:                       # padding par la 1ere ligne
                w = np.vstack([np.repeat(w[:1], L - len(w), axis=0), w])
            W.append(w); yy.append(yi[j]); wt.append(tk); wd.append(di[j])
    W  = np.asarray(W, dtype="float32")
    yy = np.asarray(yy, dtype="float32")
    wd = pd.to_datetime(pd.Series(wd))
    print(f"  Fenetres : {W.shape}")

    tr_m = (wd <= TRAIN_END).values
    va_m = ((wd > TRAIN_END) & (wd <= VAL_END)).values
    te_m = ((wd > VAL_END) & (wd <= TEST_END)).values

    # Borne memoire : sous-echantillonne le train si gigantesque (CPU)
    MAX_TR = 400_000 if device.type == "cuda" else 120_000
    tr_idx = np.where(tr_m)[0]
    if len(tr_idx) > MAX_TR:
        rng = np.random.default_rng(42)
        tr_idx = rng.choice(tr_idx, MAX_TR, replace=False)
    va_idx = np.where(va_m)[0]
    te_idx = np.where(te_m)[0]
    print(f"  train={len(tr_idx):,}  val={len(va_idx):,}  test={len(te_idx):,}")

    # ── Modele : BiGRU bidirectionnel + attention additive ───────────
    class BiGRUAttn(nn.Module):
        def __init__(self, n_feat, hidden=64, layers=1, p=0.3):
            super().__init__()
            self.gru = nn.GRU(n_feat, hidden, num_layers=layers,
                              batch_first=True, bidirectional=True,
                              dropout=p if layers > 1 else 0.0)
            self.attn = nn.Linear(hidden * 2, 1)
            self.head = nn.Sequential(
                nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(p),
                nn.Linear(64, 1))
        def forward(self, x):
            h, _ = self.gru(x)                       # (B, L, 2H)
            a = torch.softmax(self.attn(h).squeeze(-1), dim=1)  # (B, L)
            ctx = (h * a.unsqueeze(-1)).sum(dim=1)   # (B, 2H)
            return self.head(ctx).squeeze(-1)

    model = BiGRUAttn(n_feat).to(device)
    pos_w = float((yy[tr_idx] == 0).sum()) / max(float((yy[tr_idx] == 1).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    Wt = torch.from_numpy(W)   # garde en CPU, on transfere par batch

    def run_eval(idx):
        model.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(idx), SEQ_BATCH):
                b = idx[i:i + SEQ_BATCH]
                xb = Wt[b].to(device)
                outs.append(torch.sigmoid(model(xb)).cpu().numpy())
        return np.concatenate(outs) if outs else np.array([])

    best_auc, best_state, patience, bad = -1.0, None, 5, 0
    print("  Entrainement BiGRU...")
    for ep in range(1, int(SEQ_EPOCHS) + 1):
        model.train()
        rng = np.random.default_rng(ep)
        perm = rng.permutation(tr_idx)
        t0 = time.time()
        for i in range(0, len(perm), SEQ_BATCH):
            b = perm[i:i + SEQ_BATCH]
            xb = Wt[b].to(device)
            yb = torch.from_numpy(yy[b]).to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        p_va = run_eval(va_idx)
        auc = roc_auc_score(yy[va_idx], p_va) if len(set(yy[va_idx])) > 1 else 0.5
        print(f"    epoch {ep:>2}  val_AUC={auc:.4f}  ({time.time()-t0:.1f}s)")
        if auc > best_auc:
            best_auc, best_state, bad = auc, {k: v.cpu().clone()
                                              for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print("    early stopping"); break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  ✅ meilleur val_AUC = {best_auc:.4f}")

    # ── Probabilites val + test → parquet pour le stacking ───────────
    out = pd.DataFrame({
        "ticker": np.r_[np.array(wt)[va_idx], np.array(wt)[te_idx]],
        "date":   pd.to_datetime(np.r_[wd.values[va_idx], wd.values[te_idx]]),
        "p_seq":  np.r_[run_eval(va_idx), run_eval(te_idx)].astype("float32"),
        "split":  (["val"] * len(va_idx)) + (["test"] * len(te_idx)),
    })
    out["date"] = out["date"].dt.normalize()
    out.to_parquet(SEQ_OUT, index=False)
    torch.save(model.state_dict(), f"{MODELS_DIR}/model_seq.pt")
    with open(f"{MODELS_DIR}/seq_summary.json", "w") as f:
        json.dump({"val_auc": float(best_auc), "seq_len": L,
                   "n_feat": int(n_feat)}, f, indent=2)
    print(f"  💾 {Path(SEQ_OUT).name}  ({len(out):,} rows)  + model_seq.pt")
    mem.free()


  ✅ seq_proba.parquet existe — skip BiGRU


## Cell 13c — Stacking hybride + calibration  ⭐ NOUVEAU

Combine les **4 modeles de base** (LightGBM, XGBoost, CatBoost, BiGRU) via un **meta-modele logistique**
ajuste sur la validation, puis applique une **calibration isotone** (probabilites mieux calibrees → meilleures
decisions de trading). Produit `stack_proba.parquet` (proba finale alignee sur les lignes de test) que la
CELL 14 utilise automatiquement.

In [36]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13c — STACKING (meta-logistique) + CALIBRATION ISOTONE     ║
# ╚══════════════════════════════════════════════════════════════════╝
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from catboost import CatBoostClassifier

if not USE_STACKING:
    print("  USE_STACKING=False → stacking desactive (CELL 14 utilisera la moyenne fixe).")
else:
    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    val  = ckpt.load(PATH_VAL).sort_values(["ticker", "date"]).reset_index(drop=True)
    test = ckpt.load(PATH_TEST).sort_values(["ticker", "date"]).reset_index(drop=True)
    for d in (val, test):
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()

    clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
    clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
    clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

    def tree_probas(d):
        X = d[FEATURE_COLS].values
        return np.column_stack([
            clf_lgb.predict_proba(X)[:, 1],
            clf_xgb.predict_proba(X)[:, 1],
            clf_cb.predict_proba(X)[:, 1],
        ])

    Pva, Pte = tree_probas(val), tree_probas(test)
    base_names = ["lgb", "xgb", "cb"]

    # ── Ajout du BiGRU si disponible ─────────────────────────────────
    seqp = f"{OUTPUT_DIR}/seq_proba.parquet"
    if Path(seqp).exists():
        sp = pd.read_parquet(seqp)
        sp["date"] = pd.to_datetime(sp["date"]).dt.normalize()
        def add_seq(d, P):
            m = d[["ticker", "date"]].merge(
                sp[["ticker", "date", "p_seq"]], on=["ticker", "date"], how="left")
            return np.column_stack([P, m["p_seq"].fillna(0.5).values])
        Pva, Pte = add_seq(val, Pva), add_seq(test, Pte)
        base_names.append("seq")
        print("  ✅ BiGRU integre au stacking")
    else:
        print("  ℹ️  Pas de seq_proba.parquet — stacking sur les 3 arbres seulement")

    yva = val["target_direction"].astype(int).values
    yte = test["target_direction"].astype(int).values

    # ── Meta-modele logistique (ajuste sur la validation) ────────────
    meta = LogisticRegression(max_iter=2000, C=1.0)
    meta.fit(Pva, yva)
    raw_va = meta.predict_proba(Pva)[:, 1]
    raw_te = meta.predict_proba(Pte)[:, 1]

    # ── Calibration isotone (sur la validation) ──────────────────────
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va, yva)
    p_stack    = iso.transform(raw_te).astype("float32")
    p_stack_va = iso.transform(raw_va).astype("float32")
    # Seuil de décision calibré sur la VALIDATION (quantile = taux de hausse val).
    # Évite l effondrement de l ensemble (sinon il prédit presque tout "baisse").
    _thr_stack = float(np.quantile(p_stack_va, 1.0 - float(np.mean(yva))))
    print(f"  Seuil ensemble calibré (val) : {_thr_stack:.4f}")

    # ── Rapport AUC base vs stack ────────────────────────────────────
    print("\n  AUC (test) par modele de base :")
    for i, nm in enumerate(base_names):
        print(f"    {nm:<5} {roc_auc_score(yte, Pte[:, i]):.4f}")
    print(f"    {'STACK':<5} {roc_auc_score(yte, p_stack):.4f}  "
          f"(acc={accuracy_score(yte, (p_stack>0.5).astype(int)):.4f})")
    print(f"  Poids meta (logistique) : "
          f"{dict(zip(base_names, np.round(meta.coef_[0], 3)))}")

    # ── Sauvegardes ──────────────────────────────────────────────────
    out = test[["ticker", "date"]].copy()
    out["p_stack"] = p_stack
    out.to_parquet(f"{OUTPUT_DIR}/stack_proba.parquet", index=False)
    joblib.dump({"meta": meta, "iso": iso, "names": base_names},
                f"{MODELS_DIR}/model_stack.joblib")
    with open(f"{MODELS_DIR}/stack_summary.json", "w") as f:
        json.dump({"base_names": base_names,
                   "weights": dict(zip(base_names, meta.coef_[0].tolist())),
                   "stack_test_auc": float(roc_auc_score(yte, p_stack)),
                   "threshold": _thr_stack}, f, indent=2)
    with open(f"{MODELS_DIR}/stack_threshold.json", "w") as f:
        json.dump({"threshold": _thr_stack,
                   "val_pos_rate": float(np.mean(yva))}, f)
    print(f"  💾 stack_proba.parquet + model_stack.joblib")


  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  ✅ BiGRU integre au stacking
  Seuil ensemble calibré (val) : 0.4286

  AUC (test) par modele de base :
    lgb   0.6113
    xgb   0.6207
    cb    0.6003
    seq   0.5380
    STACK 0.6213  (acc=0.3823)
  Poids meta (logistique) : {'lgb': np.float64(0.709), 'xgb': np.float64(1.24), 'cb': np.float64(-1.012), 'seq': np.float64(4.366)}
  💾 stack_proba.parquet + model_stack.joblib


## Cell 14 — Ensemble (stacking hybride), evaluation & backtest

In [37]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — ENSEMBLE + EVAL + BACKTEST                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import joblib
from catboost import CatBoostClassifier

with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

test = ckpt.load(PATH_TEST)
X_te = test[FEATURE_COLS].values
y_te = test["target_direction"].values.astype(int)

# Load models
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

# Probabilities
p_lgb = clf_lgb.predict_proba(X_te)[:, 1]
p_xgb = clf_xgb.predict_proba(X_te)[:, 1]
p_cb  = clf_cb.predict_proba(X_te)[:, 1]

# ── Ensemble HYBRIDE : on privilegie la pile apprise (CELL 13c :
#    arbres + BiGRU + meta-logistique calibree) si elle existe, sinon
#    on retombe sur la moyenne ponderee fixe.
from pathlib import Path as _P
_stack_path = f"{OUTPUT_DIR}/stack_proba.parquet"
_p_fixed = 0.4 * p_lgb + 0.3 * p_xgb + 0.3 * p_cb
if _P(_stack_path).exists():
    _sp = pd.read_parquet(_stack_path)
    _sp["date"] = pd.to_datetime(_sp["date"]).dt.normalize()
    _key = test[["ticker", "date"]].copy()
    _key["date"] = pd.to_datetime(_key["date"]).dt.normalize()
    _m = _key.merge(_sp[["ticker", "date", "p_stack"]], on=["ticker", "date"], how="left")
    p_ens = _m["p_stack"].fillna(pd.Series(_p_fixed)).values
    print("  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)")
else:
    p_ens = _p_fixed
    print("  ℹ️  Ensemble = moyenne ponderee fixe (stacking absent)")

# Seuil ensemble par CALAGE DE TAUX : on aligne la proportion de "hausse"
# prédite sur le taux de hausse de la VALIDATION (a priori connu, pas de
# fuite des labels test). Robuste au fort déséquilibre des horizons longs.
try:
    with open(f"{MODELS_DIR}/stack_threshold.json") as _f:
        _vpr = float(json.load(_f).get("val_pos_rate", 0.5))
    _THR_ENS = float(np.quantile(p_ens, 1.0 - _vpr))
except Exception:
    _THR_ENS = 0.5

# Per-model metrics (seuil par modèle : ensemble = seuil calibré)
def metrics_row(name, p, thr=0.5):
    yp = (p > thr).astype(int)
    return {
        "name":      name,
        "accuracy":  float(accuracy_score(y_te, yp)),
        "auc":       float(roc_auc_score(y_te, p)),
        "precision": float(precision_score(y_te, yp, zero_division=0)),
        "recall":    float(recall_score(y_te, yp, zero_division=0)),
        "f1":        float(f1_score(y_te, yp, zero_division=0)),
    }
rows = [metrics_row("lightgbm", p_lgb), metrics_row("xgboost", p_xgb),
        metrics_row("catboost", p_cb)]
# Ensemble (stacking) exclu : dégénère sous fort déséquilibre et n'apporte
# rien (AUC ≈ modèles de base). Modèles retenus : LightGBM/XGBoost/CatBoost.
results = pd.DataFrame(rows).set_index("name").round(4)

print("=" * 60)
print("  TEST SET METRICS")
print("=" * 60)
print(results.to_string())

# ── BACKTEST ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BACKTEST  (threshold 0.55 long / 0.45 short)")
print("=" * 60)
_is_vol = globals().get("TARGET", "direction") == "volatility"
if _is_vol:
    print("  ⚠️ Cible = VOLATILITÉ : ce backtest directionnel (long/short) n a PAS")
    print("     de sens (on prédit l amplitude, pas le sens du mouvement).")
    print("     → À IGNORER pour le papier. Utiliser AUC + ablation à la place.")
    print("     (Calculé seulement pour ne pas casser la figure du dashboard.)")
bt = test[["ticker", "date", "close"]].copy()
bt["pred_ens"] = p_lgb   # backtest sur LightGBM (modèle principal)
bt = bt.sort_values(["ticker", "date"]).reset_index(drop=True)
# Rendement RÉALISÉ à 1 jour (close-to-close). On n'utilise JAMAIS le target
# à H jours pour composer : sinon on cumulerait un rendement H-jours chaque
# jour (fenêtres chevauchantes) → explosion irréaliste du cumul.
bt["ret_1d"] = bt.groupby("ticker", observed=True)["close"].pct_change().fillna(0.0)
bt["signal"]   = 0
bt.loc[bt["pred_ens"] > 0.55, "signal"] = 1
bt.loc[bt["pred_ens"] < 0.45, "signal"] = -1

# Strategy return = signal de la veille × rendement réalisé du jour
bt["signal_lag"]      = bt.groupby("ticker", observed=True)["signal"].shift(1).fillna(0)
bt["strategy_return"] = bt["signal_lag"] * bt["ret_1d"]

# ── Transaction costs (10 bps per round-trip = realistic retail broker)
TXN_COST_BPS = 10
bt["pos_change"]      = (bt["signal"] != bt["signal_lag"]).astype(np.int8)
bt["txn_cost"]        = bt["pos_change"] * (TXN_COST_BPS / 10000.0)
bt["strategy_net"]    = bt["strategy_return"] - bt["txn_cost"]

bt["cum_strategy"]    = bt.groupby("ticker", observed=True)["strategy_net"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())
bt["cum_market"]      = bt.groupby("ticker", observed=True)["ret_1d"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())

# Per-ticker summary
backtest_summary = {}
for tk, g in bt.groupby("ticker", observed=True):
    ret_s   = (g["cum_strategy"].iloc[-1] - 1) * 100
    ret_m   = (g["cum_market"].iloc[-1]   - 1) * 100
    sr_std  = g["strategy_net"].std()
    sharpe  = (g["strategy_net"].mean() / sr_std * np.sqrt(252)) if sr_std > 0 else 0.0
    n_tr    = int(g["pos_change"].sum())
    act     = g[g["signal"] != 0]["strategy_net"]
    win_rt  = float((act > 0).mean()) if len(act) else 0.0
    txn_drag = float(g["txn_cost"].sum() * 100)
    backtest_summary[tk] = {
        "return_strategy":  round(float(ret_s), 2),
        "return_market":    round(float(ret_m), 2),
        "alpha":            round(float(ret_s - ret_m), 2),
        "sharpe":           round(float(sharpe), 3),
        "n_trades":         n_tr,
        "win_rate":         round(win_rt, 4),
        "txn_cost_drag":    round(txn_drag, 2),     # cumulative cost in %
    }

# Aggregate
total_s = (bt.groupby("date")["strategy_net"].mean().fillna(0) + 1).cumprod().iloc[-1] - 1
total_m = (bt.groupby("date")["ret_1d"].mean().fillna(0)   + 1).cumprod().iloc[-1] - 1
_daily_strat = bt.groupby("date")["strategy_net"].mean().fillna(0)
_eq = (1 + _daily_strat).cumprod()
_max_dd = float((_eq / _eq.cummax() - 1).min())
_sh_overall = (float(_daily_strat.mean() / _daily_strat.std() * np.sqrt(252))
               if _daily_strat.std() > 0 else 0.0)
backtest_summary["__overall__"] = {
    "return_strategy": round(float(total_s * 100), 2),
    "return_market":   round(float(total_m * 100), 2),
    "alpha":           round(float((total_s - total_m) * 100), 2),
    "sharpe":          round(_sh_overall, 3),
    "max_drawdown":    round(_max_dd * 100, 2),
}

print(f"\n  Strategy total : {total_s*100:+.2f}%")
print(f"  Market   total : {total_m*100:+.2f}%")
print(f"  Alpha          : {(total_s - total_m)*100:+.2f}%\n")
print(f"  {'Ticker':<8} {'Strategy':>10} {'Market':>10} {'Alpha':>9} {'Sharpe':>7} {'Trades':>7} {'WinR':>6}")
print("  " + "─" * 60)
for tk in STOCKS:
    if tk not in backtest_summary: continue
    s = backtest_summary[tk]
    print(f"  {tk:<8} {s['return_strategy']:>+9.2f}% {s['return_market']:>+9.2f}% "
          f"{s['alpha']:>+8.2f}% {s['sharpe']:>7.2f} {s['n_trades']:>7d} {s['win_rate']*100:>5.1f}%")

# Save
ckpt.save(bt, PATH_BACKTEST)
results.to_json(f"{MODELS_DIR}/test_metrics.json", indent=2)
with open(f"{MODELS_DIR}/backtest_summary.json", "w") as f:
    json.dump(backtest_summary, f, indent=2)

# Confusion matrix (LightGBM)
print("\n  Confusion matrix (LightGBM):")
yp_ens = (p_lgb > 0.5).astype(int)
cm = confusion_matrix(y_te, yp_ens)
print(f"             pred_down  pred_up")
print(f"  true_down   {cm[0,0]:>7,}  {cm[0,1]:>7,}")
print(f"  true_up     {cm[1,0]:>7,}  {cm[1,1]:>7,}")

mem.free()


# ════════════════════════════════════════════════════════════════════
#  BASELINES + SIGNIFICATIVITÉ STATISTIQUE — pour l'acceptation du papier
# ════════════════════════════════════════════════════════════════════
import numpy as _np
from sklearn.metrics import roc_auc_score as _auc

print("\n" + "═" * 64)
print("  BASELINES & SIGNIFICATIVITÉ (test set)")
print("═" * 64)

# Baselines naïves
_base_up   = float((y_te == 1).mean())          # "toujours hausse"
_base_maj  = max(_base_up, 1 - _base_up)         # classe majoritaire
print(f"  Baseline 'toujours hausse' : acc = {_base_up:.4f}")
print(f"  Baseline classe majoritaire: acc = {_base_maj:.4f}")
print(f"  (un modèle utile doit battre {_base_maj:.4f} de façon significative)")

def _auc_safe(yt, p):
    try: return _auc(yt, p)
    except Exception: return 0.5

# Bootstrap IC 95% + test de permutation pour l'AUC de chaque modèle
_rng = _np.random.default_rng(42)
_N_BOOT, _N_PERM = 1000, 1000
print(f"\n  {'Modèle':<11} {'AUC':>7}  {'IC 95% bootstrap':>22}  {'p (perm)':>9}")
print("  " + "-" * 56)
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    _p = _np.asarray(_p, dtype=float)
    _obs = _auc_safe(y_te, _p)
    # bootstrap
    _bs = []
    _n = len(y_te)
    for _ in range(_N_BOOT):
        _idx = _rng.integers(0, _n, _n)
        if len(_np.unique(y_te[_idx])) < 2: continue
        _bs.append(_auc_safe(y_te[_idx], _p[_idx]))
    _lo, _hi = (_np.percentile(_bs, [2.5, 97.5]) if _bs else (_np.nan, _np.nan))
    # permutation : H0 = pas de lien (on mélange y)
    _null = _np.empty(_N_PERM)
    for _i in range(_N_PERM):
        _null[_i] = _auc_safe(_rng.permutation(y_te), _p)
    _pval = float((_np.sum(_np.abs(_null - 0.5) >= abs(_obs - 0.5)) + 1) / (_N_PERM + 1))
    _star = "***" if _pval < 0.01 else "**" if _pval < 0.05 else "*" if _pval < 0.10 else "ns"
    print(f"  {_name:<11} {_obs:>7.4f}  [{_lo:.4f}, {_hi:.4f}]  {_pval:>8.4f} {_star}")

print("\n  Lecture : un AUC est crédible si son IC 95% exclut 0.50 et p<0.05.")
print("  *** p<0.01  ** p<0.05  * p<0.10  ns = non significatif")
print("═" * 64)


# ════════════════════════════════════════════════════════════════════
#  MODE ABSTENTION — accuracy vs couverture (sélectivité de confiance)
#  Le modèle n'agit que sur ses prédictions les plus confiantes.
#  Courbe publiable : "accuracy croît quand on réduit la couverture".
# ════════════════════════════════════════════════════════════════════
import numpy as _np
print("\n" + "═" * 64)
print("  MODE ABSTENTION — accuracy sur les prédictions les + confiantes")
print("═" * 64)

def _conf_curve(name, p, thr=0.5):
    p = _np.asarray(p, dtype=float)
    conf = _np.abs(p - thr)                 # distance au seuil de décision = confiance
    order = _np.argsort(-conf)              # plus confiant d'abord
    yp_all = (p > thr).astype(int)
    rows = []
    for cov in (1.00, 0.50, 0.30, 0.20, 0.10, 0.05):
        k = max(1, int(round(len(p) * cov)))
        idx = order[:k]
        acc = float((yp_all[idx] == y_te[idx]).mean())
        rows.append((cov, k, acc))
    return rows

print(f"  {'Couverture':>10} {'N trades':>9} {'Accuracy':>9}")
print("  " + "-" * 32)
_abst_out = {}
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    print(f"  ── {_name} ──")
    _thr_a = _THR_ENS if _name == "ensemble" else 0.5
    _rows = _conf_curve(_name, _p, _thr_a)
    _abst_out[_name] = [{"coverage": c, "n": k, "accuracy": a} for c, k, a in _rows]
    for cov, k, acc in _rows:
        print(f"  {cov*100:>9.0f}% {k:>9,} {acc:>9.4f}")

try:
    with open(f"{MODELS_DIR}/abstention_curve.json", "w") as _f:
        json.dump(_abst_out, _f, indent=2)
except Exception:
    pass

print("\n  Lecture : si l'accuracy monte quand la couverture baisse, le modèle")
print("  'sait quand il sait' — argument fort pour le papier (trading sélectif).")
print("  À reporter en figure : accuracy (y) vs couverture (x), par modèle.")
print("═" * 64)


  📂 loaded 09_test.parquet (1,750 rows)
  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)
  TEST SET METRICS
          accuracy     auc  precision  recall      f1
name                                                 
lightgbm    0.5994  0.6113     0.7589  0.6648  0.7088
xgboost     0.6434  0.6207     0.7725  0.7280  0.7496
catboost    0.6149  0.6003     0.7756  0.6680  0.7178

  BACKTEST  (threshold 0.55 long / 0.45 short)

  Strategy total : +63.55%
  Market   total : +112.33%
  Alpha          : -48.78%

  Ticker     Strategy     Market     Alpha  Sharpe  Trades   WinR
  ────────────────────────────────────────────────────────────
  AAPL         +9.06%    +54.80%   -45.74%    0.57      67  46.8%
  MSFT        +14.23%    +58.35%   -44.11%    0.71      51  51.4%
  GOOGL       +86.21%    +56.74%   +29.46%    2.51      73  47.5%
  AMZN        +64.56%    +77.04%   -12.48%    1.83      65  46.2%
  META         +3.00%   +183.76%  -180.76%    0.28      76  41.3%
  TSLA       +239.62% 

## Cell 15 — Analyse causale (Pearson lags + Granger + SHAP + Ablation)

In [38]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — CAUSAL ANALYSIS                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
for pkg in ("statsmodels", "shap"):
    try: __import__(pkg)
    except ImportError: pip_install(pkg)

from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
import shap

df = ckpt.load(PATH_FEATURES)
df["date"] = pd.to_datetime(df["date"])

# ── A. PEARSON LAG CORRELATIONS (per ticker) ──────────────────────
print("=" * 60)
print("  A. PEARSON LAG CORRELATIONS")
print("=" * 60)
lag_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 50: continue
    for lag in range(0, 6):
        s_lagged = ds["sent_mean"].shift(lag).dropna()
        ret_aligned = ds["return_1d"].iloc[lag:].reset_index(drop=True)
        s_lagged    = s_lagged.reset_index(drop=True)
        n = min(len(s_lagged), len(ret_aligned))
        if n < 30:
            r_val, p_val = np.nan, np.nan
        else:
            r_val, p_val = stats.pearsonr(s_lagged[:n], ret_aligned[:n])
        lag_rows.append({
            "ticker":      tk,
            "lag_days":    lag,
            "correlation": round(float(r_val), 4) if not np.isnan(r_val) else None,
            "p_value":     round(float(p_val), 4) if not np.isnan(p_val) else None,
            "significant": (p_val < 0.05) if not np.isnan(p_val) else False,
        })
df_lags = pd.DataFrame(lag_rows)
df_lags.to_csv(f"{MODELS_DIR}/lag_correlations.csv", index=False)
print("\n  Lag-correlation summary:")
print(df_lags.groupby("lag_days")["correlation"].describe()[
      ["mean", "std", "min", "max"]].round(4).to_string())
print(f"\n  Significant lags (p<0.05): {df_lags['significant'].sum()} / {len(df_lags)}")

# ── B. GRANGER CAUSALITY ──────────────────────────────────────────
print("\n" + "=" * 60)
print("  B. GRANGER CAUSALITY  (sent -> return)")
print("=" * 60)
granger_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 100: continue
    try:
        data = ds[["return_1d", "sent_mean"]].values
        gc_res = grangercausalitytests(data, maxlag=5, verbose=False)
        for lag, tests in gc_res.items():
            granger_rows.append({
                "ticker":  tk,
                "lag":     lag,
                "p_value": round(float(tests[0]["ssr_ftest"][1]), 4),
                "causal":  bool(tests[0]["ssr_ftest"][1] < 0.05),
            })
    except Exception as e:
        print(f"  ⚠️  {tk}: {str(e)[:80]}")
df_granger = pd.DataFrame(granger_rows)
df_granger.to_csv(f"{MODELS_DIR}/granger_results.csv", index=False)
print("  Verdict per ticker (significant lags / 5):")
for tk in STOCKS:
    g = df_granger[df_granger["ticker"] == tk]
    if g.empty: continue
    n_sig = int(g["causal"].sum())
    verdict = "✅ causal" if n_sig >= 2 else "⚠️  weak" if n_sig == 1 else "❌ not causal"
    print(f"    {tk:<6} {n_sig}/5   {verdict}")

# ── C. SHAP VALUES ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  C. SHAP VALUES (LightGBM)")
print("=" * 60)
import joblib
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
test = ckpt.load(PATH_TEST)
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

# Sample for speed
sample_n   = min(3000, len(test))
X_sample   = test[FEATURE_COLS].sample(sample_n, random_state=42).values
explainer  = shap.TreeExplainer(clf_lgb)
shap_vals  = explainer.shap_values(X_sample)
if isinstance(shap_vals, list):    # for binary classifier returns [neg, pos]
    sv = shap_vals[1]
else:
    sv = shap_vals
mean_abs   = np.abs(sv).mean(axis=0)
shap_df    = pd.DataFrame({"feature": FEATURE_COLS, "mean_abs_shap": mean_abs}) \
                .sort_values("mean_abs_shap", ascending=False)
shap_df.to_csv(f"{MODELS_DIR}/shap_importance.csv", index=False)

print("\n  Top 15 features:")
total = shap_df["mean_abs_shap"].sum()
SENT_KEY = ("sent", "finbert", "sarcasm", "fear", "anger", "joy", "sadness", "mentions", "emoji")
sent_share = shap_df[shap_df["feature"].str.contains("|".join(SENT_KEY))]["mean_abs_shap"].sum()
for _, r in shap_df.head(15).iterrows():
    tag = " 🟣NLP" if any(k in r["feature"] for k in SENT_KEY) else ""
    bar = "█" * int(r["mean_abs_shap"] / shap_df["mean_abs_shap"].max() * 30)
    print(f"    {r['feature']:<25} {r['mean_abs_shap']:.5f}  {bar}{tag}")
print(f"\n  Sentiment/NLP share of total SHAP : {sent_share/total*100:.1f}%")

# ── D. ABLATION STUDY ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  D. ABLATION  (with vs without sentiment features)")
print("=" * 60)
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

tech_only = [c for c in FEATURE_COLS if not any(k in c for k in SENT_KEY)]
sent_only = [c for c in FEATURE_COLS if any(k in c for k in SENT_KEY)]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test_ = test  # alias

ablation = {}
for label, cols in (
    ("technical_only", tech_only),
    ("sentiment_only", sent_only),
    ("full_model",     FEATURE_COLS),
):
    if not cols:
        continue
    X_tr_a = train[cols].values
    y_tr_a = train["target_direction"].values.astype(int)
    X_te_a = test_[cols].values
    y_te_a = test_["target_direction"].values.astype(int)
    clf = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    clf.fit(X_tr_a, y_tr_a)
    auc = roc_auc_score(y_te_a, clf.predict_proba(X_te_a)[:, 1])
    ablation[label] = {
        "n_features": len(cols),
        "auc":        round(float(auc), 4),
    }
    print(f"  {label:<20} ({len(cols):>2} feats)   AUC = {auc:.4f}")

if "technical_only" in ablation and "full_model" in ablation:
    lift = ablation["full_model"]["auc"] - ablation["technical_only"]["auc"]
    ablation["sentiment_lift"] = round(float(lift), 4)
    print(f"\n  ▶  Sentiment lift over price-only: {lift:+.4f} AUC points "
          f"({lift*100:+.2f}%)")

with open(f"{MODELS_DIR}/ablation.json", "w") as f:
    json.dump(ablation, f, indent=2)
# ── E. PLACEBO : sentiment remplacé par du bruit (sanity check) ───
print("\n" + "=" * 60)
print("  E. PLACEBO  (sentiment features = bruit aléatoire)")
print("=" * 60)
rng = np.random.default_rng(42)
train_p, test_p = train.copy(), test_.copy()
for c in sent_only:                      # mêmes colonnes NLP, remplacées par du bruit
    mu, sd = train[c].mean(), train[c].std() + 1e-9
    train_p[c] = rng.normal(mu, sd, len(train_p))
    test_p[c]  = rng.normal(mu, sd, len(test_p))

clf_p = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    random_state=42, n_jobs=-1, verbose=-1,
)
clf_p.fit(train_p[FEATURE_COLS].values, train_p["target_direction"].values.astype(int))
auc_placebo = roc_auc_score(
    test_p["target_direction"].values.astype(int),
    clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1],
)
auc_full = ablation.get("full_model", {}).get("auc", float("nan"))
auc_tech = ablation.get("technical_only", {}).get("auc", float("nan"))

# ── F. SAUVEGARDE PROBA PLACEBO (sentiment=bruit) pour le Projet 2 ──
# clf_p et test_p viennent de la section E ci-dessus.
proba_placebo = clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1]
out_P = test_[["ticker","date"]].copy()
out_P["pred_placebo"] = proba_placebo

# nom daté selon l'année de test courante
yr = int(pd.to_datetime(test_["date"]).dt.year.mode()[0])
out_P.to_parquet(f"{OUTPUT_DIR}/proba_placebo_{yr}.parquet", index=False)

# ── G. PLACEBO MULTI-GRAINES ──────────────────────────────────────
print("\n" + "="*60); print("  G. PLACEBO MULTI-GRAINES (10 tirages)"); print("="*60)
seeds = range(42, 52); records = []
for seed in seeds:
    rng_g = np.random.default_rng(seed)
    tr_p, te_p = train.copy(), test_.copy()
    for c in sent_only:
        mu, sd = train[c].mean(), train[c].std() + 1e-9
        tr_p[c] = rng_g.normal(mu, sd, len(tr_p)); te_p[c] = rng_g.normal(mu, sd, len(te_p))
    clf_g = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               random_state=42, n_jobs=-1, verbose=-1)
    clf_g.fit(tr_p[FEATURE_COLS].values, tr_p["target_direction"].values.astype(int))
    records.append(clf_g.predict_proba(te_p[FEATURE_COLS].values)[:, 1])
proba_mean = np.mean(records, axis=0)
out_Pm = test_[["ticker","date"]].copy(); out_Pm["pred_placebo"] = proba_mean
out_Pm.to_parquet(f"{OUTPUT_DIR}/proba_placebo_multi_{yr}.parquet", index=False)
print(f"✅ proba_placebo_multi_{yr}.parquet sauvegardé")
print(f"✅ proba_placebo_{yr}.parquet sauvegardé : {out_P.shape}")
print("Vérif année :", sorted(pd.to_datetime(out_P['date']).dt.year.unique()))
print(f"  technical_only      AUC = {auc_tech:.4f}")
print(f"  full_model (réel)   AUC = {auc_full:.4f}")
print(f"  full_model PLACEBO  AUC = {auc_placebo:.4f}  (sentiment = bruit)")
print(f"\n  ▶  Lift réel    : {(auc_full-auc_tech)*100:+.2f}%")
print(f"  ▶  Lift placebo : {(auc_placebo-auc_tech)*100:+.2f}%")
print("  Lecture : si le lift placebo ≈ 0 alors que le lift réel est positif,")
print("            le gain vient bien du sentiment réel (pas d'un simple ajout de colonnes).")

  📂 loaded 08_features.parquet (19,215 rows)
  A. PEARSON LAG CORRELATIONS

  Lag-correlation summary:
            mean     std     min     max
lag_days                                
0         0.1840  0.0484  0.1436  0.2661
1         0.0283  0.0236 -0.0014  0.0751
2         0.0020  0.0282 -0.0464  0.0381
3         0.0032  0.0294 -0.0276  0.0568
4         0.0126  0.0174 -0.0178  0.0311
5        -0.0037  0.0127 -0.0228  0.0175

  Significant lags (p<0.05): 11 / 42

  B. GRANGER CAUSALITY  (sent -> return)
  Verdict per ticker (significant lags / 5):
    AAPL   2/5   ✅ causal
    MSFT   3/5   ✅ causal
    GOOGL  0/5   ❌ not causal
    AMZN   0/5   ❌ not causal
    META   5/5   ✅ causal
    TSLA   0/5   ❌ not causal
    NVDA   3/5   ✅ causal

  C. SHAP VALUES (LightGBM)
  📂 loaded 09_test.parquet (1,750 rows)

  Top 15 features:
    month                     0.29553  ██████████████████████████████
    atr_14                    0.16110  ████████████████
    vol_20d                   0.147

In [39]:
# ═══ SNAPSHOT — consigne les métriques de CE run dans RESULTS.txt ═══
import json, glob, os, shutil, traceback
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score
try:
    YEAR = int(pd.Timestamp(TEST_END).year); H = int(HORIZON)
    RES  = f"{BASE_DIR}/RESULTS.txt"
    test = pd.read_parquet(f"{OUTPUT_DIR}/09_test.parquet")
    with open(f"{MODELS_DIR}/feature_names.json") as fh:
        FEATS = json.load(fh)["features"]
    ycol = "target_direction"
    X = test[FEATS]; y = test[ycol].values.astype(int)
    rng = np.random.default_rng(42)
    def boot_ci(y_, p_, n=1000):
        idx = np.arange(len(y_)); a = []
        for _ in range(n):
            b = rng.choice(idx, len(idx), replace=True)
            if y_[b].min() == y_[b].max(): continue
            a.append(roc_auc_score(y_[b], p_[b]))
        return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))
    def perm_p(y_, p_, n=500):
        base = roc_auc_score(y_, p_); c = 0
        for _ in range(n):
            yp = rng.permutation(y_)
            if yp.min() == yp.max(): continue
            if abs(roc_auc_score(yp, p_) - 0.5) >= abs(base - 0.5): c += 1
        return (c + 1) / (n + 1)
    L = ["", "="*66,
         f"RUN  test_year={YEAR}  horizon={H}j   n_test={len(y)}  pos_rate={y.mean():.3f}",
         "-"*66,
         f"baseline classe majoritaire (acc) = {max(y.mean(), 1-y.mean()):.4f}"]
    for path in sorted(glob.glob(f"{MODELS_DIR}/*.joblib")):
        name = os.path.basename(path).replace(".joblib", "")
        try:
            mdl = joblib.load(path); p = mdl.predict_proba(X)[:, 1]
        except Exception as e:
            L.append(f"{name:22s} (skip: {type(e).__name__})"); continue
        auc = roc_auc_score(y, p); lo, hi = boot_ci(y, p); pv = perm_p(y, p)
        L.append(f"{name:22s} AUC={auc:.4f}  IC95=[{lo:.4f}, {hi:.4f}]  p_perm={pv:.4f}")
    try:
        bt = pd.read_parquet(f"{OUTPUT_DIR}/11_backtest.parquet")
        bt["y"] = (bt["ret_1d"] > 0).astype(int)
        per = bt.groupby("ticker").apply(
            lambda g: roc_auc_score(g["y"], g["pred_ens"]) if g["y"].nunique() > 1 else np.nan)
        L.append("per-ticker AUC (ensemble, ret_1d) : "
                 + "  ".join(f"{t}={v:.3f}" for t, v in per.sort_values().items()))
    except Exception as e:
        L.append(f"per-ticker indisponible ({type(e).__name__}: {e})")
    tagfull = f"_{YEAR}_h{H}"
    for f_ in ["09_test.parquet", "10_predictions.parquet", "11_backtest.parquet"]:
        s = f"{OUTPUT_DIR}/{f_}"
        if os.path.exists(s):
            shutil.copy2(s, s.replace(".parquet", tagfull + ".parquet"))
    phase2_tag = "" if (YEAR == 2023 and H == 10) else ("_2022" if (YEAR == 2022 and H == 10) else None)
    if phase2_tag is not None:
        if phase2_tag:
            shutil.copy2(f"{OUTPUT_DIR}/11_backtest.parquet",
                         f"{OUTPUT_DIR}/11_backtest{phase2_tag}.parquet")
        import lightgbm as lgb
        from sklearn.utils.class_weight import compute_sample_weight
        SENT_KEY = ("sent","finbert","sarcasm","fear","anger","joy","sadness","mentions","emoji")
        tech = [c for c in FEATS if not any(k in c for k in SENT_KEY)]
        tr = pd.read_parquet(f"{OUTPUT_DIR}/09_train.parquet")
        va = pd.read_parquet(f"{OUTPUT_DIR}/09_val.parquet")
        trv = pd.concat([tr, va])
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               max_depth=8, subsample=0.8, colsample_bytree=0.7,
                               random_state=42, verbosity=-1)
        m.fit(trv[tech], trv[ycol].astype(int),
              sample_weight=compute_sample_weight("balanced", trv[ycol].astype(int)))
        out = test[["ticker", "date"]].copy()
        out["pred_tech"] = m.predict_proba(test[tech])[:, 1]
        out.to_parquet(f"{OUTPUT_DIR}/proba_technical_only{phase2_tag}.parquet")
        L.append(f"proba_technical_only{phase2_tag}.parquet sauvegardé "
                 f"({len(out)} lignes, {len(tech)} features techniques)")
    txt = "\n".join(L)
    print(txt)
    with open(RES, "a") as fh:
        fh.write(txt + "\n")
except Exception:
    traceback.print_exc()



RUN  test_year=2023  horizon=20j   n_test=1750  pos_rate=0.733
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.7331
model_lgb              AUC=0.6113  IC95=[0.5825, 0.6425]  p_perm=0.0020
model_stack            (skip: AttributeError)
model_xgb              AUC=0.6207  IC95=[0.5908, 0.6489]  p_perm=0.0020
per-ticker AUC (ensemble, ret_1d) : NVDA=0.450  META=0.469  AMZN=0.475  TSLA=0.496  GOOGL=0.508  MSFT=0.510  AAPL=0.514


## ▶ RUN test 2022 — horizon 10 j


In [40]:
# ═══ BASCULE CONFIG — test 2022, horizon 10 j ═══
import pandas as pd, glob, os
TRAIN_END = pd.Timestamp("2020-12-31")
VAL_END   = pd.Timestamp("2021-12-31")
TEST_END  = pd.Timestamp("2022-12-31")
HORIZON   = 10
try:
    CONFIG["horizon"] = HORIZON
except Exception:
    pass
KEEP = ("_2022", "_2023", "_h1.", "_h10.", "_h20.")
for pat in ["07_", "08_", "09_train", "09_val", "09_test", "10_predictions", "11_backtest"]:
    for f in glob.glob(f"{OUTPUT_DIR}/{pat}*.parquet"):
        b = os.path.basename(f)
        if any(s in b for s in KEEP):
            continue
        os.remove(f); print("rm", b)
for f in glob.glob(f"{MODELS_DIR}/*"):
    os.remove(f)
print(f"✅ CONFIG ACTIVE → train≤{TRAIN_END.date()}  val≤{VAL_END.date()}  test≤{TEST_END.date()}  H={HORIZON}j")


rm 07_merged.parquet
rm 08_features.parquet
rm 09_train.parquet
rm 09_val.parquet
rm 09_test.parquet
rm 11_backtest.parquet
✅ CONFIG ACTIVE → train≤2020-12-31  val≤2021-12-31  test≤2022-12-31  H=10j


## Cell 11 — Merge sentiment ⨉ prix + targets

In [41]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — MERGE SENTIMENT + PRICES + TARGETS                    ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_MERGED, min_rows=10):
    print(f"  ✅ {Path(PATH_MERGED).name} exists - skipping")
    df_merged = ckpt.load(PATH_MERGED)
else:
    df_sent  = ckpt.load(PATH_DAILY)
    df_price = ckpt.load(PATH_TECH)

    df_sent["date"]  = normalize_date_col(df_sent["date"])
    df_price["date"] = normalize_date_col(df_price["date"])
    df_sent["ticker"]  = df_sent["ticker"].astype(str)
    df_price["ticker"] = df_price["ticker"].astype(str)

    print(f"  Sentiment rows : {len(df_sent):,}")
    print(f"  Price    rows  : {len(df_price):,}")

    # Left join: keep every trading day, fill sentiment with neutral if missing
    df = df_price.merge(df_sent, on=["ticker", "date"], how="left")

    # Sentiment columns to fill
    sent_cols = ["sent_mean","sent_std","sent_min","sent_max","mentions",
                 "finbert_mean","finbert_std","sarcasm_mean","sarcasm_max",
                 "fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic",
                 "emoji_mean"]
    for c in sent_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    print(f"  Merged rows    : {len(df):,}")
    days_with_sent = (df["mentions"] > 0).sum()
    print(f"  Trading days WITH sentiment: {days_with_sent:,} "
          f"({100*days_with_sent/len(df):.1f}%)")

    # ── TARGETS ────────────────────────────────────────────────────
    # Forward returns at multiple horizons (per ticker, no cross-leakage)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    for h in CONFIG["horizons"]:
        df[f"close_t+{h}"]     = df.groupby("ticker", observed=True)["close"].shift(-h)
        df[f"return_t+{h}"]    = (df[f"close_t+{h}"] - df["close"]) / df["close"]
        df[f"direction_t+{h}"] = (df[f"return_t+{h}"] > 0).astype(np.int8)
    # Primary target
    _H = globals().get("HORIZON", 1)
    if f"return_t+{_H}" not in df.columns:
        print(f"  ⚠️ horizon {_H} absent de CONFIG[horizons] → repli sur 1")
        _H = 1
    _TARGET = globals().get("TARGET", "direction")
    if _TARGET == "volatility":
        # Rendement journalier (depuis close, robuste au nom de colonne)
        df["_r1"] = df.groupby("ticker", observed=True)["close"].pct_change()
        # Volatilité réalisée FORWARD sur t+1..t+H (= la cible, légitimement future)
        _fwd_sq = None
        for _k in range(1, _H + 1):
            _rk = df.groupby("ticker", observed=True)["_r1"].shift(-_k)
            _fwd_sq = _rk.pow(2) if _fwd_sq is None else _fwd_sq + _rk.pow(2)
        df["_fwd_vol"] = (_fwd_sq / _H) ** 0.5
        # Seuil = médiane GLISSANTE PASSÉE de la vol, fenêtre courte (~1 trimestre)
        # → cible "vol élevée RELATIVE au régime récent" : classes ~50/50 à toute
        # période (évite l effet régime calme/turbulent), et aucune info du futur.
        _VOL_WIN = 63
        _past = df.groupby("ticker", observed=True)["_r1"].transform(
            lambda x: x.rolling(_H, min_periods=max(3, _H // 2)).std())
        df["_thr"] = _past.groupby(df["ticker"], observed=True).transform(
            lambda x: x.rolling(_VOL_WIN, min_periods=20).median())
        df["target_return"]    = df["_fwd_vol"].where(df["_thr"].notna())
        df["target_direction"] = (df["_fwd_vol"] > df["_thr"]).astype(np.int8)
        df = df.drop(columns=["_r1", "_fwd_vol", "_thr"])
        print(f"  🎯 Cible = VOLATILITÉ à +{_H}j (1 = vol future > médiane glissante passée)")
    else:
        df["target_return"]    = df[f"return_t+{_H}"]
        df["target_direction"] = df[f"direction_t+{_H}"]
        print(f"  🎯 Cible = direction à +{_H} jour(s)")
    # Drop rows without target_return (last few rows per ticker)
    df = df[df["target_return"].notna()].copy()

    df = mem.optimize_df(df)
    ckpt.save(df, PATH_MERGED)
    df_merged = df

print(f"\n  Final rows : {len(df_merged):,}")
print(f"  Columns    : {len(df_merged.columns)}")
print(f"\n  Direction balance (t+1):")
print(f"    Up   : {(df_merged['target_direction']==1).sum():,}  "
      f"({100*(df_merged['target_direction']==1).mean():.1f}%)")
print(f"    Down : {(df_merged['target_direction']==0).sum():,}  "
      f"({100*(df_merged['target_direction']==0).mean():.1f}%)")


  📂 loaded 05_daily.parquet (23,258 rows)
  📂 loaded 06_prices_tech.parquet (19,355 rows)
  Sentiment rows : 23,258
  Price    rows  : 19,355
  Merged rows    : 19,355
  Trading days WITH sentiment: 16,506 (85.3%)
  🎯 Cible = direction à +10 jour(s)
  💾 saved 07_merged.parquet (19,285 rows | 6.4 MB)

  Final rows : 19,285
  Columns    : 75

  Direction balance (t+1):
    Up   : 11,535  (59.8%)
    Down : 7,750  (40.2%)


## Cell 12 — Feature engineering + split temporel

In [42]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — FEATURE ENGINEERING + TEMPORAL SPLIT                  ║
# ╚══════════════════════════════════════════════════════════════════╝
if all(ckpt.exists_and_valid(p) for p in (PATH_FEATURES, PATH_TRAIN, PATH_VAL, PATH_TEST)):
    print(f"  ✅ Feature splits exist - skipping")
    df_feat = ckpt.load(PATH_FEATURES)
else:
    df = ckpt.load(PATH_MERGED)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    print(f"  Input rows : {len(df):,}")

    SENT_COLS = ["sent_mean","sent_std","mentions","finbert_mean","emoji_mean",
                 "sarcasm_mean","fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic"]

    print("\n  Lagging sentiment features (anti-leakage)...")
    # Shift each sentiment column by 1 day per ticker -> "yesterday's sentiment"
    for c in SENT_COLS:
        if c not in df.columns:
            continue   # robustesse : colonne absente (vieux checkpoint)
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # Rolling sentiment features (shift + rolling BOTH inside groupby to avoid
    # cross-ticker leakage at boundaries — transform() preserves original index).
    print("  Building sentiment rolling features...")
    for w in (3, 7, 14, 30):
        df[f"sent_ma_{w}d"] = (
            df.groupby("ticker", observed=True)["sent_mean"]
              .transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean())
        ).fillna(0)

    # Sentiment momentum (lagged)
    df["sent_momentum"] = (df["sent_ma_3d"] - df["sent_ma_14d"]).fillna(0)

    # Sarcasm-adjusted sentiment (lagged)
    df["sent_adj_lag1"] = (
        df["sent_mean_lag1"] * (1 - 0.5 * df["sarcasm_mean_lag1"])
    ).fillna(0).astype(np.float32)

    # Calendar features
    df["dow"]        = df["date"].dt.dayofweek.astype(np.int8)
    df["month"]      = df["date"].dt.month.astype(np.int8)
    df["quarter"]    = df["date"].dt.quarter.astype(np.int8)
    df["is_monday"]  = (df["dow"] == 0).astype(np.int8)
    df["is_friday"]  = (df["dow"] == 4).astype(np.int8)

    # Ticker as integer
    stock_map = {s: i for i, s in enumerate(sorted(STOCKS))}
    df["stock_id"] = df["ticker"].map(stock_map).astype(np.int8)

    # Fill NaN from lags (first row per ticker)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # Drop rows where target is missing
    df = df.dropna(subset=["target_return", "target_direction"]).copy()

    # ── MARKET-WIDE SENTIMENT FEATURES (cross-ticker) ─────────────
    # On any given day, what was the AVERAGE sentiment across ALL tickers?
    # This captures macro mood that no single ticker can show. We then
    # compute each ticker's DEVIATION from the market mood = their unique
    # signal.
    print("\n  Building market-wide cross-ticker features...")
    market_daily = (df.groupby("date", as_index=False)
                      .agg(market_sent_mean = ("sent_mean",        "mean"),
                           market_share_neg = ("share_negative",   "mean"),
                           market_share_pos = ("share_positive",   "mean"),
                           market_share_sarc= ("share_sarcastic",  "mean"),
                           market_mentions  = ("mentions",         "sum")))
    df = df.merge(market_daily, on="date", how="left")

    # Per-ticker deviation from market mean (negative → ticker more
    # pessimistic than the market, positive → more optimistic)
    df["sent_vs_market"] = (df["sent_mean"] - df["market_sent_mean"]).astype(np.float32)

    # Lag market features by 1 day (anti-leakage like the rest)
    for c in ("market_sent_mean", "market_share_neg", "market_share_pos",
              "market_share_sarc", "sent_vs_market"):
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # ── FEATURE LIST ──────────────────────────────────────────────
    FEATURE_COLS = [
        # Technical
        "ret_1d","ret_3d","ret_5d","ret_10d","ret_20d",
        "vol_5d","vol_20d", "rsi_14",
        "macd","macd_signal","macd_hist",
        "bb_width","bb_pos","atr_14",
        "price_vs_sma20","price_vs_sma50",
        "vol_ratio","hl_range",
        # Sentiment (all lagged or rolling - no same-day)
        "sent_mean_lag1","sent_std_lag1","mentions_lag1",
        "finbert_mean_lag1","sarcasm_mean_lag1",
        "fear_mean_lag1","anger_mean_lag1",
        "joy_mean_lag1","sadness_mean_lag1",
        # 4-class category proportions (lagged) — captures whether
        # the previous day's chatter was bullish, bearish, neutral or ironic
        "share_positive_lag1","share_negative_lag1",
        "share_neutral_lag1","share_sarcastic_lag1",
    "emoji_mean_lag1",
        # Market-wide cross-ticker features (NEW)
        "market_sent_mean_lag1","market_share_neg_lag1",
        "market_share_pos_lag1","market_share_sarc_lag1",
        "sent_vs_market_lag1",
        # Rolling sentiment trends
        "sent_ma_3d","sent_ma_7d","sent_ma_14d","sent_ma_30d",
        "sent_momentum","sent_adj_lag1",
        # Calendar
        "dow","month","quarter","is_monday","is_friday",
        # Identifier
        "stock_id",
    ]
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    print(f"\n  Features  : {len(FEATURE_COLS)}")

    # Save list
    with open(f"{MODELS_DIR}/feature_names.json", "w") as f:
        json.dump({"features": FEATURE_COLS}, f, indent=2)

    # ── TEMPORAL SPLIT ────────────────────────────────────────────
    train = df[df["date"] <= TRAIN_END].copy()
    val   = df[(df["date"] > TRAIN_END) & (df["date"] <= VAL_END)].copy()
    test  = df[(df["date"] > VAL_END) & (df["date"] <= TEST_END)].copy()

    print(f"\n  Train : {len(train):>7,} rows   "
          f"({train['date'].min().date()} -> {train['date'].max().date()})")
    print(f"  Val   : {len(val):>7,} rows   "
          f"({val['date'].min().date()} -> {val['date'].max().date()})")
    print(f"  Test  : {len(test):>7,} rows   "
          f"({test['date'].min().date()} -> {test['date'].max().date()})")

    # Save everything
    df = mem.optimize_df(df)
    ckpt.save(df,    PATH_FEATURES)
    ckpt.save(train, PATH_TRAIN)
    ckpt.save(val,   PATH_VAL)
    ckpt.save(test,  PATH_TEST)

    # ── Liberation disque : les parquets ligne-par-ligne (raw/clean/finbert/sent)
    #    ne sont plus lus en aval (tout passe par les agregats journaliers). ──
    if globals().get("FREE_ROWLEVEL_AFTER_FEATURES", True):
        import os as _os
        # Sauvegarde d un petit résumé "volume par source" pour la Fig 9
        # (sinon perdu quand on supprime les parquets ligne-par-ligne).
        try:
            _srcp = globals().get("PATH_SENT", "") or PATH_CLEAN
            if _srcp and _os.path.exists(_srcp):
                _sc = pd.read_parquet(_srcp, columns=["source", "dtype"])
                _scg = _sc.groupby(["source", "dtype"]).size().reset_index(name="n")
                _scg.to_parquet(f"{OUTPUT_DIR}/source_counts.parquet", index=False)
                del _sc, _scg
        except Exception as _e:
            print(f"  (résumé sources non sauvé: {str(_e)[:40]})")
        for _p in (PATH_RAW, PATH_CLEAN, PATH_FB_OUT, globals().get("PATH_SENT", "")):
            try:
                if _p and _os.path.exists(_p):
                    _sz = _os.path.getsize(_p) / 1e9
                    _os.remove(_p)
                    print(f"  🧹 {Path(_p).name} supprimé (~{_sz:.2f} GB libérés)")
            except Exception: pass

    df_feat = df

print(f"\n  Features file : {Path(PATH_FEATURES).name} ({len(df_feat):,} rows)")


# ════════════════════════════════════════════════════════════════════
#  AUDIT ANTI-FUITE (leakage) — pour la section méthodo du papier
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 64)
print("  AUDIT ANTI-FUITE (look-ahead leakage)")
print("═" * 64)
import json as _json
_audit_ok = True
try:
    with open(f"{MODELS_DIR}/feature_names.json") as _f:
        _FC = _json.load(_f)["features"]
except Exception:
    _FC = FEATURE_COLS

# (a) Aucune colonne dérivée du futur/cible ne doit être une feature
_forbidden = ("target_", "return_t+", "close_t+", "direction_t+")
_leaky = [c for c in _FC if any(k in c for k in _forbidden)]
if _leaky:
    _audit_ok = False
    print(f"  ❌ Features suspectes (dérivées de la cible) : {_leaky}")
else:
    print(f"  ✅ Aucune feature dérivée de la cible ({len(_FC)} features)")

# (b) Intégrité temporelle du split (train < val < test)
try:
    _tr = ckpt.load(PATH_TRAIN); _va = ckpt.load(PATH_VAL); _te = ckpt.load(PATH_TEST)
    _trmax, _vamin = _tr["date"].max(), _va["date"].min()
    _vamax, _temin = _va["date"].max(), _te["date"].min()
    if _trmax < _vamin and _vamax < _temin:
        print(f"  ✅ Split chronologique strict : "
              f"train≤{_trmax.date()} < val≤{_vamax.date()} < test≤{_te['date'].max().date()}")
    else:
        _audit_ok = False
        print(f"  ❌ Chevauchement temporel : trmax={_trmax}, vamin={_vamin}, "
              f"vamax={_vamax}, temin={_temin}")

    # (c) Corrélation feature↔cible sur le TRAIN : flag |r|>0.30 (signe de fuite)
    import numpy as _np
    _y = _tr["target_direction"].astype(float).values
    _susp = []
    for _c in _FC:
        try:
            _x = _tr[_c].astype(float).values
            if _np.nanstd(_x) == 0: continue
            _r = _np.corrcoef(_np.nan_to_num(_x), _y)[0, 1]
            _voltgt = globals().get("TARGET", "direction") == "volatility"
            _volfam = ("vol_", "atr", "bb_", "rsi", "macd", "ret_", "return_", "hl_range")
            # En cible "volatility", les features de vol/momentum SONT légitimement
            # prédictives (la vol se regroupe) → ce n est pas une fuite.
            if abs(_r) > 0.30 and not (_voltgt and any(_w in _c for _w in _volfam)):
                _susp.append((_c, round(float(_r), 3)))
        except Exception:
            pass
    if _susp:
        _audit_ok = False
        print(f"  ⚠️ Features très corrélées à la cible (|r|>0.30) — à vérifier :")
        for _c, _r in sorted(_susp, key=lambda t: -abs(t[1]))[:10]:
            print(f"       {_c:<28} r={_r:+.3f}")
    else:
        print(f"  ✅ Aucune feature anormalement corrélée à la cible (|r|≤0.30)")
    del _tr, _va, _te
except Exception as _e:
    print(f"  (audit partiel : {str(_e)[:60]})")

print("─" * 64)
print(f"  VERDICT : {'✅ PAS DE FUITE DÉTECTÉE' if _audit_ok else '⚠️ POINTS À VÉRIFIER (voir ci-dessus)'}")
print("═" * 64)


  📂 loaded 07_merged.parquet (19,285 rows)
  Input rows : 19,285

  Lagging sentiment features (anti-leakage)...
  Building sentiment rolling features...

  Building market-wide cross-ticker features...

  Features  : 49

  Train :  10,577 rows   (2015-01-02 -> 2020-12-31)
  Val   :   1,764 rows   (2021-01-04 -> 2021-12-31)
  Test  :   1,757 rows   (2022-01-03 -> 2022-12-30)
  💾 saved 08_features.parquet (19,285 rows | 8.1 MB)
  💾 saved 09_train.parquet (10,577 rows | 4.9 MB)
  💾 saved 09_val.parquet (1,764 rows | 0.8 MB)
  💾 saved 09_test.parquet (1,757 rows | 0.8 MB)

  Features file : 08_features.parquet (19,285 rows)

════════════════════════════════════════════════════════════════
  AUDIT ANTI-FUITE (look-ahead leakage)
════════════════════════════════════════════════════════════════
  ✅ Aucune feature dérivée de la cible (49 features)
  📂 loaded 09_train.parquet (10,577 rows)
  📂 loaded 09_val.parquet (1,764 rows)
  📂 loaded 09_test.parquet (1,757 rows)
  ✅ Split chronologique st

## Cell 13 — Arbres : LightGBM + XGBoost + CatBoost

In [43]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — TRAIN MODELS                                          ║
# ╚══════════════════════════════════════════════════════════════════╝
import time

# Lazy installs
for pkg in ("lightgbm", "xgboost", "catboost"):
    try:
        __import__(pkg)
    except ImportError:
        pip_install(pkg)

import lightgbm as lgb
import xgboost  as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import joblib

# Load
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test  = ckpt.load(PATH_TEST)

# CORRECTION ANTI-FUITE : tri chronologique GLOBAL avant TimeSeriesSplit.
# (les fichiers sont tries [ticker, date] → la CV temporelle melangerait
#  les tickers et casserait l'ordre du temps.)
train = train.sort_values("date").reset_index(drop=True)
val   = val.sort_values("date").reset_index(drop=True)
test  = test.sort_values("date").reset_index(drop=True)

X_tr, y_tr = train[FEATURE_COLS].values, train["target_direction"].values.astype(int)
X_va, y_va = val[FEATURE_COLS].values,   val["target_direction"].values.astype(int)
X_te, y_te = test[FEATURE_COLS].values,  test["target_direction"].values.astype(int)

print(f"  Train : {X_tr.shape}  |  pos rate = {y_tr.mean():.3f}")
print(f"  Val   : {X_va.shape}  |  pos rate = {y_va.mean():.3f}")
print(f"  Test  : {X_te.shape}  |  pos rate = {y_te.mean():.3f}")

sw_tr = compute_sample_weight("balanced", y_tr)

# ── 1. LIGHTGBM ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  LightGBM")
print("─" * 60)
lgb_params = dict(
    objective="binary", metric="auc",
    n_estimators=800, learning_rate=0.03,
    num_leaves=63, max_depth=8,
    min_child_samples=40,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1,
)
tscv = TimeSeriesSplit(n_splits=5)
cv_lgb = []
clf_lgb = lgb.LGBMClassifier(**lgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_lgb.fit(
        X_tr[tr_i], y_tr[tr_i],
        sample_weight=sw_tr[tr_i],
        eval_set=[(X_tr[val_i], y_tr[val_i])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    auc = roc_auc_score(y_tr[val_i], clf_lgb.predict_proba(X_tr[val_i])[:, 1])
    cv_lgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_lgb):.4f} ± {np.std(cv_lgb):.4f}")
# Final fit
clf_lgb.fit(X_tr, y_tr, sample_weight=sw_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)])
joblib.dump(clf_lgb, f"{MODELS_DIR}/model_lgb.joblib")
auc_lgb_val = roc_auc_score(y_va, clf_lgb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_lgb_val:.4f}")

# ── 2. XGBOOST ─────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  XGBoost")
print("─" * 60)
neg, pos = int((y_tr == 0).sum()), int((y_tr == 1).sum())
spw = neg / max(pos, 1)
xgb_params = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=600, learning_rate=0.05,
    max_depth=7, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw,
    random_state=42, n_jobs=-1, verbosity=0,
    early_stopping_rounds=50,
)
cv_xgb = []
clf_xgb = xgb.XGBClassifier(**xgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_xgb.fit(X_tr[tr_i], y_tr[tr_i],
                eval_set=[(X_tr[val_i], y_tr[val_i])],
                verbose=False)
    auc = roc_auc_score(y_tr[val_i], clf_xgb.predict_proba(X_tr[val_i])[:, 1])
    cv_xgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_xgb):.4f} ± {np.std(cv_xgb):.4f}")
# Final fit (without early-stopping callback now)
xgb_params_final = {k: v for k, v in xgb_params.items() if k != "early_stopping_rounds"}
clf_xgb_final = xgb.XGBClassifier(**xgb_params_final)
clf_xgb_final.fit(X_tr, y_tr)
joblib.dump(clf_xgb_final, f"{MODELS_DIR}/model_xgb.joblib")
auc_xgb_val = roc_auc_score(y_va, clf_xgb_final.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_xgb_val:.4f}")

# ── 3. CATBOOST ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  CatBoost")
print("─" * 60)
clf_cb = CatBoostClassifier(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=42, verbose=0,
    early_stopping_rounds=50,
)
clf_cb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=(X_va, y_va), use_best_model=True)
clf_cb.save_model(f"{MODELS_DIR}/model_cb.cbm")
auc_cb_val = roc_auc_score(y_va, clf_cb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_cb_val:.4f}")
print(f"  Best iter     = {clf_cb.best_iteration_}")

# Save model summary
model_summary = {
    "lgb": {"cv_auc_mean": float(np.mean(cv_lgb)),
            "cv_auc_std":  float(np.std(cv_lgb)),
            "val_auc":     float(auc_lgb_val)},
    "xgb": {"cv_auc_mean": float(np.mean(cv_xgb)),
            "cv_auc_std":  float(np.std(cv_xgb)),
            "val_auc":     float(auc_xgb_val)},
    "cb":  {"val_auc":     float(auc_cb_val),
            "best_iter":   int(clf_cb.best_iteration_)},
}
with open(f"{MODELS_DIR}/training_summary.json", "w") as f:
    json.dump(model_summary, f, indent=2)

print("\n" + "=" * 60)
print(f"  ✅ 3 models trained and saved to {MODELS_DIR}")
print("=" * 60)
mem.free()


  📂 loaded 09_train.parquet (10,577 rows)
  📂 loaded 09_val.parquet (1,764 rows)
  📂 loaded 09_test.parquet (1,757 rows)
  Train : (10577, 49)  |  pos rate = 0.621
  Val   : (1764, 49)  |  pos rate = 0.601
  Test  : (1757, 49)  |  pos rate = 0.392

────────────────────────────────────────────────────────────
  LightGBM
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.5631
  fold 2/5  AUC=0.5433
  fold 3/5  AUC=0.5504
  fold 4/5  AUC=0.4630
  fold 5/5  AUC=0.5403
  CV AUC mean   = 0.5320 ± 0.0354
  Val AUC       = 0.5201

────────────────────────────────────────────────────────────
  XGBoost
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.5410
  fold 2/5  AUC=0.5341
  fold 3/5  AUC=0.5791
  fold 4/5  AUC=0.4782
  fold 5/5  AUC=0.5345
  CV AUC mean   = 0.5334 ± 0.0322
  Val AUC       = 0.5145

────────────────────────────────────────────────────────────
  CatBoost
────────────────────────────────────────────────────────────
  V

## Cell 13b — Modele sequentiel hybride (BiGRU + attention)  ⭐ NOUVEAU

Apporte la composante « deep » de l'ensemble hybride. Pour chaque (ticker, jour) on construit une
**fenetre glissante de `SEQ_LEN` jours** du vecteur de features (sentiment lagge + techniques), normalisee
sur le **train uniquement** (anti-fuite). Un **BiGRU bidirectionnel + attention** apprend les dynamiques
temporelles que les arbres ne capturent pas. Les probabilites val/test sont sauvegardees pour le stacking.
GPU recommande ; degrade proprement en CPU (ou se desactive si torch absent).

In [44]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13b — BiGRU + ATTENTION (modele sequentiel)                ║
# ║  Sortie : {OUTPUT_DIR}/seq_proba.parquet  [ticker,date,p_seq,split]║
# ╚══════════════════════════════════════════════════════════════════╝
import json, time
import numpy as np
import pandas as pd
from pathlib import Path

SEQ_OUT = f"{OUTPUT_DIR}/seq_proba.parquet"

if not USE_SEQ_MODEL:
    print("  USE_SEQ_MODEL=False → modele sequentiel desactive.")
elif Path(SEQ_OUT).exists():
    print(f"  ✅ {Path(SEQ_OUT).name} existe — skip BiGRU")
else:
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        pip_install("torch"); import torch; import torch.nn as nn

    from sklearn.metrics import roc_auc_score

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device : {device}")

    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    df = ckpt.load(PATH_FEATURES)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # ── Standardisation (stats du TRAIN uniquement) ──────────────────
    train_mask = df["date"] <= TRAIN_END
    mu = df.loc[train_mask, FEATURE_COLS].mean()
    sd = df.loc[train_mask, FEATURE_COLS].std().replace(0, 1.0)
    Xall = ((df[FEATURE_COLS] - mu) / sd).fillna(0.0).values.astype("float32")
    y    = df["target_direction"].values.astype("float32")
    tks  = df["ticker"].values
    dts  = df["date"].values
    n_feat = Xall.shape[1]
    L = int(SEQ_LEN)

    # ── Construction des fenetres glissantes par ticker ──────────────
    print(f"  Construction des fenetres (L={L}, n_feat={n_feat})...")
    W, yy, wt, wd = [], [], [], []
    for tk in sorted(set(tks)):
        idx = np.where(tks == tk)[0]
        Xi, yi, di = Xall[idx], y[idx], dts[idx]
        for j in range(len(idx)):
            lo = max(0, j - L + 1)
            w = Xi[lo:j + 1]
            if len(w) < L:                       # padding par la 1ere ligne
                w = np.vstack([np.repeat(w[:1], L - len(w), axis=0), w])
            W.append(w); yy.append(yi[j]); wt.append(tk); wd.append(di[j])
    W  = np.asarray(W, dtype="float32")
    yy = np.asarray(yy, dtype="float32")
    wd = pd.to_datetime(pd.Series(wd))
    print(f"  Fenetres : {W.shape}")

    tr_m = (wd <= TRAIN_END).values
    va_m = ((wd > TRAIN_END) & (wd <= VAL_END)).values
    te_m = ((wd > VAL_END) & (wd <= TEST_END)).values

    # Borne memoire : sous-echantillonne le train si gigantesque (CPU)
    MAX_TR = 400_000 if device.type == "cuda" else 120_000
    tr_idx = np.where(tr_m)[0]
    if len(tr_idx) > MAX_TR:
        rng = np.random.default_rng(42)
        tr_idx = rng.choice(tr_idx, MAX_TR, replace=False)
    va_idx = np.where(va_m)[0]
    te_idx = np.where(te_m)[0]
    print(f"  train={len(tr_idx):,}  val={len(va_idx):,}  test={len(te_idx):,}")

    # ── Modele : BiGRU bidirectionnel + attention additive ───────────
    class BiGRUAttn(nn.Module):
        def __init__(self, n_feat, hidden=64, layers=1, p=0.3):
            super().__init__()
            self.gru = nn.GRU(n_feat, hidden, num_layers=layers,
                              batch_first=True, bidirectional=True,
                              dropout=p if layers > 1 else 0.0)
            self.attn = nn.Linear(hidden * 2, 1)
            self.head = nn.Sequential(
                nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(p),
                nn.Linear(64, 1))
        def forward(self, x):
            h, _ = self.gru(x)                       # (B, L, 2H)
            a = torch.softmax(self.attn(h).squeeze(-1), dim=1)  # (B, L)
            ctx = (h * a.unsqueeze(-1)).sum(dim=1)   # (B, 2H)
            return self.head(ctx).squeeze(-1)

    model = BiGRUAttn(n_feat).to(device)
    pos_w = float((yy[tr_idx] == 0).sum()) / max(float((yy[tr_idx] == 1).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    Wt = torch.from_numpy(W)   # garde en CPU, on transfere par batch

    def run_eval(idx):
        model.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(idx), SEQ_BATCH):
                b = idx[i:i + SEQ_BATCH]
                xb = Wt[b].to(device)
                outs.append(torch.sigmoid(model(xb)).cpu().numpy())
        return np.concatenate(outs) if outs else np.array([])

    best_auc, best_state, patience, bad = -1.0, None, 5, 0
    print("  Entrainement BiGRU...")
    for ep in range(1, int(SEQ_EPOCHS) + 1):
        model.train()
        rng = np.random.default_rng(ep)
        perm = rng.permutation(tr_idx)
        t0 = time.time()
        for i in range(0, len(perm), SEQ_BATCH):
            b = perm[i:i + SEQ_BATCH]
            xb = Wt[b].to(device)
            yb = torch.from_numpy(yy[b]).to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        p_va = run_eval(va_idx)
        auc = roc_auc_score(yy[va_idx], p_va) if len(set(yy[va_idx])) > 1 else 0.5
        print(f"    epoch {ep:>2}  val_AUC={auc:.4f}  ({time.time()-t0:.1f}s)")
        if auc > best_auc:
            best_auc, best_state, bad = auc, {k: v.cpu().clone()
                                              for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print("    early stopping"); break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  ✅ meilleur val_AUC = {best_auc:.4f}")

    # ── Probabilites val + test → parquet pour le stacking ───────────
    out = pd.DataFrame({
        "ticker": np.r_[np.array(wt)[va_idx], np.array(wt)[te_idx]],
        "date":   pd.to_datetime(np.r_[wd.values[va_idx], wd.values[te_idx]]),
        "p_seq":  np.r_[run_eval(va_idx), run_eval(te_idx)].astype("float32"),
        "split":  (["val"] * len(va_idx)) + (["test"] * len(te_idx)),
    })
    out["date"] = out["date"].dt.normalize()
    out.to_parquet(SEQ_OUT, index=False)
    torch.save(model.state_dict(), f"{MODELS_DIR}/model_seq.pt")
    with open(f"{MODELS_DIR}/seq_summary.json", "w") as f:
        json.dump({"val_auc": float(best_auc), "seq_len": L,
                   "n_feat": int(n_feat)}, f, indent=2)
    print(f"  💾 {Path(SEQ_OUT).name}  ({len(out):,} rows)  + model_seq.pt")
    mem.free()


  ✅ seq_proba.parquet existe — skip BiGRU


## Cell 13c — Stacking hybride + calibration  ⭐ NOUVEAU

Combine les **4 modeles de base** (LightGBM, XGBoost, CatBoost, BiGRU) via un **meta-modele logistique**
ajuste sur la validation, puis applique une **calibration isotone** (probabilites mieux calibrees → meilleures
decisions de trading). Produit `stack_proba.parquet` (proba finale alignee sur les lignes de test) que la
CELL 14 utilise automatiquement.

In [45]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13c — STACKING (meta-logistique) + CALIBRATION ISOTONE     ║
# ╚══════════════════════════════════════════════════════════════════╝
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from catboost import CatBoostClassifier

if not USE_STACKING:
    print("  USE_STACKING=False → stacking desactive (CELL 14 utilisera la moyenne fixe).")
else:
    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    val  = ckpt.load(PATH_VAL).sort_values(["ticker", "date"]).reset_index(drop=True)
    test = ckpt.load(PATH_TEST).sort_values(["ticker", "date"]).reset_index(drop=True)
    for d in (val, test):
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()

    clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
    clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
    clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

    def tree_probas(d):
        X = d[FEATURE_COLS].values
        return np.column_stack([
            clf_lgb.predict_proba(X)[:, 1],
            clf_xgb.predict_proba(X)[:, 1],
            clf_cb.predict_proba(X)[:, 1],
        ])

    Pva, Pte = tree_probas(val), tree_probas(test)
    base_names = ["lgb", "xgb", "cb"]

    # ── Ajout du BiGRU si disponible ─────────────────────────────────
    seqp = f"{OUTPUT_DIR}/seq_proba.parquet"
    if Path(seqp).exists():
        sp = pd.read_parquet(seqp)
        sp["date"] = pd.to_datetime(sp["date"]).dt.normalize()
        def add_seq(d, P):
            m = d[["ticker", "date"]].merge(
                sp[["ticker", "date", "p_seq"]], on=["ticker", "date"], how="left")
            return np.column_stack([P, m["p_seq"].fillna(0.5).values])
        Pva, Pte = add_seq(val, Pva), add_seq(test, Pte)
        base_names.append("seq")
        print("  ✅ BiGRU integre au stacking")
    else:
        print("  ℹ️  Pas de seq_proba.parquet — stacking sur les 3 arbres seulement")

    yva = val["target_direction"].astype(int).values
    yte = test["target_direction"].astype(int).values

    # ── Meta-modele logistique (ajuste sur la validation) ────────────
    meta = LogisticRegression(max_iter=2000, C=1.0)
    meta.fit(Pva, yva)
    raw_va = meta.predict_proba(Pva)[:, 1]
    raw_te = meta.predict_proba(Pte)[:, 1]

    # ── Calibration isotone (sur la validation) ──────────────────────
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va, yva)
    p_stack    = iso.transform(raw_te).astype("float32")
    p_stack_va = iso.transform(raw_va).astype("float32")
    # Seuil de décision calibré sur la VALIDATION (quantile = taux de hausse val).
    # Évite l effondrement de l ensemble (sinon il prédit presque tout "baisse").
    _thr_stack = float(np.quantile(p_stack_va, 1.0 - float(np.mean(yva))))
    print(f"  Seuil ensemble calibré (val) : {_thr_stack:.4f}")

    # ── Rapport AUC base vs stack ────────────────────────────────────
    print("\n  AUC (test) par modele de base :")
    for i, nm in enumerate(base_names):
        print(f"    {nm:<5} {roc_auc_score(yte, Pte[:, i]):.4f}")
    print(f"    {'STACK':<5} {roc_auc_score(yte, p_stack):.4f}  "
          f"(acc={accuracy_score(yte, (p_stack>0.5).astype(int)):.4f})")
    print(f"  Poids meta (logistique) : "
          f"{dict(zip(base_names, np.round(meta.coef_[0], 3)))}")

    # ── Sauvegardes ──────────────────────────────────────────────────
    out = test[["ticker", "date"]].copy()
    out["p_stack"] = p_stack
    out.to_parquet(f"{OUTPUT_DIR}/stack_proba.parquet", index=False)
    joblib.dump({"meta": meta, "iso": iso, "names": base_names},
                f"{MODELS_DIR}/model_stack.joblib")
    with open(f"{MODELS_DIR}/stack_summary.json", "w") as f:
        json.dump({"base_names": base_names,
                   "weights": dict(zip(base_names, meta.coef_[0].tolist())),
                   "stack_test_auc": float(roc_auc_score(yte, p_stack)),
                   "threshold": _thr_stack}, f, indent=2)
    with open(f"{MODELS_DIR}/stack_threshold.json", "w") as f:
        json.dump({"threshold": _thr_stack,
                   "val_pos_rate": float(np.mean(yva))}, f)
    print(f"  💾 stack_proba.parquet + model_stack.joblib")


  📂 loaded 09_val.parquet (1,764 rows)
  📂 loaded 09_test.parquet (1,757 rows)
  ✅ BiGRU integre au stacking
  Seuil ensemble calibré (val) : 0.5985

  AUC (test) par modele de base :
    lgb   0.4630
    xgb   0.5273
    cb    0.4882
    seq   0.5583
    STACK 0.4975  (acc=0.3921)
  Poids meta (logistique) : {'lgb': np.float64(0.049), 'xgb': np.float64(0.084), 'cb': np.float64(0.564), 'seq': np.float64(-0.021)}
  💾 stack_proba.parquet + model_stack.joblib


## Cell 14 — Ensemble (stacking hybride), evaluation & backtest

In [46]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — ENSEMBLE + EVAL + BACKTEST                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import joblib
from catboost import CatBoostClassifier

with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

test = ckpt.load(PATH_TEST)
X_te = test[FEATURE_COLS].values
y_te = test["target_direction"].values.astype(int)

# Load models
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

# Probabilities
p_lgb = clf_lgb.predict_proba(X_te)[:, 1]
p_xgb = clf_xgb.predict_proba(X_te)[:, 1]
p_cb  = clf_cb.predict_proba(X_te)[:, 1]

# ── Ensemble HYBRIDE : on privilegie la pile apprise (CELL 13c :
#    arbres + BiGRU + meta-logistique calibree) si elle existe, sinon
#    on retombe sur la moyenne ponderee fixe.
from pathlib import Path as _P
_stack_path = f"{OUTPUT_DIR}/stack_proba.parquet"
_p_fixed = 0.4 * p_lgb + 0.3 * p_xgb + 0.3 * p_cb
if _P(_stack_path).exists():
    _sp = pd.read_parquet(_stack_path)
    _sp["date"] = pd.to_datetime(_sp["date"]).dt.normalize()
    _key = test[["ticker", "date"]].copy()
    _key["date"] = pd.to_datetime(_key["date"]).dt.normalize()
    _m = _key.merge(_sp[["ticker", "date", "p_stack"]], on=["ticker", "date"], how="left")
    p_ens = _m["p_stack"].fillna(pd.Series(_p_fixed)).values
    print("  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)")
else:
    p_ens = _p_fixed
    print("  ℹ️  Ensemble = moyenne ponderee fixe (stacking absent)")

# Seuil ensemble par CALAGE DE TAUX : on aligne la proportion de "hausse"
# prédite sur le taux de hausse de la VALIDATION (a priori connu, pas de
# fuite des labels test). Robuste au fort déséquilibre des horizons longs.
try:
    with open(f"{MODELS_DIR}/stack_threshold.json") as _f:
        _vpr = float(json.load(_f).get("val_pos_rate", 0.5))
    _THR_ENS = float(np.quantile(p_ens, 1.0 - _vpr))
except Exception:
    _THR_ENS = 0.5

# Per-model metrics (seuil par modèle : ensemble = seuil calibré)
def metrics_row(name, p, thr=0.5):
    yp = (p > thr).astype(int)
    return {
        "name":      name,
        "accuracy":  float(accuracy_score(y_te, yp)),
        "auc":       float(roc_auc_score(y_te, p)),
        "precision": float(precision_score(y_te, yp, zero_division=0)),
        "recall":    float(recall_score(y_te, yp, zero_division=0)),
        "f1":        float(f1_score(y_te, yp, zero_division=0)),
    }
rows = [metrics_row("lightgbm", p_lgb), metrics_row("xgboost", p_xgb),
        metrics_row("catboost", p_cb)]
# Ensemble (stacking) exclu : dégénère sous fort déséquilibre et n'apporte
# rien (AUC ≈ modèles de base). Modèles retenus : LightGBM/XGBoost/CatBoost.
results = pd.DataFrame(rows).set_index("name").round(4)

print("=" * 60)
print("  TEST SET METRICS")
print("=" * 60)
print(results.to_string())

# ── BACKTEST ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BACKTEST  (threshold 0.55 long / 0.45 short)")
print("=" * 60)
_is_vol = globals().get("TARGET", "direction") == "volatility"
if _is_vol:
    print("  ⚠️ Cible = VOLATILITÉ : ce backtest directionnel (long/short) n a PAS")
    print("     de sens (on prédit l amplitude, pas le sens du mouvement).")
    print("     → À IGNORER pour le papier. Utiliser AUC + ablation à la place.")
    print("     (Calculé seulement pour ne pas casser la figure du dashboard.)")
bt = test[["ticker", "date", "close"]].copy()
bt["pred_ens"] = p_lgb   # backtest sur LightGBM (modèle principal)
bt = bt.sort_values(["ticker", "date"]).reset_index(drop=True)
# Rendement RÉALISÉ à 1 jour (close-to-close). On n'utilise JAMAIS le target
# à H jours pour composer : sinon on cumulerait un rendement H-jours chaque
# jour (fenêtres chevauchantes) → explosion irréaliste du cumul.
bt["ret_1d"] = bt.groupby("ticker", observed=True)["close"].pct_change().fillna(0.0)
bt["signal"]   = 0
bt.loc[bt["pred_ens"] > 0.55, "signal"] = 1
bt.loc[bt["pred_ens"] < 0.45, "signal"] = -1

# Strategy return = signal de la veille × rendement réalisé du jour
bt["signal_lag"]      = bt.groupby("ticker", observed=True)["signal"].shift(1).fillna(0)
bt["strategy_return"] = bt["signal_lag"] * bt["ret_1d"]

# ── Transaction costs (10 bps per round-trip = realistic retail broker)
TXN_COST_BPS = 10
bt["pos_change"]      = (bt["signal"] != bt["signal_lag"]).astype(np.int8)
bt["txn_cost"]        = bt["pos_change"] * (TXN_COST_BPS / 10000.0)
bt["strategy_net"]    = bt["strategy_return"] - bt["txn_cost"]

bt["cum_strategy"]    = bt.groupby("ticker", observed=True)["strategy_net"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())
bt["cum_market"]      = bt.groupby("ticker", observed=True)["ret_1d"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())

# Per-ticker summary
backtest_summary = {}
for tk, g in bt.groupby("ticker", observed=True):
    ret_s   = (g["cum_strategy"].iloc[-1] - 1) * 100
    ret_m   = (g["cum_market"].iloc[-1]   - 1) * 100
    sr_std  = g["strategy_net"].std()
    sharpe  = (g["strategy_net"].mean() / sr_std * np.sqrt(252)) if sr_std > 0 else 0.0
    n_tr    = int(g["pos_change"].sum())
    act     = g[g["signal"] != 0]["strategy_net"]
    win_rt  = float((act > 0).mean()) if len(act) else 0.0
    txn_drag = float(g["txn_cost"].sum() * 100)
    backtest_summary[tk] = {
        "return_strategy":  round(float(ret_s), 2),
        "return_market":    round(float(ret_m), 2),
        "alpha":            round(float(ret_s - ret_m), 2),
        "sharpe":           round(float(sharpe), 3),
        "n_trades":         n_tr,
        "win_rate":         round(win_rt, 4),
        "txn_cost_drag":    round(txn_drag, 2),     # cumulative cost in %
    }

# Aggregate
total_s = (bt.groupby("date")["strategy_net"].mean().fillna(0) + 1).cumprod().iloc[-1] - 1
total_m = (bt.groupby("date")["ret_1d"].mean().fillna(0)   + 1).cumprod().iloc[-1] - 1
_daily_strat = bt.groupby("date")["strategy_net"].mean().fillna(0)
_eq = (1 + _daily_strat).cumprod()
_max_dd = float((_eq / _eq.cummax() - 1).min())
_sh_overall = (float(_daily_strat.mean() / _daily_strat.std() * np.sqrt(252))
               if _daily_strat.std() > 0 else 0.0)
backtest_summary["__overall__"] = {
    "return_strategy": round(float(total_s * 100), 2),
    "return_market":   round(float(total_m * 100), 2),
    "alpha":           round(float((total_s - total_m) * 100), 2),
    "sharpe":          round(_sh_overall, 3),
    "max_drawdown":    round(_max_dd * 100, 2),
}

print(f"\n  Strategy total : {total_s*100:+.2f}%")
print(f"  Market   total : {total_m*100:+.2f}%")
print(f"  Alpha          : {(total_s - total_m)*100:+.2f}%\n")
print(f"  {'Ticker':<8} {'Strategy':>10} {'Market':>10} {'Alpha':>9} {'Sharpe':>7} {'Trades':>7} {'WinR':>6}")
print("  " + "─" * 60)
for tk in STOCKS:
    if tk not in backtest_summary: continue
    s = backtest_summary[tk]
    print(f"  {tk:<8} {s['return_strategy']:>+9.2f}% {s['return_market']:>+9.2f}% "
          f"{s['alpha']:>+8.2f}% {s['sharpe']:>7.2f} {s['n_trades']:>7d} {s['win_rate']*100:>5.1f}%")

# Save
ckpt.save(bt, PATH_BACKTEST)
results.to_json(f"{MODELS_DIR}/test_metrics.json", indent=2)
with open(f"{MODELS_DIR}/backtest_summary.json", "w") as f:
    json.dump(backtest_summary, f, indent=2)

# Confusion matrix (LightGBM)
print("\n  Confusion matrix (LightGBM):")
yp_ens = (p_lgb > 0.5).astype(int)
cm = confusion_matrix(y_te, yp_ens)
print(f"             pred_down  pred_up")
print(f"  true_down   {cm[0,0]:>7,}  {cm[0,1]:>7,}")
print(f"  true_up     {cm[1,0]:>7,}  {cm[1,1]:>7,}")

mem.free()


# ════════════════════════════════════════════════════════════════════
#  BASELINES + SIGNIFICATIVITÉ STATISTIQUE — pour l'acceptation du papier
# ════════════════════════════════════════════════════════════════════
import numpy as _np
from sklearn.metrics import roc_auc_score as _auc

print("\n" + "═" * 64)
print("  BASELINES & SIGNIFICATIVITÉ (test set)")
print("═" * 64)

# Baselines naïves
_base_up   = float((y_te == 1).mean())          # "toujours hausse"
_base_maj  = max(_base_up, 1 - _base_up)         # classe majoritaire
print(f"  Baseline 'toujours hausse' : acc = {_base_up:.4f}")
print(f"  Baseline classe majoritaire: acc = {_base_maj:.4f}")
print(f"  (un modèle utile doit battre {_base_maj:.4f} de façon significative)")

def _auc_safe(yt, p):
    try: return _auc(yt, p)
    except Exception: return 0.5

# Bootstrap IC 95% + test de permutation pour l'AUC de chaque modèle
_rng = _np.random.default_rng(42)
_N_BOOT, _N_PERM = 1000, 1000
print(f"\n  {'Modèle':<11} {'AUC':>7}  {'IC 95% bootstrap':>22}  {'p (perm)':>9}")
print("  " + "-" * 56)
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    _p = _np.asarray(_p, dtype=float)
    _obs = _auc_safe(y_te, _p)
    # bootstrap
    _bs = []
    _n = len(y_te)
    for _ in range(_N_BOOT):
        _idx = _rng.integers(0, _n, _n)
        if len(_np.unique(y_te[_idx])) < 2: continue
        _bs.append(_auc_safe(y_te[_idx], _p[_idx]))
    _lo, _hi = (_np.percentile(_bs, [2.5, 97.5]) if _bs else (_np.nan, _np.nan))
    # permutation : H0 = pas de lien (on mélange y)
    _null = _np.empty(_N_PERM)
    for _i in range(_N_PERM):
        _null[_i] = _auc_safe(_rng.permutation(y_te), _p)
    _pval = float((_np.sum(_np.abs(_null - 0.5) >= abs(_obs - 0.5)) + 1) / (_N_PERM + 1))
    _star = "***" if _pval < 0.01 else "**" if _pval < 0.05 else "*" if _pval < 0.10 else "ns"
    print(f"  {_name:<11} {_obs:>7.4f}  [{_lo:.4f}, {_hi:.4f}]  {_pval:>8.4f} {_star}")

print("\n  Lecture : un AUC est crédible si son IC 95% exclut 0.50 et p<0.05.")
print("  *** p<0.01  ** p<0.05  * p<0.10  ns = non significatif")
print("═" * 64)


# ════════════════════════════════════════════════════════════════════
#  MODE ABSTENTION — accuracy vs couverture (sélectivité de confiance)
#  Le modèle n'agit que sur ses prédictions les plus confiantes.
#  Courbe publiable : "accuracy croît quand on réduit la couverture".
# ════════════════════════════════════════════════════════════════════
import numpy as _np
print("\n" + "═" * 64)
print("  MODE ABSTENTION — accuracy sur les prédictions les + confiantes")
print("═" * 64)

def _conf_curve(name, p, thr=0.5):
    p = _np.asarray(p, dtype=float)
    conf = _np.abs(p - thr)                 # distance au seuil de décision = confiance
    order = _np.argsort(-conf)              # plus confiant d'abord
    yp_all = (p > thr).astype(int)
    rows = []
    for cov in (1.00, 0.50, 0.30, 0.20, 0.10, 0.05):
        k = max(1, int(round(len(p) * cov)))
        idx = order[:k]
        acc = float((yp_all[idx] == y_te[idx]).mean())
        rows.append((cov, k, acc))
    return rows

print(f"  {'Couverture':>10} {'N trades':>9} {'Accuracy':>9}")
print("  " + "-" * 32)
_abst_out = {}
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    print(f"  ── {_name} ──")
    _thr_a = _THR_ENS if _name == "ensemble" else 0.5
    _rows = _conf_curve(_name, _p, _thr_a)
    _abst_out[_name] = [{"coverage": c, "n": k, "accuracy": a} for c, k, a in _rows]
    for cov, k, acc in _rows:
        print(f"  {cov*100:>9.0f}% {k:>9,} {acc:>9.4f}")

try:
    with open(f"{MODELS_DIR}/abstention_curve.json", "w") as _f:
        json.dump(_abst_out, _f, indent=2)
except Exception:
    pass

print("\n  Lecture : si l'accuracy monte quand la couverture baisse, le modèle")
print("  'sait quand il sait' — argument fort pour le papier (trading sélectif).")
print("  À reporter en figure : accuracy (y) vs couverture (x), par modèle.")
print("═" * 64)


  📂 loaded 09_test.parquet (1,757 rows)
  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)
  TEST SET METRICS
          accuracy     auc  precision  recall      f1
name                                                 
lightgbm    0.4474  0.4630     0.3723  0.5965  0.4584
xgboost     0.4126  0.5273     0.3927  0.9115  0.5490
catboost    0.4143  0.4882     0.3916  0.8911  0.5441

  BACKTEST  (threshold 0.55 long / 0.45 short)

  Strategy total : +0.00%
  Market   total : -47.14%
  Alpha          : +47.14%

  Ticker     Strategy     Market     Alpha  Sharpe  Trades   WinR
  ────────────────────────────────────────────────────────────
  AAPL         +0.00%    -28.20%   +28.20%    0.00       0   0.0%
  MSFT         +0.00%    -27.69%   +27.69%    0.00       0   0.0%
  GOOGL        +0.00%    -39.15%   +39.15%    0.00       0   0.0%
  AMZN         +0.00%    -50.71%   +50.71%    0.00       0   0.0%
  META         +0.00%    -64.45%   +64.45%    0.00       0   0.0%
  TSLA         +0.00%   

## Cell 15 — Analyse causale (Pearson lags + Granger + SHAP + Ablation)

In [47]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — CAUSAL ANALYSIS                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
for pkg in ("statsmodels", "shap"):
    try: __import__(pkg)
    except ImportError: pip_install(pkg)

from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
import shap

df = ckpt.load(PATH_FEATURES)
df["date"] = pd.to_datetime(df["date"])

# ── A. PEARSON LAG CORRELATIONS (per ticker) ──────────────────────
print("=" * 60)
print("  A. PEARSON LAG CORRELATIONS")
print("=" * 60)
lag_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 50: continue
    for lag in range(0, 6):
        s_lagged = ds["sent_mean"].shift(lag).dropna()
        ret_aligned = ds["return_1d"].iloc[lag:].reset_index(drop=True)
        s_lagged    = s_lagged.reset_index(drop=True)
        n = min(len(s_lagged), len(ret_aligned))
        if n < 30:
            r_val, p_val = np.nan, np.nan
        else:
            r_val, p_val = stats.pearsonr(s_lagged[:n], ret_aligned[:n])
        lag_rows.append({
            "ticker":      tk,
            "lag_days":    lag,
            "correlation": round(float(r_val), 4) if not np.isnan(r_val) else None,
            "p_value":     round(float(p_val), 4) if not np.isnan(p_val) else None,
            "significant": (p_val < 0.05) if not np.isnan(p_val) else False,
        })
df_lags = pd.DataFrame(lag_rows)
df_lags.to_csv(f"{MODELS_DIR}/lag_correlations.csv", index=False)
print("\n  Lag-correlation summary:")
print(df_lags.groupby("lag_days")["correlation"].describe()[
      ["mean", "std", "min", "max"]].round(4).to_string())
print(f"\n  Significant lags (p<0.05): {df_lags['significant'].sum()} / {len(df_lags)}")

# ── B. GRANGER CAUSALITY ──────────────────────────────────────────
print("\n" + "=" * 60)
print("  B. GRANGER CAUSALITY  (sent -> return)")
print("=" * 60)
granger_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 100: continue
    try:
        data = ds[["return_1d", "sent_mean"]].values
        gc_res = grangercausalitytests(data, maxlag=5, verbose=False)
        for lag, tests in gc_res.items():
            granger_rows.append({
                "ticker":  tk,
                "lag":     lag,
                "p_value": round(float(tests[0]["ssr_ftest"][1]), 4),
                "causal":  bool(tests[0]["ssr_ftest"][1] < 0.05),
            })
    except Exception as e:
        print(f"  ⚠️  {tk}: {str(e)[:80]}")
df_granger = pd.DataFrame(granger_rows)
df_granger.to_csv(f"{MODELS_DIR}/granger_results.csv", index=False)
print("  Verdict per ticker (significant lags / 5):")
for tk in STOCKS:
    g = df_granger[df_granger["ticker"] == tk]
    if g.empty: continue
    n_sig = int(g["causal"].sum())
    verdict = "✅ causal" if n_sig >= 2 else "⚠️  weak" if n_sig == 1 else "❌ not causal"
    print(f"    {tk:<6} {n_sig}/5   {verdict}")

# ── C. SHAP VALUES ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  C. SHAP VALUES (LightGBM)")
print("=" * 60)
import joblib
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
test = ckpt.load(PATH_TEST)
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

# Sample for speed
sample_n   = min(3000, len(test))
X_sample   = test[FEATURE_COLS].sample(sample_n, random_state=42).values
explainer  = shap.TreeExplainer(clf_lgb)
shap_vals  = explainer.shap_values(X_sample)
if isinstance(shap_vals, list):    # for binary classifier returns [neg, pos]
    sv = shap_vals[1]
else:
    sv = shap_vals
mean_abs   = np.abs(sv).mean(axis=0)
shap_df    = pd.DataFrame({"feature": FEATURE_COLS, "mean_abs_shap": mean_abs}) \
                .sort_values("mean_abs_shap", ascending=False)
shap_df.to_csv(f"{MODELS_DIR}/shap_importance.csv", index=False)

print("\n  Top 15 features:")
total = shap_df["mean_abs_shap"].sum()
SENT_KEY = ("sent", "finbert", "sarcasm", "fear", "anger", "joy", "sadness", "mentions", "emoji")
sent_share = shap_df[shap_df["feature"].str.contains("|".join(SENT_KEY))]["mean_abs_shap"].sum()
for _, r in shap_df.head(15).iterrows():
    tag = " 🟣NLP" if any(k in r["feature"] for k in SENT_KEY) else ""
    bar = "█" * int(r["mean_abs_shap"] / shap_df["mean_abs_shap"].max() * 30)
    print(f"    {r['feature']:<25} {r['mean_abs_shap']:.5f}  {bar}{tag}")
print(f"\n  Sentiment/NLP share of total SHAP : {sent_share/total*100:.1f}%")

# ── D. ABLATION STUDY ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  D. ABLATION  (with vs without sentiment features)")
print("=" * 60)
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

tech_only = [c for c in FEATURE_COLS if not any(k in c for k in SENT_KEY)]
sent_only = [c for c in FEATURE_COLS if any(k in c for k in SENT_KEY)]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test_ = test  # alias

ablation = {}
for label, cols in (
    ("technical_only", tech_only),
    ("sentiment_only", sent_only),
    ("full_model",     FEATURE_COLS),
):
    if not cols:
        continue
    X_tr_a = train[cols].values
    y_tr_a = train["target_direction"].values.astype(int)
    X_te_a = test_[cols].values
    y_te_a = test_["target_direction"].values.astype(int)
    clf = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    clf.fit(X_tr_a, y_tr_a)
    auc = roc_auc_score(y_te_a, clf.predict_proba(X_te_a)[:, 1])
    ablation[label] = {
        "n_features": len(cols),
        "auc":        round(float(auc), 4),
    }
    print(f"  {label:<20} ({len(cols):>2} feats)   AUC = {auc:.4f}")

if "technical_only" in ablation and "full_model" in ablation:
    lift = ablation["full_model"]["auc"] - ablation["technical_only"]["auc"]
    ablation["sentiment_lift"] = round(float(lift), 4)
    print(f"\n  ▶  Sentiment lift over price-only: {lift:+.4f} AUC points "
          f"({lift*100:+.2f}%)")

with open(f"{MODELS_DIR}/ablation.json", "w") as f:
    json.dump(ablation, f, indent=2)
# ── E. PLACEBO : sentiment remplacé par du bruit (sanity check) ───
print("\n" + "=" * 60)
print("  E. PLACEBO  (sentiment features = bruit aléatoire)")
print("=" * 60)
rng = np.random.default_rng(42)
train_p, test_p = train.copy(), test_.copy()
for c in sent_only:                      # mêmes colonnes NLP, remplacées par du bruit
    mu, sd = train[c].mean(), train[c].std() + 1e-9
    train_p[c] = rng.normal(mu, sd, len(train_p))
    test_p[c]  = rng.normal(mu, sd, len(test_p))

clf_p = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    random_state=42, n_jobs=-1, verbose=-1,
)
clf_p.fit(train_p[FEATURE_COLS].values, train_p["target_direction"].values.astype(int))
auc_placebo = roc_auc_score(
    test_p["target_direction"].values.astype(int),
    clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1],
)
auc_full = ablation.get("full_model", {}).get("auc", float("nan"))
auc_tech = ablation.get("technical_only", {}).get("auc", float("nan"))

# ── F. SAUVEGARDE PROBA PLACEBO (sentiment=bruit) pour le Projet 2 ──
# clf_p et test_p viennent de la section E ci-dessus.
proba_placebo = clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1]
out_P = test_[["ticker","date"]].copy()
out_P["pred_placebo"] = proba_placebo

# nom daté selon l'année de test courante
yr = int(pd.to_datetime(test_["date"]).dt.year.mode()[0])
out_P.to_parquet(f"{OUTPUT_DIR}/proba_placebo_{yr}.parquet", index=False)

# ── G. PLACEBO MULTI-GRAINES ──────────────────────────────────────
print("\n" + "="*60); print("  G. PLACEBO MULTI-GRAINES (10 tirages)"); print("="*60)
seeds = range(42, 52); records = []
for seed in seeds:
    rng_g = np.random.default_rng(seed)
    tr_p, te_p = train.copy(), test_.copy()
    for c in sent_only:
        mu, sd = train[c].mean(), train[c].std() + 1e-9
        tr_p[c] = rng_g.normal(mu, sd, len(tr_p)); te_p[c] = rng_g.normal(mu, sd, len(te_p))
    clf_g = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               random_state=42, n_jobs=-1, verbose=-1)
    clf_g.fit(tr_p[FEATURE_COLS].values, tr_p["target_direction"].values.astype(int))
    records.append(clf_g.predict_proba(te_p[FEATURE_COLS].values)[:, 1])
proba_mean = np.mean(records, axis=0)
out_Pm = test_[["ticker","date"]].copy(); out_Pm["pred_placebo"] = proba_mean
out_Pm.to_parquet(f"{OUTPUT_DIR}/proba_placebo_multi_{yr}.parquet", index=False)
print(f"✅ proba_placebo_multi_{yr}.parquet sauvegardé")
print(f"✅ proba_placebo_{yr}.parquet sauvegardé : {out_P.shape}")
print("Vérif année :", sorted(pd.to_datetime(out_P['date']).dt.year.unique()))
print(f"  technical_only      AUC = {auc_tech:.4f}")
print(f"  full_model (réel)   AUC = {auc_full:.4f}")
print(f"  full_model PLACEBO  AUC = {auc_placebo:.4f}  (sentiment = bruit)")
print(f"\n  ▶  Lift réel    : {(auc_full-auc_tech)*100:+.2f}%")
print(f"  ▶  Lift placebo : {(auc_placebo-auc_tech)*100:+.2f}%")
print("  Lecture : si le lift placebo ≈ 0 alors que le lift réel est positif,")
print("            le gain vient bien du sentiment réel (pas d'un simple ajout de colonnes).")

  📂 loaded 08_features.parquet (19,285 rows)
  A. PEARSON LAG CORRELATIONS

  Lag-correlation summary:
            mean     std     min     max
lag_days                                
0         0.1840  0.0485  0.1434  0.2662
1         0.0283  0.0236 -0.0013  0.0751
2         0.0020  0.0281 -0.0462  0.0381
3         0.0032  0.0294 -0.0275  0.0568
4         0.0127  0.0173 -0.0177  0.0311
5        -0.0036  0.0127 -0.0227  0.0175

  Significant lags (p<0.05): 11 / 42

  B. GRANGER CAUSALITY  (sent -> return)
  Verdict per ticker (significant lags / 5):
    AAPL   2/5   ✅ causal
    MSFT   3/5   ✅ causal
    GOOGL  0/5   ❌ not causal
    AMZN   0/5   ❌ not causal
    META   5/5   ✅ causal
    TSLA   0/5   ❌ not causal
    NVDA   3/5   ✅ causal

  C. SHAP VALUES (LightGBM)
  📂 loaded 09_test.parquet (1,757 rows)

  Top 15 features:
    market_share_pos_lag1     0.01297  ██████████████████████████████
    quarter                   0.00569  █████████████
    market_share_neg_lag1     0.00535 

In [48]:
# ═══ SNAPSHOT — consigne les métriques de CE run dans RESULTS.txt ═══
import json, glob, os, shutil, traceback
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score
try:
    YEAR = int(pd.Timestamp(TEST_END).year); H = int(HORIZON)
    RES  = f"{BASE_DIR}/RESULTS.txt"
    test = pd.read_parquet(f"{OUTPUT_DIR}/09_test.parquet")
    with open(f"{MODELS_DIR}/feature_names.json") as fh:
        FEATS = json.load(fh)["features"]
    ycol = "target_direction"
    X = test[FEATS]; y = test[ycol].values.astype(int)
    rng = np.random.default_rng(42)
    def boot_ci(y_, p_, n=1000):
        idx = np.arange(len(y_)); a = []
        for _ in range(n):
            b = rng.choice(idx, len(idx), replace=True)
            if y_[b].min() == y_[b].max(): continue
            a.append(roc_auc_score(y_[b], p_[b]))
        return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))
    def perm_p(y_, p_, n=500):
        base = roc_auc_score(y_, p_); c = 0
        for _ in range(n):
            yp = rng.permutation(y_)
            if yp.min() == yp.max(): continue
            if abs(roc_auc_score(yp, p_) - 0.5) >= abs(base - 0.5): c += 1
        return (c + 1) / (n + 1)
    L = ["", "="*66,
         f"RUN  test_year={YEAR}  horizon={H}j   n_test={len(y)}  pos_rate={y.mean():.3f}",
         "-"*66,
         f"baseline classe majoritaire (acc) = {max(y.mean(), 1-y.mean()):.4f}"]
    for path in sorted(glob.glob(f"{MODELS_DIR}/*.joblib")):
        name = os.path.basename(path).replace(".joblib", "")
        try:
            mdl = joblib.load(path); p = mdl.predict_proba(X)[:, 1]
        except Exception as e:
            L.append(f"{name:22s} (skip: {type(e).__name__})"); continue
        auc = roc_auc_score(y, p); lo, hi = boot_ci(y, p); pv = perm_p(y, p)
        L.append(f"{name:22s} AUC={auc:.4f}  IC95=[{lo:.4f}, {hi:.4f}]  p_perm={pv:.4f}")
    try:
        bt = pd.read_parquet(f"{OUTPUT_DIR}/11_backtest.parquet")
        bt["y"] = (bt["ret_1d"] > 0).astype(int)
        per = bt.groupby("ticker").apply(
            lambda g: roc_auc_score(g["y"], g["pred_ens"]) if g["y"].nunique() > 1 else np.nan)
        L.append("per-ticker AUC (ensemble, ret_1d) : "
                 + "  ".join(f"{t}={v:.3f}" for t, v in per.sort_values().items()))
    except Exception as e:
        L.append(f"per-ticker indisponible ({type(e).__name__}: {e})")
    tagfull = f"_{YEAR}_h{H}"
    for f_ in ["09_test.parquet", "10_predictions.parquet", "11_backtest.parquet"]:
        s = f"{OUTPUT_DIR}/{f_}"
        if os.path.exists(s):
            shutil.copy2(s, s.replace(".parquet", tagfull + ".parquet"))
    phase2_tag = "" if (YEAR == 2023 and H == 10) else ("_2022" if (YEAR == 2022 and H == 10) else None)
    if phase2_tag is not None:
        if phase2_tag:
            shutil.copy2(f"{OUTPUT_DIR}/11_backtest.parquet",
                         f"{OUTPUT_DIR}/11_backtest{phase2_tag}.parquet")
        import lightgbm as lgb
        from sklearn.utils.class_weight import compute_sample_weight
        SENT_KEY = ("sent","finbert","sarcasm","fear","anger","joy","sadness","mentions","emoji")
        tech = [c for c in FEATS if not any(k in c for k in SENT_KEY)]
        tr = pd.read_parquet(f"{OUTPUT_DIR}/09_train.parquet")
        va = pd.read_parquet(f"{OUTPUT_DIR}/09_val.parquet")
        trv = pd.concat([tr, va])
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               max_depth=8, subsample=0.8, colsample_bytree=0.7,
                               random_state=42, verbosity=-1)
        m.fit(trv[tech], trv[ycol].astype(int),
              sample_weight=compute_sample_weight("balanced", trv[ycol].astype(int)))
        out = test[["ticker", "date"]].copy()
        out["pred_tech"] = m.predict_proba(test[tech])[:, 1]
        out.to_parquet(f"{OUTPUT_DIR}/proba_technical_only{phase2_tag}.parquet")
        L.append(f"proba_technical_only{phase2_tag}.parquet sauvegardé "
                 f"({len(out)} lignes, {len(tech)} features techniques)")
    txt = "\n".join(L)
    print(txt)
    with open(RES, "a") as fh:
        fh.write(txt + "\n")
except Exception:
    traceback.print_exc()



RUN  test_year=2022  horizon=10j   n_test=1757  pos_rate=0.392
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.6079
model_lgb              AUC=0.4630  IC95=[0.4362, 0.4884]  p_perm=0.0040
model_stack            (skip: AttributeError)
model_xgb              AUC=0.5273  IC95=[0.4989, 0.5572]  p_perm=0.0499
per-ticker AUC (ensemble, ret_1d) : META=0.410  AAPL=0.410  AMZN=0.423  NVDA=0.439  GOOGL=0.447  MSFT=0.483  TSLA=0.498
proba_technical_only_2022.parquet sauvegardé (1757 lignes, 31 features techniques)


## ▶ RUN test 2023 — horizon 10 j


In [49]:
# ═══ BASCULE CONFIG — test 2023, horizon 10 j ═══
import pandas as pd, glob, os
TRAIN_END = pd.Timestamp("2021-12-31")
VAL_END   = pd.Timestamp("2022-12-31")
TEST_END  = pd.Timestamp("2023-12-31")
HORIZON   = 10
try:
    CONFIG["horizon"] = HORIZON
except Exception:
    pass
KEEP = ("_2022", "_2023", "_h1.", "_h10.", "_h20.")
for pat in ["07_", "08_", "09_train", "09_val", "09_test", "10_predictions", "11_backtest"]:
    for f in glob.glob(f"{OUTPUT_DIR}/{pat}*.parquet"):
        b = os.path.basename(f)
        if any(s in b for s in KEEP):
            continue
        os.remove(f); print("rm", b)
for f in glob.glob(f"{MODELS_DIR}/*"):
    os.remove(f)
print(f"✅ CONFIG ACTIVE → train≤{TRAIN_END.date()}  val≤{VAL_END.date()}  test≤{TEST_END.date()}  H={HORIZON}j")


rm 07_merged.parquet
rm 08_features.parquet
rm 09_train.parquet
rm 09_val.parquet
rm 09_test.parquet
rm 11_backtest.parquet
✅ CONFIG ACTIVE → train≤2021-12-31  val≤2022-12-31  test≤2023-12-31  H=10j


## Cell 11 — Merge sentiment ⨉ prix + targets

In [50]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — MERGE SENTIMENT + PRICES + TARGETS                    ║
# ╚══════════════════════════════════════════════════════════════════╝
if ckpt.exists_and_valid(PATH_MERGED, min_rows=10):
    print(f"  ✅ {Path(PATH_MERGED).name} exists - skipping")
    df_merged = ckpt.load(PATH_MERGED)
else:
    df_sent  = ckpt.load(PATH_DAILY)
    df_price = ckpt.load(PATH_TECH)

    df_sent["date"]  = normalize_date_col(df_sent["date"])
    df_price["date"] = normalize_date_col(df_price["date"])
    df_sent["ticker"]  = df_sent["ticker"].astype(str)
    df_price["ticker"] = df_price["ticker"].astype(str)

    print(f"  Sentiment rows : {len(df_sent):,}")
    print(f"  Price    rows  : {len(df_price):,}")

    # Left join: keep every trading day, fill sentiment with neutral if missing
    df = df_price.merge(df_sent, on=["ticker", "date"], how="left")

    # Sentiment columns to fill
    sent_cols = ["sent_mean","sent_std","sent_min","sent_max","mentions",
                 "finbert_mean","finbert_std","sarcasm_mean","sarcasm_max",
                 "fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic",
                 "emoji_mean"]
    for c in sent_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    print(f"  Merged rows    : {len(df):,}")
    days_with_sent = (df["mentions"] > 0).sum()
    print(f"  Trading days WITH sentiment: {days_with_sent:,} "
          f"({100*days_with_sent/len(df):.1f}%)")

    # ── TARGETS ────────────────────────────────────────────────────
    # Forward returns at multiple horizons (per ticker, no cross-leakage)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    for h in CONFIG["horizons"]:
        df[f"close_t+{h}"]     = df.groupby("ticker", observed=True)["close"].shift(-h)
        df[f"return_t+{h}"]    = (df[f"close_t+{h}"] - df["close"]) / df["close"]
        df[f"direction_t+{h}"] = (df[f"return_t+{h}"] > 0).astype(np.int8)
    # Primary target
    _H = globals().get("HORIZON", 1)
    if f"return_t+{_H}" not in df.columns:
        print(f"  ⚠️ horizon {_H} absent de CONFIG[horizons] → repli sur 1")
        _H = 1
    _TARGET = globals().get("TARGET", "direction")
    if _TARGET == "volatility":
        # Rendement journalier (depuis close, robuste au nom de colonne)
        df["_r1"] = df.groupby("ticker", observed=True)["close"].pct_change()
        # Volatilité réalisée FORWARD sur t+1..t+H (= la cible, légitimement future)
        _fwd_sq = None
        for _k in range(1, _H + 1):
            _rk = df.groupby("ticker", observed=True)["_r1"].shift(-_k)
            _fwd_sq = _rk.pow(2) if _fwd_sq is None else _fwd_sq + _rk.pow(2)
        df["_fwd_vol"] = (_fwd_sq / _H) ** 0.5
        # Seuil = médiane GLISSANTE PASSÉE de la vol, fenêtre courte (~1 trimestre)
        # → cible "vol élevée RELATIVE au régime récent" : classes ~50/50 à toute
        # période (évite l effet régime calme/turbulent), et aucune info du futur.
        _VOL_WIN = 63
        _past = df.groupby("ticker", observed=True)["_r1"].transform(
            lambda x: x.rolling(_H, min_periods=max(3, _H // 2)).std())
        df["_thr"] = _past.groupby(df["ticker"], observed=True).transform(
            lambda x: x.rolling(_VOL_WIN, min_periods=20).median())
        df["target_return"]    = df["_fwd_vol"].where(df["_thr"].notna())
        df["target_direction"] = (df["_fwd_vol"] > df["_thr"]).astype(np.int8)
        df = df.drop(columns=["_r1", "_fwd_vol", "_thr"])
        print(f"  🎯 Cible = VOLATILITÉ à +{_H}j (1 = vol future > médiane glissante passée)")
    else:
        df["target_return"]    = df[f"return_t+{_H}"]
        df["target_direction"] = df[f"direction_t+{_H}"]
        print(f"  🎯 Cible = direction à +{_H} jour(s)")
    # Drop rows without target_return (last few rows per ticker)
    df = df[df["target_return"].notna()].copy()

    df = mem.optimize_df(df)
    ckpt.save(df, PATH_MERGED)
    df_merged = df

print(f"\n  Final rows : {len(df_merged):,}")
print(f"  Columns    : {len(df_merged.columns)}")
print(f"\n  Direction balance (t+1):")
print(f"    Up   : {(df_merged['target_direction']==1).sum():,}  "
      f"({100*(df_merged['target_direction']==1).mean():.1f}%)")
print(f"    Down : {(df_merged['target_direction']==0).sum():,}  "
      f"({100*(df_merged['target_direction']==0).mean():.1f}%)")


  📂 loaded 05_daily.parquet (23,258 rows)
  📂 loaded 06_prices_tech.parquet (19,355 rows)
  Sentiment rows : 23,258
  Price    rows  : 19,355
  Merged rows    : 19,355
  Trading days WITH sentiment: 16,506 (85.3%)
  🎯 Cible = direction à +10 jour(s)
  💾 saved 07_merged.parquet (19,285 rows | 6.4 MB)

  Final rows : 19,285
  Columns    : 75

  Direction balance (t+1):
    Up   : 11,535  (59.8%)
    Down : 7,750  (40.2%)


## Cell 12 — Feature engineering + split temporel

In [51]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — FEATURE ENGINEERING + TEMPORAL SPLIT                  ║
# ╚══════════════════════════════════════════════════════════════════╝
if all(ckpt.exists_and_valid(p) for p in (PATH_FEATURES, PATH_TRAIN, PATH_VAL, PATH_TEST)):
    print(f"  ✅ Feature splits exist - skipping")
    df_feat = ckpt.load(PATH_FEATURES)
else:
    df = ckpt.load(PATH_MERGED)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    print(f"  Input rows : {len(df):,}")

    SENT_COLS = ["sent_mean","sent_std","mentions","finbert_mean","emoji_mean",
                 "sarcasm_mean","fear_mean","anger_mean","joy_mean","sadness_mean",
                 "share_positive","share_negative","share_neutral","share_sarcastic"]

    print("\n  Lagging sentiment features (anti-leakage)...")
    # Shift each sentiment column by 1 day per ticker -> "yesterday's sentiment"
    for c in SENT_COLS:
        if c not in df.columns:
            continue   # robustesse : colonne absente (vieux checkpoint)
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # Rolling sentiment features (shift + rolling BOTH inside groupby to avoid
    # cross-ticker leakage at boundaries — transform() preserves original index).
    print("  Building sentiment rolling features...")
    for w in (3, 7, 14, 30):
        df[f"sent_ma_{w}d"] = (
            df.groupby("ticker", observed=True)["sent_mean"]
              .transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).mean())
        ).fillna(0)

    # Sentiment momentum (lagged)
    df["sent_momentum"] = (df["sent_ma_3d"] - df["sent_ma_14d"]).fillna(0)

    # Sarcasm-adjusted sentiment (lagged)
    df["sent_adj_lag1"] = (
        df["sent_mean_lag1"] * (1 - 0.5 * df["sarcasm_mean_lag1"])
    ).fillna(0).astype(np.float32)

    # Calendar features
    df["dow"]        = df["date"].dt.dayofweek.astype(np.int8)
    df["month"]      = df["date"].dt.month.astype(np.int8)
    df["quarter"]    = df["date"].dt.quarter.astype(np.int8)
    df["is_monday"]  = (df["dow"] == 0).astype(np.int8)
    df["is_friday"]  = (df["dow"] == 4).astype(np.int8)

    # Ticker as integer
    stock_map = {s: i for i, s in enumerate(sorted(STOCKS))}
    df["stock_id"] = df["ticker"].map(stock_map).astype(np.int8)

    # Fill NaN from lags (first row per ticker)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # Drop rows where target is missing
    df = df.dropna(subset=["target_return", "target_direction"]).copy()

    # ── MARKET-WIDE SENTIMENT FEATURES (cross-ticker) ─────────────
    # On any given day, what was the AVERAGE sentiment across ALL tickers?
    # This captures macro mood that no single ticker can show. We then
    # compute each ticker's DEVIATION from the market mood = their unique
    # signal.
    print("\n  Building market-wide cross-ticker features...")
    market_daily = (df.groupby("date", as_index=False)
                      .agg(market_sent_mean = ("sent_mean",        "mean"),
                           market_share_neg = ("share_negative",   "mean"),
                           market_share_pos = ("share_positive",   "mean"),
                           market_share_sarc= ("share_sarcastic",  "mean"),
                           market_mentions  = ("mentions",         "sum")))
    df = df.merge(market_daily, on="date", how="left")

    # Per-ticker deviation from market mean (negative → ticker more
    # pessimistic than the market, positive → more optimistic)
    df["sent_vs_market"] = (df["sent_mean"] - df["market_sent_mean"]).astype(np.float32)

    # Lag market features by 1 day (anti-leakage like the rest)
    for c in ("market_sent_mean", "market_share_neg", "market_share_pos",
              "market_share_sarc", "sent_vs_market"):
        df[f"{c}_lag1"] = df.groupby("ticker", observed=True)[c].shift(1)

    # ── FEATURE LIST ──────────────────────────────────────────────
    FEATURE_COLS = [
        # Technical
        "ret_1d","ret_3d","ret_5d","ret_10d","ret_20d",
        "vol_5d","vol_20d", "rsi_14",
        "macd","macd_signal","macd_hist",
        "bb_width","bb_pos","atr_14",
        "price_vs_sma20","price_vs_sma50",
        "vol_ratio","hl_range",
        # Sentiment (all lagged or rolling - no same-day)
        "sent_mean_lag1","sent_std_lag1","mentions_lag1",
        "finbert_mean_lag1","sarcasm_mean_lag1",
        "fear_mean_lag1","anger_mean_lag1",
        "joy_mean_lag1","sadness_mean_lag1",
        # 4-class category proportions (lagged) — captures whether
        # the previous day's chatter was bullish, bearish, neutral or ironic
        "share_positive_lag1","share_negative_lag1",
        "share_neutral_lag1","share_sarcastic_lag1",
    "emoji_mean_lag1",
        # Market-wide cross-ticker features (NEW)
        "market_sent_mean_lag1","market_share_neg_lag1",
        "market_share_pos_lag1","market_share_sarc_lag1",
        "sent_vs_market_lag1",
        # Rolling sentiment trends
        "sent_ma_3d","sent_ma_7d","sent_ma_14d","sent_ma_30d",
        "sent_momentum","sent_adj_lag1",
        # Calendar
        "dow","month","quarter","is_monday","is_friday",
        # Identifier
        "stock_id",
    ]
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    print(f"\n  Features  : {len(FEATURE_COLS)}")

    # Save list
    with open(f"{MODELS_DIR}/feature_names.json", "w") as f:
        json.dump({"features": FEATURE_COLS}, f, indent=2)

    # ── TEMPORAL SPLIT ────────────────────────────────────────────
    train = df[df["date"] <= TRAIN_END].copy()
    val   = df[(df["date"] > TRAIN_END) & (df["date"] <= VAL_END)].copy()
    test  = df[(df["date"] > VAL_END) & (df["date"] <= TEST_END)].copy()

    print(f"\n  Train : {len(train):>7,} rows   "
          f"({train['date'].min().date()} -> {train['date'].max().date()})")
    print(f"  Val   : {len(val):>7,} rows   "
          f"({val['date'].min().date()} -> {val['date'].max().date()})")
    print(f"  Test  : {len(test):>7,} rows   "
          f"({test['date'].min().date()} -> {test['date'].max().date()})")

    # Save everything
    df = mem.optimize_df(df)
    ckpt.save(df,    PATH_FEATURES)
    ckpt.save(train, PATH_TRAIN)
    ckpt.save(val,   PATH_VAL)
    ckpt.save(test,  PATH_TEST)

    # ── Liberation disque : les parquets ligne-par-ligne (raw/clean/finbert/sent)
    #    ne sont plus lus en aval (tout passe par les agregats journaliers). ──
    if globals().get("FREE_ROWLEVEL_AFTER_FEATURES", True):
        import os as _os
        # Sauvegarde d un petit résumé "volume par source" pour la Fig 9
        # (sinon perdu quand on supprime les parquets ligne-par-ligne).
        try:
            _srcp = globals().get("PATH_SENT", "") or PATH_CLEAN
            if _srcp and _os.path.exists(_srcp):
                _sc = pd.read_parquet(_srcp, columns=["source", "dtype"])
                _scg = _sc.groupby(["source", "dtype"]).size().reset_index(name="n")
                _scg.to_parquet(f"{OUTPUT_DIR}/source_counts.parquet", index=False)
                del _sc, _scg
        except Exception as _e:
            print(f"  (résumé sources non sauvé: {str(_e)[:40]})")
        for _p in (PATH_RAW, PATH_CLEAN, PATH_FB_OUT, globals().get("PATH_SENT", "")):
            try:
                if _p and _os.path.exists(_p):
                    _sz = _os.path.getsize(_p) / 1e9
                    _os.remove(_p)
                    print(f"  🧹 {Path(_p).name} supprimé (~{_sz:.2f} GB libérés)")
            except Exception: pass

    df_feat = df

print(f"\n  Features file : {Path(PATH_FEATURES).name} ({len(df_feat):,} rows)")


# ════════════════════════════════════════════════════════════════════
#  AUDIT ANTI-FUITE (leakage) — pour la section méthodo du papier
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 64)
print("  AUDIT ANTI-FUITE (look-ahead leakage)")
print("═" * 64)
import json as _json
_audit_ok = True
try:
    with open(f"{MODELS_DIR}/feature_names.json") as _f:
        _FC = _json.load(_f)["features"]
except Exception:
    _FC = FEATURE_COLS

# (a) Aucune colonne dérivée du futur/cible ne doit être une feature
_forbidden = ("target_", "return_t+", "close_t+", "direction_t+")
_leaky = [c for c in _FC if any(k in c for k in _forbidden)]
if _leaky:
    _audit_ok = False
    print(f"  ❌ Features suspectes (dérivées de la cible) : {_leaky}")
else:
    print(f"  ✅ Aucune feature dérivée de la cible ({len(_FC)} features)")

# (b) Intégrité temporelle du split (train < val < test)
try:
    _tr = ckpt.load(PATH_TRAIN); _va = ckpt.load(PATH_VAL); _te = ckpt.load(PATH_TEST)
    _trmax, _vamin = _tr["date"].max(), _va["date"].min()
    _vamax, _temin = _va["date"].max(), _te["date"].min()
    if _trmax < _vamin and _vamax < _temin:
        print(f"  ✅ Split chronologique strict : "
              f"train≤{_trmax.date()} < val≤{_vamax.date()} < test≤{_te['date'].max().date()}")
    else:
        _audit_ok = False
        print(f"  ❌ Chevauchement temporel : trmax={_trmax}, vamin={_vamin}, "
              f"vamax={_vamax}, temin={_temin}")

    # (c) Corrélation feature↔cible sur le TRAIN : flag |r|>0.30 (signe de fuite)
    import numpy as _np
    _y = _tr["target_direction"].astype(float).values
    _susp = []
    for _c in _FC:
        try:
            _x = _tr[_c].astype(float).values
            if _np.nanstd(_x) == 0: continue
            _r = _np.corrcoef(_np.nan_to_num(_x), _y)[0, 1]
            _voltgt = globals().get("TARGET", "direction") == "volatility"
            _volfam = ("vol_", "atr", "bb_", "rsi", "macd", "ret_", "return_", "hl_range")
            # En cible "volatility", les features de vol/momentum SONT légitimement
            # prédictives (la vol se regroupe) → ce n est pas une fuite.
            if abs(_r) > 0.30 and not (_voltgt and any(_w in _c for _w in _volfam)):
                _susp.append((_c, round(float(_r), 3)))
        except Exception:
            pass
    if _susp:
        _audit_ok = False
        print(f"  ⚠️ Features très corrélées à la cible (|r|>0.30) — à vérifier :")
        for _c, _r in sorted(_susp, key=lambda t: -abs(t[1]))[:10]:
            print(f"       {_c:<28} r={_r:+.3f}")
    else:
        print(f"  ✅ Aucune feature anormalement corrélée à la cible (|r|≤0.30)")
    del _tr, _va, _te
except Exception as _e:
    print(f"  (audit partiel : {str(_e)[:60]})")

print("─" * 64)
print(f"  VERDICT : {'✅ PAS DE FUITE DÉTECTÉE' if _audit_ok else '⚠️ POINTS À VÉRIFIER (voir ci-dessus)'}")
print("═" * 64)


  📂 loaded 07_merged.parquet (19,285 rows)
  Input rows : 19,285

  Lagging sentiment features (anti-leakage)...
  Building sentiment rolling features...

  Building market-wide cross-ticker features...

  Features  : 49

  Train :  12,341 rows   (2015-01-02 -> 2021-12-31)
  Val   :   1,757 rows   (2022-01-03 -> 2022-12-30)
  Test  :   1,750 rows   (2023-01-03 -> 2023-12-29)
  💾 saved 08_features.parquet (19,285 rows | 8.1 MB)
  💾 saved 09_train.parquet (12,341 rows | 5.7 MB)
  💾 saved 09_val.parquet (1,757 rows | 0.8 MB)
  💾 saved 09_test.parquet (1,750 rows | 0.8 MB)

  Features file : 08_features.parquet (19,285 rows)

════════════════════════════════════════════════════════════════
  AUDIT ANTI-FUITE (look-ahead leakage)
════════════════════════════════════════════════════════════════
  ✅ Aucune feature dérivée de la cible (49 features)
  📂 loaded 09_train.parquet (12,341 rows)
  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  ✅ Split chronologique st

## Cell 13 — Arbres : LightGBM + XGBoost + CatBoost

In [52]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — TRAIN MODELS                                          ║
# ╚══════════════════════════════════════════════════════════════════╝
import time

# Lazy installs
for pkg in ("lightgbm", "xgboost", "catboost"):
    try:
        __import__(pkg)
    except ImportError:
        pip_install(pkg)

import lightgbm as lgb
import xgboost  as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import joblib

# Load
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test  = ckpt.load(PATH_TEST)

# CORRECTION ANTI-FUITE : tri chronologique GLOBAL avant TimeSeriesSplit.
# (les fichiers sont tries [ticker, date] → la CV temporelle melangerait
#  les tickers et casserait l'ordre du temps.)
train = train.sort_values("date").reset_index(drop=True)
val   = val.sort_values("date").reset_index(drop=True)
test  = test.sort_values("date").reset_index(drop=True)

X_tr, y_tr = train[FEATURE_COLS].values, train["target_direction"].values.astype(int)
X_va, y_va = val[FEATURE_COLS].values,   val["target_direction"].values.astype(int)
X_te, y_te = test[FEATURE_COLS].values,  test["target_direction"].values.astype(int)

print(f"  Train : {X_tr.shape}  |  pos rate = {y_tr.mean():.3f}")
print(f"  Val   : {X_va.shape}  |  pos rate = {y_va.mean():.3f}")
print(f"  Test  : {X_te.shape}  |  pos rate = {y_te.mean():.3f}")

sw_tr = compute_sample_weight("balanced", y_tr)

# ── 1. LIGHTGBM ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  LightGBM")
print("─" * 60)
lgb_params = dict(
    objective="binary", metric="auc",
    n_estimators=800, learning_rate=0.03,
    num_leaves=63, max_depth=8,
    min_child_samples=40,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1,
)
tscv = TimeSeriesSplit(n_splits=5)
cv_lgb = []
clf_lgb = lgb.LGBMClassifier(**lgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_lgb.fit(
        X_tr[tr_i], y_tr[tr_i],
        sample_weight=sw_tr[tr_i],
        eval_set=[(X_tr[val_i], y_tr[val_i])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    auc = roc_auc_score(y_tr[val_i], clf_lgb.predict_proba(X_tr[val_i])[:, 1])
    cv_lgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_lgb):.4f} ± {np.std(cv_lgb):.4f}")
# Final fit
clf_lgb.fit(X_tr, y_tr, sample_weight=sw_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False)])
joblib.dump(clf_lgb, f"{MODELS_DIR}/model_lgb.joblib")
auc_lgb_val = roc_auc_score(y_va, clf_lgb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_lgb_val:.4f}")

# ── 2. XGBOOST ─────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  XGBoost")
print("─" * 60)
neg, pos = int((y_tr == 0).sum()), int((y_tr == 1).sum())
spw = neg / max(pos, 1)
xgb_params = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=600, learning_rate=0.05,
    max_depth=7, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw,
    random_state=42, n_jobs=-1, verbosity=0,
    early_stopping_rounds=50,
)
cv_xgb = []
clf_xgb = xgb.XGBClassifier(**xgb_params)
for fold, (tr_i, val_i) in enumerate(tscv.split(X_tr), 1):
    clf_xgb.fit(X_tr[tr_i], y_tr[tr_i],
                eval_set=[(X_tr[val_i], y_tr[val_i])],
                verbose=False)
    auc = roc_auc_score(y_tr[val_i], clf_xgb.predict_proba(X_tr[val_i])[:, 1])
    cv_xgb.append(auc)
    print(f"  fold {fold}/5  AUC={auc:.4f}")
print(f"  CV AUC mean   = {np.mean(cv_xgb):.4f} ± {np.std(cv_xgb):.4f}")
# Final fit (without early-stopping callback now)
xgb_params_final = {k: v for k, v in xgb_params.items() if k != "early_stopping_rounds"}
clf_xgb_final = xgb.XGBClassifier(**xgb_params_final)
clf_xgb_final.fit(X_tr, y_tr)
joblib.dump(clf_xgb_final, f"{MODELS_DIR}/model_xgb.joblib")
auc_xgb_val = roc_auc_score(y_va, clf_xgb_final.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_xgb_val:.4f}")

# ── 3. CATBOOST ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("  CatBoost")
print("─" * 60)
clf_cb = CatBoostClassifier(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=42, verbose=0,
    early_stopping_rounds=50,
)
clf_cb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=(X_va, y_va), use_best_model=True)
clf_cb.save_model(f"{MODELS_DIR}/model_cb.cbm")
auc_cb_val = roc_auc_score(y_va, clf_cb.predict_proba(X_va)[:, 1])
print(f"  Val AUC       = {auc_cb_val:.4f}")
print(f"  Best iter     = {clf_cb.best_iteration_}")

# Save model summary
model_summary = {
    "lgb": {"cv_auc_mean": float(np.mean(cv_lgb)),
            "cv_auc_std":  float(np.std(cv_lgb)),
            "val_auc":     float(auc_lgb_val)},
    "xgb": {"cv_auc_mean": float(np.mean(cv_xgb)),
            "cv_auc_std":  float(np.std(cv_xgb)),
            "val_auc":     float(auc_xgb_val)},
    "cb":  {"val_auc":     float(auc_cb_val),
            "best_iter":   int(clf_cb.best_iteration_)},
}
with open(f"{MODELS_DIR}/training_summary.json", "w") as f:
    json.dump(model_summary, f, indent=2)

print("\n" + "=" * 60)
print(f"  ✅ 3 models trained and saved to {MODELS_DIR}")
print("=" * 60)
mem.free()


  📂 loaded 09_train.parquet (12,341 rows)
  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  Train : (12341, 49)  |  pos rate = 0.618
  Val   : (1757, 49)  |  pos rate = 0.392
  Test  : (1750, 49)  |  pos rate = 0.683

────────────────────────────────────────────────────────────
  LightGBM
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.5214
  fold 2/5  AUC=0.5568
  fold 3/5  AUC=0.4926
  fold 4/5  AUC=0.5077
  fold 5/5  AUC=0.5376
  CV AUC mean   = 0.5232 ± 0.0224
  Val AUC       = 0.5976

────────────────────────────────────────────────────────────
  XGBoost
────────────────────────────────────────────────────────────
  fold 1/5  AUC=0.4861
  fold 2/5  AUC=0.5723
  fold 3/5  AUC=0.5363
  fold 4/5  AUC=0.5569
  fold 5/5  AUC=0.5381
  CV AUC mean   = 0.5379 ± 0.0291
  Val AUC       = 0.5848

────────────────────────────────────────────────────────────
  CatBoost
────────────────────────────────────────────────────────────
  V

## Cell 13b — Modele sequentiel hybride (BiGRU + attention)  ⭐ NOUVEAU

Apporte la composante « deep » de l'ensemble hybride. Pour chaque (ticker, jour) on construit une
**fenetre glissante de `SEQ_LEN` jours** du vecteur de features (sentiment lagge + techniques), normalisee
sur le **train uniquement** (anti-fuite). Un **BiGRU bidirectionnel + attention** apprend les dynamiques
temporelles que les arbres ne capturent pas. Les probabilites val/test sont sauvegardees pour le stacking.
GPU recommande ; degrade proprement en CPU (ou se desactive si torch absent).

In [53]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13b — BiGRU + ATTENTION (modele sequentiel)                ║
# ║  Sortie : {OUTPUT_DIR}/seq_proba.parquet  [ticker,date,p_seq,split]║
# ╚══════════════════════════════════════════════════════════════════╝
import json, time
import numpy as np
import pandas as pd
from pathlib import Path

SEQ_OUT = f"{OUTPUT_DIR}/seq_proba.parquet"

if not USE_SEQ_MODEL:
    print("  USE_SEQ_MODEL=False → modele sequentiel desactive.")
elif Path(SEQ_OUT).exists():
    print(f"  ✅ {Path(SEQ_OUT).name} existe — skip BiGRU")
else:
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        pip_install("torch"); import torch; import torch.nn as nn

    from sklearn.metrics import roc_auc_score

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device : {device}")

    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    df = ckpt.load(PATH_FEATURES)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # ── Standardisation (stats du TRAIN uniquement) ──────────────────
    train_mask = df["date"] <= TRAIN_END
    mu = df.loc[train_mask, FEATURE_COLS].mean()
    sd = df.loc[train_mask, FEATURE_COLS].std().replace(0, 1.0)
    Xall = ((df[FEATURE_COLS] - mu) / sd).fillna(0.0).values.astype("float32")
    y    = df["target_direction"].values.astype("float32")
    tks  = df["ticker"].values
    dts  = df["date"].values
    n_feat = Xall.shape[1]
    L = int(SEQ_LEN)

    # ── Construction des fenetres glissantes par ticker ──────────────
    print(f"  Construction des fenetres (L={L}, n_feat={n_feat})...")
    W, yy, wt, wd = [], [], [], []
    for tk in sorted(set(tks)):
        idx = np.where(tks == tk)[0]
        Xi, yi, di = Xall[idx], y[idx], dts[idx]
        for j in range(len(idx)):
            lo = max(0, j - L + 1)
            w = Xi[lo:j + 1]
            if len(w) < L:                       # padding par la 1ere ligne
                w = np.vstack([np.repeat(w[:1], L - len(w), axis=0), w])
            W.append(w); yy.append(yi[j]); wt.append(tk); wd.append(di[j])
    W  = np.asarray(W, dtype="float32")
    yy = np.asarray(yy, dtype="float32")
    wd = pd.to_datetime(pd.Series(wd))
    print(f"  Fenetres : {W.shape}")

    tr_m = (wd <= TRAIN_END).values
    va_m = ((wd > TRAIN_END) & (wd <= VAL_END)).values
    te_m = ((wd > VAL_END) & (wd <= TEST_END)).values

    # Borne memoire : sous-echantillonne le train si gigantesque (CPU)
    MAX_TR = 400_000 if device.type == "cuda" else 120_000
    tr_idx = np.where(tr_m)[0]
    if len(tr_idx) > MAX_TR:
        rng = np.random.default_rng(42)
        tr_idx = rng.choice(tr_idx, MAX_TR, replace=False)
    va_idx = np.where(va_m)[0]
    te_idx = np.where(te_m)[0]
    print(f"  train={len(tr_idx):,}  val={len(va_idx):,}  test={len(te_idx):,}")

    # ── Modele : BiGRU bidirectionnel + attention additive ───────────
    class BiGRUAttn(nn.Module):
        def __init__(self, n_feat, hidden=64, layers=1, p=0.3):
            super().__init__()
            self.gru = nn.GRU(n_feat, hidden, num_layers=layers,
                              batch_first=True, bidirectional=True,
                              dropout=p if layers > 1 else 0.0)
            self.attn = nn.Linear(hidden * 2, 1)
            self.head = nn.Sequential(
                nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(p),
                nn.Linear(64, 1))
        def forward(self, x):
            h, _ = self.gru(x)                       # (B, L, 2H)
            a = torch.softmax(self.attn(h).squeeze(-1), dim=1)  # (B, L)
            ctx = (h * a.unsqueeze(-1)).sum(dim=1)   # (B, 2H)
            return self.head(ctx).squeeze(-1)

    model = BiGRUAttn(n_feat).to(device)
    pos_w = float((yy[tr_idx] == 0).sum()) / max(float((yy[tr_idx] == 1).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    Wt = torch.from_numpy(W)   # garde en CPU, on transfere par batch

    def run_eval(idx):
        model.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(idx), SEQ_BATCH):
                b = idx[i:i + SEQ_BATCH]
                xb = Wt[b].to(device)
                outs.append(torch.sigmoid(model(xb)).cpu().numpy())
        return np.concatenate(outs) if outs else np.array([])

    best_auc, best_state, patience, bad = -1.0, None, 5, 0
    print("  Entrainement BiGRU...")
    for ep in range(1, int(SEQ_EPOCHS) + 1):
        model.train()
        rng = np.random.default_rng(ep)
        perm = rng.permutation(tr_idx)
        t0 = time.time()
        for i in range(0, len(perm), SEQ_BATCH):
            b = perm[i:i + SEQ_BATCH]
            xb = Wt[b].to(device)
            yb = torch.from_numpy(yy[b]).to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        p_va = run_eval(va_idx)
        auc = roc_auc_score(yy[va_idx], p_va) if len(set(yy[va_idx])) > 1 else 0.5
        print(f"    epoch {ep:>2}  val_AUC={auc:.4f}  ({time.time()-t0:.1f}s)")
        if auc > best_auc:
            best_auc, best_state, bad = auc, {k: v.cpu().clone()
                                              for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print("    early stopping"); break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  ✅ meilleur val_AUC = {best_auc:.4f}")

    # ── Probabilites val + test → parquet pour le stacking ───────────
    out = pd.DataFrame({
        "ticker": np.r_[np.array(wt)[va_idx], np.array(wt)[te_idx]],
        "date":   pd.to_datetime(np.r_[wd.values[va_idx], wd.values[te_idx]]),
        "p_seq":  np.r_[run_eval(va_idx), run_eval(te_idx)].astype("float32"),
        "split":  (["val"] * len(va_idx)) + (["test"] * len(te_idx)),
    })
    out["date"] = out["date"].dt.normalize()
    out.to_parquet(SEQ_OUT, index=False)
    torch.save(model.state_dict(), f"{MODELS_DIR}/model_seq.pt")
    with open(f"{MODELS_DIR}/seq_summary.json", "w") as f:
        json.dump({"val_auc": float(best_auc), "seq_len": L,
                   "n_feat": int(n_feat)}, f, indent=2)
    print(f"  💾 {Path(SEQ_OUT).name}  ({len(out):,} rows)  + model_seq.pt")
    mem.free()


  ✅ seq_proba.parquet existe — skip BiGRU


## Cell 13c — Stacking hybride + calibration  ⭐ NOUVEAU

Combine les **4 modeles de base** (LightGBM, XGBoost, CatBoost, BiGRU) via un **meta-modele logistique**
ajuste sur la validation, puis applique une **calibration isotone** (probabilites mieux calibrees → meilleures
decisions de trading). Produit `stack_proba.parquet` (proba finale alignee sur les lignes de test) que la
CELL 14 utilise automatiquement.

In [54]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13c — STACKING (meta-logistique) + CALIBRATION ISOTONE     ║
# ╚══════════════════════════════════════════════════════════════════╝
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from catboost import CatBoostClassifier

if not USE_STACKING:
    print("  USE_STACKING=False → stacking desactive (CELL 14 utilisera la moyenne fixe).")
else:
    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]

    val  = ckpt.load(PATH_VAL).sort_values(["ticker", "date"]).reset_index(drop=True)
    test = ckpt.load(PATH_TEST).sort_values(["ticker", "date"]).reset_index(drop=True)
    for d in (val, test):
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()

    clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
    clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
    clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

    def tree_probas(d):
        X = d[FEATURE_COLS].values
        return np.column_stack([
            clf_lgb.predict_proba(X)[:, 1],
            clf_xgb.predict_proba(X)[:, 1],
            clf_cb.predict_proba(X)[:, 1],
        ])

    Pva, Pte = tree_probas(val), tree_probas(test)
    base_names = ["lgb", "xgb", "cb"]

    # ── Ajout du BiGRU si disponible ─────────────────────────────────
    seqp = f"{OUTPUT_DIR}/seq_proba.parquet"
    if Path(seqp).exists():
        sp = pd.read_parquet(seqp)
        sp["date"] = pd.to_datetime(sp["date"]).dt.normalize()
        def add_seq(d, P):
            m = d[["ticker", "date"]].merge(
                sp[["ticker", "date", "p_seq"]], on=["ticker", "date"], how="left")
            return np.column_stack([P, m["p_seq"].fillna(0.5).values])
        Pva, Pte = add_seq(val, Pva), add_seq(test, Pte)
        base_names.append("seq")
        print("  ✅ BiGRU integre au stacking")
    else:
        print("  ℹ️  Pas de seq_proba.parquet — stacking sur les 3 arbres seulement")

    yva = val["target_direction"].astype(int).values
    yte = test["target_direction"].astype(int).values

    # ── Meta-modele logistique (ajuste sur la validation) ────────────
    meta = LogisticRegression(max_iter=2000, C=1.0)
    meta.fit(Pva, yva)
    raw_va = meta.predict_proba(Pva)[:, 1]
    raw_te = meta.predict_proba(Pte)[:, 1]

    # ── Calibration isotone (sur la validation) ──────────────────────
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va, yva)
    p_stack    = iso.transform(raw_te).astype("float32")
    p_stack_va = iso.transform(raw_va).astype("float32")
    # Seuil de décision calibré sur la VALIDATION (quantile = taux de hausse val).
    # Évite l effondrement de l ensemble (sinon il prédit presque tout "baisse").
    _thr_stack = float(np.quantile(p_stack_va, 1.0 - float(np.mean(yva))))
    print(f"  Seuil ensemble calibré (val) : {_thr_stack:.4f}")

    # ── Rapport AUC base vs stack ────────────────────────────────────
    print("\n  AUC (test) par modele de base :")
    for i, nm in enumerate(base_names):
        print(f"    {nm:<5} {roc_auc_score(yte, Pte[:, i]):.4f}")
    print(f"    {'STACK':<5} {roc_auc_score(yte, p_stack):.4f}  "
          f"(acc={accuracy_score(yte, (p_stack>0.5).astype(int)):.4f})")
    print(f"  Poids meta (logistique) : "
          f"{dict(zip(base_names, np.round(meta.coef_[0], 3)))}")

    # ── Sauvegardes ──────────────────────────────────────────────────
    out = test[["ticker", "date"]].copy()
    out["p_stack"] = p_stack
    out.to_parquet(f"{OUTPUT_DIR}/stack_proba.parquet", index=False)
    joblib.dump({"meta": meta, "iso": iso, "names": base_names},
                f"{MODELS_DIR}/model_stack.joblib")
    with open(f"{MODELS_DIR}/stack_summary.json", "w") as f:
        json.dump({"base_names": base_names,
                   "weights": dict(zip(base_names, meta.coef_[0].tolist())),
                   "stack_test_auc": float(roc_auc_score(yte, p_stack)),
                   "threshold": _thr_stack}, f, indent=2)
    with open(f"{MODELS_DIR}/stack_threshold.json", "w") as f:
        json.dump({"threshold": _thr_stack,
                   "val_pos_rate": float(np.mean(yva))}, f)
    print(f"  💾 stack_proba.parquet + model_stack.joblib")


  📂 loaded 09_val.parquet (1,757 rows)
  📂 loaded 09_test.parquet (1,750 rows)
  ✅ BiGRU integre au stacking
  Seuil ensemble calibré (val) : 0.4145

  AUC (test) par modele de base :
    lgb   0.6548
    xgb   0.6467
    cb    0.5942
    seq   0.5841
    STACK 0.6572  (acc=0.3217)
  Poids meta (logistique) : {'lgb': np.float64(1.481), 'xgb': np.float64(0.449), 'cb': np.float64(0.517), 'seq': np.float64(0.962)}
  💾 stack_proba.parquet + model_stack.joblib


## Cell 14 — Ensemble (stacking hybride), evaluation & backtest

In [55]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — ENSEMBLE + EVAL + BACKTEST                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import joblib
from catboost import CatBoostClassifier

with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

test = ckpt.load(PATH_TEST)
X_te = test[FEATURE_COLS].values
y_te = test["target_direction"].values.astype(int)

# Load models
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")

# Probabilities
p_lgb = clf_lgb.predict_proba(X_te)[:, 1]
p_xgb = clf_xgb.predict_proba(X_te)[:, 1]
p_cb  = clf_cb.predict_proba(X_te)[:, 1]

# ── Ensemble HYBRIDE : on privilegie la pile apprise (CELL 13c :
#    arbres + BiGRU + meta-logistique calibree) si elle existe, sinon
#    on retombe sur la moyenne ponderee fixe.
from pathlib import Path as _P
_stack_path = f"{OUTPUT_DIR}/stack_proba.parquet"
_p_fixed = 0.4 * p_lgb + 0.3 * p_xgb + 0.3 * p_cb
if _P(_stack_path).exists():
    _sp = pd.read_parquet(_stack_path)
    _sp["date"] = pd.to_datetime(_sp["date"]).dt.normalize()
    _key = test[["ticker", "date"]].copy()
    _key["date"] = pd.to_datetime(_key["date"]).dt.normalize()
    _m = _key.merge(_sp[["ticker", "date", "p_stack"]], on=["ticker", "date"], how="left")
    p_ens = _m["p_stack"].fillna(pd.Series(_p_fixed)).values
    print("  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)")
else:
    p_ens = _p_fixed
    print("  ℹ️  Ensemble = moyenne ponderee fixe (stacking absent)")

# Seuil ensemble par CALAGE DE TAUX : on aligne la proportion de "hausse"
# prédite sur le taux de hausse de la VALIDATION (a priori connu, pas de
# fuite des labels test). Robuste au fort déséquilibre des horizons longs.
try:
    with open(f"{MODELS_DIR}/stack_threshold.json") as _f:
        _vpr = float(json.load(_f).get("val_pos_rate", 0.5))
    _THR_ENS = float(np.quantile(p_ens, 1.0 - _vpr))
except Exception:
    _THR_ENS = 0.5

# Per-model metrics (seuil par modèle : ensemble = seuil calibré)
def metrics_row(name, p, thr=0.5):
    yp = (p > thr).astype(int)
    return {
        "name":      name,
        "accuracy":  float(accuracy_score(y_te, yp)),
        "auc":       float(roc_auc_score(y_te, p)),
        "precision": float(precision_score(y_te, yp, zero_division=0)),
        "recall":    float(recall_score(y_te, yp, zero_division=0)),
        "f1":        float(f1_score(y_te, yp, zero_division=0)),
    }
rows = [metrics_row("lightgbm", p_lgb), metrics_row("xgboost", p_xgb),
        metrics_row("catboost", p_cb)]
# Ensemble (stacking) exclu : dégénère sous fort déséquilibre et n'apporte
# rien (AUC ≈ modèles de base). Modèles retenus : LightGBM/XGBoost/CatBoost.
results = pd.DataFrame(rows).set_index("name").round(4)

print("=" * 60)
print("  TEST SET METRICS")
print("=" * 60)
print(results.to_string())

# ── BACKTEST ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BACKTEST  (threshold 0.55 long / 0.45 short)")
print("=" * 60)
_is_vol = globals().get("TARGET", "direction") == "volatility"
if _is_vol:
    print("  ⚠️ Cible = VOLATILITÉ : ce backtest directionnel (long/short) n a PAS")
    print("     de sens (on prédit l amplitude, pas le sens du mouvement).")
    print("     → À IGNORER pour le papier. Utiliser AUC + ablation à la place.")
    print("     (Calculé seulement pour ne pas casser la figure du dashboard.)")
bt = test[["ticker", "date", "close"]].copy()
bt["pred_ens"] = p_lgb   # backtest sur LightGBM (modèle principal)
bt = bt.sort_values(["ticker", "date"]).reset_index(drop=True)
# Rendement RÉALISÉ à 1 jour (close-to-close). On n'utilise JAMAIS le target
# à H jours pour composer : sinon on cumulerait un rendement H-jours chaque
# jour (fenêtres chevauchantes) → explosion irréaliste du cumul.
bt["ret_1d"] = bt.groupby("ticker", observed=True)["close"].pct_change().fillna(0.0)
bt["signal"]   = 0
bt.loc[bt["pred_ens"] > 0.55, "signal"] = 1
bt.loc[bt["pred_ens"] < 0.45, "signal"] = -1

# Strategy return = signal de la veille × rendement réalisé du jour
bt["signal_lag"]      = bt.groupby("ticker", observed=True)["signal"].shift(1).fillna(0)
bt["strategy_return"] = bt["signal_lag"] * bt["ret_1d"]

# ── Transaction costs (10 bps per round-trip = realistic retail broker)
TXN_COST_BPS = 10
bt["pos_change"]      = (bt["signal"] != bt["signal_lag"]).astype(np.int8)
bt["txn_cost"]        = bt["pos_change"] * (TXN_COST_BPS / 10000.0)
bt["strategy_net"]    = bt["strategy_return"] - bt["txn_cost"]

bt["cum_strategy"]    = bt.groupby("ticker", observed=True)["strategy_net"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())
bt["cum_market"]      = bt.groupby("ticker", observed=True)["ret_1d"].transform(
    lambda x: (1 + x.fillna(0)).cumprod())

# Per-ticker summary
backtest_summary = {}
for tk, g in bt.groupby("ticker", observed=True):
    ret_s   = (g["cum_strategy"].iloc[-1] - 1) * 100
    ret_m   = (g["cum_market"].iloc[-1]   - 1) * 100
    sr_std  = g["strategy_net"].std()
    sharpe  = (g["strategy_net"].mean() / sr_std * np.sqrt(252)) if sr_std > 0 else 0.0
    n_tr    = int(g["pos_change"].sum())
    act     = g[g["signal"] != 0]["strategy_net"]
    win_rt  = float((act > 0).mean()) if len(act) else 0.0
    txn_drag = float(g["txn_cost"].sum() * 100)
    backtest_summary[tk] = {
        "return_strategy":  round(float(ret_s), 2),
        "return_market":    round(float(ret_m), 2),
        "alpha":            round(float(ret_s - ret_m), 2),
        "sharpe":           round(float(sharpe), 3),
        "n_trades":         n_tr,
        "win_rate":         round(win_rt, 4),
        "txn_cost_drag":    round(txn_drag, 2),     # cumulative cost in %
    }

# Aggregate
total_s = (bt.groupby("date")["strategy_net"].mean().fillna(0) + 1).cumprod().iloc[-1] - 1
total_m = (bt.groupby("date")["ret_1d"].mean().fillna(0)   + 1).cumprod().iloc[-1] - 1
_daily_strat = bt.groupby("date")["strategy_net"].mean().fillna(0)
_eq = (1 + _daily_strat).cumprod()
_max_dd = float((_eq / _eq.cummax() - 1).min())
_sh_overall = (float(_daily_strat.mean() / _daily_strat.std() * np.sqrt(252))
               if _daily_strat.std() > 0 else 0.0)
backtest_summary["__overall__"] = {
    "return_strategy": round(float(total_s * 100), 2),
    "return_market":   round(float(total_m * 100), 2),
    "alpha":           round(float((total_s - total_m) * 100), 2),
    "sharpe":          round(_sh_overall, 3),
    "max_drawdown":    round(_max_dd * 100, 2),
}

print(f"\n  Strategy total : {total_s*100:+.2f}%")
print(f"  Market   total : {total_m*100:+.2f}%")
print(f"  Alpha          : {(total_s - total_m)*100:+.2f}%\n")
print(f"  {'Ticker':<8} {'Strategy':>10} {'Market':>10} {'Alpha':>9} {'Sharpe':>7} {'Trades':>7} {'WinR':>6}")
print("  " + "─" * 60)
for tk in STOCKS:
    if tk not in backtest_summary: continue
    s = backtest_summary[tk]
    print(f"  {tk:<8} {s['return_strategy']:>+9.2f}% {s['return_market']:>+9.2f}% "
          f"{s['alpha']:>+8.2f}% {s['sharpe']:>7.2f} {s['n_trades']:>7d} {s['win_rate']*100:>5.1f}%")

# Save
ckpt.save(bt, PATH_BACKTEST)
results.to_json(f"{MODELS_DIR}/test_metrics.json", indent=2)
with open(f"{MODELS_DIR}/backtest_summary.json", "w") as f:
    json.dump(backtest_summary, f, indent=2)

# Confusion matrix (LightGBM)
print("\n  Confusion matrix (LightGBM):")
yp_ens = (p_lgb > 0.5).astype(int)
cm = confusion_matrix(y_te, yp_ens)
print(f"             pred_down  pred_up")
print(f"  true_down   {cm[0,0]:>7,}  {cm[0,1]:>7,}")
print(f"  true_up     {cm[1,0]:>7,}  {cm[1,1]:>7,}")

mem.free()


# ════════════════════════════════════════════════════════════════════
#  BASELINES + SIGNIFICATIVITÉ STATISTIQUE — pour l'acceptation du papier
# ════════════════════════════════════════════════════════════════════
import numpy as _np
from sklearn.metrics import roc_auc_score as _auc

print("\n" + "═" * 64)
print("  BASELINES & SIGNIFICATIVITÉ (test set)")
print("═" * 64)

# Baselines naïves
_base_up   = float((y_te == 1).mean())          # "toujours hausse"
_base_maj  = max(_base_up, 1 - _base_up)         # classe majoritaire
print(f"  Baseline 'toujours hausse' : acc = {_base_up:.4f}")
print(f"  Baseline classe majoritaire: acc = {_base_maj:.4f}")
print(f"  (un modèle utile doit battre {_base_maj:.4f} de façon significative)")

def _auc_safe(yt, p):
    try: return _auc(yt, p)
    except Exception: return 0.5

# Bootstrap IC 95% + test de permutation pour l'AUC de chaque modèle
_rng = _np.random.default_rng(42)
_N_BOOT, _N_PERM = 1000, 1000
print(f"\n  {'Modèle':<11} {'AUC':>7}  {'IC 95% bootstrap':>22}  {'p (perm)':>9}")
print("  " + "-" * 56)
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    _p = _np.asarray(_p, dtype=float)
    _obs = _auc_safe(y_te, _p)
    # bootstrap
    _bs = []
    _n = len(y_te)
    for _ in range(_N_BOOT):
        _idx = _rng.integers(0, _n, _n)
        if len(_np.unique(y_te[_idx])) < 2: continue
        _bs.append(_auc_safe(y_te[_idx], _p[_idx]))
    _lo, _hi = (_np.percentile(_bs, [2.5, 97.5]) if _bs else (_np.nan, _np.nan))
    # permutation : H0 = pas de lien (on mélange y)
    _null = _np.empty(_N_PERM)
    for _i in range(_N_PERM):
        _null[_i] = _auc_safe(_rng.permutation(y_te), _p)
    _pval = float((_np.sum(_np.abs(_null - 0.5) >= abs(_obs - 0.5)) + 1) / (_N_PERM + 1))
    _star = "***" if _pval < 0.01 else "**" if _pval < 0.05 else "*" if _pval < 0.10 else "ns"
    print(f"  {_name:<11} {_obs:>7.4f}  [{_lo:.4f}, {_hi:.4f}]  {_pval:>8.4f} {_star}")

print("\n  Lecture : un AUC est crédible si son IC 95% exclut 0.50 et p<0.05.")
print("  *** p<0.01  ** p<0.05  * p<0.10  ns = non significatif")
print("═" * 64)


# ════════════════════════════════════════════════════════════════════
#  MODE ABSTENTION — accuracy vs couverture (sélectivité de confiance)
#  Le modèle n'agit que sur ses prédictions les plus confiantes.
#  Courbe publiable : "accuracy croît quand on réduit la couverture".
# ════════════════════════════════════════════════════════════════════
import numpy as _np
print("\n" + "═" * 64)
print("  MODE ABSTENTION — accuracy sur les prédictions les + confiantes")
print("═" * 64)

def _conf_curve(name, p, thr=0.5):
    p = _np.asarray(p, dtype=float)
    conf = _np.abs(p - thr)                 # distance au seuil de décision = confiance
    order = _np.argsort(-conf)              # plus confiant d'abord
    yp_all = (p > thr).astype(int)
    rows = []
    for cov in (1.00, 0.50, 0.30, 0.20, 0.10, 0.05):
        k = max(1, int(round(len(p) * cov)))
        idx = order[:k]
        acc = float((yp_all[idx] == y_te[idx]).mean())
        rows.append((cov, k, acc))
    return rows

print(f"  {'Couverture':>10} {'N trades':>9} {'Accuracy':>9}")
print("  " + "-" * 32)
_abst_out = {}
for _name, _p in (("lightgbm", p_lgb), ("xgboost", p_xgb),
                  ("catboost", p_cb)):
    print(f"  ── {_name} ──")
    _thr_a = _THR_ENS if _name == "ensemble" else 0.5
    _rows = _conf_curve(_name, _p, _thr_a)
    _abst_out[_name] = [{"coverage": c, "n": k, "accuracy": a} for c, k, a in _rows]
    for cov, k, acc in _rows:
        print(f"  {cov*100:>9.0f}% {k:>9,} {acc:>9.4f}")

try:
    with open(f"{MODELS_DIR}/abstention_curve.json", "w") as _f:
        json.dump(_abst_out, _f, indent=2)
except Exception:
    pass

print("\n  Lecture : si l'accuracy monte quand la couverture baisse, le modèle")
print("  'sait quand il sait' — argument fort pour le papier (trading sélectif).")
print("  À reporter en figure : accuracy (y) vs couverture (x), par modèle.")
print("═" * 64)


  📂 loaded 09_test.parquet (1,750 rows)
  ✅ Ensemble = STACKING hybride (arbres + BiGRU, calibre)
  TEST SET METRICS
          accuracy     auc  precision  recall      f1
name                                                 
lightgbm    0.6451  0.6548     0.7424  0.7356  0.7390
xgboost     0.6491  0.6467     0.7419  0.7456  0.7437
catboost    0.6246  0.5942     0.7010  0.7849  0.7406

  BACKTEST  (threshold 0.55 long / 0.45 short)

  Strategy total : +53.85%
  Market   total : +112.33%
  Alpha          : -58.48%

  Ticker     Strategy     Market     Alpha  Sharpe  Trades   WinR
  ────────────────────────────────────────────────────────────
  AAPL        +74.78%    +54.80%   +19.99%    3.38      73  48.0%
  MSFT        +60.45%    +58.35%    +2.10%    2.27      79  50.5%
  GOOGL       +56.43%    +56.74%    -0.31%    1.81      75  43.6%
  AMZN        +43.46%    +77.04%   -33.59%    1.44      90  41.8%
  META        -42.05%   +183.76%  -225.81%   -1.42      92  34.1%
  TSLA        +91.91% 

## Cell 15 — Analyse causale (Pearson lags + Granger + SHAP + Ablation)

In [56]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — CAUSAL ANALYSIS                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
for pkg in ("statsmodels", "shap"):
    try: __import__(pkg)
    except ImportError: pip_install(pkg)

from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
import shap

df = ckpt.load(PATH_FEATURES)
df["date"] = pd.to_datetime(df["date"])

# ── A. PEARSON LAG CORRELATIONS (per ticker) ──────────────────────
print("=" * 60)
print("  A. PEARSON LAG CORRELATIONS")
print("=" * 60)
lag_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 50: continue
    for lag in range(0, 6):
        s_lagged = ds["sent_mean"].shift(lag).dropna()
        ret_aligned = ds["return_1d"].iloc[lag:].reset_index(drop=True)
        s_lagged    = s_lagged.reset_index(drop=True)
        n = min(len(s_lagged), len(ret_aligned))
        if n < 30:
            r_val, p_val = np.nan, np.nan
        else:
            r_val, p_val = stats.pearsonr(s_lagged[:n], ret_aligned[:n])
        lag_rows.append({
            "ticker":      tk,
            "lag_days":    lag,
            "correlation": round(float(r_val), 4) if not np.isnan(r_val) else None,
            "p_value":     round(float(p_val), 4) if not np.isnan(p_val) else None,
            "significant": (p_val < 0.05) if not np.isnan(p_val) else False,
        })
df_lags = pd.DataFrame(lag_rows)
df_lags.to_csv(f"{MODELS_DIR}/lag_correlations.csv", index=False)
print("\n  Lag-correlation summary:")
print(df_lags.groupby("lag_days")["correlation"].describe()[
      ["mean", "std", "min", "max"]].round(4).to_string())
print(f"\n  Significant lags (p<0.05): {df_lags['significant'].sum()} / {len(df_lags)}")

# ── B. GRANGER CAUSALITY ──────────────────────────────────────────
print("\n" + "=" * 60)
print("  B. GRANGER CAUSALITY  (sent -> return)")
print("=" * 60)
granger_rows = []
for tk in STOCKS:
    ds = df[df["ticker"] == tk][["date", "sent_mean", "return_1d"]].dropna()
    if len(ds) < 100: continue
    try:
        data = ds[["return_1d", "sent_mean"]].values
        gc_res = grangercausalitytests(data, maxlag=5, verbose=False)
        for lag, tests in gc_res.items():
            granger_rows.append({
                "ticker":  tk,
                "lag":     lag,
                "p_value": round(float(tests[0]["ssr_ftest"][1]), 4),
                "causal":  bool(tests[0]["ssr_ftest"][1] < 0.05),
            })
    except Exception as e:
        print(f"  ⚠️  {tk}: {str(e)[:80]}")
df_granger = pd.DataFrame(granger_rows)
df_granger.to_csv(f"{MODELS_DIR}/granger_results.csv", index=False)
print("  Verdict per ticker (significant lags / 5):")
for tk in STOCKS:
    g = df_granger[df_granger["ticker"] == tk]
    if g.empty: continue
    n_sig = int(g["causal"].sum())
    verdict = "✅ causal" if n_sig >= 2 else "⚠️  weak" if n_sig == 1 else "❌ not causal"
    print(f"    {tk:<6} {n_sig}/5   {verdict}")

# ── C. SHAP VALUES ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  C. SHAP VALUES (LightGBM)")
print("=" * 60)
import joblib
clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
test = ckpt.load(PATH_TEST)
with open(f"{MODELS_DIR}/feature_names.json") as f:
    FEATURE_COLS = json.load(f)["features"]

# Sample for speed
sample_n   = min(3000, len(test))
X_sample   = test[FEATURE_COLS].sample(sample_n, random_state=42).values
explainer  = shap.TreeExplainer(clf_lgb)
shap_vals  = explainer.shap_values(X_sample)
if isinstance(shap_vals, list):    # for binary classifier returns [neg, pos]
    sv = shap_vals[1]
else:
    sv = shap_vals
mean_abs   = np.abs(sv).mean(axis=0)
shap_df    = pd.DataFrame({"feature": FEATURE_COLS, "mean_abs_shap": mean_abs}) \
                .sort_values("mean_abs_shap", ascending=False)
shap_df.to_csv(f"{MODELS_DIR}/shap_importance.csv", index=False)

print("\n  Top 15 features:")
total = shap_df["mean_abs_shap"].sum()
SENT_KEY = ("sent", "finbert", "sarcasm", "fear", "anger", "joy", "sadness", "mentions", "emoji")
sent_share = shap_df[shap_df["feature"].str.contains("|".join(SENT_KEY))]["mean_abs_shap"].sum()
for _, r in shap_df.head(15).iterrows():
    tag = " 🟣NLP" if any(k in r["feature"] for k in SENT_KEY) else ""
    bar = "█" * int(r["mean_abs_shap"] / shap_df["mean_abs_shap"].max() * 30)
    print(f"    {r['feature']:<25} {r['mean_abs_shap']:.5f}  {bar}{tag}")
print(f"\n  Sentiment/NLP share of total SHAP : {sent_share/total*100:.1f}%")

# ── D. ABLATION STUDY ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  D. ABLATION  (with vs without sentiment features)")
print("=" * 60)
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

tech_only = [c for c in FEATURE_COLS if not any(k in c for k in SENT_KEY)]
sent_only = [c for c in FEATURE_COLS if any(k in c for k in SENT_KEY)]

train = ckpt.load(PATH_TRAIN)
val   = ckpt.load(PATH_VAL)
test_ = test  # alias

ablation = {}
for label, cols in (
    ("technical_only", tech_only),
    ("sentiment_only", sent_only),
    ("full_model",     FEATURE_COLS),
):
    if not cols:
        continue
    X_tr_a = train[cols].values
    y_tr_a = train["target_direction"].values.astype(int)
    X_te_a = test_[cols].values
    y_te_a = test_["target_direction"].values.astype(int)
    clf = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    clf.fit(X_tr_a, y_tr_a)
    auc = roc_auc_score(y_te_a, clf.predict_proba(X_te_a)[:, 1])
    ablation[label] = {
        "n_features": len(cols),
        "auc":        round(float(auc), 4),
    }
    print(f"  {label:<20} ({len(cols):>2} feats)   AUC = {auc:.4f}")

if "technical_only" in ablation and "full_model" in ablation:
    lift = ablation["full_model"]["auc"] - ablation["technical_only"]["auc"]
    ablation["sentiment_lift"] = round(float(lift), 4)
    print(f"\n  ▶  Sentiment lift over price-only: {lift:+.4f} AUC points "
          f"({lift*100:+.2f}%)")

with open(f"{MODELS_DIR}/ablation.json", "w") as f:
    json.dump(ablation, f, indent=2)
# ── E. PLACEBO : sentiment remplacé par du bruit (sanity check) ───
print("\n" + "=" * 60)
print("  E. PLACEBO  (sentiment features = bruit aléatoire)")
print("=" * 60)
rng = np.random.default_rng(42)
train_p, test_p = train.copy(), test_.copy()
for c in sent_only:                      # mêmes colonnes NLP, remplacées par du bruit
    mu, sd = train[c].mean(), train[c].std() + 1e-9
    train_p[c] = rng.normal(mu, sd, len(train_p))
    test_p[c]  = rng.normal(mu, sd, len(test_p))

clf_p = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    random_state=42, n_jobs=-1, verbose=-1,
)
clf_p.fit(train_p[FEATURE_COLS].values, train_p["target_direction"].values.astype(int))
auc_placebo = roc_auc_score(
    test_p["target_direction"].values.astype(int),
    clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1],
)
auc_full = ablation.get("full_model", {}).get("auc", float("nan"))
auc_tech = ablation.get("technical_only", {}).get("auc", float("nan"))

# ── F. SAUVEGARDE PROBA PLACEBO (sentiment=bruit) pour le Projet 2 ──
# clf_p et test_p viennent de la section E ci-dessus.
proba_placebo = clf_p.predict_proba(test_p[FEATURE_COLS].values)[:, 1]
out_P = test_[["ticker","date"]].copy()
out_P["pred_placebo"] = proba_placebo

# nom daté selon l'année de test courante
yr = int(pd.to_datetime(test_["date"]).dt.year.mode()[0])
out_P.to_parquet(f"{OUTPUT_DIR}/proba_placebo_{yr}.parquet", index=False)

# ── G. PLACEBO MULTI-GRAINES ──────────────────────────────────────
print("\n" + "="*60); print("  G. PLACEBO MULTI-GRAINES (10 tirages)"); print("="*60)
seeds = range(42, 52); records = []
for seed in seeds:
    rng_g = np.random.default_rng(seed)
    tr_p, te_p = train.copy(), test_.copy()
    for c in sent_only:
        mu, sd = train[c].mean(), train[c].std() + 1e-9
        tr_p[c] = rng_g.normal(mu, sd, len(tr_p)); te_p[c] = rng_g.normal(mu, sd, len(te_p))
    clf_g = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               random_state=42, n_jobs=-1, verbose=-1)
    clf_g.fit(tr_p[FEATURE_COLS].values, tr_p["target_direction"].values.astype(int))
    records.append(clf_g.predict_proba(te_p[FEATURE_COLS].values)[:, 1])
proba_mean = np.mean(records, axis=0)
out_Pm = test_[["ticker","date"]].copy(); out_Pm["pred_placebo"] = proba_mean
out_Pm.to_parquet(f"{OUTPUT_DIR}/proba_placebo_multi_{yr}.parquet", index=False)
print(f"✅ proba_placebo_multi_{yr}.parquet sauvegardé")
print(f"✅ proba_placebo_{yr}.parquet sauvegardé : {out_P.shape}")
print("Vérif année :", sorted(pd.to_datetime(out_P['date']).dt.year.unique()))
print(f"  technical_only      AUC = {auc_tech:.4f}")
print(f"  full_model (réel)   AUC = {auc_full:.4f}")
print(f"  full_model PLACEBO  AUC = {auc_placebo:.4f}  (sentiment = bruit)")
print(f"\n  ▶  Lift réel    : {(auc_full-auc_tech)*100:+.2f}%")
print(f"  ▶  Lift placebo : {(auc_placebo-auc_tech)*100:+.2f}%")
print("  Lecture : si le lift placebo ≈ 0 alors que le lift réel est positif,")
print("            le gain vient bien du sentiment réel (pas d'un simple ajout de colonnes).")

  📂 loaded 08_features.parquet (19,285 rows)
  A. PEARSON LAG CORRELATIONS

  Lag-correlation summary:
            mean     std     min     max
lag_days                                
0         0.1840  0.0485  0.1434  0.2662
1         0.0283  0.0236 -0.0013  0.0751
2         0.0020  0.0281 -0.0462  0.0381
3         0.0032  0.0294 -0.0275  0.0568
4         0.0127  0.0173 -0.0177  0.0311
5        -0.0036  0.0127 -0.0227  0.0175

  Significant lags (p<0.05): 11 / 42

  B. GRANGER CAUSALITY  (sent -> return)
  Verdict per ticker (significant lags / 5):
    AAPL   2/5   ✅ causal
    MSFT   3/5   ✅ causal
    GOOGL  0/5   ❌ not causal
    AMZN   0/5   ❌ not causal
    META   5/5   ✅ causal
    TSLA   0/5   ❌ not causal
    NVDA   3/5   ✅ causal

  C. SHAP VALUES (LightGBM)
  📂 loaded 09_test.parquet (1,750 rows)

  Top 15 features:
    market_share_pos_lag1     0.17189  ██████████████████████████████
    month                     0.17031  █████████████████████████████
    macd_signal       

In [57]:
# ═══ SNAPSHOT — consigne les métriques de CE run dans RESULTS.txt ═══
import json, glob, os, shutil, traceback
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score
try:
    YEAR = int(pd.Timestamp(TEST_END).year); H = int(HORIZON)
    RES  = f"{BASE_DIR}/RESULTS.txt"
    test = pd.read_parquet(f"{OUTPUT_DIR}/09_test.parquet")
    with open(f"{MODELS_DIR}/feature_names.json") as fh:
        FEATS = json.load(fh)["features"]
    ycol = "target_direction"
    X = test[FEATS]; y = test[ycol].values.astype(int)
    rng = np.random.default_rng(42)
    def boot_ci(y_, p_, n=1000):
        idx = np.arange(len(y_)); a = []
        for _ in range(n):
            b = rng.choice(idx, len(idx), replace=True)
            if y_[b].min() == y_[b].max(): continue
            a.append(roc_auc_score(y_[b], p_[b]))
        return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))
    def perm_p(y_, p_, n=500):
        base = roc_auc_score(y_, p_); c = 0
        for _ in range(n):
            yp = rng.permutation(y_)
            if yp.min() == yp.max(): continue
            if abs(roc_auc_score(yp, p_) - 0.5) >= abs(base - 0.5): c += 1
        return (c + 1) / (n + 1)
    L = ["", "="*66,
         f"RUN  test_year={YEAR}  horizon={H}j   n_test={len(y)}  pos_rate={y.mean():.3f}",
         "-"*66,
         f"baseline classe majoritaire (acc) = {max(y.mean(), 1-y.mean()):.4f}"]
    for path in sorted(glob.glob(f"{MODELS_DIR}/*.joblib")):
        name = os.path.basename(path).replace(".joblib", "")
        try:
            mdl = joblib.load(path); p = mdl.predict_proba(X)[:, 1]
        except Exception as e:
            L.append(f"{name:22s} (skip: {type(e).__name__})"); continue
        auc = roc_auc_score(y, p); lo, hi = boot_ci(y, p); pv = perm_p(y, p)
        L.append(f"{name:22s} AUC={auc:.4f}  IC95=[{lo:.4f}, {hi:.4f}]  p_perm={pv:.4f}")
    try:
        bt = pd.read_parquet(f"{OUTPUT_DIR}/11_backtest.parquet")
        bt["y"] = (bt["ret_1d"] > 0).astype(int)
        per = bt.groupby("ticker").apply(
            lambda g: roc_auc_score(g["y"], g["pred_ens"]) if g["y"].nunique() > 1 else np.nan)
        L.append("per-ticker AUC (ensemble, ret_1d) : "
                 + "  ".join(f"{t}={v:.3f}" for t, v in per.sort_values().items()))
    except Exception as e:
        L.append(f"per-ticker indisponible ({type(e).__name__}: {e})")
    tagfull = f"_{YEAR}_h{H}"
    for f_ in ["09_test.parquet", "10_predictions.parquet", "11_backtest.parquet"]:
        s = f"{OUTPUT_DIR}/{f_}"
        if os.path.exists(s):
            shutil.copy2(s, s.replace(".parquet", tagfull + ".parquet"))
    phase2_tag = "" if (YEAR == 2023 and H == 10) else ("_2022" if (YEAR == 2022 and H == 10) else None)
    if phase2_tag is not None:
        if phase2_tag:
            shutil.copy2(f"{OUTPUT_DIR}/11_backtest.parquet",
                         f"{OUTPUT_DIR}/11_backtest{phase2_tag}.parquet")
        import lightgbm as lgb
        from sklearn.utils.class_weight import compute_sample_weight
        SENT_KEY = ("sent","finbert","sarcasm","fear","anger","joy","sadness","mentions","emoji")
        tech = [c for c in FEATS if not any(k in c for k in SENT_KEY)]
        tr = pd.read_parquet(f"{OUTPUT_DIR}/09_train.parquet")
        va = pd.read_parquet(f"{OUTPUT_DIR}/09_val.parquet")
        trv = pd.concat([tr, va])
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                               max_depth=8, subsample=0.8, colsample_bytree=0.7,
                               random_state=42, verbosity=-1)
        m.fit(trv[tech], trv[ycol].astype(int),
              sample_weight=compute_sample_weight("balanced", trv[ycol].astype(int)))
        out = test[["ticker", "date"]].copy()
        out["pred_tech"] = m.predict_proba(test[tech])[:, 1]
        out.to_parquet(f"{OUTPUT_DIR}/proba_technical_only{phase2_tag}.parquet")
        L.append(f"proba_technical_only{phase2_tag}.parquet sauvegardé "
                 f"({len(out)} lignes, {len(tech)} features techniques)")
    txt = "\n".join(L)
    print(txt)
    with open(RES, "a") as fh:
        fh.write(txt + "\n")
except Exception:
    traceback.print_exc()



RUN  test_year=2023  horizon=10j   n_test=1750  pos_rate=0.683
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.6829
model_lgb              AUC=0.6548  IC95=[0.6276, 0.6806]  p_perm=0.0020
model_stack            (skip: AttributeError)
model_xgb              AUC=0.6467  IC95=[0.6190, 0.6745]  p_perm=0.0020
per-ticker AUC (ensemble, ret_1d) : META=0.424  NVDA=0.443  AMZN=0.468  TSLA=0.473  GOOGL=0.490  MSFT=0.540  AAPL=0.546
proba_technical_only.parquet sauvegardé (1750 lignes, 31 features techniques)


## Figure « learning behavior » (random vs chronologique)


In [58]:
# (bloc protégé — une erreur ici n'arrête pas le run)
import traceback
try:
    # ╔══════════════════════════════════════════════════════════════════╗
    # ║  CELL 13d (NOUVEAU) — Figure « Learning behavior » : preuve de fuite ║
    # ╚══════════════════════════════════════════════════════════════════╝
    # Entraîne LE MÊME LightGBM sous deux découpages :
    #   (a) ALÉATOIRE (mélangé)        -> exploite la structure de régime partagée -> AUC val gonflée
    #   (b) CHRONOLOGIQUE (sans fuite)  -> AUC val ~ 0.50 (hasard) = le vrai résultat
    import json
    import numpy as np, pandas as pd
    import matplotlib.pyplot as plt
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.utils.class_weight import compute_sample_weight
    
    # ── 1) Données (mêmes que tes modèles) ──
    df_feat = ckpt.load(PATH_FEATURES)
    with open(f"{MODELS_DIR}/feature_names.json") as f:
        FEATURE_COLS = json.load(f)["features"]
    
    df_feat = df_feat.dropna(subset=["target_direction"]).copy()
    df_feat["date"] = pd.to_datetime(df_feat["date"])
    
    X     = df_feat[FEATURE_COLS].values
    y     = df_feat["target_direction"].values.astype(int)
    dates = df_feat["date"].values
    print(f"Échantillons={len(X):,}  features={len(FEATURE_COLS)}  taux de positifs={y.mean():.3f}")
    
    # ── 2) Mêmes hyperparamètres que ta Cellule 13 (sans early-stopping, n fixe) ──
    NUM_ROUNDS = 600
    LGB_PARAMS = dict(
        objective="binary", metric="auc",
        n_estimators=NUM_ROUNDS, learning_rate=0.03,
        num_leaves=63, max_depth=8, min_child_samples=40,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        reg_alpha=0.1, reg_lambda=0.2, n_jobs=-1, verbose=-1,
    )
    
    def fit_curve(Xtr, ytr, Xval, yval, seed=42):
        clf = lgb.LGBMClassifier(**dict(LGB_PARAMS, random_state=seed))
        evals = {}
        clf.fit(
            Xtr, ytr,
            sample_weight=compute_sample_weight("balanced", ytr),
            eval_set=[(Xtr, ytr), (Xval, yval)],
            eval_names=["train", "valid"], eval_metric="auc",
            callbacks=[lgb.record_evaluation(evals)],
        )
        return np.array(evals["train"]["auc"]), np.array(evals["valid"]["auc"])
    
    # ── 3) (a) SPLIT ALÉATOIRE — moyenné sur 5 graines ──
    SEEDS = [0, 1, 2, 3, 4]
    tr_r, va_r = [], []
    for sd in SEEDS:
        Xtr, Xva, ytr, yva = train_test_split(
            X, y, test_size=0.25, shuffle=True, random_state=sd, stratify=y)
        a, b = fit_curve(Xtr, ytr, Xva, yva, seed=sd)
        tr_r.append(a); va_r.append(b)
    tr_r, va_r = np.array(tr_r), np.array(va_r)
    
    # ── 4) (b) SPLIT CHRONOLOGIQUE — 75% passé / 25% futur ──
    order = np.argsort(dates)
    Xs, ys = X[order], y[order]
    cut = int(len(Xs) * 0.75)
    tr_t, va_t = fit_curve(Xs[:cut], ys[:cut], Xs[cut:], ys[cut:], seed=42)
    
    # ── 5) Figure ──
    BLUE, REDc, GRAY = "#3B6EA5", "#C1392B", "#7E8C8D"
    it = np.arange(1, NUM_ROUNDS + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), dpi=130, sharey=True)
    
    ax = axes[0]
    ax.plot(it, tr_r.mean(0), color=BLUE, lw=1.8, label="Train AUC")
    ax.fill_between(it, tr_r.mean(0)-tr_r.std(0), tr_r.mean(0)+tr_r.std(0), color=BLUE, alpha=.15)
    ax.plot(it, va_r.mean(0), color=REDc, lw=1.8, label="Validation AUC")
    ax.fill_between(it, va_r.mean(0)-va_r.std(0), va_r.mean(0)+va_r.std(0), color=REDc, alpha=.15)
    ax.axhline(0.5, color=GRAY, ls="--", lw=1.0, label="Chance (0.5)")
    ax.set_title("Random split (shuffled)")
    ax.set_xlabel("Boosting iteration"); ax.set_ylabel("AUC")
    ax.legend(frameon=False, fontsize=9, loc="lower right")
    
    ax = axes[1]
    ax.plot(it, tr_t, color=BLUE, lw=1.8, label="Train AUC")
    ax.plot(it, va_t, color=REDc, lw=1.8, label="Validation AUC")
    ax.axhline(0.5, color=GRAY, ls="--", lw=1.0)
    ax.set_title("Chronological split (no leakage)")
    ax.set_xlabel("Boosting iteration")
    
    for ax in axes:
        ax.set_ylim(0.45, 1.0)
        ax.grid(True, color="#e6e6e6", lw=0.7); ax.set_axisbelow(True)
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    
    fig.suptitle("Learning behavior: validation AUC under random vs chronological splitting", fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    OUT = f"{OUTPUT_DIR}/fig5_learning_behavior.png"
    fig.savefig(OUT, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Figure sauvegardée :", OUT)
    
    # ── 6) Chiffres à citer dans le texte ──
    rv, tv = va_r.mean(0)[-1], va_t[-1]
    print(f"\n[À citer] Val AUC — split aléatoire    : {rv:.3f}  (± {va_r.std(0)[-1]:.3f} sur {len(SEEDS)} graines)")
    print(f"[À citer] Val AUC — split chronologique : {tv:.3f}")
    print(f"[À citer] Écart de fuite (aléatoire − chronologique) : {rv - tv:+.3f}")
    print(f"[À citer] Écart train−val (chronologique) : {tr_t[-1] - tv:+.3f}")
except Exception:
    print('⚠️ Cell 13d (learning behavior) a échoué :')
    traceback.print_exc()


  📂 loaded 08_features.parquet (19,285 rows)
Échantillons=19,285  features=49  taux de positifs=0.598
Figure sauvegardée : /kaggle/working/sentiment_pipeline/data/fig5_learning_behavior.png

[À citer] Val AUC — split aléatoire    : 0.828  (± 0.008 sur 5 graines)
[À citer] Val AUC — split chronologique : 0.542
[À citer] Écart de fuite (aléatoire − chronologique) : +0.286
[À citer] Écart train−val (chronologique) : +0.457


## PHASE 2 — robustesse de régime (2022 vs 2023)


In [59]:
# ═══ PHASE 2 — ROBUSTESSE DE RÉGIME : 2022 (bear) vs 2023 (bull) ═══
# Auto-contenue : A = stratégie technique seule, B = technique + sentiment (ensemble).
# Seuils 0.55 long / 0.45 short ; coûts de transaction 1.0 bps par changement de position.
import numpy as np, pandas as pd, traceback
def _pos(p): return np.where(p > 0.55, 1, np.where(p < 0.45, -1, 0))
def compute_net(m, pred, ret, cost_bps=1.0):
    d = m.sort_values(["ticker", "date"]).copy()
    d["pos"] = _pos(d[pred].values)
    d["chg"] = d.groupby("ticker")["pos"].diff().abs().fillna(d["pos"].abs())
    d["net"] = d["pos"] * d[ret] - d["chg"] * cost_bps / 1e4
    return d
def portfolio_daily(d): return d.groupby("date")["net"].mean().sort_index()
def perf(s):
    cum = float((1 + s).prod() - 1)
    sh = float(s.mean() / s.std() * np.sqrt(252)) if s.std() > 0 else 0.0
    return {"cum": cum, "sharpe": round(sh, 2)}
def cvar(s, a=.05):
    q = s.quantile(a); return float(s[s <= q].mean())
def analyze_year(tag):
    dfB = pd.read_parquet(f"{OUTPUT_DIR}/11_backtest{tag}.parquet")
    dfA = pd.read_parquet(f"{OUTPUT_DIR}/proba_technical_only{tag}.parquet")
    for d in (dfB, dfA): d["date"] = pd.to_datetime(d["date"])
    m = dfB.merge(dfA[["ticker", "date", "pred_tech"]], on=["ticker", "date"], how="inner")
    dB = portfolio_daily(compute_net(m, "pred_ens", "ret_1d"))
    dA = portfolio_daily(compute_net(m, "pred_tech", "ret_1d"))
    mkt = m.groupby("date")["ret_1d"].mean().sort_index()
    return {"year": int(m["date"].dt.year.mode()[0]),
            "A": perf(dA), "B": perf(dB), "cvA": cvar(dA), "cvB": cvar(dB),
            "mkt": float((1 + mkt).prod() - 1)}
try:
    lines = ["", "="*66,
             "PHASE 2 — RÉGIME 2022 (bear) vs 2023 (bull) — backtest coûts 1 bps",
             "-"*66]
    for tag in ("", "_2022"):
        r = analyze_year(tag); reg = "bull" if r["mkt"] > 0 else "BEAR"
        lines += [f"{r['year']} ({reg}, marché {r['mkt']:+.1%})",
                  f"   A technique       : cum {r['A']['cum']:+.2%} | Sharpe {r['A']['sharpe']} | CVaR5 {r['cvA']:.4f}",
                  f"   B tech+sentiment  : cum {r['B']['cum']:+.2%} | Sharpe {r['B']['sharpe']} | CVaR5 {r['cvB']:.4f}"]
    txt = "\n".join(lines); print(txt)
    with open(f"{BASE_DIR}/RESULTS.txt", "a") as fh:
        fh.write(txt + "\n")
except Exception:
    traceback.print_exc()



PHASE 2 — RÉGIME 2022 (bear) vs 2023 (bull) — backtest coûts 1 bps
------------------------------------------------------------------
2023 (bull, marché +112.3%)
   A technique       : cum -1.80% | Sharpe -0.02 | CVaR5 -0.0243
   B tech+sentiment  : cum +2.91% | Sharpe 0.25 | CVaR5 -0.0258
2022 (BEAR, marché -47.1%)
   A technique       : cum -68.77% | Sharpe -3.52 | CVaR5 -0.0512
   B tech+sentiment  : cum +0.00% | Sharpe 0.0 | CVaR5 0.0000


## Cell 17 — Visualisation (figures matplotlib)

In [60]:
# (bloc protégé — une erreur ici n'arrête pas le run)
import traceback
try:
    # ╔══════════════════════════════════════════════════════════════════╗
    # ║  CELL 17 — VISUALIZATION                                         ║
    # ╚══════════════════════════════════════════════════════════════════╝
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import seaborn as sns
    
    FIG_DIR = f"{DASHBOARD_DIR}/figures"
    Path(FIG_DIR).mkdir(parents=True, exist_ok=True)
    
    # Thème clair, classique et peu contrasté (palette sobre)
    BG, BG2 = "#ffffff", "#f5f5f5"
    GRID    = "#dcdcdc"
    WHITE   = "#333333"          # = couleur de texte/avant-plan (foncé sur fond clair)
    # Palette catégorielle sobre (type "muted", faible saturation)
    COLORS  = ["#4C72B0", "#55A868", "#C44E52", "#8172B2",
               "#937860", "#64738C", "#B07AA1"]
    POS, NEG, ACC = "#6F9E6E", "#B5616A", "#4C72B0"   # vert/rouge/bleu atténués
    
    plt.rcParams.update({
        "figure.facecolor": BG, "axes.facecolor": BG,
        "axes.edgecolor":   GRID, "axes.labelcolor": WHITE,
        "axes.titlecolor":  WHITE, "axes.titleweight": "bold",
        "xtick.color": "#666666", "ytick.color": "#666666",
        "grid.color":  GRID, "grid.linewidth": 0.6,
        "text.color":  WHITE, "lines.linewidth": 1.6,
        "legend.facecolor": BG2, "legend.edgecolor": GRID,
    })
    
    df_daily   = ckpt.load(PATH_DAILY)
    df_merged  = ckpt.load(PATH_MERGED)
    df_bt      = ckpt.load(PATH_BACKTEST)
    df_lags    = pd.read_csv(f"{MODELS_DIR}/lag_correlations.csv")
    shap_df    = pd.read_csv(f"{MODELS_DIR}/shap_importance.csv")
    with open(f"{MODELS_DIR}/realtime_predictions.json") as f:
        rt = json.load(f)
    predictions = rt["predictions"]
    
    # ── Fig 1 — Sentiment evolution ───────────────────────────────────
    print("Fig 1 — sentiment evolution...")
    n = len(STOCKS)
    fig, axes = plt.subplots(n, 1, figsize=(16, 3.5*n), facecolor=BG)
    if n == 1: axes = [axes]
    fig.subplots_adjust(hspace=0.6)
    for ax, tk, c in zip(axes, STOCKS, COLORS):
        ds = df_daily[df_daily["ticker"] == tk].sort_values("date").copy()
        if ds.empty: continue
        ds["ma30"] = ds["sent_mean"].rolling(30, min_periods=1).mean()
        ds["ma90"] = ds["sent_mean"].rolling(90, min_periods=1).mean()
        ax.fill_between(ds["date"], ds["sent_mean"], 0,
                        where=ds["sent_mean"] >= 0, color=POS, alpha=0.12)
        ax.fill_between(ds["date"], ds["sent_mean"], 0,
                        where=ds["sent_mean"] <  0, color=NEG, alpha=0.12)
        ax.plot(ds["date"], ds["ma30"], color=c,     lw=2.0, label="MA-30")
        ax.plot(ds["date"], ds["ma90"], color=WHITE, lw=1.0, ls="--", alpha=0.5, label="MA-90")
        ax.axhline(0, color="#777", lw=0.6, ls=":")
        ax.set_title(f"{tk} — {NAMES[tk]}", color=c, loc="left")
        ax.set_ylabel("Sentiment")
        ax.legend(loc="upper right", fontsize=8)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle("Sentiment Evolution per Stock", color=WHITE, fontsize=14, y=1.001)
    fig.savefig(f"{FIG_DIR}/01_sentiment_evolution.png", dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 2 — Sentiment vs price ────────────────────────────────────
    print("Fig 2 — sentiment vs price...")
    fig, axes = plt.subplots(n, 1, figsize=(16, 4*n), facecolor=BG)
    if n == 1: axes = [axes]
    fig.subplots_adjust(hspace=0.6)
    for ax, tk, c in zip(axes, STOCKS, COLORS):
        ds = df_merged[df_merged["ticker"] == tk].sort_values("date").copy()
        if ds.empty: continue
        ds["sent_ma"] = ds["sent_mean"].rolling(30, min_periods=1).mean()
        ax.plot(ds["date"], ds["sent_ma"], color=c, lw=2.0, label="Sentiment MA-30")
        ax.fill_between(ds["date"], ds["sent_ma"], 0, color=c, alpha=0.08)
        ax.axhline(0, color="#777", lw=0.5, ls=":")
        ax.set_ylabel("Sentiment", color=c)
        ax.tick_params(axis="y", colors=c)
        ax2 = ax.twinx()
        ax2.plot(ds["date"], ds["close"], color="#8C7B3B", lw=1.0, alpha=0.7, label="Close $")
        ax2.set_ylabel("Price $", color="#8C7B3B")
        ax2.tick_params(axis="y", colors="#8C7B3B")
        corr = ds[["sent_mean","close"]].corr().iloc[0,1]
        col_c = POS if corr >= 0 else NEG
        ax.text(0.01, 0.96, f"Pearson p = {corr:+.3f}",
                transform=ax.transAxes, color=col_c, va="top", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor=BG2, alpha=0.9))
        ax.set_title(f"{tk} — Sentiment vs Price", color=c, loc="left")
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle("Sentiment (MA-30) vs Price", color=WHITE, fontsize=14, y=1.001)
    fig.savefig(f"{FIG_DIR}/02_sentiment_vs_price.png", dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 3 — Annual correlation heatmap ────────────────────────────
    print("Fig 3 — annual correlation heatmap...")
    df_merged["year"] = df_merged["date"].dt.year
    years = sorted(df_merged["year"].unique())
    mat = pd.DataFrame(index=years, columns=STOCKS, dtype=float)
    for yr in years:
        for tk in STOCKS:
            ds = df_merged[(df_merged["year"] == yr) & (df_merged["ticker"] == tk)]
            if len(ds) < 10: continue
            mat.loc[yr, tk] = round(ds[["sent_mean", "return_1d"]].corr().iloc[0, 1], 3)
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(years)*0.6)), facecolor=BG)
    sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap="RdBu_r", center=0,
                vmin=-0.35, vmax=0.35, ax=ax,
                cbar_kws={"label": "Pearson p", "shrink": 0.7},
                annot_kws={"size": 10, "weight": "bold"})
    ax.set_title("Annual Correlation: Sentiment -> Daily Return",
                 color=WHITE, fontsize=13)
    ax.set_xlabel("Ticker"); ax.set_ylabel("Year")
    fig.savefig(f"{FIG_DIR}/03_correlation_heatmap.png", dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 4 — Real-time signals ─────────────────────────────────────
    print("Fig 4 — real-time signals...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
    fig.subplots_adjust(wspace=0.3)
    tks_p = list(predictions.keys())
    probs = [predictions[t]["probability_up"] for t in tks_p]
    sents = [predictions[t]["sentiment_score"] for t in tks_p]
    sigs  = [predictions[t]["signal"]          for t in tks_p]
    bcs = [POS if p > 60 else NEG if p < 40 else "#C9A227" for p in probs]
    ax1.barh(tks_p, probs, color=bcs, alpha=0.85, edgecolor=WHITE, linewidth=0.5)
    ax1.axvline(50, color=WHITE, lw=1, ls="--", alpha=0.6)
    for i, (p, s) in enumerate(zip(probs, sigs)):
        ax1.text(p + 1, i, f" {p:.1f}%  {s}", va="center", fontsize=9.5, fontweight="bold")
    ax1.set_xlim(0, 120)
    ax1.set_xlabel("Prob. of upward move (%)")
    ax1.set_title("Real-Time Signals")
    sc = ax2.scatter(sents, probs, s=220, c=probs, cmap="RdBu_r",
                     vmin=30, vmax=70, edgecolors=WHITE, linewidth=1.5)
    for t, sv, pv in zip(tks_p, sents, probs):
        ax2.annotate(f"  {t}", (sv, pv), color=WHITE, fontsize=10, fontweight="bold", va="center")
    if len(sents) >= 3:
        z = np.polyfit(sents, probs, 1)
        xl = np.linspace(min(sents)-0.05, max(sents)+0.05, 50)
        ax2.plot(xl, np.poly1d(z)(xl), color=ACC, lw=2, ls="--", alpha=0.85, label="Trend")
    ax2.axhline(50, color=WHITE, lw=0.8, ls=":")
    ax2.axvline(0,  color=WHITE, lw=0.8, ls=":")
    ax2.set_xlabel("Sentiment (latest)")
    ax2.set_ylabel("Prob. up (%)")
    ax2.set_title("Sentiment <-> Predicted Probability")
    ax2.legend()
    plt.colorbar(sc, ax=ax2, shrink=0.7, label="Prob. up (%)")
    fig.suptitle("Real-Time Sentiment Impact", color=WHITE, fontsize=14, y=1.01)
    fig.savefig(f"{FIG_DIR}/04_realtime_signals.png", dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 5 — Backtest cumulative returns ───────────────────────────
    print("Fig 5 — backtest...")
    fig, axes = plt.subplots(n, 1, figsize=(16, 3.5*n), facecolor=BG)
    if n == 1: axes = [axes]
    fig.subplots_adjust(hspace=0.55)
    for ax, tk, c in zip(axes, STOCKS, COLORS):
        ds = df_bt[df_bt["ticker"] == tk].sort_values("date")
        if ds.empty: continue
        rs = (ds["cum_strategy"] - 1) * 100
        rm = (ds["cum_market"]   - 1) * 100
        ax.plot(ds["date"], rs, color=c,     lw=2.0, label="Strategy")
        ax.plot(ds["date"], rm, color=WHITE, lw=1.2, ls="--", alpha=0.6, label="Buy & Hold")
        ax.fill_between(ds["date"], rs, rm, where=rs >= rm, color=POS, alpha=0.18)
        ax.fill_between(ds["date"], rs, rm, where=rs <  rm, color=NEG, alpha=0.18)
        final_alpha = float(rs.iloc[-1] - rm.iloc[-1])
        col_a = POS if final_alpha >= 0 else NEG
        ax.text(0.99, 0.96, f"Alpha {final_alpha:+.1f}%",
                transform=ax.transAxes, ha="right", va="top",
                color=col_a, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor=BG2, alpha=0.9))
        ax.axhline(0, color="#777", lw=0.5, ls=":")
        ax.set_title(f"{tk}", color=c, loc="left")
        ax.set_ylabel("Cum return %")
        ax.legend(loc="upper left", fontsize=8)
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.suptitle("Backtest — Strategy vs Buy & Hold", color=WHITE, fontsize=14, y=1.001)
    fig.savefig(f"{FIG_DIR}/05_backtest.png", dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 6 — SHAP top features ─────────────────────────────────────
    print("Fig 6 — SHAP top features...")
    top = shap_df.head(20)
    SENT_KEY = ("sent", "finbert", "sarcasm", "fear", "anger", "joy", "sadness", "mentions")
    colors_b = [POS if any(k in f for k in SENT_KEY) else ACC for f in top["feature"]]
    fig, ax = plt.subplots(figsize=(12, 8), facecolor=BG)
    ax.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color=colors_b[::-1],
            alpha=0.85, edgecolor=WHITE, linewidth=0.5)
    ax.set_xlabel("|SHAP| mean")
    ax.set_title("Top-20 features — SHAP importance\n"
                 "Green = Sentiment/Emotion · Blue = Technical",
                 color=WHITE)
    fig.savefig(f"{FIG_DIR}/06_shap.png", dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 7 — 4-class sentiment composition over time (stacked) ────
    print("Fig 7 — sentiment 4-class composition over time...")
    fig, axes = plt.subplots(n, 1, figsize=(16, 3.2*n), facecolor=BG)
    if n == 1: axes = [axes]
    fig.subplots_adjust(hspace=0.55)
    cat_colors = {"share_positive": POS, "share_negative": NEG,
                  "share_neutral": "#888", "share_sarcastic": "#a78bfa"}
    cat_labels = {"share_positive": "Positive", "share_negative": "Negative",
                  "share_neutral": "Neutral",   "share_sarcastic": "Sarcastic"}
    for ax, tk in zip(axes, STOCKS):
        ds = df_daily[df_daily["ticker"] == tk].sort_values("date").copy()
        if ds.empty or "share_positive" not in ds.columns:
            ax.text(0.5, 0.5, "no category data", transform=ax.transAxes,
                    ha="center", va="center", color=WHITE, fontsize=11)
            ax.set_title(tk, color=WHITE, loc="left"); continue
        # Smooth each share over a 14-day window
        smoothed = {k: ds[k].rolling(14, min_periods=1).mean().fillna(0)
                    for k in cat_colors}
        # Renormalize to sum to 1 (smoothing can break that)
        sum_s = sum(smoothed.values()).replace(0, 1)
        for k in cat_colors: smoothed[k] = smoothed[k] / sum_s
        ax.stackplot(ds["date"],
                     smoothed["share_positive"], smoothed["share_negative"],
                     smoothed["share_neutral"],  smoothed["share_sarcastic"],
                     labels=[cat_labels[k] for k in cat_colors],
                     colors=[cat_colors[k] for k in cat_colors],
                     alpha=0.85)
        ax.set_ylim(0, 1); ax.set_ylabel("Share")
        ax.set_title(f"{tk} — daily sentiment mix (14-day smoothed)",
                     color=WHITE, loc="left")
        ax.legend(loc="upper right", fontsize=8, ncol=4)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle("4-Class Sentiment Composition Over Time",
                 color=WHITE, fontsize=14, y=1.001)
    fig.savefig(f"{FIG_DIR}/07_category_composition.png",
                dpi=120, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    
    # ── Fig 8 — Sarcasm impact on prediction accuracy ────────────────
    # Compare model accuracy on days with high sarcasm vs low sarcasm.
    print("Fig 8 — sarcasm impact on prediction accuracy...")
    try:
        df_bt_merged = df_bt.merge(
            df_merged[["ticker","date","sarcasm_mean"]], on=["ticker","date"], how="left")
        df_bt_merged["sarc_bucket"] = pd.cut(
            df_bt_merged["sarcasm_mean"].fillna(0),
            bins=[-0.01, 0.05, 0.15, 0.30, 1.0],
            labels=["very low", "low", "medium", "high"])
        df_bt_merged["correct"] = (df_bt_merged["signal_lag"] * df_bt_merged["target_return"] > 0).astype(int)
        df_bt_merged = df_bt_merged[df_bt_merged["signal_lag"] != 0]   # only days with active trades
        acc_by_bucket = df_bt_merged.groupby("sarc_bucket", observed=True).agg(
            n_trades=("correct", "count"),
            accuracy=("correct", "mean"),
            mean_return=("strategy_return", "mean"),
        ).reset_index()
    
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)
        fig.subplots_adjust(wspace=0.3)
        bcs = [POS if a > 0.5 else NEG for a in acc_by_bucket["accuracy"]]
        ax1.bar(acc_by_bucket["sarc_bucket"].astype(str), acc_by_bucket["accuracy"] * 100,
                color=bcs, alpha=0.85, edgecolor=WHITE)
        ax1.axhline(50, color=WHITE, lw=0.8, ls="--", alpha=0.6, label="random (50%)")
        for i, r in enumerate(acc_by_bucket.itertuples(index=False)):
            ax1.text(i, r.accuracy*100 + 0.5, f"n={int(r.n_trades)}",
                     ha="center", color=WHITE, fontsize=9)
        ax1.set_ylim(0, 100); ax1.set_ylabel("Hit rate %")
        ax1.set_xlabel("Sarcasm level on prediction day")
        ax1.set_title("Prediction accuracy by sarcasm level")
        ax1.legend(loc="upper right", fontsize=9)
        ax2.bar(acc_by_bucket["sarc_bucket"].astype(str),
                acc_by_bucket["mean_return"] * 100,
                color=[POS if v > 0 else NEG for v in acc_by_bucket["mean_return"]],
                alpha=0.85, edgecolor=WHITE)
        ax2.axhline(0, color=WHITE, lw=0.6, ls=":")
        ax2.set_ylabel("Mean strategy return per trade %")
        ax2.set_xlabel("Sarcasm level")
        ax2.set_title("Realized return by sarcasm bucket")
        fig.suptitle("Does sarcasm hurt or help predictions?",
                     color=WHITE, fontsize=14, y=1.01)
        fig.savefig(f"{FIG_DIR}/08_sarcasm_impact.png",
                    dpi=120, bbox_inches="tight", facecolor=BG)
        plt.close(fig)
    except Exception as e:
        print(f"  ⚠️  Fig 8 skipped: {str(e)[:120]}")
    
    # ── Fig 9 — Source contribution (data volume per source) ─────────
    print("Fig 9 — data volume per source...")
    try:
        _scp = f"{OUTPUT_DIR}/source_counts.parquet"
        if Path(_scp).exists():
            _sc = pd.read_parquet(_scp)
            src_counts = _sc.pivot_table(index="source", columns="dtype",
                                         values="n", aggfunc="sum", fill_value=0)
        else:
            df_clean = pd.read_parquet(PATH_CLEAN, columns=["source", "dtype"])  # léger
            src_counts = df_clean.groupby(["source","dtype"]).size().unstack(fill_value=0)
        if "news" not in src_counts.columns:   src_counts["news"]   = 0
        if "social" not in src_counts.columns: src_counts["social"] = 0
        src_counts["total"] = src_counts.sum(axis=1)
        src_counts = src_counts.sort_values("total", ascending=True).tail(15)
        fig, ax = plt.subplots(figsize=(12, max(4, len(src_counts)*0.5)), facecolor=BG)
        y = np.arange(len(src_counts))
        ax.barh(y, src_counts["news"],   color=ACC,   alpha=0.85, label="News")
        ax.barh(y, src_counts["social"], left=src_counts["news"],
                color="#FF9F43", alpha=0.85, label="Social")
        ax.set_yticks(y); ax.set_yticklabels(src_counts.index, fontsize=9)
        ax.set_xlabel("Row count")
        ax.set_title("Data volume per source (after cleaning)", color=WHITE)
        for i, r in enumerate(src_counts.itertuples()):
            ax.text(r.total + max(src_counts["total"])*0.005, i,
                    f" {int(r.total):,}", color=WHITE, fontsize=9, va="center")
        ax.legend(loc="lower right")
        fig.savefig(f"{FIG_DIR}/09_sources.png",
                    dpi=120, bbox_inches="tight", facecolor=BG)
        plt.close(fig)
    except Exception as e:
        print(f"  ⚠️  Fig 9 skipped: {str(e)[:120]}")
    
    # ── Fig 10 — Confusion matrix + Calibration plot ─────────────────
    print("Fig 10 — confusion + calibration...")
    try:
        import joblib
        from catboost import CatBoostClassifier
        from sklearn.metrics import confusion_matrix
        from sklearn.calibration import calibration_curve
    
        with open(f"{MODELS_DIR}/feature_names.json") as f:
            FEATURE_COLS = json.load(f)["features"]
        test = ckpt.load(PATH_TEST)
        X_te = test[FEATURE_COLS].values
        y_te = test["target_direction"].values.astype(int)
        clf_lgb = joblib.load(f"{MODELS_DIR}/model_lgb.joblib")
        clf_xgb = joblib.load(f"{MODELS_DIR}/model_xgb.joblib")
        clf_cb  = CatBoostClassifier(); clf_cb.load_model(f"{MODELS_DIR}/model_cb.cbm")
        p_ens = (0.4 * clf_lgb.predict_proba(X_te)[:, 1] +
                 0.3 * clf_xgb.predict_proba(X_te)[:, 1] +
                 0.3 * clf_cb.predict_proba(X_te)[:, 1])
        yp = (p_ens > 0.5).astype(int)
        cm = confusion_matrix(y_te, yp)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG)
        fig.subplots_adjust(wspace=0.4)
        sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=ax1, cbar=False,
                    xticklabels=["pred ↓", "pred ↑"],
                    yticklabels=["true ↓", "true ↑"],
                    annot_kws={"size": 14, "color":"#000"})
        ax1.set_title("Ensemble confusion matrix (test)", color=WHITE)
        pt, pp = calibration_curve(y_te, p_ens, n_bins=10, strategy="quantile")
        ax2.plot([0, 1], [0, 1], color=WHITE, ls="--", lw=0.8, label="Perfect")
        ax2.plot(pp, pt, color=ACC, lw=2, marker="o", label="Ensemble")
        ax2.fill_between(pp, pt, pp, color=ACC, alpha=0.15)
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
        ax2.set_xlabel("Predicted prob"); ax2.set_ylabel("Empirical freq")
        ax2.set_title("Calibration curve", color=WHITE); ax2.legend()
        fig.savefig(f"{FIG_DIR}/10_diagnostic.png",
                    dpi=120, bbox_inches="tight", facecolor=BG)
        plt.close(fig)
    except Exception as e:
        print(f"  ⚠️  Fig 10 skipped: {str(e)[:120]}")
    
    print(f"\n  ✅ All figures saved -> {FIG_DIR}")
    print("     " + "\n     ".join(sorted(os.listdir(FIG_DIR))))
    
except Exception:
    print('⚠️ Cell 17 (figures) a échoué :')
    traceback.print_exc()


  📂 loaded 05_daily.parquet (23,258 rows)
  📂 loaded 07_merged.parquet (19,285 rows)
  📂 loaded 11_backtest.parquet (1,750 rows)
⚠️ Cell 17 (figures) a échoué :


Traceback (most recent call last):
  File "/tmp/ipykernel_58/1626767505.py", line 40, in <cell line: 0>
    with open(f"{MODELS_DIR}/realtime_predictions.json") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/sentiment_pipeline/models/realtime_predictions.json'


In [61]:
# ═══ RÉCAPITULATIF FINAL — tout RESULTS.txt dans les logs ═══
print(open(f"{BASE_DIR}/RESULTS.txt").read())



RUN  test_year=2023  horizon=1j   n_test=1750  pos_rate=0.549
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.5491
model_lgb              AUC=0.4970  IC95=[0.4705, 0.5250]  p_perm=0.8104
model_stack            (skip: AttributeError)
model_xgb              AUC=0.5141  IC95=[0.4869, 0.5411]  p_perm=0.2994
per-ticker AUC (ensemble, ret_1d) : AAPL=0.370  TSLA=0.385  MSFT=0.412  NVDA=0.436  AMZN=0.455  GOOGL=0.456  META=0.456

RUN  test_year=2023  horizon=20j   n_test=1750  pos_rate=0.733
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.7331
model_lgb              AUC=0.6113  IC95=[0.5825, 0.6425]  p_perm=0.0020
model_stack            (skip: AttributeError)
model_xgb              AUC=0.6207  IC95=[0.5908, 0.6489]  p_perm=0.0020
per-ticker AUC (ensemble, ret_1d) : NVDA=0.450  META=0.469  AMZN=0.475  TSLA=0.496  GOOGL=0.508  MSFT=0.510  AAPL=0.514

RUN  test_year=2022  horizon=10j

In [62]:
# ═══ ARCHIVE FINALE (Zenodo + sauvegarde) ═══
import shutil
shutil.make_archive("/kaggle/working/artefacts", "zip", "/kaggle/working/sentiment_pipeline")
print("✅ artefacts.zip prêt — onglet Output de la version committée")


✅ artefacts.zip prêt — onglet Output de la version committée


In [63]:
# ═══ TOUT AFFICHER POUR COPIER-COLLER ═══
import glob, os

BASE = "/kaggle/working/sentiment_pipeline"

# 1) Le fichier de résultats (l'essentiel)
try:
    print(open(f"{BASE}/RESULTS.txt").read())
except FileNotFoundError:
    print("❌ RESULTS.txt absent — les cellules SNAPSHOT n'ont pas encore tourné")

# 2) Inventaire de ce qui a été produit
print("\n===== FICHIERS PRODUITS =====")
for f in sorted(glob.glob(f"{BASE}/data/*.parquet")):
    print(os.path.basename(f))
print("\n===== FIGURES =====")
for f in sorted(glob.glob(f"{BASE}/figures/*.png")):
    print(os.path.basename(f))
print("\n===== CSV causaux =====")
for f in sorted(glob.glob(f"{BASE}/models/*.csv")):
    print(os.path.basename(f))


RUN  test_year=2023  horizon=1j   n_test=1750  pos_rate=0.549
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.5491
model_lgb              AUC=0.4970  IC95=[0.4705, 0.5250]  p_perm=0.8104
model_stack            (skip: AttributeError)
model_xgb              AUC=0.5141  IC95=[0.4869, 0.5411]  p_perm=0.2994
per-ticker AUC (ensemble, ret_1d) : AAPL=0.370  TSLA=0.385  MSFT=0.412  NVDA=0.436  AMZN=0.455  GOOGL=0.456  META=0.456

RUN  test_year=2023  horizon=20j   n_test=1750  pos_rate=0.733
------------------------------------------------------------------
baseline classe majoritaire (acc) = 0.7331
model_lgb              AUC=0.6113  IC95=[0.5825, 0.6425]  p_perm=0.0020
model_stack            (skip: AttributeError)
model_xgb              AUC=0.6207  IC95=[0.5908, 0.6489]  p_perm=0.0020
per-ticker AUC (ensemble, ret_1d) : NVDA=0.450  META=0.469  AMZN=0.475  TSLA=0.496  GOOGL=0.508  MSFT=0.510  AAPL=0.514

RUN  test_year=2022  horizon=10j

In [64]:
import shutil; shutil.copy("/kaggle/working/sentiment_pipeline/RESULTS.txt", "/kaggle/working/RESULTS.txt")
print("→ RESULTS.txt dispo dans Output, clic droit → Download")

→ RESULTS.txt dispo dans Output, clic droit → Download


In [65]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELLULE A — FIGURE 4 : ABLATION (thèse C : lift négatif partout = placebo)
#  À coller dans une NOUVELLE cellule, à exécuter APRÈS les 4 runs.
#  Ne recalcule rien : les valeurs viennent de tes logs (ablation + placebo).
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, os
 
FIG_OUT = f"{BASE_DIR}/figures"          # BASE_DIR déjà défini en Cell 1
os.makedirs(FIG_OUT, exist_ok=True)
 
# ── Chiffres du run (issus de RESULTS.txt / Cell 15) ──────────────────────────
# lift = AUC(full) - AUC(technical_only) ; placebo = AUC(full_placebo) - AUC(tech)
groups   = ["Direction\n10d · 2023 (bull)", "Direction\n10d · 2022 (bear)"]
real     = [-0.99, -0.61]      # % — lift réel
placebo  = [-0.32, -0.73]      # % — lift placebo (bruit)
 
x = np.arange(len(groups)); w = 0.36
fig, ax = plt.subplots(figsize=(6.6, 3.4), dpi=150)
b1 = ax.bar(x - w/2, real,    w, label="Real sentiment", color="#00629B", zorder=3)
b2 = ax.bar(x + w/2, placebo, w, label="Noise placebo",  color="#9E9E9E",
            hatch="////", edgecolor="white", zorder=3)
ax.axhline(0, color="#222", lw=1)
for bars in (b1, b2):
    for r in bars:
        h = r.get_height()
        ax.text(r.get_x()+r.get_width()/2, h + (0.03 if h >= 0 else -0.03),
                f"{h:+.2f}%", ha="center", va="bottom" if h >= 0 else "top",
                fontsize=9, weight="bold", color="#222")
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=9)
ax.set_ylabel("Sentiment AUC lift over technical-only (%)", fontsize=9)
ax.set_ylim(-1.25, 0.35)
ax.set_title("Sentiment/sarcasm ablation: real features match a noise placebo",
             fontsize=10, weight="bold")
ax.legend(frameon=False, fontsize=9, loc="lower right")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
ax.grid(axis="y", ls=":", alpha=0.4)
fig.tight_layout()
fig.savefig(f"{FIG_OUT}/fig4.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(f"{FIG_OUT}/fig4.pdf", bbox_inches="tight", facecolor="white")
plt.close(fig)
print("✅ fig4.png / fig4.pdf écrits dans", FIG_OUT)
 
 
# ═══════════════════════════════════════════════════════════════════════════
#  CELLULE B — FIGURE 8 : WALK-FORWARD (AUC par régime + lift par régime)
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np, os
FIG_OUT = f"{BASE_DIR}/figures"; os.makedirs(FIG_OUT, exist_ok=True)
 
years   = ["2022\n(bear)", "2023\n(bull)"]
auc_lgb = [0.479, 0.672]        # best 10j LightGBM AUC par régime
lift    = [-0.61, -0.99]        # % — lift sentiment par régime
 
fig, (axL, axR) = plt.subplots(1, 2, figsize=(9.2, 3.4), dpi=150)
 
# — gauche : AUC direction 10j par régime —
cols = ["#C62828" if a < 0.5 else "#00629B" for a in auc_lgb]
bL = axL.bar(years, auc_lgb, width=0.55, color=cols, zorder=3)
axL.axhline(0.5, color="#9E9E9E", ls="--", lw=1, zorder=2)
axL.text(1.45, 0.505, "chance", fontsize=7, color="#9E9E9E", ha="right", va="bottom")
for r, a in zip(bL, auc_lgb):
    axL.text(r.get_x()+r.get_width()/2, a+0.006, f"{a:.3f}",
             ha="center", va="bottom", fontsize=10, weight="bold", color="#222")
axL.set_ylim(0.44, 0.71); axL.set_ylabel("Best 10-day AUC (test)", fontsize=9)
axL.set_title("Direction AUC collapses in the bear regime", fontsize=9.5, weight="bold")
for s in ["top", "right"]: axL.spines[s].set_visible(False)
axL.grid(axis="y", ls=":", alpha=0.4)
 
# — droite : lift sentiment par régime (négatif partout) —
bR = axR.bar(years, lift, width=0.55, color="#00629B", zorder=3)
axR.axhline(0, color="#222", lw=1)
for r, v in zip(bR, lift):
    axR.text(r.get_x()+r.get_width()/2, v-0.03, f"{v:+.2f}%",
             ha="center", va="top", fontsize=10, weight="bold", color="#222")
axR.set_ylim(-1.2, 0.3); axR.set_ylabel("Sentiment ablation lift (%)", fontsize=9)
axR.set_title("Sentiment lift is negative in both regimes", fontsize=9.5, weight="bold")
for s in ["top", "right"]: axR.spines[s].set_visible(False)
axR.grid(axis="y", ls=":", alpha=0.4)
 
fig.suptitle("Temporal robustness (walk-forward): performance tracks the regime, "
             "sentiment adds nothing", fontsize=10, weight="bold", y=1.02)
fig.tight_layout()
fig.savefig(f"{FIG_OUT}/fig8.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(f"{FIG_OUT}/fig8.pdf", bbox_inches="tight", facecolor="white")
plt.close(fig)
print("✅ fig8.png / fig8.pdf écrits dans", FIG_OUT)
 
# ── Télécharge-les depuis l'Output, ou zippe tout ──
import shutil
shutil.make_archive("/kaggle/working/figures_fig4_fig8", "zip", FIG_OUT)
print("📦 figures_fig4_fig8.zip prêt (Output → Download)")
 

✅ fig4.png / fig4.pdf écrits dans /kaggle/working/sentiment_pipeline/figures
✅ fig8.png / fig8.pdf écrits dans /kaggle/working/sentiment_pipeline/figures
📦 figures_fig4_fig8.zip prêt (Output → Download)


In [66]:
# ═══ FIGURE 6 — real-time snapshot (IEEE, fond blanc) ═══
# À coller en fin de notebook (après les runs). Lit 11_backtest.parquet.
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np, pandas as pd, os
FIG_OUT=f"{BASE_DIR}/figures"; os.makedirs(FIG_OUT,exist_ok=True)
BLUE="#00629B"; GREEN="#2E7D32"; RED="#C62828"; GREY="#8a8f94"; INK="#222"
 
bt=pd.read_parquet(f"{OUTPUT_DIR}/11_backtest.parquet")
# proba hausse moyenne récente par ticker (ensemble) -> % ; sentiment latest si dispo
g=bt.groupby("ticker")
prob=(g["pred_ens"].mean()*100)
# sentiment: si une colonne existe (ex: 'market_sent_mean_lag1' ou 'sent_ma_30d'), sinon 0
sc=[c for c in bt.columns if "sent" in c.lower()]
sent=(g[sc[0]].mean() if sc else prob*0)
tk=list(prob.index)
def signal(p): return ("BUY",GREEN) if p>=60 else (("SELL",RED) if p<45 else ("HOLD","#C9A227"))
fig,(axL,axR)=plt.subplots(1,2,figsize=(9.4,3.6),dpi=150)
order=np.argsort(prob.values)
for i,idx in enumerate(order):
    p=prob.values[idx]; lab,col=signal(p)
    axL.barh(i,p,color=col,height=0.62,zorder=3)
    axL.text(p+1.5,i,f"{p:.1f}%  {lab}",va="center",fontsize=7.8,weight="bold",color=INK)
axL.axvline(50,color=GREY,ls="--",lw=1); axL.set_yticks(range(len(tk)))
axL.set_yticklabels([tk[i] for i in order],fontsize=8); axL.set_xlim(0,max(prob)*1.25)
axL.set_xlabel("Predicted probability of upward move (%)",fontsize=8.5)
axL.set_title("Per-ticker real-time signal",fontsize=9.5,weight="bold")
[axL.spines[s].set_visible(False) for s in ["top","right"]]; axL.grid(axis="x",ls=":",alpha=0.4)
axR.axhline(50,color=GREY,ls=":",lw=1); axR.axvline(0,color=GREY,ls=":",lw=1)
axR.scatter(sent.values,prob.values,s=70,color=BLUE,edgecolor="white",linewidth=1,zorder=3)
for i,t in enumerate(tk): axR.annotate(t,(sent.values[i],prob.values[i]),textcoords="offset points",xytext=(6,4),fontsize=7.2)
if np.std(sent.values)>0:
    b,a=np.polyfit(sent.values,prob.values,1); xs=np.linspace(sent.min(),sent.max(),50)
    axR.plot(xs,a+b*xs,"--",color=RED,lw=1.3,label="trend"); axR.legend(frameon=False,fontsize=7.5,loc="lower right")
axR.set_xlabel("Latest sentiment",fontsize=8.5); axR.set_ylabel("Predicted prob. up (%)",fontsize=8.5)
axR.set_title("Sentiment vs predicted probability",fontsize=9.5,weight="bold")
[axR.spines[s].set_visible(False) for s in ["top","right"]]; axR.grid(ls=":",alpha=0.35)
fig.suptitle("Real-time snapshot: signals are driven by technical variables, not sentiment",fontsize=9.5,weight="bold",y=1.02)
fig.tight_layout()
fig.savefig(f"{FIG_OUT}/fig6.png",dpi=300,bbox_inches="tight",facecolor="white")
fig.savefig(f"{FIG_OUT}/fig6.pdf",bbox_inches="tight",facecolor="white")
plt.close(fig); print("✅ fig6.png / fig6.pdf")
 

✅ fig6.png / fig6.pdf


In [5]:
import shutil, os
os.makedirs("/kaggle/working", exist_ok=True)
shutil.make_archive("/kaggle/working/artefacts_ALL", "zip", "/kaggle/working/sentiment_pipelin")
print("✅ artefacts_ALL.zip prêt")
print("→ panneau DROIT (Output) → artefacts_ALL.zip → clic droit → Download")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/sentiment_pipelin'

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
#  UNE SEULE CELLULE — GÉNÈRE fig2, fig3, fig4, fig5, fig6, fig8, fig9  (+ zip)
#  100% autonome : définit BASE_DIR, valeurs en dur du run. Colle → exécute.
#  Style IEEE : fond blanc, 300 dpi, PNG + PDF. Aucune dépendance de fichier
#  sauf fig6 (lit 11_backtest si présent, sinon utilise des valeurs de secours).
# ═══════════════════════════════════════════════════════════════════════════════
import os, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR   = os.environ.get("BASE_DIR", "/kaggle/working/sentiment_pipeline")
OUTPUT_DIR = f"{BASE_DIR}/data"
FIG_OUT    = f"{BASE_DIR}/figures"
os.makedirs(FIG_OUT, exist_ok=True)

BLUE="#00629B"; GREEN="#2E7D32"; ORANGE="#E08214"; RED="#C62828"; GREY="#9E9E9E"; INK="#222"
def save(fig, name):
    fig.savefig(f"{FIG_OUT}/{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(f"{FIG_OUT}/{name}.pdf", bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"  ✅ {name}")

# ───────────────────────── FIG 3 — AUC by horizon ─────────────────────────
H=[1,10,20]; lgb=[0.527,0.672,0.592]; xgb=[0.511,0.654,0.633]; cb=[0.503,0.608,0.562]
fig,ax=plt.subplots(figsize=(5.4,3.6),dpi=150)
ax.axhline(0.5,color=GREY,ls=":",lw=1); ax.text(20,0.503,"chance",fontsize=7,color=GREY,ha="right")
ax.plot(H,lgb,"-o",color=BLUE,lw=2,label="LightGBM")
ax.plot(H,xgb,"-s",color=GREEN,lw=2,label="XGBoost")
ax.plot(H,cb,"-^",color=ORANGE,lw=2,label="CatBoost")
ax.annotate("0.672",(10,0.672),textcoords="offset points",xytext=(0,8),ha="center",fontsize=8,weight="bold",color=BLUE)
ax.set_xticks(H); ax.set_xlabel("Prediction horizon (days)",fontsize=9)
ax.set_ylabel("AUC (test)",fontsize=9); ax.set_ylim(0.48,0.71)
ax.set_title("Direction AUC by model and horizon (peaks at ten days)",fontsize=9,weight="bold")
ax.legend(frameon=False,fontsize=8,loc="lower center",ncol=3)
for s in ["top","right"]: ax.spines[s].set_visible(False)
ax.grid(axis="y",ls=":",alpha=0.4); fig.tight_layout(); save(fig,"fig3")

# ───────────────────────── FIG 5 — lag 0 vs lag 1 ─────────────────────────
fig,ax=plt.subplots(figsize=(5.4,3.6),dpi=150)
ax.bar([0,1],[0.196,0.030],width=0.55,color=[BLUE,RED],zorder=3)
for xi,v in zip([0,1],[0.196,0.030]):
    ax.text(xi,v+0.005,f"{v:.3f}",ha="center",va="bottom",fontsize=10,weight="bold",color=INK)
ax.set_xticks([0,1]); ax.set_xticklabels(["Same day\n(lag 0)","Next day\n(lag 1)"],fontsize=9)
ax.set_ylabel("Mean |Pearson|  (sentiment vs return)",fontsize=9); ax.set_ylim(0,0.23)
ax.set_title("Sentiment is contemporaneous, not predictive",fontsize=9.5,weight="bold")
for s in ["top","right"]: ax.spines[s].set_visible(False)
ax.grid(axis="y",ls=":",alpha=0.4); fig.tight_layout(); save(fig,"fig5")

# ───────────────────────── FIG 2 — abstention curve ───────────────────────
cov=[100,50,30,20,10,5]
lgb_a=[0.605,0.612,0.651,0.672,0.686,0.748]
xgb_a=[0.655,0.742,0.808,0.865,0.922,0.965]
cb_a =[0.641,0.740,0.792,0.812,0.905,0.940]
fig,ax=plt.subplots(figsize=(5.4,3.6),dpi=150)
ax.axhline(0.683,color=GREY,ls="--",lw=1,label="Baseline accuracy (0.683)")
ax.plot(cov,lgb_a,"-o",color=BLUE,lw=2,label="LightGBM")
ax.plot(cov,xgb_a,"-s",color=GREEN,lw=2,label="XGBoost")
ax.plot(cov,cb_a,"-^",color=ORANGE,lw=2,label="CatBoost")
ax.invert_xaxis(); ax.set_xticks(cov)
ax.set_xlabel("Coverage: % most-confident predictions kept",fontsize=9)
ax.set_ylabel("Accuracy",fontsize=9); ax.set_ylim(0.48,1.0)
ax.set_title("Abstention (direction, ten-day horizon)",fontsize=9.5,weight="bold")
ax.legend(frameon=False,fontsize=7.5,loc="upper left")
for s in ["top","right"]: ax.spines[s].set_visible(False)
ax.grid(axis="y",ls=":",alpha=0.4); fig.tight_layout(); save(fig,"fig2")

# ───────────────────────── FIG 4 — ablation (= placebo) ───────────────────
groups=["Direction\n10d · 2023 (bull)","Direction\n10d · 2022 (bear)"]
real=[-0.99,-0.61]; placebo=[-0.32,-0.73]; x=np.arange(2); w=0.36
fig,ax=plt.subplots(figsize=(6.6,3.4),dpi=150)
b1=ax.bar(x-w/2,real,w,label="Real sentiment",color=BLUE,zorder=3)
b2=ax.bar(x+w/2,placebo,w,label="Noise placebo",color=GREY,hatch="////",edgecolor="white",zorder=3)
ax.axhline(0,color=INK,lw=1)
for bars in (b1,b2):
    for r in bars:
        h=r.get_height()
        ax.text(r.get_x()+r.get_width()/2,h+(0.03 if h>=0 else -0.03),f"{h:+.2f}%",
                ha="center",va="bottom" if h>=0 else "top",fontsize=9,weight="bold",color=INK)
ax.set_xticks(x); ax.set_xticklabels(groups,fontsize=9)
ax.set_ylabel("Sentiment AUC lift over technical-only (%)",fontsize=9); ax.set_ylim(-1.25,0.35)
ax.set_title("Sentiment/sarcasm ablation: real features match a noise placebo",fontsize=10,weight="bold")
ax.legend(frameon=False,fontsize=9,loc="lower right")
for s in ["top","right"]: ax.spines[s].set_visible(False)
ax.grid(axis="y",ls=":",alpha=0.4); fig.tight_layout(); save(fig,"fig4")

# ───────────────────────── FIG 8 — walk-forward ───────────────────────────
years=["2022\n(bear)","2023\n(bull)"]; auc=[0.479,0.672]; lift=[-0.61,-0.99]
fig,(axL,axR)=plt.subplots(1,2,figsize=(9.2,3.4),dpi=150)
cols=["#C62828" if a<0.5 else "#00629B" for a in auc]
bL=axL.bar(years,auc,width=0.55,color=cols,zorder=3); axL.axhline(0.5,color=GREY,ls="--",lw=1)
axL.text(1.45,0.505,"chance",fontsize=7,color=GREY,ha="right",va="bottom")
for r,a in zip(bL,auc): axL.text(r.get_x()+r.get_width()/2,a+0.006,f"{a:.3f}",ha="center",va="bottom",fontsize=10,weight="bold",color=INK)
axL.set_ylim(0.44,0.71); axL.set_ylabel("Best 10-day AUC (test)",fontsize=9)
axL.set_title("Direction AUC collapses in the bear regime",fontsize=9.5,weight="bold")
for s in ["top","right"]: axL.spines[s].set_visible(False)
axL.grid(axis="y",ls=":",alpha=0.4)
bR=axR.bar(years,lift,width=0.55,color="#00629B",zorder=3); axR.axhline(0,color=INK,lw=1)
for r,v in zip(bR,lift): axR.text(r.get_x()+r.get_width()/2,v-0.03,f"{v:+.2f}%",ha="center",va="top",fontsize=10,weight="bold",color=INK)
axR.set_ylim(-1.2,0.3); axR.set_ylabel("Sentiment ablation lift (%)",fontsize=9)
axR.set_title("Sentiment lift is negative in both regimes",fontsize=9.5,weight="bold")
for s in ["top","right"]: axR.spines[s].set_visible(False)
axR.grid(axis="y",ls=":",alpha=0.4)
fig.suptitle("Temporal robustness (walk-forward): performance tracks the regime, sentiment adds nothing",fontsize=10,weight="bold",y=1.02)
fig.tight_layout(); save(fig,"fig8")

# ───────────────────────── FIG 9 — convergent 3 probes ────────────────────
fig,axs=plt.subplots(1,3,figsize=(9.6,3.3),dpi=150)
axs[0].bar([0,1],[-0.99,-0.32],width=0.6,color=[BLUE,GREY],zorder=3)
axs[0].bar([1],[-0.32],width=0.6,color=GREY,hatch="////",edgecolor="white",zorder=3)
axs[0].axhline(0,color=INK,lw=1)
for xi,v in zip([0,1],[-0.99,-0.32]): axs[0].text(xi,v-0.03,f"{v:+.2f}%",ha="center",va="top",fontsize=8,weight="bold")
axs[0].set_xticks([0,1]); axs[0].set_xticklabels(["real","placebo"],fontsize=8); axs[0].set_ylim(-1.25,0.25)
axs[0].set_title("Probe 1 — falsification\n(ablation vs placebo)",fontsize=8.5,weight="bold"); axs[0].set_ylabel("Sentiment lift (%)",fontsize=8)
axs[1].bar([0,1],[0.479,0.672],width=0.6,color=["#C62828","#00629B"],zorder=3); axs[1].axhline(0.5,color=GREY,ls="--",lw=1)
for xi,v in zip([0,1],[0.479,0.672]): axs[1].text(xi,v+0.006,f"{v:.3f}",ha="center",va="bottom",fontsize=8,weight="bold")
axs[1].set_xticks([0,1]); axs[1].set_xticklabels(["2022\nbear","2023\nbull"],fontsize=8); axs[1].set_ylim(0.44,0.71)
axs[1].set_title("Probe 2 — test year\n(direction, walk-forward)",fontsize=8.5,weight="bold"); axs[1].set_ylabel("Best 10-day AUC",fontsize=8)
axs[2].bar([0,1],[0.196,0.030],width=0.6,color=[BLUE,RED],zorder=3)
for xi,v in zip([0,1],[0.196,0.030]): axs[2].text(xi,v+0.005,f"{v:.3f}",ha="center",va="bottom",fontsize=8,weight="bold")
axs[2].set_xticks([0,1]); axs[2].set_xticklabels(["same\nday","next\nday"],fontsize=8); axs[2].set_ylim(0,0.23)
axs[2].set_title("Probe 3 — lag structure\n(contemporaneous?)",fontsize=8.5,weight="bold"); axs[2].set_ylabel("Mean |Pearson|",fontsize=8)
for ax in axs:
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    ax.grid(axis="y",ls=":",alpha=0.4)
fig.suptitle("Three independent probes — one conclusion: affective text adds no forward-looking skill",fontsize=9.5,weight="bold",y=1.03)
fig.tight_layout(); save(fig,"fig9")

# ───────────────────────── FIG 6 — real-time (best-effort) ────────────────
try:
    import pandas as pd
    bt=pd.read_parquet(f"{OUTPUT_DIR}/11_backtest.parquet")
    g=bt.groupby("ticker"); prob=(g["pred_ens"].mean()*100)
    sc=[c for c in bt.columns if "sent" in c.lower()]
    sent=(g[sc[0]].mean() if sc else prob*0); tk=list(prob.index)
    prob=prob.values; sent=sent.values
except Exception as e:
    print("  (fig6: 11_backtest absent, valeurs de secours)", e)
    tk=["AAPL","NVDA","MSFT","AMZN","GOOGL","META","TSLA"]
    prob=np.array([65.3,53.5,56.5,54.8,39.7,37.5,38.3]); sent=np.array([0.22,0.20,0.06,0.05,-0.02,-0.01,-0.02])
def sig(p): return ("BUY",GREEN) if p>=60 else (("SELL",RED) if p<45 else ("HOLD","#C9A227"))
fig,(axL,axR)=plt.subplots(1,2,figsize=(9.4,3.6),dpi=150); order=np.argsort(prob)
for i,idx in enumerate(order):
    lab,col=sig(prob[idx]); axL.barh(i,prob[idx],color=col,height=0.62,zorder=3)
    axL.text(prob[idx]+1.5,i,f"{prob[idx]:.1f}%  {lab}",va="center",fontsize=7.8,weight="bold",color=INK)
axL.axvline(50,color=GREY,ls="--",lw=1); axL.set_yticks(range(len(tk))); axL.set_yticklabels([tk[i] for i in order],fontsize=8)
axL.set_xlim(0,max(prob)*1.3); axL.set_xlabel("Predicted probability of upward move (%)",fontsize=8.5)
axL.set_title("Per-ticker real-time signal",fontsize=9.5,weight="bold")
for s in ["top","right"]: axL.spines[s].set_visible(False)
axL.grid(axis="x",ls=":",alpha=0.4)
axR.axhline(50,color=GREY,ls=":",lw=1); axR.axvline(0,color=GREY,ls=":",lw=1)
axR.scatter(sent,prob,s=70,color=BLUE,edgecolor="white",linewidth=1,zorder=3)
for i,t in enumerate(tk): axR.annotate(t,(sent[i],prob[i]),textcoords="offset points",xytext=(6,4),fontsize=7.2)
if np.std(sent)>0:
    b,a=np.polyfit(sent,prob,1); xs=np.linspace(sent.min(),sent.max(),50)
    axR.plot(xs,a+b*xs,"--",color=RED,lw=1.3,label="trend"); axR.legend(frameon=False,fontsize=7.5,loc="lower right")
axR.set_xlabel("Latest sentiment",fontsize=8.5); axR.set_ylabel("Predicted prob. up (%)",fontsize=8.5)
axR.set_title("Sentiment vs predicted probability",fontsize=9.5,weight="bold")
for s in ["top","right"]: axR.spines[s].set_visible(False)
axR.grid(ls=":",alpha=0.35)
fig.suptitle("Real-time snapshot: signals are driven by technical variables, not sentiment",fontsize=9.5,weight="bold",y=1.02)
fig.tight_layout(); save(fig,"fig6")

# ───────────────────────── ZIP FINAL ─────────────────────────
import shutil
shutil.make_archive("/kaggle/working/figures_ALL","zip",FIG_OUT)
print("\n📦 figures_ALL.zip prêt → panneau Output → Download")
print("Figures écrites :", sorted(f for f in os.listdir(FIG_OUT) if f.endswith(".png")))

  ✅ fig3
  ✅ fig5
  ✅ fig2
  ✅ fig4
  ✅ fig8
  ✅ fig9
  (fig6: 11_backtest absent, valeurs de secours) [Errno 2] No such file or directory: '/kaggle/working/sentiment_pipeline/data/11_backtest.parquet'
  ✅ fig6

📦 figures_ALL.zip prêt → panneau Output → Download
Figures écrites : ['fig2.png', 'fig3.png', 'fig4.png', 'fig5.png', 'fig6.png', 'fig8.png', 'fig9.png']


In [6]:
import os
print(os.listdir("/kaggle/working"))

['.virtual_documents', 'sentiment_pipeline', 'figures_ALL.zip']


In [7]:
import shutil, os
shutil.make_archive("/kaggle/working/artefacts_ALL", "zip",
                    "/kaggle/working/sentiment_pipeline")
print("✅ artefacts_ALL.zip prêt")
print("→ panneau DROIT (Output) → artefacts_ALL.zip → clic droit → Download")

✅ artefacts_ALL.zip prêt
→ panneau DROIT (Output) → artefacts_ALL.zip → clic droit → Download
